# CircMTL – Exploration and Curation of the CircR2Disease v2.0 Data and ENCORI datasets

**Purpose.** This notebook documents the exploratory analysis of the CircR2Disease v2.0 resource, which serves as the source of circRNA–disease associations (CDA) and auxiliary miRNA/RBP labels for the CircMTL thesis project. Additionally the exploratory analysis of the ENCORI resource, which serves as the source of auxiliary miRNA/RBP labels for the CircMTL thesis project

**Structure.** The notebook has 47 code cells, numbered in the order in which they appear here (Cell 1 to Cell 47); every reference such as "Cell 22" follows this numbering. Each code cell is preceded by a heading with its objective, carries inline comments, and is followed by a markdown cell that (i) summarises the principal insight obtained from its output and (ii) justifies the decision that motivates the next cell.

**Reading the comments.** Counts and percentages in the interpretation cells were compared with the printed output of the cell they describe. Statements marked *Inference*, *not verified* or *not checked* are reasoning that was not tested in this notebook. Figures typed from the CircR2Disease publication (for example the CircDis universe of 2,099 pairs, 1,577 circRNAs and 202 diseases) come from the reference paper and were not re-read here.

### 1. Configuration and Raw Data Inventory

**Objective.** Resolve all project paths from the central configuration file (`configs/paths.yaml`) and verify that every raw input required by the pipeline is present before any processing starts. The inventory records the number of files and the size of each source (CircR2Disease, ENCORI miRNA, ENCORI RBP), so that later loading decisions, such as chunked reading, are based on the actual file sizes.

In [1]:
# Cell 1 - Configuration and raw data inventory
from pathlib import Path
import yaml
import pandas as pd

# Display settings: show all columns and full-width strings for wide tables.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# The notebook resides in notebooks/; the project root is its parent directory.
PROJECT_ROOT = Path.cwd().parent
CONFIG_FILE = PROJECT_ROOT / "configs" / "paths.yaml"

# All paths come from the configuration file, so no location is hard-coded in the notebook.
with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ds = cfg["datasets"]
RAW_DIR       = PROJECT_ROOT / cfg["inputs"]["raw_dir"]           # source data (read-only)
PROCESSED_DIR = PROJECT_ROOT / cfg["outputs"]["processed_dir"]    # derived artefacts
CDA_DIR       = PROJECT_ROOT / ds["circr2disease"]["dir"]         # CircR2Disease v2.0 resources
ENC_MIRNA_DIR = PROJECT_ROOT / ds["encori_mirna"]["dir"]          # ENCORI circRNA-miRNA (hg38)
ENC_RBP_DIR   = PROJECT_ROOT / ds["encori_rbp"]["dir"]            # ENCORI circRNA-RBP (hg38)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Integrity check: every input must exist before any analysis starts.
inputs = {"RAW_DIR": RAW_DIR, "CircR2Disease": CDA_DIR,
          "ENCORI miRNA dir": ENC_MIRNA_DIR, "ENCORI RBP dir": ENC_RBP_DIR}
for name, p in inputs.items():
    print(f"{name:18s} exists: {p.exists()}")
missing = [n for n, p in inputs.items() if not p.exists()]
assert not missing, f"missing inputs: {missing}"

# Inventory: file list and total size per source (recursive).
def inventory(path):
    files = sorted(f for f in path.rglob("*") if f.is_file())
    return files, sum(f.stat().st_size for f in files) / 1e6

sources = {"CircR2Disease": CDA_DIR, "ENCORI miRNA": ENC_MIRNA_DIR, "ENCORI RBP": ENC_RBP_DIR}

print("\nInventory:")
for name, p in sources.items():
    files, mb = inventory(p)
    print(f"{name:14s} {len(files):3d} files {mb:10.1f} MB")

# File-level detail, so that each resource can be assigned a role in later cells.
for name, p in sources.items():
    files, _ = inventory(p)
    print(f"\n{name}:")
    for f in files:
        print(f"  {f.stat().st_size / 1e6:9.2f} MB   {f.relative_to(p)}")

RAW_DIR            exists: True
CircR2Disease      exists: True
ENCORI miRNA dir   exists: True
ENCORI RBP dir     exists: True

Inventory:
CircR2Disease    8 files     1700.8 MB
ENCORI miRNA     1 files     2205.7 MB
ENCORI RBP       2 files     2273.9 MB

CircR2Disease:
       2.91 MB   1-s2.0-S1672022921002461-main.pdf
     500.37 MB   human_hg19_circRNAs_putative_spliced_sequence.fa
    1194.42 MB   RNA-FM_pretrained.pth
       2.12 MB   The_candidate_circRNA-disease_associations_that_predicted_by_CircDis_method.xlsx
       0.73 MB   The_circRNA-disease_entries.xlsx
       0.06 MB   The_circRNA_list.xlsx
       0.03 MB   The_disease_list.xlsx
       0.12 MB   The_manually_curated_associations_between_circRNAs_and_other_associations.xlsx

ENCORI miRNA:
    2205.67 MB   hg38_miRcircRNA.txt

ENCORI RBP:
     239.48 MB   hg38_RBPcircRNA.txt
    2034.39 MB   hg38_RBPcircRNA_cluster.txt


**Interpretation.** The configuration file was parsed correctly and all four input locations exist. The three sources differ strongly in size and in kind:

- **CircR2Disease (8 files, ~1.7 GB).** Most of the volume is not tabular data. It consists of the hg19 putative spliced circRNA sequences (~500 MB, FASTA) and the pretrained RNA-FM checkpoint (~1.2 GB). The association data are five small Excel workbooks (entries, circRNA list, disease list, manually curated associations, and the CircDis-predicted candidates). The publication PDF is stored in the same folder.
- **ENCORI miRNA (1 file, ~2.2 GB).** A single genome-wide table, which is too large to load at once and must be read in chunks with column selection.
- **ENCORI RBP (2 files, ~2.3 GB).** `hg38_RBPcircRNA.txt` (~240 MB) and `hg38_RBPcircRNA_cluster.txt` (~2.0 GB). The difference in content between the two is not yet known.

Two points affect the later design. First, the FASTA is hg19 while both ENCORI tables are hg38, so the two sources can only be linked by circBase ID, not by genomic coordinates. Second, the predicted CircDis associations are model outputs and must not enter the training labels.

**Decision.** The raw inputs are complete, so processing can start. The next cell inspects the *structure* of each resource before any content-level analysis: sheet names, dimensions and column names for the Excel workbooks, and the header and first rows for the three ENCORI tables. The choice between the two RBP files, and the definition of the evidence filter (CLIP support), will be made after that inspection.

## CircR2Disease_V2.0 dataset

### 2. Structural Inspection of the Excel Workbooks

**Objective.** Load every sheet of every Excel workbook and report its dimensions and column names, in order to understand the schema of each resource before any content-level analysis.

In [22]:
# Cell 2 - Structural inspection of the Excel workbooks

# Container: tables[file_stem][sheet_name] -> DataFrame, retained in memory for subsequent cells.
tables = {}

for f in sorted(CDA_DIR.glob("*.xlsx")):
    # sheet_name=None loads all sheets of the workbook rather than only the first.
    sheets = pd.read_excel(f, sheet_name=None)
    tables[f.stem] = sheets

    # Report the file name, then the dimensions and column names of each sheet.
    print("=" * 100)
    print(f.name)
    for sheet, d in sheets.items():
        print(f"  sheet '{sheet}': {d.shape[0]} rows x {d.shape[1]} cols")
        print(f"  columns: {list(d.columns)}")

The_candidate_circRNA-disease_associations_that_predicted_by_CircDis_method.xlsx
  sheet 'Sheet3': 1577 rows x 203 cols
  columns: ['Unnamed: 0', 'Cardiovascular disease', 'Lung cancer', 'Osteoarthritis', 'Intervertebral disc degeneration', 'Rheumatoid arthritis', 'Atherosclerosis', 'Prostate cancer', 'Glioblastoma', 'Hepatocellular carcinoma', 'Repeated implantation failure', 'Bladder cancer', 'Gastric cancer', 'Breast cancer', 'Liver cancer', 'Osteosarcoma', 'Colorectal cancer', 'Moyamoya disease', 'PulmoNAry tuberculosis', 'Triple negative breast cancer', 'Glioma', 'Cervical cancer', 'Hepatic fibrosis', 'Multiple sclerosis', 'Oral squamous cell carcinoma', 'Hypopharyngeal squamous cell carcinoma', 'Lung adenocarcinoma', 'Clear cell reNAl cell carcinoma', 'Hypertension', 'Senescence', 'Cervical carcinoma', 'Acute myeloid leukemia', 'Papillary thyroid carcinoma', 'Type 2 diabetes mellitus', 'CoroNAry artery disease', 'Hirschsprung disease', 'Pancreatic ductal adenocarcinoma', 'Preecla

**Interpretation.** Five principal findings emerged: (1) the two list files appear to lack a header row, so pandas used the first data record as the column name, giving 3,076 and 311 rows instead of 3,077 and 312; (2) the main entries table contains 4,201 curated records, which are *not* necessarily unique circRNA–disease pairs; (3) miRNA and RBP annotations reside in a separate file (2,160 rows); (4) the CircDis prediction file is a matrix of 1,577 circRNAs × 202 diseases, whose column headers contain a text corruption (e.g., `PulmoNAry`) and an empty second sheet; (5) disease names contain trailing whitespace and spelling variants (e.g., `disc` vs. `disk`).

**Decision.** The list files must be re-read with `header=None` (Cell 3), and the main table must be profiled in terms of missingness, species and identifier formats (Cell 4). The FASTA headers must also be inspected (Cell 5), because they determine how circRNAs can be matched to their sequences.

### 3. Re-reading the circRNA and Disease Lists

**Objective.** Re-read the two single-column list files without treating the first record as a header, and characterise the naming schemes of the circRNA identifiers.

In [100]:
# Cell 3 - Re-reading the circRNA and disease lists

# header=None prevents the first record from being consumed as a column name.
circ_list = pd.read_excel(CDA_DIR / "The_circRNA_list.xlsx", header=None, names=["circRNA_id"])
dis_list  = pd.read_excel(CDA_DIR / "The_disease_list.xlsx",  header=None, names=["disease"])

# Expected counts after correction: 3,077 circRNAs and 312 diseases, all unique.
print("circRNAs:", len(circ_list), "| unique:", circ_list["circRNA_id"].nunique())
print("diseases:", len(dis_list),  "| unique:", dis_list["disease"].nunique())

print("\nFirst circRNA IDs:")
print(circ_list["circRNA_id"].head(8).tolist())

# Extract the leading alphabetic prefix of each identifier to characterise the naming schemes;
# identifiers starting with a digit (coordinate-based) yield NaN.
print("\nID prefix distribution (top 10):")
print(circ_list["circRNA_id"].str.extract(r"^([A-Za-z]+)")[0].value_counts(dropna=False).head(20))

circRNAs: 3077 | unique: 3077
diseases: 312 | unique: 312

First circRNA IDs:
['11_33286412_33287511_+', '17_50189167_50194626_-', '17_50189167_50198002_-', '17_50189858_50195330_-', '17_50190326_50194041_-', '2_72718102_72733118_-', 'ASB3-has_circ_0003959', 'BCRC-3']

ID prefix distribution (top 10):
0
hsa           1825
circ           341
circRNA        213
mmu             95
Circ            79
rno             36
chr             27
CircRNA         14
NaN              6
CBT              6
has              6
ciRNA            5
mm               5
circular         4
Chr              3
circARHGAP       3
circEIF          3
Circular         3
circZNF          3
hsacirc          3
Name: count, dtype: int64


**Interpretation.** The corrected counts are 3,077 circRNAs and 312 diseases, all unique; each is exactly one more than the row count read with a header (3,076 and 311), which confirms that the first record had been consumed as a column name. Agreement with the counts printed in the CircR2Disease publication is assumed and was not rechecked in this notebook. An earlier hypothesis that the identifiers were genomic coordinates was **revised**: only six identifiers begin with a digit; the majority use heterogeneous schemes (`hsa`, `circ`, `circRNA`, `mmu`, `rno`, `chr`, …).

**Decision.** Identifier heterogeneity implies that name normalisation cannot rely on a single rule. The main entries table is therefore profiled next (Cell 4), together with the FASTA headers (Cell 5).

### 4. Overview of the Main Entries Table

**Objective.** Summarise the main circRNA–disease entries table (shape, one transposed record, missing values per column, species labels and the numbers of unique circRNA and disease names) in order to identify which fields can be used for identifier matching.

In [24]:
# Cell 4 - Overview of the main circRNA-disease entries table
entries = tables["The_circRNA-disease_entries"]["The CircRNA-Disease Entries"]

print(entries.shape)
# Transposing one record makes all 13 fields readable at a glance
print("\nFirst record (transposed):")
print(entries.head(1).T)

# Missingness indicates which fields (Region, Strand, Synonyms) can be used for identifier matching
print("\nMissing values per column:")
print(entries.isna().sum())

print("\nSpecies:")
print(entries["Species"].value_counts(dropna=False))

# Raw uniqueness counts exceed those of the official lists, which suggests spelling/whitespace variants
print("\nUnique circRNA names:", entries["circRNA Name"].nunique())
print("Unique disease names:", entries["Disease Name"].nunique())

(4201, 13)

First record (transposed):
                                                                                   0
CRD ID                                                                             1
circRNA Name                                                      hsa_circRNA_001937
Synonyms                                                                         NaN
Gene Symbol                                                                     CHD9
Disease Name                                                            Tuberculosis
Expression pattern                                                       Upregulated
PubMed ID                                                                 29448254.0
Region                                                                           NaN
Strand                                                                           NaN
Species                                                                        Human
Experimental techniques   

**Interpretation.** The table holds 4,201 rows, of which 3,853 + 1 are human (labelled "Human"/"human"), with minor non-human subsets. `Region` is missing for 1,295 rows and `Strand` for 1,536. The numbers of unique names (3,126 circRNAs; 313 diseases) exceed those in the official lists (3,077; 312), indicating whitespace or case duplicates.

**Decision.** Species labels must be harmonised and name hygiene tested (Cell 7). Before that, the FASTA headers are inspected (Cell 5), as these define the identifier space against which circRNAs will be matched.

### 5. Inspection of the FASTA Header Layout

**Objective.** Read only the first records of the hg19 FASTA file (about 500 MB, too large to inspect in full) to establish the header layout: circBase identifier, coordinates with strand, transcript and gene.

In [26]:
# Cell 5 - Inspect the first lines of the FASTA file
# Header layout: circBaseID | chr:start-end(strand) | transcript | gene
fa = next(CDA_DIR.glob("*.fa"))
print(fa.name, f"{fa.stat().st_size / 1e6:.1f} MB")

# Only six lines are read, because the file is large (about 500 MB)
with open(fa) as f:
    for _ in range(6):
        print(f.readline().rstrip()[:150])

human_hg19_circRNAs_putative_spliced_sequence.fa 500.4 MB
>hsa_circ_0000001|chr1:1080738-1080845-|None|None
ATGGGGTTGGGTCAGCCGTGCGGTCAGGTCAGGTCGGCCATGAGGTCAGGTGGGGTCGGCCATGAAGGTGGTGGGGGTCATGAGGTCACAAGGGGGTCGGCCATGTG
>hsa_circ_0000002|chr1:1158623-1159348-|NM_016176|SDF4
GGTGGATGTGAACACTGACCGGAAGATCAGTGCCAAGGAGATGCAGCGCTGGATCATGGAGAAGACGGCCGAGCACTTCCAGGAGGCCATGGAGGAGAGCAAGACACACTTCCGCGCCGTGGACCCTGACGGGGACGGTCACGTGTCTTG
>hsa_circ_0000003|chr1:1423242-1459777+|NM_031921|ATAD3B
CCTCCTGCTCTTCATGGATGAAGCAGACGCCTTCCTTCGGAAGCGAGCCACTGAGGAGATAAGCAAGGACCTCAGAGCCACACTGAACGCCTTCCTGTACCACATGGGCCAACACAGCAACAAATTCATGCTGGTCCTGGCCAGCAATCT


**Interpretation.** The sequences are keyed by circBase identifiers and carry genomic coordinates (hg19), a transcript identifier and a gene symbol. Any entry that can be tied to a circBase ID therefore obtains its sequence directly, and coordinates provide a second matching route.

**Decision.** The whole FASTA is indexed once (headers and sequence lengths only) and saved to disk (Cell 6), so that later steps do not require repeated parsing of a 500 MB file.

### 6. Index of the FASTA Headers

**Objective.** Stream the FASTA file once and build a compact index (identifier, chromosome, start, end, strand, transcript, gene, sequence length), save it as `circbase_hg19_index.csv`, and report parsing failures, strand balance and the sequence-length distribution.

In [27]:
# Cell 6 - Build a compact index of the FASTA headers (identifier, coordinates, gene, length)
import re

fa_path = next(CDA_DIR.glob("*.fa"))

def parse_header(h):
    """Split a FASTA header into its fields; coordinates are parsed with a strict regular expression."""
    # e.g. >hsa_circ_0000001|chr1:1080738-1080845-|None|None
    parts = h[1:].split("|")
    loc = parts[1] if len(parts) > 1 else ""
    m = re.match(r"^(chr\w+):(\d+)-(\d+)([+-])$", loc)
    chrom, start, end, strand = m.groups() if m else (None, None, None, None)
    return {
        "circbase_id": parts[0],
        "chrom": chrom, "start": start, "end": end, "strand": strand,
        "transcript": parts[2] if len(parts) > 2 else None,
        "gene": parts[3] if len(parts) > 3 else None,
    }

# Single streaming pass: sequence lines are only counted, never stored, to limit memory usage
records, cur, seq_len = [], None, 0
with open(fa_path) as f:
    for line in f:
        if line.startswith(">"):
            if cur is not None:              # close the previous record
                cur["seq_len"] = seq_len
                records.append(cur)
            cur, seq_len = parse_header(line.strip()), 0
        else:
            seq_len += len(line.strip())
    if cur is not None:                      # close the last record
        cur["seq_len"] = seq_len
        records.append(cur)

fa_index = pd.DataFrame(records)
for c in ["start", "end"]:
    fa_index[c] = pd.to_numeric(fa_index[c], errors="coerce").astype("Int64")

# Persist the index so that it can be reloaded without re-reading the FASTA
fa_index.to_csv(PROCESSED_DIR / "circbase_hg19_index.csv", index=False)

print("records:", len(fa_index), "| unique IDs:", fa_index["circbase_id"].nunique())
print("header parse failures:", fa_index["chrom"].isna().sum())
print("\nstrand:\n", fa_index["strand"].value_counts(dropna=False))
print("\ngene field is 'None':", (fa_index["gene"] == "None").mean().round(3))
print("\nsequence length (nt):")
print(fa_index["seq_len"].describe())
# RNA-FM accepts at most 1,022 nucleotides; this share refers to the whole file, not to our circRNAs
print("share <= 1022 nt (RNA-FM limit):", (fa_index["seq_len"] <= 1022).mean().round(3))

records: 140790 | unique IDs: 140790
header parse failures: 0

strand:
 strand
-    70436
+    70354
Name: count, dtype: int64

gene field is 'None': 0.017

sequence length (nt):
count    1.407900e+05
mean     3.491749e+03
std      3.444732e+04
min      0.000000e+00
25%      3.290000e+02
50%      6.840000e+02
75%      1.545000e+03
max      1.847059e+06
Name: seq_len, dtype: float64
share <= 1022 nt (RNA-FM limit): 0.635


**Interpretation.** The index contains 140,790 unique circBase IDs with no parsing failures and a balanced strand distribution. Sequence lengths are highly skewed (median 684 nt; maximum 1.85 Mb), 58 sequences are empty, and only 63.5% of all records fit the RNA-FM limit. The latter figure concerns the *entire* file and is not yet informative for the circRNAs of interest.

**Decision.** A length policy is deferred until the length distribution of the *matched* circRNAs is known (Cell 16). The next step is to harmonise species labels, classify naming schemes and test name hygiene in the entries table (Cell 7).

### 7. Species Harmonisation, Naming Schemes and Name Hygiene

**Objective.** Harmonise the species labels, classify the naming scheme of every circRNA name, inspect the format of `Region` and `Strand`, and test name hygiene (whitespace and case variants; agreement with the official disease and circRNA lists).

In [28]:
# Cell 7 - Species harmonisation, naming-scheme classification and name hygiene
# 1. Normalise species labels (case, plural forms) to a controlled vocabulary
species_map = {"human": "Human", "mouse": "Mouse", "rat": "Rat", "rats": "Rat",
               "chicken": "Chicken", "c. elegans": "C. elegans"}
entries["species_clean"] = entries["Species"].str.strip().str.lower().map(species_map)
print(entries["species_clean"].value_counts(dropna=False), "\n")

# 2. Classify the naming scheme of each circRNA name (rules are applied in order of specificity)
def name_pattern(name):
    n = str(name).strip()
    if re.search(r"hsa_circ_\d+", n, re.I):          return "circbase_full"        # e.g. hsa_circ_0001946
    if re.match(r"^(chr)?[\dXYM]+[_:]\d+[_-]\d+", n): return "coordinate"          # e.g. 11_33286412_33287511_+
    if re.search(r"circ_\d+", n, re.I):              return "circbase_no_prefix"   # e.g. circ_0001946
    if re.search(r"circRNA_\d+", n, re.I):           return "arraystar_like"       # microarray probe names
    return "other"

entries["name_pattern"] = entries["circRNA Name"].map(name_pattern)
print(pd.crosstab(entries["name_pattern"], entries["species_clean"], margins=True), "\n")

# 3. Inspect the format of Region/Strand on a random sample (fixed seed for reproducibility)
print(entries.loc[entries["Region"].notna(),
                  ["circRNA Name", "Region", "Strand", "species_clean"]]
      .sample(8, random_state=0), "\n")

# 4. Name hygiene: raw vs. whitespace/case-normalised uniqueness, and comparison with the official lists
names = entries["circRNA Name"].astype(str)
print("unique circRNA names raw:", names.nunique(),
      "| stripped+casefold:", names.str.strip().str.casefold().nunique())
d = entries["Disease Name"].astype(str)
print("unique diseases raw:", d.nunique(),
      "| stripped+casefold:", d.str.strip().str.casefold().nunique())
print("\nin entries but not in disease_list:", sorted(set(entries["Disease Name"]) - set(dis_list["disease"])))
print("in disease_list but not in entries:", sorted(set(dis_list["disease"]) - set(entries["Disease Name"])))
in_list = entries["circRNA Name"].isin(circ_list["circRNA_id"])
print("\nentry rows whose circRNA name is in circRNA_list:", in_list.sum(), "/", len(entries))

species_clean
Human         3854
Mouse          248
Rat             93
Chicken          4
C. elegans       2
Name: count, dtype: int64 

species_clean       C. elegans  Chicken  Human  Mouse  Rat   All
name_pattern                                                    
arraystar_like               0        0    339     76   38   453
circbase_full                0        0   2317      3    2  2322
circbase_no_prefix           2        0    217     61    6   286
coordinate                   0        4     14      0    0    18
other                        0        0    967    108   47  1122
All                          2        4   3854    248   93  4201 

            circRNA Name                    Region Strand species_clean
789     hsa_circ_0006215    chr3:27478878-27490288      -         Human
2811    hsa_circ_0006772      chr7:5410005-5410995      -         Human
1681    hsa_circ_0079593    chr7:23381682-23383472      -         Human
2994    hsa_circ_0006156  chr3:171965322-171969331   

**Interpretation.** After merging label variants, 3,854 rows are human (92%). Among them, 2,317 carry a full circBase-style name (directly matchable), 339 are Arraystar-like probe names (not convertible by string editing), and about 1,200 are heterogeneous. Cleaning reduces unique circRNA names from 3,126 to 3,071 and unique diseases from 313 to 312; the single extra disease is "Breast Carcinoma" versus "Breast carcinoma".

**Decision.** A tiered identifier-resolution strategy is adopted: Tier 1 – exact circBase ID (name or synonyms); Tier 2 – genomic coordinates; Cell 8 implements Tier 1 and deliberately refuses to resolve ambiguous rows.

### 8. Human Subset and Tier 1 (circBase-ID) Resolution

**Objective.** Restrict the entries to human, clean the text fields, map disease names onto the official list, and resolve circRNAs by an exact circBase ID written in the name or in `Synonyms` (Tier 1). Ambiguous rows are deliberately left unresolved.

In [29]:
# Cell 8 - Human subset, text cleaning and Tier 1 (circBase-ID) resolution
# ---- 1. Human subset and text cleaning ------------------------------------
def clean_txt(s):
    """Collapse repeated whitespace and strip leading/trailing spaces."""
    return re.sub(r"\s+", " ", str(s)).strip()

hum = entries[entries["species_clean"] == "Human"].copy()
hum["circ_raw"]      = hum["circRNA Name"].map(clean_txt)
hum["disease_clean"] = hum["Disease Name"].map(clean_txt)

# Map each cleaned disease name onto the official list, ignoring case
dis_std = {clean_txt(d).casefold(): clean_txt(d) for d in dis_list["disease"]}
hum["disease_std"] = hum["disease_clean"].str.casefold().map(dis_std)
print("human rows:", len(hum))
print("diseases not matched to official list:", hum["disease_std"].isna().sum())

# Check whether the 'NA' text corruption also affects the official disease list
bad = dis_list[dis_list["disease"].str.contains(r"[a-z]NA|\bNA[a-z]", regex=True)]
print("official disease names with 'NA' corruption:", len(bad), bad["disease"].head(8).tolist())

# ---- 2. Tier 1: a circBase ID written in the name or in Synonyms ----------
# 'has_' is a frequent typographical variant of 'hsa_'; exactly seven digits are required
pat = re.compile(r"(?:hsa|has)_circ_(\d{7})(?!\d)", re.I)

def ids_in(text):
    """Return the sorted set of canonical circBase IDs found in a text field."""
    return sorted({f"hsa_circ_{m}" for m in pat.findall(text)}) if isinstance(text, str) else []

hum["ids_name"] = hum["circ_raw"].map(ids_in)
hum["ids_syn"]  = hum["Synonyms"].map(ids_in)
has_name = hum["ids_name"].map(len) > 0
has_syn  = hum["ids_syn"].map(len) > 0

print("\nrows with circBase ID in name:      ", has_name.sum())
print("rows with ID only in Synonyms:      ", (~has_name & has_syn).sum())
print("rows with >1 distinct ID (name+syn):",
      ((hum["ids_name"] + hum["ids_syn"]).map(lambda x: len(set(x))) > 1).sum())

# 'Near misses': contain 'hsa_circ_' but not a clean 7-digit number (candidate typographical errors)
near = hum["circ_raw"].str.contains(r"(?:hsa|has)_circ_\d+", case=False, regex=True) & ~has_name
print("rows with 'hsa_circ_' but not a clean 7-digit ID:", near.sum(),
      hum.loc[near, "circ_raw"].head(6).tolist())

def pick(row):
    """Accept an ID only when it is unique; ambiguous rows stay unresolved on purpose."""
    cand = row["ids_name"] or row["ids_syn"]
    return cand[0] if len(cand) == 1 else None

hum["circbase_id"] = hum.apply(pick, axis=1)
fa_ids = set(fa_index["circbase_id"])
hum["in_fasta"] = hum["circbase_id"].isin(fa_ids)   # an ID is only usable if a sequence exists

# ---- 3. Summary of what survives -------------------------------------------
res   = hum[hum["in_fasta"]]
pairs = res[["circbase_id", "disease_std"]].drop_duplicates()
print("\nID found but NOT in FASTA:", (hum["circbase_id"].notna() & ~hum["in_fasta"]).sum())
print("resolved human rows:", len(res), "/", len(hum))
print("unique circRNA-disease pairs:", len(pairs))
print("unique circRNAs:", pairs["circbase_id"].nunique(),
      "| unique diseases:", pairs["disease_std"].nunique())
print("paper's circDis universe:  2099 pairs | 1577 circRNAs | 202 diseases")

# ---- 4. Quality issues in the FASTA -----------------------------------------
print("\nFASTA zero-length sequences:", (fa_index["seq_len"] == 0).sum())
print("FASTA > 10,000 nt:", (fa_index["seq_len"] > 10_000).sum(),
      "| > 100,000 nt:", (fa_index["seq_len"] > 100_000).sum())

human rows: 3854
diseases not matched to official list: 0
official disease names with 'NA' corruption: 0 []

rows with circBase ID in name:       2259
rows with ID only in Synonyms:       158
rows with >1 distinct ID (name+syn): 0
rows with 'hsa_circ_' but not a clean 7-digit ID: 65 ['hsa_circ_402458', 'hsa_circ_406329', 'hsa_circ_407176', 'hsa_circ_082319', 'hsa_circ_10072088', 'hsa_circ_101396']

ID found but NOT in FASTA: 5
resolved human rows: 2412 / 3854
unique circRNA-disease pairs: 2096
unique circRNAs: 1579 | unique diseases: 200
paper's circDis universe:  2099 pairs | 1577 circRNAs | 202 diseases

FASTA zero-length sequences: 58
FASTA > 10,000 nt: 5486 | > 100,000 nt: 335


**Interpretation.** Tier 1 alone resolves 2,412 of 3,854 human rows, yielding **2,096 pairs, 1,579 circRNAs and 200 diseases**, against 2,099 / 1,577 / 202 in the original benchmark. This close agreement is consistent with, but does not prove, the same matching logic as the authors' (the CircDis figures are typed from the reference paper and were not rechecked here). A previous claim that the "NA" corruption affected disease names was **refuted**: it occurs only in the CircDis matrix headers. Synonyms add 158 IDs without any conflict. Sixty-five "near-miss" IDs (mostly six digits) and five IDs absent from the FASTA remain.

**Decision.** The 1,442 unresolved rows are characterised next (Cell 9): which are recoverable through coordinates, and which are hopeless owing to a missing `Region`. In parallel, the reliability of coordinate matching must be calibrated against rows whose true ID is known (Cell 10).

### 9. Characterisation of the Unresolved Rows

**Objective.** Characterise the rows that Tier 1 could not resolve (IDs absent from the FASTA, near-miss IDs, format of `Region`) to see which of them are recoverable by coordinates.

In [30]:
# Cell 9 - Who remains unresolved, and why?
# A. The five IDs that were extracted but are absent from the FASTA
miss = hum[hum["circbase_id"].notna() & ~hum["in_fasta"]]
print("IDs not in FASTA:")
print(miss[["circ_raw", "circbase_id", "Region", "Strand", "Gene Symbol"]], "\n")

# B. The near-miss IDs: digit counts reveal whether they follow a recognisable pattern
nm = hum.loc[near, ["circ_raw", "Synonyms", "Region", "Strand", "Gene Symbol"]]
print("near-miss rows:", len(nm), "| with Region:", nm["Region"].notna().sum())
print("digit-count of the number after 'circ_':")
print(nm["circ_raw"].str.extract(r"circ_(\d+)")[0].str.len().value_counts(), "\n")
print(nm.head(10), "\n")

# C. Format of Region among unresolved rows: is coordinate matching feasible?
reg_pat = re.compile(r"^(chr\w+):(\d+)-(\d+)$")

def region_kind(r):
    """Categorise a Region string by how much positional information it carries."""
    if not isinstance(r, str) or not r.strip(): return "missing"
    r = r.strip()
    if reg_pat.match(r):                        return "full chr:start-end"
    if re.match(r"^chr\w+$", r):                return "chromosome only"
    return "other format"

un = hum[~hum["in_fasta"]].copy()
un["region_kind"] = un["Region"].map(region_kind)
print("unresolved rows:", len(un))
print(un["region_kind"].value_counts(), "\n")
print(pd.crosstab(un["name_pattern"], un["region_kind"], margins=True), "\n")
print("examples of 'other format':")
print(un.loc[un["region_kind"] == "other format", "Region"].drop_duplicates().head(8).tolist())
print("\nunresolved rows that have a Gene Symbol:", un["Gene Symbol"].notna().sum())

IDs not in FASTA:
              circ_raw       circbase_id                     Region Strand Gene Symbol
547   hsa_circ_0141539  hsa_circ_0141539  chr1:2072008388-207201024      -    C1orf116
1099  hsa_circ_0141940  hsa_circ_0141940   chr3:123332641-123332832      -        MYLK
1739  hsa_circ_0142527  hsa_circ_0142527     chr8:68934270-68950527      +       PREX2
2193  hsa_circ_0141206  hsa_circ_0141206    chr14:97312431-97327072      +        VRK1
3616  hsa_circ_0141940  hsa_circ_0141940   chr3:123332641-123332832      -        MYLK 

near-miss rows: 65 | with Region: 21
digit-count of the number after 'circ_':
0
6    58
5     5
8     1
4     1
Name: count, dtype: int64 

              circ_raw                    Synonyms Region Strand Gene Symbol
106    hsa_circ_402458                         NaN   chr2      +      CCNYL1
107    hsa_circ_406329                         NaN   chr3      +       FSTL1
108    hsa_circ_407176                         NaN   chr9      +      ZCCHC7
109    hsa

**Interpretation.** Of the 1,442 unresolved rows, 1,077 (75%) have no `Region` at all and 124 have only a chromosome; neither can be resolved from this file. Only about 240 rows (219 with a full `chr:start-end` and 22 in a recoverable "other format", e.g. with an en-dash) are candidates for Tier 2. Four of the five IDs missing from the FASTA have clean coordinates, whereas one is malformed. The 58 six-digit near-miss IDs may resemble microarray numbering with an incorrect prefix (inference, not tested).

**Decision.** *Inference:* the realistic size ceiling of the dataset is therefore about 1,650–1,800 circRNAs. Before applying coordinate matching to unresolved rows, its tolerance and trustworthiness are calibrated on rows with a trusted ID (Cell 10).

### 10. Calibration of Coordinate Matching

**Objective.** Calibrate coordinate matching on rows whose circBase ID is already trusted (Tier 1): how often do the entry `Region` and the FASTA coordinates agree, and by how many nucleotides do they differ?

In [31]:
# Cell 10 - Calibrate coordinate matching on rows whose ID is already trusted (Tier 1)
def parse_region(r):
    """Strict parser: returns (chromosome, start, end) or (None, None, None)."""
    m = reg_pat.match(r.strip()) if isinstance(r, str) else None
    return (m.group(1), int(m.group(2)), int(m.group(3))) if m else (None, None, None)

cal = res.copy()
parsed = cal["Region"].map(parse_region)
cal["chrom_e"] = parsed.map(lambda t: t[0])
cal["start_e"] = pd.to_numeric(parsed.map(lambda t: t[1]))
cal["end_e"]   = pd.to_numeric(parsed.map(lambda t: t[2]))
cal = cal.dropna(subset=["chrom_e"]).drop_duplicates("circbase_id")

# Attach the FASTA coordinates of the same ID, so that both sources can be compared
cal = cal.merge(fa_index[["circbase_id", "chrom", "start", "end", "strand"]]
                .rename(columns={"start": "start_f", "end": "end_f", "strand": "strand_f"}),
                on="circbase_id")
cal["start_f"] = pd.to_numeric(cal["start_f"])
cal["end_f"]   = pd.to_numeric(cal["end_f"])
# Signed differences reveal off-by-one conventions or genome-build shifts
cal["d_start"] = cal["start_e"] - cal["start_f"]
cal["d_end"]   = cal["end_e"] - cal["end_f"]

print("calibration set (resolved circRNAs with a full Region):", len(cal))
print("chromosome agrees:", (cal["chrom_e"] == cal["chrom"]).mean().round(3))
print("\nstart difference (entry - FASTA):")
print(cal["d_start"].value_counts().head(6))
print("\nend difference (entry - FASTA):")
print(cal["d_end"].value_counts().head(6))

# A match is accepted within +/-1 nt (0-based vs 1-based coordinate conventions)
same = (cal["d_start"].abs() <= 1) & (cal["d_end"].abs() <= 1) & (cal["chrom_e"] == cal["chrom"])
print("\nwithin +/-1 nt on both ends:", same.mean().round(3))

st = cal["Strand"].notna()
print("strand agrees (where given):", (cal.loc[st, "Strand"] == cal.loc[st, "strand_f"]).mean().round(3))
print("\nexamples of disagreement:")
print(cal.loc[~same, ["circbase_id", "Region", "chrom", "start_f", "end_f"]].head(8))

calibration set (resolved circRNAs with a full Region): 1564
chromosome agrees: 0.998

start difference (entry - FASTA):
d_start
 0.0          1539
 1.0             8
-15172.0         1
-8268529.0       1
-393778.0        1
-856510.0        1
Name: count, dtype: Int64

end difference (entry - FASTA):
d_end
 0.0          1547
 8400.0          1
-8275486.0       1
-393778.0        1
-856510.0        1
-288928.0        1
Name: count, dtype: Int64

within +/-1 nt on both ends: 0.989
strand agrees (where given): 0.996

examples of disagreement:
          circbase_id                    Region  chrom    start_f      end_f
358  hsa_circ_0015758  chr1:196695832-196723529   chr1  196711004  196715129
379  hsa_circ_0018955    chr3:71493425-71501988  chr10   79761954   79777474
380  hsa_circ_0027821   chr12:97492460-97530859  chr12   97886238   97924637
381  hsa_circ_0005280  chr2:168012633-168075231   chr2  168869143  168931741
382  hsa_circ_0000231   chr10:31908171-31910563  chr10   32197099   3

**Interpretation.** On 1,564 resolved circRNAs with a full `Region`, coordinates agree within ±1 nt in **98.9%** of cases (1,539 exact; 8 off by one) and strand agrees in 99.6%. Coordinate matching is therefore a reasonable second tier. The calibration set consists of rows that already carry a trusted circBase ID, so its agreement rate need not transfer exactly to the unresolved rows (selection effect; not testable here). About 1% of rows disagree in two distinct ways: same-length but shifted segments (suggesting another genome build) and genuinely inconsistent entries (different chromosome or length).

**Decision.** Tier 2 is run with a ±1 nt tolerance on the unresolved rows, verified by an independent signal (the host-gene symbol). Matches are accepted only when they are unique (Cell 11). The conflicting rows are triaged in Cell 12.

### 11. Tier 2: Coordinate Matching with Gene-Symbol Verification

**Objective.** Resolve still-unresolved rows by genomic coordinates with a ±1 nt tolerance (Tier 2), accept only unique matches, and verify each match with the host-gene symbol.

In [32]:
# Cell 11 - Tier 2: coordinate matching with gene-symbol verification
import numpy as np

# ---- tolerant parser: handles en/em dashes, case and spaces -----------------
VALID_CHR = {str(i) for i in range(1, 23)} | {"X", "Y", "M"}

def parse_region2(r):
    """Parse 'chrN:start-end' tolerantly; reject invalid chromosomes and start >= end."""
    if not isinstance(r, str):
        return (None, None, None)
    s = r.strip().replace("\u2013", "-").replace("\u2014", "-").replace(" ", "")
    m = re.match(r"^chr([0-9XYMxym]+):(\d+)-(\d+)$", s, re.I)
    if not m:
        return (None, None, None)
    tok = m.group(1).upper()
    start, end = int(m.group(2)), int(m.group(3))
    if tok not in VALID_CHR or start >= end:
        return (None, None, None)
    return ("chr" + tok, start, end)

# ---- FASTA lookup structures (grouped by chromosome for speed) ---------------
fa = fa_index.copy()
fa["start"] = pd.to_numeric(fa["start"]).astype(float)
fa["end"]   = pd.to_numeric(fa["end"]).astype(float)
fa_by_chr   = {c: g for c, g in fa.groupby("chrom")}
fa_gene     = fa.set_index("circbase_id")["gene"].to_dict()

def match_coords(chrom, s, e, strand, tol=1):
    """Return all circBase IDs whose coordinates match within `tol` nt (and strand, when given)."""
    g = fa_by_chr.get(chrom)
    if g is None:
        return []
    hit = g[((g["start"] - s).abs() <= tol) & ((g["end"] - e).abs() <= tol)]
    if strand in ("+", "-"):
        hit = hit[hit["strand"] == strand]
    return hit["circbase_id"].tolist()

# ---- apply to every row that Tier 1 could not resolve -------------------------
cand = hum[~hum["in_fasta"]].copy()
parsed = cand["Region"].map(parse_region2)
cand["chrom_e"] = parsed.map(lambda t: t[0])
cand["start_e"] = parsed.map(lambda t: t[1])
cand["end_e"]   = parsed.map(lambda t: t[2])

cand_ok = cand.dropna(subset=["chrom_e"]).copy()
cand_ok["hits"] = [match_coords(c, s, e, st)
                   for c, s, e, st in zip(cand_ok["chrom_e"], cand_ok["start_e"],
                                          cand_ok["end_e"], cand_ok["Strand"])]
cand_ok["n_hits"] = cand_ok["hits"].map(len)

print("unresolved rows:", len(cand))
print("rows with a parseable full Region:", len(cand_ok))
print("\nnumber of FASTA matches per row:")
print(cand_ok["n_hits"].value_counts().sort_index())

# Only unambiguous (exactly one) matches are considered further
uniq = cand_ok[cand_ok["n_hits"] == 1].copy()
uniq["matched_id"] = uniq["hits"].str[0]

# ---- independent verification with the host-gene symbol -------------------------
def gene_check(row):
    """Compare the entry's gene symbol with the FASTA gene of the matched ID."""
    g_e = row["Gene Symbol"]
    g_f = fa_gene.get(row["matched_id"])
    if not isinstance(g_e, str) or g_f in (None, "None"):
        return "n/a"
    return "agree" if g_e.strip().casefold() == str(g_f).strip().casefold() else "disagree"

uniq["gene_check"] = uniq.apply(gene_check, axis=1)
print("\nunique matches:", len(uniq))
print(uniq["gene_check"].value_counts())
print("\nexamples of gene disagreement:")
print(uniq.loc[uniq["gene_check"] == "disagree",
               ["circ_raw", "Region", "Gene Symbol", "matched_id"]].head(6))

# Accept unique matches that do not contradict the gene annotation
accept = uniq[uniq["gene_check"] != "disagree"]
hum["circbase_final"] = hum["circbase_id"].where(hum["in_fasta"])
hum["match_tier"] = hum["in_fasta"].map({True: "tier1_id", False: None})
hum.loc[accept.index, "circbase_final"] = accept["matched_id"]
hum.loc[accept.index, "match_tier"] = "tier2_coords"

# ---- updated totals ----------------------------------------------------------------
res2   = hum[hum["circbase_final"].notna()]
pairs2 = res2[["circbase_final", "disease_std"]].drop_duplicates()
print("\nmatch tiers:")
print(hum["match_tier"].value_counts(dropna=False))
print("\nunique pairs:", len(pairs2),
      "| circRNAs:", pairs2["circbase_final"].nunique(),
      "| diseases:", pairs2["disease_std"].nunique())
print("before tier 2:  2096 pairs | 1579 circRNAs | 200 diseases")

# Inspect regions that could not be parsed but contain long digit strings (possible recoverable typos)
left = cand[cand["chrom_e"].isna() & cand["Region"].notna()
            & cand["Region"].astype(str).str.contains(r"\d{5,}")]
print("\nunparseable but number-like regions:", left["Region"].drop_duplicates().tolist()[:10])

unresolved rows: 1442
rows with a parseable full Region: 228

number of FASTA matches per row:
n_hits
0     86
1    142
Name: count, dtype: int64

unique matches: 142
gene_check
agree       118
n/a          18
disagree      6
Name: count, dtype: int64

examples of gene disagreement:
                circ_raw                    Region  Gene Symbol        matched_id
572          circSMARC1A  chr4:144464662-144465125      SMARC1A  hsa_circ_0001445
855   hsa_circRNA_002086    chr7:30590251-30601744    LOC401320  hsa_circ_0001693
1640           circ_2137   chr13:64564228-64608670     AK057471  hsa_circ_0030388
2049     hsa_circ_003785   chr18:32902351-32907027  XLOC_012658  hsa_circ_0003785
3158            circPVT1  chr8:128902834-128903244         PVT1  hsa_circ_0001821
3825           circ-Rtn4    chr3:17059499-17059748         RTN4  hsa_circ_0001275

match tiers:
match_tier
tier1_id        2412
NaN             1306
tier2_coords     136
Name: count, dtype: int64

unique pairs: 2208 | circRN

**Interpretation.** Of 228 parseable unresolved rows, 142 hit exactly one FASTA circRNA, 86 hit none, and **none was ambiguous**. Gene verification agreed in 118 cases, was not applicable in 18, and disagreed in 6; 136 matches were accepted. Totals rose to **2,208 pairs, 1,647 circRNAs and 209 diseases**. Several "disagreements" appear to be annotation artefacts (placeholder labels such as `LOC…`, `AK…`, `XLOC…`) rather than wrong matches; this requires closer inspection. Some unparseable regions show *reversed* coordinates (start > end).

**Decision.** The conflicting Tier 1 rows (Cell 12) and the six gene disagreements (Cell 13) are examined further. The triage code of Cell 12 contains a limitation (reversed coordinates are not swapped) that is corrected in Cell 13.

### 12. First Triage of Tier 1 Conflicts (Provisional)

**Objective.** Make a first triage of the Tier 1 rows whose coordinates disagree with the FASTA. This version is provisional and is superseded by Cell 13.

In [33]:
# Cell 12 - First triage of Tier 1 conflicts (NOTE: superseded by Cell 13, see limitation below)
cf = cal.loc[~same].copy()
# Compare interval lengths: equal lengths with shifted positions suggest a different genome build
same_len = ((cf["end_e"] - cf["start_e"]) - (cf["end_f"] - cf["start_f"])).abs() <= 1
chrom_ok = cf["chrom_e"] == cf["chrom"]

cf["kind"] = np.select(
    [~chrom_ok, same_len & chrom_ok],
    ["chromosome mismatch", "same length, shifted (likely other genome build)"],
    default="different length/coordinates")

print(cf["kind"].value_counts(), "\n")
print(cf[["circbase_id", "Region", "chrom", "start_f", "end_f", "kind"]].to_string())

kind
same length, shifted (likely other genome build)    8
different length/coordinates                        6
chromosome mismatch                                 3
Name: count, dtype: int64 

          circbase_id                    Region  chrom    start_f      end_f                                              kind
358  hsa_circ_0015758  chr1:196695832-196723529   chr1  196711004  196715129                      different length/coordinates
379  hsa_circ_0018955    chr3:71493425-71501988  chr10   79761954   79777474                               chromosome mismatch
380  hsa_circ_0027821   chr12:97492460-97530859  chr12   97886238   97924637  same length, shifted (likely other genome build)
381  hsa_circ_0005280  chr2:168012633-168075231   chr2  168869143  168931741  same length, shifted (likely other genome build)
382  hsa_circ_0000231   chr10:31908171-31910563  chr10   32197099   32199491  same length, shifted (likely other genome build)
383  hsa_circ_0007930  chr5:136147831-13615

**Interpretation.** The triage yielded 8 "same length, shifted", 6 "different length" and 3 "chromosome mismatch" rows. Four entries, however, have start > end (coordinates written in reverse), which this code did not swap; consequently, their length was miscomputed and three rows were wrongly labelled "different length". The classification is therefore *provisional*.

**Decision.** Cell 13 repeats the triage after swapping reversed coordinates, inspects the FASTA gene fields for the six gene disagreements, applies a lenient parser to the remaining rows, and tests a weaker length-plus-gene fallback.

### 13. Corrected Triage, Gene Disagreements and Lenient Re-parsing

**Objective.** Repeat the conflict triage after swapping reversed coordinates, inspect the six gene disagreements, re-parse the remaining regions with a lenient parser and test a weaker length-plus-gene fallback. The cell only reports; the table `hum` is not modified.

In [34]:
# Cell 13 - Corrected triage, gene disagreements and lenient re-parsing (report only; `hum` is not modified)
# ---- A. The six gene disagreements: what does the FASTA annotation contain? -------
fa_tx = fa_index.set_index("circbase_id")["transcript"].to_dict()
dis6 = uniq[uniq["gene_check"] == "disagree"].copy()
dis6["fasta_gene"] = dis6["matched_id"].map(fa_gene)
dis6["fasta_tx"]   = dis6["matched_id"].map(fa_tx)
print("A. gene disagreements")
print(dis6[["circ_raw", "Gene Symbol", "fasta_gene", "fasta_tx", "matched_id"]].to_string(), "\n")

# ---- B. Conflicts re-triaged with reversed coordinates swapped -----------------------
cf2 = cal.loc[~same].copy()
cf2["reversed_in_entry"] = cf2["start_e"] > cf2["end_e"]
cf2["len_e"] = (cf2["end_e"] - cf2["start_e"]).abs()      # absolute value handles reversed intervals
cf2["len_f"] = cf2["end_f"] - cf2["start_f"]
cf2["same_len"] = (cf2["len_e"] - cf2["len_f"]).abs() <= 1
cf2["kind"] = np.select(
    [cf2["chrom_e"] != cf2["chrom"], cf2["same_len"]],
    ["chromosome mismatch", "same length, shifted"],
    default="different length")
print("B. corrected conflict triage")
print(cf2["kind"].value_counts(), "\n")
print(cf2.loc[cf2["kind"] != "same length, shifted",
              ["circbase_id", "Region", "chrom", "start_f", "end_f", "len_e", "len_f", "kind"]].to_string(), "\n")

# ---- C. Lenient re-parse of rows still unresolved ----------------------------------------
def parse_region3(r):
    """Lenient parser: normalises dashes, strips stray ')', swaps reversed coordinates and flags them."""
    if not isinstance(r, str):
        return (None, None, None, False)
    s = r.strip().replace("\u2013", "-").replace("\u2014", "-").replace(" ", "").rstrip(")")
    m = re.match(r"^chr([0-9XYMxym]+):(\d+)-(\d+)$", s, re.I)
    if not m:
        return (None, None, None, False)
    tok, a, b = m.group(1).upper(), int(m.group(2)), int(m.group(3))
    if tok not in VALID_CHR or a == b:
        return (None, None, None, False)
    return ("chr" + tok, min(a, b), max(a, b), a > b)

rest = hum[hum["circbase_final"].isna()].copy()
p3 = rest["Region"].map(parse_region3)
rest["chrom_e"] = p3.map(lambda t: t[0])
rest["s"]       = p3.map(lambda t: t[1])
rest["e"]       = p3.map(lambda t: t[2])
rest["rev"]     = p3.map(lambda t: t[3])
rest = rest.dropna(subset=["chrom_e"])
rest = rest[(rest["e"] - rest["s"]) <= 1_000_000]          # discard implausibly large spans

rest["hits"] = [match_coords(c, s, e, st)
                for c, s, e, st in zip(rest["chrom_e"], rest["s"], rest["e"], rest["Strand"])]
rest["n_hits"] = rest["hits"].map(len)
print("C. lenient parse of unresolved rows:", len(rest), "| reversed:", int(rest["rev"].sum()))
print(rest["n_hits"].value_counts().sort_index(), "\n")

def length_gene_hits(chrom, s, e, gene, tol=1):
    """Weaker fallback: same chromosome, same interval length (+/- tol) and same gene symbol."""
    g = fa_by_chr.get(chrom)
    if g is None or not isinstance(gene, str):
        return []
    hit = g[((g["end"] - g["start"]) - (e - s)).abs() <= tol]
    return hit.loc[hit["gene"].astype(str).str.casefold() == gene.strip().casefold(),
                   "circbase_id"].tolist()

zero = rest[rest["n_hits"] == 0].copy()
zero["lg_hits"] = [length_gene_hits(c, s, e, g)
                   for c, s, e, g in zip(zero["chrom_e"], zero["s"], zero["e"], zero["Gene Symbol"])]
zero["n_lg"] = zero["lg_hits"].map(len)
print("zero exact hits:", len(zero), "| has a Gene Symbol:", zero["Gene Symbol"].notna().sum())
print("same chromosome + same length + same gene (possible other-build entry):")
print(zero["n_lg"].value_counts().sort_index(), "\n")
print(zero.loc[zero["n_lg"] >= 1, ["circ_raw", "Region", "Gene Symbol", "lg_hits"]].head(8).to_string())

print("\nstill unresolved overall:", hum["circbase_final"].isna().sum(), "/", len(hum))

A. gene disagreements
                circ_raw  Gene Symbol         fasta_gene           fasta_tx        matched_id
572          circSMARC1A      SMARC1A            SMARCA5          NM_003601  hsa_circ_0001445
855   hsa_circRNA_002086    LOC401320     TCONS_00012989     TCONS_00012989  hsa_circ_0001693
1640           circ_2137     AK057471  TCONS_l2_00007569  TCONS_l2_00007569  hsa_circ_0030388
2049     hsa_circ_003785  XLOC_012658     TCONS_00026319     TCONS_00026319  hsa_circ_0003785
3158            circPVT1         PVT1     TCONS_00015354     TCONS_00015354  hsa_circ_0001821
3825           circ-Rtn4         RTN4              PLCL2       NM_001144382  hsa_circ_0001275 

B. corrected conflict triage
kind
same length, shifted    11
different length         3
chromosome mismatch      3
Name: count, dtype: int64 

          circbase_id                    Region  chrom    start_f      end_f     len_e  len_f                 kind
358  hsa_circ_0015758  chr1:196695832-196723529   chr1  1967

**Interpretation.** (A) Five of the six gene "disagreements" appear not to be genuine: the entry contains a probable typographical error (`SMARC1A` for `SMARCA5`, inferred) or either side holds a placeholder (`TCONS`, `XLOC`, `LOC`, `AK`) rather than a gene symbol. Only `circ-Rtn4` (RTN4 vs PLCL2, both real symbols) is a true conflict. (B) After swapping reversed coordinates, the conflicts are 11 *same-length shifted* (kept, flagged as probable genome-build differences), 3 *different length* and 3 *chromosome mismatch* (excluded). (C) The lenient parse recovers almost nothing new (7 single hits; their overlap with previously rejected rows was not checked), whereas 27 zero-hit rows have a unique same-length, same-gene candidate.

**Decision.** The gene check must ignore placeholder annotations and accept a documented manual exception; all decisions are encoded as explicit flags in a saved table (Cell 14). The length-plus-gene fallback and the zero-padding of near-miss IDs are not calibrated in this notebook and are not used for the primary set; the 27 candidate rows therefore stay unresolved.

### 14. Applying the Decisions: Flagged Resolved Table

**Objective.** Apply the decisions of Cells 11 and 13 (placeholder gene labels ignored, one documented manual exception), flag every row, define the primary analysis set (`keep`) and save the first reproducible table, `cda_entries_resolved_v1.csv`.

In [35]:
# Cell 14 - Apply the decisions, flag every row and save the first reproducible table
# Strings that denote assembly/placeholder labels rather than genuine gene symbols
PLACEHOLDER = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$)", re.I)
# Documented manual exception: the entry gene 'SMARC1A' is a typographical error for 'SMARCA5'
MANUAL_OK = {"hsa_circ_0001445"}

def gene_status(g_entry, g_fa, cid):
    """Compare entry and FASTA gene symbols; placeholders yield 'n/a' because they cannot be compared."""
    if cid in MANUAL_OK:
        return "manual_ok"
    if not isinstance(g_entry, str) or not isinstance(g_fa, str):
        return "n/a"
    if PLACEHOLDER.match(g_entry.strip()) or PLACEHOLDER.match(g_fa.strip()):
        return "n/a"
    return "agree" if g_entry.strip().casefold() == g_fa.strip().casefold() else "disagree"

# ---- Tier 2 candidates: union of the strict (uniq) and the lenient (rest) parses --------
a  = uniq[["Gene Symbol", "matched_id"]]
r1 = rest[rest["n_hits"] == 1]
b  = r1[["Gene Symbol"]].assign(matched_id=r1["hits"].str[0])
t2 = pd.concat([a, b])
t2 = t2[~t2.index.duplicated(keep="first")].copy()       # one candidate per original row
t2["gene_status"] = [gene_status(g, fa_gene.get(i), i)
                     for g, i in zip(t2["Gene Symbol"], t2["matched_id"])]
print("tier-2 candidates:", len(t2))
print(t2["gene_status"].value_counts(), "\n")
t2_keep = t2[t2["gene_status"] != "disagree"]

# ---- assemble the final table: nothing is deleted, every decision is recorded in a flag ---
final = hum.copy()
final["circbase_final"] = final["circbase_id"].where(final["in_fasta"])
final["match_tier"] = final["in_fasta"].map({True: "tier1_id", False: None})
final.loc[t2_keep.index, "circbase_final"] = t2_keep["matched_id"]
final.loc[t2_keep.index, "match_tier"] = "tier2_coords"

final["flag"] = final["circbase_final"].notna().map({True: "ok", False: "unresolved"})
na_idx = t2_keep.index[(t2_keep["gene_status"] == "n/a").to_numpy()]
final.loc[na_idx, "flag"] = "gene_unverifiable"
gc_idx = t2.index[(t2["gene_status"] == "disagree").to_numpy()]
final.loc[gc_idx, "flag"] = "gene_conflict"

# Flags derived from the Tier 1 calibration (Cell 13, part B)
shifted_ids  = set(cf2.loc[cf2["kind"] == "same length, shifted", "circbase_id"])
conflict_ids = set(cf2.loc[cf2["kind"] != "same length, shifted", "circbase_id"])
final.loc[final["circbase_final"].isin(shifted_ids),  "flag"] = "shifted_build"
final.loc[final["circbase_final"].isin(conflict_ids), "flag"] = "id_conflict"
# `keep` defines the primary analysis set
final["keep"] = final["circbase_final"].notna() & ~final["circbase_final"].isin(conflict_ids)

# ---- report and save ----------------------------------------------------------------------------
print(final["flag"].value_counts(), "\n")
core  = final[final["keep"]]
pairs = core[["circbase_final", "disease_std"]].drop_duplicates()
print("KEEP rows:", len(core), "| pairs:", len(pairs),
      "| circRNAs:", pairs["circbase_final"].nunique(),
      "| diseases:", pairs["disease_std"].nunique())
print("tier-1 only was: 2096 pairs | 1579 circRNAs | 200 diseases")

cols = ["CRD ID", "circ_raw", "circbase_final", "disease_std", "match_tier",
        "flag", "keep", "PubMed ID", "Region", "Strand", "Gene Symbol"]
final[cols].to_csv(PROCESSED_DIR / "cda_entries_resolved_v1.csv", index=False)
print("saved:", PROCESSED_DIR / "cda_entries_resolved_v1.csv")

tier-2 candidates: 143
gene_status
agree        115
n/a           23
manual_ok      3
disagree       2
Name: count, dtype: int64 

flag
ok                   2506
unresolved           1299
gene_unverifiable      23
shifted_build          18
id_conflict             6
gene_conflict           2
Name: count, dtype: int64 

KEEP rows: 2547 | pairs: 2206 | circRNAs: 1644 | diseases: 209
tier-1 only was: 2096 pairs | 1579 circRNAs | 200 diseases
saved: d:\Queens_university\Project\Working\Notebook\circmtl\data\processed\cda_entries_resolved_v1.csv


**Interpretation.** The primary (`keep`) set comprises 2,547 rows, i.e. **2,206 pairs, 1,644 circRNAs and 209 diseases**. Flags: `ok` 2,506; `gene_unverifiable` 23; `shifted_build` 18; excluded – `id_conflict` 6, `gene_conflict` 2, `unresolved` 1,299. The usable CDA graph therefore contains about 2,200 pairs rather than 4,201 entries, at a density of roughly 0.64%.

**Decision.** The length-plus-gene fallback (27 candidate rows in Cell 13) and the zero-padding of six-digit near-miss IDs are not evaluated in this notebook, so those rows stay unresolved and outside the primary set. The flags allow every later result to be reported with and without the lower-confidence groups (`gene_unverifiable`, `shifted_build`). The file `cda_entries_resolved_v1.csv` is written here, before the override of Cell 15, and therefore does not contain that override.

### 15. Row-Level Override of the False-Alarm Gene Conflict

**Objective.** Correct one false-alarm gene conflict with an explicit, assertion-protected row override and check that the counts are unaffected.

In [36]:
# Cell 15 - Row-level override for the single false-alarm gene conflict
# Evidence: the name implies the ID, the coordinates match uniquely, and the entry gene 'ST'
# is a truncation of 'ST5' (the gene rule had wrongly classified this row as 'disagree').
OVERRIDES = {
    2826: ("circ_0021093", "hsa_circ_0021093"),   # index: (expected circ_raw, circBase ID)
}

before = set(map(tuple, final.loc[final["keep"], ["circbase_final", "disease_std"]]
                 .drop_duplicates().values))

for idx, (raw, cid) in OVERRIDES.items():
    # The assertion protects against silently patching the wrong row if the index changes
    assert final.loc[idx, "circ_raw"] == raw, f"index {idx} changed - check before patching"
    final.loc[idx, "circbase_final"] = cid
    final.loc[idx, "match_tier"] = "tier2_coords"
    final.loc[idx, "flag"] = "ok"
    final.loc[idx, "keep"] = True

core = final[final["keep"]]
after = core[["circbase_final", "disease_std"]].drop_duplicates()
print("pairs:", len(before), "->", len(after),
      "| circRNAs:", after["circbase_final"].nunique(),
      "| diseases:", after["disease_std"].nunique())
print(final.loc[list(OVERRIDES), ["circ_raw", "circbase_final", "disease_std", "flag", "keep"]])
print("gene_conflict rows left:", int((final["flag"] == "gene_conflict").sum()))

pairs: 2206 -> 2206 | circRNAs: 1644 | diseases: 209
          circ_raw    circbase_final               disease_std flag  keep
2826  circ_0021093  hsa_circ_0021093  Hepatocellular carcinoma   ok  True
gene_conflict rows left: 1


**Interpretation.** The override changes no counts (2,206 pairs; 1,644 circRNAs; 209 diseases), because the pair (`hsa_circ_0021093`, Hepatocellular carcinoma) already existed through another row. One `gene_conflict` row remains excluded; it is presumably `circ-Rtn4` (RTN4 versus PLCL2, both real symbols; Cell 13A). *Inference (not verified against the source paper):* the entry gene `ST` is a truncation of `ST5`. The override is therefore a single manual, documented exception.

The override is applied to the in-memory table `final` only. `cda_entries_resolved_v1.csv` (Cell 14) does not contain it, whereas the frozen pair table of Cell 26 is rebuilt from `final` and does.

**Decision.** The override is kept. The assertion on `circ_raw` stops the cell if the row index ever changes, so a wrong row cannot be patched silently. Next, what is lost during identifier resolution is analysed (Cell 16).

### 16. Losses During Identifier Resolution and Structure of the Retained Data

**Objective.** Measure which entries are lost during identifier resolution (by naming scheme and by disease), and describe the sequence lengths and the sparsity of the retained circRNA–disease graph.

In [37]:
# Cell 16 - What is lost during identifier resolution, and what do the retained sequences look like?
# ---- A. Who is lost? --------------------------------------------------------------------------
print("A1. retention by naming scheme:")
print(final.groupby("name_pattern")["keep"].agg(rows="size", kept="sum", share="mean").round(2), "\n")

d_all  = final["disease_std"].value_counts().rename("rows_all")
d_keep = final.loc[final["keep"], "disease_std"].value_counts().rename("rows_kept")
dis = pd.concat([d_all, d_keep], axis=1).fillna(0).astype(int)
dis["retention"] = (dis["rows_kept"] / dis["rows_all"]).round(2)

print("A2. ten most-reported diseases:")
print(dis.sort_values("rows_all", ascending=False).head(10), "\n")
print("diseases with zero kept rows:", int((dis["rows_kept"] == 0).sum()), "of", len(dis))
print("diseases with >= 5 kept rows:", int((dis["rows_kept"] >= 5).sum()))
print("diseases with >= 10 rows overall but retention < 0.4:")
print(dis[(dis["rows_all"] >= 10) & (dis["retention"] < 0.4)]
      .sort_values("rows_all", ascending=False), "\n")

# ---- B. Sequence status of the retained circRNAs ----------------------------------------------------
L = fa_index.set_index("circbase_id")["seq_len"]
kept = final[final["keep"]].copy()
kept["seq_len"] = kept["circbase_final"].map(L)
pp  = kept[["circbase_final", "disease_std", "seq_len"]].drop_duplicates()   # unique (circRNA, disease) pairs
ids = pp.drop_duplicates("circbase_final")                                   # unique circRNAs

print("B1. sequence length of the kept circRNAs (nt):")
print(ids["seq_len"].describe(percentiles=[.5, .75, .9, .95]).round(0))
print("zero-length:", int((ids["seq_len"] == 0).sum()),
      "| 1-1022:", int(ids["seq_len"].between(1, 1022).sum()),
      "| 1023-5000:", int(ids["seq_len"].between(1023, 5000).sum()),
      "| > 5000:", int((ids["seq_len"] > 5000).sum()))
ok_len = pp[pp["seq_len"].between(1, 1022)]
print("\nB2. if we keep only circRNAs of 1-1022 nt:")
print("pairs:", len(ok_len), "| circRNAs:", ok_len["circbase_final"].nunique(),
      "| diseases:", ok_len["disease_std"].nunique())

# ---- C. Sparsity structure of the bipartite graph --------------------------------------------------------
deg  = pp.groupby("circbase_final").size()      # diseases per circRNA
ddeg = pp.groupby("disease_std").size()         # circRNAs per disease
print("\nC1. diseases per circRNA:")
print(deg.value_counts().sort_index().head(8))
print("circRNAs with exactly one disease:", round((deg == 1).mean(), 3))
print("most-connected circRNAs:\n", deg.sort_values(ascending=False).head(5))
print("\nC2. circRNAs per disease: median", ddeg.median(),
      "| diseases with < 3 circRNAs:", int((ddeg < 3).sum()), "of", len(ddeg))

A1. retention by naming scheme:
                    rows  kept  share
name_pattern                         
arraystar_like       339    25   0.07
circbase_full       2317  2255   0.97
circbase_no_prefix   217    76   0.35
coordinate            14     3   0.21
other                967   189   0.20 

A2. ten most-reported diseases:
                            rows_all  rows_kept  retention
disease_std                                               
Gastric cancer                   277        194       0.70
Hepatocellular carcinoma         254        182       0.72
Breast cancer                    191        131       0.69
Colorectal cancer                189        138       0.73
Non-small cell lung cancer       169        115       0.68
Glioma                           133         95       0.71
Osteosarcoma                     126         77       0.61
Bladder cancer                   101         78       0.77
Ovarian cancer                    83         44       0.53
Prostate cancer    

**Interpretation.** *Selection bias:* retention is 97% for circBase-style names but only 7% for Arraystar-like and 20% for other custom names; the retained set therefore over-represents circRNAs annotated in circBase. The ten most-reported diseases keep 53–77% of their rows; 65 of 274 diseases are lost entirely. *Length:* 75% of retained circRNAs fit within 1,022 nt; applying that filter would cost about 25% of pairs and 29 diseases. *Sparsity:* **80.5% of circRNAs have exactly one disease**, and the hubs (e.g. `hsa_circ_0000284`, 27 diseases) correspond to famous circRNAs.

**Decision.** Because most circRNAs have one disease edge, a random edge split will frequently leave a test circRNA without any training edge. This is quantified by simulation (Cell 17). The proposal's 1,022 nt filter is reconsidered, and the long-sequence behaviour is checked in the same cell.

### 17. Cold-Start Share, Length Policies, Hubs and Sources of Loss

**Objective.** Quantify the cold-start share under a random edge split, compare sequence-length policies, inspect very long records and hub circRNAs, and test whether the losses are concentrated in a few publications.

In [38]:
# Cell 17 - Cold-start share under a random edge split, length policies, hubs and sources of loss
from sklearn.model_selection import KFold

pp = (final.loc[final["keep"], ["circbase_final", "disease_std"]]
      .drop_duplicates().reset_index(drop=True))
L = fa_index.set_index("circbase_id")["seq_len"]
pp["seq_len"] = pp["circbase_final"].map(L)

# ---- A. Simulate a random 5-fold edge split (three seeds) -----------------------------------
rows = []
for seed in (0, 1, 2):
    for k, (tr, te) in enumerate(KFold(5, shuffle=True, random_state=seed).split(pp)):
        t = pp.iloc[te]
        rows.append({"seed": seed, "fold": k, "test_pos": len(t),
                     # share of test positives whose circRNA / disease has no training edge at all
                     "cold_circ": (~t["circbase_final"].isin(set(pp.iloc[tr]["circbase_final"]))).mean(),
                     "cold_disease": (~t["disease_std"].isin(set(pp.iloc[tr]["disease_std"]))).mean()})
print("A. share of test positives with no training edge (mean over folds):")
print(pd.DataFrame(rows).groupby("seed")[["test_pos", "cold_circ", "cold_disease"]].mean().round(3), "\n")

# ---- B. Length policies, very long circRNAs and hubs ------------------------------------------------
print("B1. pairs / circRNAs / diseases by length band:")
for lo, hi in [(1, 1022), (1023, 5000), (5001, 10**9)]:
    s = pp[pp["seq_len"].between(lo, hi)]
    print(f"  {lo}-{hi}: {len(s)} pairs | {s['circbase_final'].nunique()} circRNAs "
          f"| {s['disease_std'].nunique()} diseases")
s = pp[pp["seq_len"].between(1, 5000)]
print(f"  <=5000 total: {len(s)} pairs | {s['circbase_final'].nunique()} circRNAs "
      f"| {s['disease_std'].nunique()} diseases\n")

# If the sequence length equals the genomic span, no intron was removed: the record is not a spliced product
span = fa.set_index("circbase_id")["end"] - fa.set_index("circbase_id")["start"]
lg = (pp[pp["seq_len"] > 5000].drop_duplicates("circbase_final")
      .sort_values("seq_len", ascending=False).head(8).copy())
lg["genomic_span"] = lg["circbase_final"].map(span)
lg["gene"] = lg["circbase_final"].map(fa_gene)
print("B2. longest kept circRNAs (seq_len vs genomic span):")
print(lg[["circbase_final", "seq_len", "genomic_span", "gene"]].to_string(), "\n")

deg = pp.groupby("circbase_final").size().sort_values(ascending=False)
hub = deg.head(8).rename("n_diseases").to_frame()
hub["gene"] = hub.index.map(fa_gene)
hub["seq_len"] = hub.index.map(L)
print("B3. hub circRNAs:")
print(hub.to_string(), "\n")

# ---- C. Are the losses concentrated in a few publications? ----------------------------------------------
lost = final[final["flag"] == "unresolved"]
vc = lost["PubMed ID"].value_counts()
print("C1. unresolved rows:", len(lost), "| distinct papers:", lost["PubMed ID"].nunique())
print("share from the 10 biggest papers:", round(vc.head(10).sum() / len(lost), 3))
print(vc.head(10), "\n")

dis_all = final["disease_std"].value_counts()
dis_kept = final.loc[final["keep"], "disease_std"].value_counts()
zero_dis = dis_all.index.difference(dis_kept.index)
print("C2. naming scheme of rows in the", len(zero_dis), "diseases with zero kept rows:")
print(final[final["disease_std"].isin(zero_dis)]["name_pattern"].value_counts())

A. share of test positives with no training edge (mean over folds):
      test_pos  cold_circ  cold_disease
seed                                   
0        441.2      0.636         0.034
1        441.2      0.652         0.035
2        441.2      0.646         0.038 

B1. pairs / circRNAs / diseases by length band:
  1-1022: 1665 pairs | 1232 circRNAs | 180 diseases
  1023-5000: 472 pairs | 350 circRNAs | 131 diseases
  5001-1000000000: 69 pairs | 62 circRNAs | 38 diseases
  <=5000 total: 2137 pairs | 1582 circRNAs | 207 diseases

B2. longest kept circRNAs (seq_len vs genomic span):
        circbase_final  seq_len  genomic_span       gene
2178  hsa_circ_0003420  1211004     1397938.0      ABCC1
422   hsa_circ_0033988   453671      453671.0  LINC00221
909   hsa_circ_0055538   368628      400079.0     RMND5A
865   hsa_circ_0043603   242759      246061.0      KRT19
1458  hsa_circ_0025580   161704      201335.0    SLCO1B3
861   hsa_circ_0062459    97097       97097.0       None
2037  hsa_

**Interpretation.** (A) about **64%** of test positives under a random edge split (64.0–64.6% over three seeds) have a circRNA without any training edge (only 3.5–3.6% on the disease side); a random split is therefore already largely a cold-start test. (B) Restricting to ≤5,000 nt retains 2,137 of 2,206 pairs and 1,582 circRNAs (96%), whereas ≤1,022 nt would lose about 25%; three of the longest records have a sequence length equal to the genomic span, which suggests unspliced intervals (inference from these three records). The 62 circRNAs above 5,000 nt are excluded as a policy choice. The hubs are well-studied circRNAs (circHIPK3, CDR1as, circZNF609, …), suggesting popularity bias. (C) An earlier guess was **refuted**: losses are distributed over 846 papers (top ten = 10.6%), so they follow naming conventions rather than a few studies.

**Decision.** (i) Primary input: circRNAs ≤ 5,000 nt, with a junction-centred crop for those above 1,022 nt; the strict 1,022 nt filter becomes a sensitivity analysis (a deviation from the proposal to be discussed with the supervisor). (ii) The evaluation will include a circRNA-grouped split, degree-stratified metrics and popularity baselines. (iii) The auxiliary-label file is inspected next (Cell 18).

### 18. Inspection of the Auxiliary-Label File

**Objective.** Inspect the curated file of auxiliary associations (shape, missingness, species, content and separators of the label columns) and check its join key against the entries table before any join is performed.

In [39]:
# Cell 18 - Inspect the auxiliary-label file before joining anything
cur = tables["The_manually_curated_associations_between_circRNAs_and_other_associations"][
    "CircRNA and Other Associations"]
lab_cols = ["miRNA validated", "miRNA predicted", "miRNA target", "RBP validated"]

print(cur.shape)
print("\nshare missing:\n", cur.isna().mean().round(3))
print("\nSpecies:\n", cur["Species"].value_counts(dropna=False).head(6))

# For each label column: sample values and count candidate list separators
for c in lab_cols:
    s = cur[c].dropna().astype(str)
    print(f"\n{c}: non-empty {len(s)}")
    print("  samples:", s.sample(min(6, len(s)), random_state=0).tolist())
    print("  separators:", {sp: int(s.str.contains(sp, regex=False).sum())
                            for sp in [",", ";", "/", "|", "\n", " and "]})

# Join-key check: does every curated (circRNA, disease) pair also occur in the entries table?
key = lambda d: (d["circRNA Name"].map(clean_txt) + "||" + d["Disease Name"].map(clean_txt)).str.casefold()
ck, ek = key(cur), key(entries)
print("\ncurated rows whose (circRNA, disease) appears in the entries table:", round(ck.isin(set(ek)).mean(), 3))
print("duplicate (circRNA, disease) keys in the curated file:", int(ck.duplicated().sum()))

(2160, 7)

share missing:
 circRNA Name       0.000
Disease Name       0.000
Species            0.000
miRNA validated    0.156
miRNA predicted    0.887
miRNA target       0.315
RBP validated      0.857
dtype: float64

Species:
 Species
Human    2027
Mouse     107
Rat        24
human       1
rats        1
Name: count, dtype: int64

miRNA validated: non-empty 1822
  samples: ['miR-421', 'miR-22-3p', 'miR-432-5p', 'miR-1179', 'miR-185-5p', 'miR-766-3p']
  separators: {',': 105, ';': 0, '/': 5, '|': 0, '\n': 0, ' and ': 5}

miRNA predicted: non-empty 244
  samples: ['hsa-miR-4778-3p, hsa-miR-5002-5p, hsa-miR-329-5p, hsa-miR-4652-3p, hsa-miR-6781-3p', 'hsa-miR-578, hsa-miR-429, hsa-miR-449a, hsa-miR-449b-5p, hsa-miR-34c-5p', 'mmu-miR-653-5p, mmu-miR-17-3p, mmu-miR-6394, mmu-miR-6948-3p, mmu-miR-148b-3p', 'hsa-miR-369-5p, hsa-miR-626, hsa-miR-326, hsa-miR-330-5p, hsa-miR-1265, hsa-miR-622', 'hsa-miR-302c-3p, hsa-miR-520d-3p, hsa-miR-302b-3p, hsa-miR-302d-3p, hsa-miR-197-5p', 'hsa-miR-138-5p,

**Interpretation.** `miRNA validated` is populated in 84% of rows and is the primary miRNA label; `miRNA predicted` (11%) is computational and `miRNA target` (69%) contains *target genes* (e.g. CREB1, MMP2), not miRNAs; both are excluded. `RBP validated` is sparse (14%) and dominated by AGO2 (including "Ago2,IgG"). All curated keys occur in the entries table (the file is, in effect, a view of it) and 168 (circRNA, disease) keys are duplicated. *Inference:* the miRNA/RBP annotations therefore come from the same publications as the disease labels, which makes them circular as auxiliary supervision for the disease task; this motivates the independent ENCORI source (Cells 27–43). Separators are inconsistent (commas, "/", "and").

**Decision.** Normalisation rules for miRNA and RBP tokens must be designed empirically: the cells that follow characterise name-to-ID consistency, separator conventions and token frequencies (Cell 19), and then estimate the auxiliary-label coverage of the retained circRNAs (Cell 20).

### 19. Name Consistency, Separator Conventions and Token Counts

**Objective.** Test whether a circRNA name maps to a single circBase ID, check how multiple values are separated inside a cell, and count miRNA and RBP tokens.

In [40]:
# Cell 19 - Name-level consistency, separator conventions and token counts
# ---- A. Is the mapping name -> circBase ID consistent? (relevant for joining the curated file) -------
k = final[final["keep"]].copy()
k["name_key"] = k["circ_raw"].str.casefold()
n_ids = k.groupby("name_key")["circbase_final"].nunique()
print("A1. kept circRNA names:", len(n_ids), "| names resolving to >1 ID:", int((n_ids > 1).sum()))
bad = n_ids[n_ids > 1].index
print(k[k["name_key"].isin(bad)][["circ_raw", "circbase_final", "match_tier", "Region"]]
      .sort_values("circ_raw").head(10).to_string())

# Names that were kept in some rows but dropped in others (candidates for ID propagation)
allr = final.assign(name_key=final["circ_raw"].str.casefold())
g = allr.groupby("name_key")["keep"].agg(kept="sum", rows="size")
mixed = g[(g["kept"] > 0) & (g["kept"] < g["rows"])]
print("\nA2. names kept in some rows but dropped in others:", len(mixed),
      "| dropped rows affected:", int((mixed["rows"] - mixed["kept"]).sum()))
print(mixed.head(8))

# ---- B. How are multiple values written inside a single cell? -----------------------------------------------
def show(col, pat, n=8):
    """Print distinct cells of `col` that match the regular expression `pat`."""
    s = cur[col].dropna().astype(str)
    hit = s[s.str.contains(pat, regex=True)].drop_duplicates()
    print(f"{col} ~ {pat!r}: {len(hit)} distinct cells;", hit.head(n).tolist())

print("\nB. separators")
show("miRNA validated", r"/")
show("miRNA validated", r"\band\b")
show("miRNA validated", r",.*,")
show("RBP validated", r"[;/\n]")

# ---- C. Token-level statistics ----------------------------------------------------------------------------------
def tokens(s):
    """Split a cell on commas, semicolons, newlines and the word 'and'."""
    if not isinstance(s, str):
        return []
    return [p.strip() for p in re.split(r"[,;\n]|\s+and\s+", s) if p.strip()]

rbp = cur["RBP validated"].map(tokens).explode().dropna()
rbp_n = rbp.str.upper().str.replace(r"\s+", "", regex=True)
print("\nC1. RBP tokens:", len(rbp_n), "| distinct:", rbp_n.nunique(),
      "| non-empty RBP cells:", int(cur["RBP validated"].notna().sum()))
print(rbp_n.value_counts().head(15).to_string())
# Rows whose RBP evidence consists solely of AGO2 and/or the IgG control antibody
only = cur["RBP validated"].map(lambda s: bool(tokens(s)) and
        {t.upper().replace(" ", "") for t in tokens(s)} <= {"AGO2", "IGG"})
print("rows whose RBP cell holds only AGO2 and/or IgG:", int(only.sum()))

mir = cur["miRNA validated"].map(tokens).explode().dropna()
print("\nC2. miRNA tokens:", len(mir), "| distinct as written:", mir.nunique())
print("with species prefix:", round(mir.str.match(r"(?i)^(hsa|mmu|rno)-").mean(), 3),
      "| with -3p/-5p arm:", round(mir.str.contains(r"(?i)-(?:3p|5p)$").mean(), 3))
odd = mir[~mir.str.match(r"(?i)^((hsa|mmu|rno)-)?(mir|let)-")]
print("tokens not shaped like miR-/let-:", len(odd), odd.drop_duplicates().head(12).tolist())

A1. kept circRNA names: 1791 | names resolving to >1 ID: 1
       circ_raw    circbase_final match_tier                Region
3997  circGLIS3  hsa_circ_0004221   tier1_id  chr9:3879426-3937189
4075  circGLIS3  hsa_circ_0002874   tier1_id  chr9:4286037-4286523

A2. names kept in some rows but dropped in others: 46 | dropped rows affected: 110
              kept  rows
name_key                
circ-foxm1       1     2
circ-prkci       1     3
circ-prmt5       1     3
circ-smad7       1     6
circ-znf609      1     4
circ_0000003     1     2
circ_0000502     1     2
circ_0003998     1     2

B. separators
miRNA validated ~ '/': 5 distinct cells; ['miR-127-5p/miR-612/miR-4688', 'miR-203/miR-502-5p', 'hsa-miR-337-3p/miR-337-3p', 'miR-375/1231', 'miR-375/409-3P/515-5p']
miRNA validated ~ '\\band\\b': 5 distinct cells; ['miR-339-5p, miR-96-3p and miR-135b-3p', 'miR-34, miR-34a, miR-34b, and miR-34c', 'hsa-miR-34a-5p, has-miR-221-3p, has-miR-222-3p, has-miR-181-5p, has-miR-449a, hsa-miR-125a-5p

**Interpretation.** A single name can denote two different circRNAs (`circGLIS3` → `hsa_circ_0004221` and `hsa_circ_0002874`, about 400 kb apart), so a name alone is **not** a safe join key; only the (name, disease) pair is acceptable. The miRNA column is heterogeneous (typographical prefix `has-`, Unicode hyphens, forms such as `miR449a`, and shorthand after "/" as in `miR-375/1231`), but only about 5% of tokens are non-standard. The RBP column is dominated by AGO2 (171 tokens) and IgG (63 tokens; a negative-control antibody, not an RBP); 56% of non-empty cells contain only AGO2 and/or IgG.

**Decision.** A systematic normaliser for miRNA and RBP tokens is required (Cell 22). First, provisional coverage is sized with a simple normalisation (Cell 20) to establish whether auxiliary supervision can reach the cold-start circRNAs at all.

### 20. Provisional Estimate of Auxiliary-Label Coverage

**Objective.** Estimate how many model-input circRNAs have auxiliary labels, using a simple provisional normalisation (superseded by Cell 22).

In [45]:
# Cell 20 - Provisional estimate of auxiliary-label coverage (superseded by the normalisation of Cell 22)
def norm_mir(t):
    """Provisional miRNA normalisation: strip the species prefix and unify the 'miR-' spelling."""
    t = re.sub(r"(?i)^(hsa|mmu|rno)-", "", t.strip())
    return re.sub(r"(?i)^mir-", "miR-", t)

# Restrict to human curated rows and map diseases onto the official list
cur_h = cur[cur["Species"].str.strip().str.lower().eq("human")].copy()
cur_h["circ_raw"] = cur_h["circRNA Name"].map(clean_txt)
cur_h["disease_std"] = cur_h["Disease Name"].map(clean_txt).str.casefold().map(dis_std)
cur_h = cur_h.dropna(subset=["disease_std"])

# Join on the (name, disease) pair, which was shown to be the only safe key
keyed = (final[final["keep"]][["circ_raw", "disease_std", "circbase_final"]].drop_duplicates())
m = cur_h.merge(keyed, on=["circ_raw", "disease_std"], how="inner")
print("human curated rows:", len(cur_h), "| joined to the kept set:", len(m),
      "| circRNAs:", m["circbase_final"].nunique())

# Long format: one row per (circRNA, miRNA) edge
mir_e = (m.assign(mirna=m["miRNA validated"].map(tokens)).explode("mirna").dropna(subset=["mirna"]))
mir_e["mirna"] = mir_e["mirna"].map(norm_mir)
mir_e = mir_e[["circbase_final", "mirna"]].drop_duplicates()

# Long format for RBP edges; the exact token 'IGG' is removed (negative-control antibody)
rbp_e = (m.assign(rbp=m["RBP validated"].map(tokens)).explode("rbp").dropna(subset=["rbp"]))
rbp_e["rbp"] = rbp_e["rbp"].str.upper().str.replace(r"\s+", "", regex=True)
rbp_e = rbp_e[rbp_e["rbp"] != "IGG"][["circbase_final", "rbp"]].drop_duplicates()
rbp_x = rbp_e[rbp_e["rbp"] != "AGO2"]               # RBP edges other than AGO2 (AGO2 is miRNA-related evidence)

# Model-input circRNAs (<= 5,000 nt) and the cold-start subgroup (a single disease)
pp5 = pp[pp["seq_len"].between(1, 5000)]
ids = set(pp5["circbase_final"])
deg = pp5.groupby("circbase_final").size()
deg1 = set(deg[deg == 1].index)
S = lambda d: set(d["circbase_final"]) & ids

print(f"\ncircRNAs in the <=5000 nt set: {len(ids)} (degree-1: {len(deg1)})")
for name, d in [("validated miRNA", mir_e), ("any RBP (IgG removed)", rbp_e), ("RBP other than AGO2", rbp_x)]:
    s = S(d)
    print(f"{name:24s}: {len(s):5d} circRNAs ({len(s)/len(ids):.1%}) | "
          f"among degree-1: {len(s & deg1):5d} ({len(s & deg1)/len(deg1):.1%})")
print("miRNA and non-AGO2 RBP both:", len(S(mir_e) & S(rbp_x)))

# Do miRNAs connect several circRNAs? Single-use miRNAs cannot link circRNAs in a graph.
mk = mir_e[mir_e["circbase_final"].isin(ids)]
share = mk.groupby("mirna")["circbase_final"].nunique()
print("\nmiRNA edges:", len(mk), "| distinct miRNAs:", len(share),
      "| miRNAs linked to >=2 circRNAs:", int((share >= 2).sum()))
rk = rbp_x[rbp_x["circbase_final"].isin(ids)]
print("non-AGO2 RBP edges:", len(rk), "| distinct RBPs:", rk["rbp"].nunique())
print(rk["rbp"].value_counts().head(10).to_string())

human curated rows: 2028 | joined to the kept set: 1321 | circRNAs: 857

circRNAs in the <=5000 nt set: 1582 (degree-1: 1267)
validated miRNA         :   697 circRNAs (44.1%) | among degree-1:   458 (36.1%)
any RBP (IgG removed)   :   179 circRNAs (11.3%) | among degree-1:   105 (8.3%)
RBP other than AGO2     :    73 circRNAs (4.6%) | among degree-1:    43 (3.4%)
miRNA and non-AGO2 RBP both: 49

miRNA edges: 1081 | distinct miRNAs: 587 | miRNAs linked to >=2 circRNAs: 222
non-AGO2 RBP edges: 124 | distinct RBPs: 94
rbp
ANTI-AGO2                5
ANTI-IGG                 5
CYP51A1                  4
FKBP5                    4
HUR                      3
EZH2                     3
AGO3                     3
ANTI-ARGONAUTE2(AGO2)    3
CDK6                     2
CYCLIND1                 2


**Interpretation.** Of 2,028 human curated rows, 1,321 join the kept set (65.1%, close to the 66.1% of the main table: 2,547 of 3,854 human rows). Among the 1,582 model-input circRNAs, 44.1% have a validated miRNA edge (36.1% of cold-start circRNAs), whereas only 11.3% have any RBP edge and 4.6% a non-AGO2 RBP edge. These RBP figures are **overstated**, because variants such as `ANTI-IGG` and `ANTI-AGO2` were not merged; likewise, the distinct-miRNA count (587) is inflated by spelling variants.

**Decision.** A stricter normalisation of miRNA and RBP names is implemented (Cell 22). Before that, the possibility of recovering dropped rows via within-paper name propagation and via the implied "no-prefix" rule is tested (Cell 21).

### 21. Within-Paper Name Propagation and the No-Prefix Hypothesis

**Objective.** Test whether unresolved rows can be recovered by propagating an ID within the same paper, and whether a name of the form `circ_<7 digits>` implies `hsa_circ_<7 digits>` (the no-prefix hypothesis).

In [46]:
# Cell 21 - Within-paper name propagation and the "no-prefix" hypothesis
final["name_key"] = final["circ_raw"].str.casefold()
core = final[final["keep"]]

# ---- A. circGLIS3: the same name denotes two different circRNAs --------------------------------
g = final[final["name_key"] == "circglis3"]
print("A. circGLIS3 rows:")
print(g[["circ_raw", "disease_std", "PubMed ID", "circbase_final", "Region", "flag"]].to_string(), "\n")

# ---- B. Propagate an ID to unresolved rows only when name AND paper (PubMed ID) coincide --------
lookup = (core.groupby(["name_key", "PubMed ID"])["circbase_final"]
          .agg(lambda s: sorted(set(s))).to_dict())
dropped = final[~final["keep"] & final["flag"].eq("unresolved")].copy()
dropped["cand"] = [lookup.get((n, p), []) for n, p in zip(dropped["name_key"], dropped["PubMed ID"])]
dropped["n_cand"] = dropped["cand"].map(len)
print("B. unresolved rows:", len(dropped))
print(dropped["n_cand"].value_counts().sort_index())

prop = dropped[dropped["n_cand"] == 1].copy()
prop["cand_id"] = prop["cand"].str[0]
have = set(map(tuple, core[["circbase_final", "disease_std"]].drop_duplicates().values))
new_pairs = {(i, d) for i, d in zip(prop["cand_id"], prop["disease_std"])} - have
print("within-paper candidates:", len(prop), "| pairs not already kept:", len(new_pairs))
print(prop[["circ_raw", "disease_std", "PubMed ID", "cand_id"]].head(8).to_string(), "\n")

# ---- C. No-prefix hypothesis: does 'circ_0001946' correspond to 'hsa_circ_0001946'? -------------------
shape = r"(?i)^circ[_-]?\d{7}$"
np_k = final[final["circ_raw"].str.match(shape) & final["keep"]].copy()
np_k["implied"] = "hsa_circ_" + np_k["circ_raw"].str.extract(r"(\d{7})")[0]
print("C. no-prefix names already resolved:", len(np_k), "\n   by tier:")
print(np_k["match_tier"].value_counts())
print("resolved ID equals the implied hsa_circ_ ID:",
      round((np_k["circbase_final"] == np_k["implied"]).mean(), 3))
print(np_k[["circ_raw", "implied", "circbase_final", "Region", "match_tier"]].head(8).to_string())
print("unresolved rows with this name shape:",
      int((final["circ_raw"].str.match(shape) & ~final["keep"]).sum()))

A. circGLIS3 rows:
       circ_raw     disease_std   PubMed ID    circbase_final                Region flag
3997  circGLIS3     Lung cancer  33520363.0  hsa_circ_0004221  chr9:3879426-3937189   ok
4075  circGLIS3  Bladder cancer  33656636.0  hsa_circ_0002874  chr9:4286037-4286523   ok 

B. unresolved rows: 1299
n_cand
0    1298
1       1
Name: count, dtype: int64
within-paper candidates: 1 | pairs not already kept: 0
        circ_raw                         disease_std   PubMed ID           cand_id
3452  circ-SMAD7  Esophageal squamous cell carcinoma  30611100.0  hsa_circ_0000848 

C. no-prefix names already resolved: 64 
   by tier:
match_tier
tier1_id        44
tier2_coords    20
Name: count, dtype: int64
resolved ID equals the implied hsa_circ_ ID: 0.969
          circ_raw           implied    circbase_final                     Region    match_tier
1256  circ_0030235  hsa_circ_0030235  hsa_circ_0030235    chr13:49075877-49077050  tier2_coords
1283  circ_0008450  hsa_circ_0008450  hs

**Interpretation.** (A) `circGLIS3` appears in two papers with regions about 400 kb apart; names are only valid *within* a paper. (B) Only one of 1,299 unresolved rows shares both name and paper with a kept row, and that pair was already present: **propagation yields nothing** and is dropped. (C) Among 63 already-resolved no-prefix names, 61 (96.8%) equal `hsa_circ_<digits>`, but these were resolved by independent evidence; the rate on unverified rows may be lower.

**Decision.** The no-prefix rule is not accepted into the primary set; its two disagreements and its verifiability on the unresolved rows are examined in Cell 23. No sensitivity table is built in this notebook. A rigorous normalisation of miRNA and RBP tokens follows (Cell 22).

### 22. Rigorous Normalisation of the Auxiliary Labels

**Objective.** Normalise miRNA and RBP tokens rigorously (hyphen variants, species prefixes, arms, anti-/IgG controls), confirm that the join key is safe, and recount the auxiliary-label coverage.

In [47]:
# Cell 22 - Rigorous normalisation of auxiliary labels and recount of coverage
# Map all Unicode hyphen/minus variants to the ASCII hyphen, so identical miRNAs are not counted twice
HYPH = dict.fromkeys(map(ord, "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"), "-")

def split_mir(s):
    """Split a miRNA cell on commas, semicolons, newlines, 'and' and '/' (shorthand lists)."""
    if not isinstance(s, str):
        return []
    s = s.translate(HYPH)
    return [p.strip() for p in re.split(r"[,;\n]|\s+and\s+|/", s) if p.strip()]

def norm_mirna(t):
    """Return (arm-stripped canonical miRNA name, arm) after repairing prefixes and spelling variants."""
    t = t.translate(HYPH).strip()
    t = re.sub(r"(?i)^(hsa|has|mmu|rno)[-_]", "", t)             # species prefix, incl. the typo 'has-'
    t = re.sub(r"(?i)^(microrna|mirna|mir)[-_ ]?", "miR-", t)    # unify 'microRNA-', 'miRNA-', 'miR'
    t = re.sub(r"^(\d)", r"miR-\1", t)                           # shorthand after '/', e.g. '1231'
    t = re.sub(r"(?i)^let[-_ ]?", "let-", t)
    arm = re.search(r"(?i)-(3p|5p)$", t)
    return re.sub(r"(?i)-(3p|5p)$", "", t), (arm.group(1).lower() if arm else None)

# Valid canonical shapes: miR-<number><letters>[-<number>] or let-<number><letters>
VALID = r"^(miR-\d+[A-Za-z]*(-\d+)?|let-\d+[A-Za-z]*)$"

def norm_rbp(t):
    """Canonicalise RBP tokens: drop IgG controls, merge anti-AGO2/Argonaute2 variants into 'AGO2'."""
    u = re.sub(r"\s+", "", t.translate(HYPH)).upper().strip("()")
    u = re.sub(r"^ANTI-?", "", u)
    if "IGG" in u:
        return None                                   # negative-control antibody, not a binding protein
    return "AGO2" if ("AGO2" in u or "ARGONAUTE2" in u) else u

# ---- join-key check: (name, disease) must map to a single circBase ID --------------------------------------
dupkey = keyed.duplicated(["circ_raw", "disease_std"], keep=False)
print("(name, disease) keys that map to >1 circBase ID:", int(dupkey.sum()))
print(keyed[dupkey].head(6).to_string(), "\n")

# ---- miRNA ----------------------------------------------------------------------------------------------------
mir = m.assign(tok=m["miRNA validated"].map(split_mir)).explode("tok").dropna(subset=["tok"]).copy()
pr = mir["tok"].map(norm_mirna)
mir["mirna"] = pr.map(lambda t: t[0])
mir["arm"]   = pr.map(lambda t: t[1])
ok = mir["mirna"].str.match(VALID)                    # tokens failing the shape test are reported, not kept
print("miRNA tokens:", len(mir), "| not matching the miR-/let- shape:", int((~ok).sum()))
print(mir.loc[~ok, "tok"].drop_duplicates().head(15).tolist())
mir_e = mir[ok][["circbase_final", "mirna"]].drop_duplicates()

mk = mir_e[mir_e["circbase_final"].isin(ids)]
cnt = mk.groupby("mirna")["circbase_final"].nunique()
shared = set(cnt[cnt >= 2].index)                     # miRNAs that can connect several circRNAs
conn = set(mk.loc[mk["mirna"].isin(shared), "circbase_final"])
cov = set(mk["circbase_final"])
print("\nmiRNA edges:", len(mk), "| distinct miRNAs (arm-stripped):", len(cnt),
      "| linked to >=2 circRNAs:", len(shared))
print("circRNAs with a miRNA edge:", len(cov), f"({len(cov)/len(ids):.1%})",
      "| degree-1:", len(cov & deg1), f"({len(cov & deg1)/len(deg1):.1%})")
print("circRNAs sharing a miRNA with another circRNA:", len(conn),
      "| degree-1:", len(conn & deg1))
print("most shared miRNAs:\n", cnt.sort_values(ascending=False).head(8).to_string())

# ---- RBP ----------------------------------------------------------------------------------------------------------
rb = m.assign(tok=m["RBP validated"].map(tokens)).explode("tok").dropna(subset=["tok"]).copy()
rb["rbp"] = rb["tok"].map(norm_rbp)
rb = rb.dropna(subset=["rbp"])
rb_e = rb[["circbase_final", "rbp"]].drop_duplicates()
rb_e = rb_e[rb_e["circbase_final"].isin(ids)]
cr = rb_e.groupby("rbp")["circbase_final"].nunique().sort_values(ascending=False)
non = rb_e[rb_e["rbp"] != "AGO2"]
print("\nRBP edges (IgG removed):", len(rb_e), "| distinct:", len(cr), "| seen with >=2 circRNAs:", int((cr >= 2).sum()))
print(cr.head(15).to_string())
print("non-AGO2: edges", len(non), "| circRNAs", non["circbase_final"].nunique(),
      "| distinct RBPs", non["rbp"].nunique())
print("rbp names that are not RBP-shaped (contain digits/dashes), first 15:",
      sorted({r for r in non["rbp"] if re.search(r"[0-9\-]", r)})[:15])

(name, disease) keys that map to >1 circBase ID: 0
Empty DataFrame
Columns: [circ_raw, disease_std, circbase_final]
Index: [] 

miRNA tokens: 1226 | not matching the miR-/let- shape: 3
['miR-101-3p.1', 'miR-203a-3p.1', 'miR-NC']

miRNA edges: 1055 | distinct miRNAs (arm-stripped): 406 | linked to >=2 circRNAs: 217
circRNAs with a miRNA edge: 695 (43.9%) | degree-1: 457 (36.1%)
circRNAs sharing a miRNA with another circRNA: 593 | degree-1: 378
most shared miRNAs:
 mirna
miR-145     24
miR-149     17
miR-338     17
miR-326     17
miR-130a    13
miR-182     13
miR-370     12
miR-330     11

RBP edges (IgG removed): 230 | distinct: 83 | seen with >=2 circRNAs: 14
rbp
AGO2        128
FKBP5         4
CYP51A1       4
HUR           3
EZH2          3
AGO3          3
FUS           2
AUF1          2
CYCLIND1      2
CDK6          2
QKI           2
MMP-2         2
AGO1          2
STAT3         2
AGO           1
non-AGO2: edges 102 | circRNAs 61 | distinct RBPs 82
rbp names that are not RBP-shaped (

**Interpretation.** The join key is safe: no (name, disease) key maps to more than one circBase ID. After normalisation, only 3 of 1,226 miRNA tokens fail the canonical shape (e.g. `miR-NC`, a negative-control mimic). The miRNA edges number 1,055 over 406 distinct (arm-stripped) miRNAs (previously 587); 217 miRNAs link two or more circRNAs. Coverage is 43.9% of circRNAs overall and 36.1% of cold-start circRNAs, but only 29.8% of the latter share a miRNA with another circRNA. For RBPs, AGO2 accounts for 128 of 230 edges; many remaining names are downstream pathway proteins (CASPASE-3, MMP-2, STAT3, …) rather than RNA-binding proteins, and only 14 names occur with two or more circRNAs.

**Decision.** (i) miRNA labels from the curated file are adopted as usable; (ii) the RBP column is **not** a usable label source, so ENCORI CLIP-seq data must be used (Cells 27–28); (iii) the unverified implied-prefix rows are examined further (Cell 23), and the question of whether miRNA sharing carries disease information is tested (Cell 24).

### 23. Test of the No-Prefix Rule on Unresolved Rows

**Objective.** Test the no-prefix rule on the unresolved rows: where it disagrees with resolved rows, whether the implied ID can be checked, and how many pairs it would add.

In [48]:
# Cell 23 - Test the "no-prefix" rule on the unresolved rows
shape = r"(?i)^circ[_-]?\d{7}$"
fa_c = fa.set_index("circbase_id")

def coord_check(region, cid):
    """True/False if the entry coordinates agree/disagree with the FASTA entry of `cid`; None if untestable."""
    c, s, e, _ = parse_region3(region)
    if c is None or cid not in fa_c.index:
        return None
    r = fa_c.loc[cid]
    return bool(r["chrom"] == c and abs(r["start"] - s) <= 1 and abs(r["end"] - e) <= 1)

npn = final[final["circ_raw"].str.match(shape)].copy()
npn["implied"] = "hsa_circ_" + npn["circ_raw"].str.extract(r"(\d{7})")[0]

# ---- A. The two disagreements among resolved rows -----------------------------------------------------
res_np = npn[npn["keep"]]
bad = res_np[res_np["circbase_final"] != res_np["implied"]]
print("A. resolved no-prefix rows where the ID differs from the name:")
print(bad[["circ_raw", "implied", "circbase_final", "match_tier", "Region", "Gene Symbol"]].to_string(), "\n")

# ---- B. FASTA ID density: is 'exists in FASTA' informative? ----------------------------------------------
num = pd.to_numeric(fa_index["circbase_id"].str.extract(r"hsa_circ_(\d+)")[0],
                    errors="coerce").dropna().astype(int)
print("B. FASTA numeric IDs:", num.min(), "-", num.max(),
      "| density:", round(len(num) / (num.max() - num.min() + 1), 3), "\n")

# ---- C. The unresolved rows with this name shape ------------------------------------------------------------
un = npn[~npn["keep"]].copy()
un["in_fasta"] = un["implied"].isin(fa_ids)
un["coord"] = [coord_check(r, i) for r, i in zip(un["Region"], un["implied"])]
un["gene"] = [gene_status(g, fa_gene.get(i), i) for g, i in zip(un["Gene Symbol"], un["implied"])]
print("C. unresolved no-prefix rows:", len(un))
print("implied ID in FASTA:\n", un["in_fasta"].value_counts())
print("\ncoordinates vs implied ID:\n", un["coord"].value_counts(dropna=False))
print("\ngene vs implied ID:\n", un["gene"].value_counts())
print("\nwith a full Region:", int(un["Region"].map(lambda r: parse_region3(r)[0] is not None).sum()))
print("new circRNA-disease pairs if all in-FASTA IDs were accepted:",
      len(set(zip(un.loc[un["in_fasta"], "implied"], un.loc[un["in_fasta"], "disease_std"]))
          - set(map(tuple, core[["circbase_final", "disease_std"]].values))))

A. resolved no-prefix rows where the ID differs from the name:
          circ_raw           implied    circbase_final    match_tier                   Region Gene Symbol
1788  circ_0095812  hsa_circ_0095812  hsa_circ_0000253  tier2_coords  chr10:97999787-97999925        BLNK
3236  circ_0007841  hsa_circ_0007841  hsa_circ_0007842  tier2_coords   chr7:73654267-73661093        RFC2 

B. FASTA numeric IDs: 1 - 140790 | density: 1.0 

C. unresolved no-prefix rows: 132
implied ID in FASTA:
 in_fasta
True    132
Name: count, dtype: int64

coordinates vs implied ID:
 coord
None     131
False      1
Name: count, dtype: int64

gene vs implied ID:
 gene
n/a      119
agree     13
Name: count, dtype: int64

with a full Region: 1
new circRNA-disease pairs if all in-FASTA IDs were accepted: 87


**Interpretation.** The FASTA identifier density equals exactly 1.0, so "the implied ID exists" holds trivially for all 133 unresolved rows and constitutes no evidence. The two resolved disagreements (`circ_0095812`, `circ_0007841`) show that coordinates can contradict a name; if circBase IDs follow genomic order (inference), a one-digit typographical error can land on a neighbouring circRNA of the same gene, which a gene check cannot detect. Only two of the 133 rows have a full `Region`, so the rule is largely unverifiable; accepting all of them would add 87 pairs (+3.9%) with a non-negligible error risk.

**Decision.** The implied-prefix rows are excluded from the primary set; no sensitivity table is built in this notebook. Meanwhile, the scientific premise of the graph-input arm is tested: do circRNAs that share a miRNA also share diseases (Cell 24)?

### 24. miRNA Sharing and Disease Sharing

**Objective.** Test the premise of using miRNA edges as graph input: do circRNAs from different papers that share a miRNA also share a disease more often than under a shuffled-disease null?

In [49]:
# Cell 24 - Does sharing a miRNA predict sharing a disease across different papers?
from itertools import combinations

# Disease set and paper set of each circRNA (model-input circRNAs only)
dis_of    = pp5.groupby("circbase_final")["disease_std"].agg(set).to_dict()
papers_of = (final[final["keep"]].groupby("circbase_final")["PubMed ID"]
             .agg(lambda s: set(s.dropna())).to_dict())

# All circRNA pairs that share at least one miRNA
pairs = set()
for members in mk.groupby("mirna")["circbase_final"].agg(lambda s: sorted(set(s))):
    pairs.update(combinations(members, 2))

df = pd.DataFrame(list(pairs), columns=["a", "b"])
df = df[df["a"].isin(dis_of) & df["b"].isin(dis_of)].copy()
df["share_disease"] = [bool(dis_of[a] & dis_of[b]) for a, b in zip(df["a"], df["b"])]
# Same-paper pairs trivially share both disease and miRNA, hence they must be analysed separately
df["share_paper"]   = [bool(papers_of.get(a, set()) & papers_of.get(b, set())) for a, b in zip(df["a"], df["b"])]

print("circRNA pairs sharing >=1 miRNA:", len(df))
print(df.groupby("share_paper")["share_disease"].agg(pairs="size", frac_share_disease="mean").round(3), "\n")

def homophily(pr, n_perm=500, seed=0):
    """Observed share of disease-sharing pairs vs a null obtained by shuffling disease sets among circRNAs."""
    names = sorted(set(pr["a"]) | set(pr["b"]))
    pos = {c: i for i, c in enumerate(names)}
    sets = [dis_of[c] for c in names]
    ai, bi = pr["a"].map(pos).to_numpy(), pr["b"].map(pos).to_numpy()
    def hit(p):
        return np.mean([bool(sets[p[i]] & sets[p[j]]) for i, j in zip(ai, bi)])
    obs = hit(np.arange(len(names)))
    rng = np.random.default_rng(seed)
    null = np.array([hit(rng.permutation(len(names))) for _ in range(n_perm)])
    return obs, null.mean(), (np.sum(null >= obs) + 1) / (n_perm + 1), len(pr), len(names)

cross = df[~df["share_paper"]]
low = cross[cross["a"].map(lambda c: len(dis_of[c]) <= 2) & cross["b"].map(lambda c: len(dis_of[c]) <= 2)]
print("different-paper pairs, observed vs shuffled-disease null:")
for name, pr in [("all different-paper pairs", cross), ("both circRNAs with <=2 diseases", low)]:
    if len(pr) == 0:
        print(name, ": no pairs"); continue
    obs, nm, p, npairs, nc = homophily(pr)
    print(f"{name}: pairs={npairs}, circRNAs={nc} | share a disease: observed={obs:.3f}, null={nm:.3f}, perm p={p:.3f}")

circRNA pairs sharing >=1 miRNA: 2250
             pairs  frac_share_disease
share_paper                           
False         2208               0.189
True            42               1.000 

different-paper pairs, observed vs shuffled-disease null:
all different-paper pairs: pairs=2208, circRNAs=591 | share a disease: observed=0.189, null=0.089, perm p=0.002
both circRNAs with <=2 diseases: pairs=1163, circRNAs=469 | share a disease: observed=0.074, null=0.049, perm p=0.002


**Interpretation.** Among circRNA pairs from different papers that share a miRNA, 18.9% share a disease, compared with 8.9% under a shuffled-disease null (lift 2.1×; 1.5× in the ≤2-disease subset). The 42 same-paper pairs share a disease in 100% of cases, which confirms that they had to be excluded. The null is **not degree-controlled**: well-studied circRNAs appear in many pairs and share diseases more often by chance, so the lift is probably inflated. The pairs are also not independent, so the permutation *p*-values (floor 0.002 with 500 permutations) are descriptive.

**Decision.** The test is repeated with permutations restricted within degree bins and with hub miRNAs removed (Cell 25).

### 25. Degree-Controlled Test of miRNA Homophily

**Objective.** Repeat the homophily test with permutations restricted within degree bins and with hub miRNAs removed, to separate the miRNA signal from popularity.

In [50]:
# Cell 25 - Degree-controlled test of miRNA homophily
def build_pairs(max_circ=None):
    """Different-paper circRNA pairs sharing a miRNA; optionally exclude hub miRNAs linked to > max_circ circRNAs."""
    cnt = mk.groupby("mirna")["circbase_final"].nunique()
    keep_m = set(cnt.index) if max_circ is None else set(cnt[cnt <= max_circ].index)
    prs = set()
    grp = mk[mk["mirna"].isin(keep_m)].groupby("mirna")["circbase_final"].agg(lambda s: sorted(set(s)))
    for members in grp:
        prs.update(combinations(members, 2))
    d = pd.DataFrame(list(prs), columns=["a", "b"])
    d = d[d["a"].isin(dis_of) & d["b"].isin(dis_of)].copy()
    d["share_paper"] = [bool(papers_of.get(a, set()) & papers_of.get(b, set())) for a, b in zip(d["a"], d["b"])]
    return d[~d["share_paper"]]

def strat_homophily(pr, n_perm=1000, seed=0):
    """Permutation test in which disease sets are exchanged only among circRNAs of equal degree bin (1, 2, 3+)."""
    names = sorted(set(pr["a"]) | set(pr["b"]))
    pos = {c: i for i, c in enumerate(names)}
    sets = [dis_of[c] for c in names]
    bins = np.array([min(len(s), 3) for s in sets])
    ai, bi = pr["a"].map(pos).to_numpy(), pr["b"].map(pos).to_numpy()
    idx_by_bin = {b: np.where(bins == b)[0] for b in np.unique(bins)}

    def hit(order):
        return np.mean([bool(sets[order[i]] & sets[order[j]]) for i, j in zip(ai, bi)])

    obs = hit(np.arange(len(names)))
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        order = np.arange(len(names))
        for idx in idx_by_bin.values():            # shuffle within each degree bin only
            order[idx] = rng.permutation(idx)
        null.append(hit(order))
    null = np.array(null)
    return obs, null.mean(), (np.sum(null >= obs) + 1) / (n_perm + 1), len(pr)

rows = []
for mlabel, mx in [("all miRNAs", None), ("miRNAs on <=10 circRNAs", 10)]:
    pr_all = build_pairs(mx)
    nd = lambda c: len(dis_of[c])
    subsets = [("all different-paper pairs", pr_all),
               ("both circRNAs <=2 diseases", pr_all[pr_all["a"].map(nd).le(2) & pr_all["b"].map(nd).le(2)]),
               ("both circRNAs exactly 1 disease", pr_all[pr_all["a"].map(nd).eq(1) & pr_all["b"].map(nd).eq(1)])]
    for slabel, pr in subsets:
        if len(pr) < 50:                          # too few pairs for a stable estimate
            rows.append((mlabel, slabel, len(pr), np.nan, np.nan, np.nan, np.nan)); continue
        obs, nm, p, n = strat_homophily(pr)
        rows.append((mlabel, slabel, n, round(obs, 3), round(nm, 3), round(obs / nm, 2), round(p, 3)))

out = pd.DataFrame(rows, columns=["miRNA set", "pair set", "pairs", "observed", "null (degree-matched)", "lift", "perm p"])
print(out.to_string(index=False))
print("\n(p-value floor with 1000 permutations = 0.001)")

              miRNA set                        pair set  pairs  observed  null (degree-matched)  lift  perm p
             all miRNAs       all different-paper pairs   2208     0.189                  0.136  1.39   0.001
             all miRNAs      both circRNAs <=2 diseases   1163     0.074                  0.054  1.38   0.005
             all miRNAs both circRNAs exactly 1 disease    554     0.043                  0.033  1.30   0.116
miRNAs on <=10 circRNAs       all different-paper pairs   1285     0.184                  0.143  1.29   0.001
miRNAs on <=10 circRNAs      both circRNAs <=2 diseases    648     0.076                  0.054  1.40   0.018
miRNAs on <=10 circRNAs both circRNAs exactly 1 disease    304     0.053                  0.035  1.50   0.073

(p-value floor with 1000 permutations = 0.001)


**Interpretation.** With degree matching, the lift falls from 2.1× (Cell 24) to **1.3–1.5×**, which shows that popularity had inflated the earlier estimate. For all different-paper pairs the permutation *p* is 0.001 (the floor with 1,000 permutations). For pairs in which both circRNAs have at most two diseases it is 0.005 (all miRNAs) and 0.018 (miRNAs on at most 10 circRNAs), so the signal survives removal of hub miRNAs. In the cold-start population (both circRNAs with exactly one disease) the observed rate exceeds the null by only 1.0 and 1.8 percentage points (0.043 versus 0.033 and 0.053 versus 0.035; lifts 1.30 and 1.50) and is not distinguishable from it at the 0.05 level (*p* = 0.116 and 0.073). The six *p*-values are not corrected for multiple testing and the pairs are not independent, so they are descriptive. *Inference:* shared biology cannot be separated from shared research topic, because most entries are cancers.

**Decision.** A modest effect of miRNA information is anticipated; statistical power will be the principal risk, hence the evaluation will use repeated grouped splits, paired comparisons and bootstrap intervals. The curated miRNA edges (same papers as the disease labels) are regarded as the weaker auxiliary source, and ENCORI is planned as the primary one. The frozen tables are saved next (Cell 26).

### 26. Frozen Tables for the CDA Task and the Curated miRNA Edges

**Objective.** Save the frozen tables for the CDA task: primary pairs, curated miRNA edges and a circRNA summary table (length band, degree, miRNA coverage).

In [51]:
# Cell 26 - Save the frozen tables (primary pairs, miRNA edges and circRNA summary)
L = fa_index.set_index("circbase_id")["seq_len"]

# ---- primary pair table (rebuilt from `final`, so it does not depend on stale objects) ---------------
prim = (final.loc[final["keep"], ["circbase_final", "disease_std"]]
        .drop_duplicates().rename(columns={"circbase_final": "circrna_id", "disease_std": "disease"}))
prim["seq_len"] = prim["circrna_id"].map(L)
prim.to_csv(PROCESSED_DIR / "cda_pairs_primary.csv", index=False)

# ---- circRNA table: length policy, degree and miRNA coverage -------------------------------------------
mir_ok = mir[mir["mirna"].str.match(VALID)]
mir_edges = (mir_ok[["circbase_final", "mirna", "arm"]].drop_duplicates()
             .rename(columns={"circbase_final": "circrna_id"}))
mir_edges.to_csv(PROCESSED_DIR / "mirna_edges_curated.csv", index=False)

circ = prim.groupby("circrna_id").agg(n_diseases=("disease", "size"), seq_len=("seq_len", "first")).reset_index()
circ["length_band"] = np.select([circ["seq_len"].between(1, 1022), circ["seq_len"].between(1023, 5000)],
                                ["<=1022", "1023-5000"], default=">5000")
circ["use_primary"] = circ["length_band"] != ">5000"         # circRNAs above 5,000 nt are excluded from model input
circ["has_mirna_edge"] = circ["circrna_id"].isin(set(mir_edges["circrna_id"]))
circ.to_csv(PROCESSED_DIR / "circrna_table.csv", index=False)

# ---- sanity prints -------------------------------------------------------------------------------------------
print("pairs:", len(prim), "| circRNAs:", len(circ), "| diseases:", prim["disease"].nunique())
print(circ["length_band"].value_counts().to_string())
print("model-input circRNAs (<=5000 nt):", int(circ["use_primary"].sum()))
print("miRNA edges:", len(mir_edges), "| circRNAs with a miRNA edge:", int(circ["has_mirna_edge"].sum()))
print("saved to", PROCESSED_DIR)

pairs: 2206 | circRNAs: 1644 | diseases: 209
length_band
<=1022       1232
1023-5000     350
>5000          62
model-input circRNAs (<=5000 nt): 1582
miRNA edges: 1105 | circRNAs with a miRNA edge: 722
saved to d:\Queens_university\Project\Working\Notebook\circmtl\data\processed


**Interpretation.** The frozen tables reproduce the expected values: **2,206 pairs, 1,644 circRNAs, 209 diseases**; length bands 1,232 (≤1,022 nt), 350 (1,023–5,000 nt) and 62 (>5,000 nt), i.e. 1,582 model-input circRNAs. The miRNA edges (1,105; 722 circRNAs) are slightly higher than reported in Cell 22 (1,055; 695). Two causes are possible: this file is not restricted to the ≤ 5,000 nt set (which affects the number of circRNAs) and it keeps the `arm` column, so a miRNA recorded for both arms counts as two edges. The split between the two causes was not measured. Sensitivity tiers are not built in this notebook, so no frozen set is reported for them.

| Frozen set | Pairs | circRNAs | Diseases |
|---|---|---|---|
| Primary (tiers 1 + 2) | 2,206 | 1,644 | 209 |
| Primary, ≤ 5,000 nt (model input) | 2,137 | 1,582 | 207 |
| Strict ≤ 1,022 nt (proposal) | 1,665 | 1,232 | 180 |

**Decision.** The CDA side is complete. Because the RBP column of the curated file is unusable, the principal remaining question is whether ENCORI (CLIP-seq-based) can supply auxiliary labels for these circRNAs. Its API is probed with three small requests before any loader is written (Cell 27).

## Brainstorming_Cells

In [ ]:
# Cell 3 - Load the CDA tables as text and profile every column of the entries table
import re

READ = dict(dtype=str, keep_default_na=False)        # keep text as text; "NA" is not a missing value
raw = pd.read_excel(CDA_DIR / "The_circRNA-disease_entries.xlsx",
                    sheet_name="The CircRNA-Disease Entries", **READ)
entries = raw.apply(lambda c: c.str.strip())
circ_list = pd.read_excel(CDA_DIR / "The_circRNA_list.xlsx", header=None, names=["circrna_name"], **READ)
dis_list = pd.read_excel(CDA_DIR / "The_disease_list.xlsx", header=None, names=["disease"], **READ)
print(f"entries {entries.shape} | circRNA list {len(circ_list)} | disease list {len(dis_list)}")

# 1. Column profile: fill rate, distinct values, cells with surrounding spaces, an example
prof = pd.DataFrame({
    "filled_%": (100 * (entries != "").mean()).round(1),
    "n_unique": entries.nunique(),
    "n_untrimmed": (raw != entries).sum(),
    "example": entries.apply(lambda c: c[c != ""].iloc[0][:45] if (c != "").any() else ""),
})
print("\n", prof.to_string())

# 2. Categorical columns: value counts
for col in ["Species", "Strand", "Expression pattern"]:
    print(f"\n{col}:\n{entries[col].value_counts().head(15).to_string()}")
tech = entries["Experimental techniques"].str.split(r"[;,/]").explode().str.strip()
print(f"\nExperimental techniques, top terms:\n{tech[tech != ''].value_counts().head(15).to_string()}")

# 3. Duplicates
print("\nduplicated rows:", int(entries.duplicated().sum()),
      "| duplicated CRD ID:", int(entries["CRD ID"].duplicated().sum()),
      "| duplicated (circRNA, disease, PMID):",
      int(entries.duplicated(["circRNA Name", "Disease Name", "PubMed ID"]).sum()),
      "| duplicated (circRNA, disease):", int(entries.duplicated(["circRNA Name", "Disease Name"]).sum()))

# 4. Identifier formats: how often is a circBase ID directly available?
cb = r"hsa_circ_\d{7}"
print("\ncircRNA Name is a circBase ID:      ", int(entries["circRNA Name"].str.fullmatch(cb, case=False).sum()))
print("circBase ID somewhere in Synonyms:  ", int(entries["Synonyms"].str.contains(cb, case=False).sum()))
print("Region empty:                       ", int((entries["Region"] == "").sum()))
for col in ["circRNA Name", "Synonyms", "Region", "Gene Symbol"]:
    print(f"\n{col} examples:", entries[col][entries[col] != ""].sample(6, random_state=0).tolist())

# 5. Disease names: raw problems before any cleaning
d = entries["Disease Name"]
print("\nunique diseases:", d.nunique(),
      "| case-insensitive unique:", d.str.lower().nunique(),
      "| 'NA' inside a word:", int(d.str.contains(r"[a-z]NA[a-z]|^NA[a-z]").sum()))

entries (4201, 13) | circRNA list 3077 | disease list 312

                          filled_%  n_unique  n_untrimmed                                        example
CRD ID                      100.0      4201            0                                              1
circRNA Name                100.0      3122           19                             hsa_circRNA_001937
Synonyms                    100.0      1058            2                                            N/A
Gene Symbol                 100.0      1640           25                                           CHD9
Disease Name                100.0       313            2                                   Tuberculosis
Expression pattern          100.0         8          143                                    Upregulated
PubMed ID                   100.0      2449            0                                       29448254
Region                      100.0      1884           24                                            N/A
Stra

In [ ]:
# Cell 4 - Mark placeholder text as missing, normalise categories, keep human rows, classify circRNA name formats
MISSING = {"", "N/A", "NA", "n/a", "-", "None"}
e = entries.copy()
for c in e.columns:
    e[c] = e[c].where(~e[c].isin(MISSING))                       # placeholder text -> NaN
print("missing per column:\n" + e.isna().sum().to_string())

e["Species"] = e["Species"].str.capitalize().replace({"Rats": "Rat"})
e["Expression pattern"] = e["Expression pattern"].str.capitalize()
print("\nspecies after normalising:\n" + e["Species"].value_counts().to_string())

hum = e[e["Species"] == "Human"].copy()
print(f"\nhuman rows: {len(hum)} | non-human removed: {len(e) - len(hum)}")

# circRNA name formats (checked in this order)
PATTERNS = [
    ("circbase",      r"hsa_circ_\d{7}"),                                  # exact circBase ID
    ("circbase_typo", r"(?i)(?:(?:has|hsa)[_-]?)?circ[_-]?\d{7}"),         # 7 digits, wrong prefix/case/separator
    ("short_number",  r"(?i)hsa_circ_\d{1,6}"),                            # fewer than 7 digits: not circBase
    ("arraystar",     r"(?i)hsa_circRNA_?\d+"),                            # Arraystar microarray ID
    ("coordinates",   r"(?i)(?:chr)?[\dXYM]{1,2}[_:]\d+[-_]\d+(?:[_:][+-])?"),
]
def name_class(s):
    for k, p in PATTERNS:
        if re.fullmatch(p, s):
            return k
    if re.search(r"(?i)(?:has|hsa)[_-]?circ[_-]?\d{7}", s):
        return "contains_circbase"                                         # e.g. ASB3-has_circ_0003959
    if re.match(r"(?i)circ", s):
        return "gene_based"                                                # e.g. circNF1-419, circPVT1
    return "other"

hum["name_class"] = hum["circRNA Name"].map(name_class)
hum["syn_circbase"] = hum["Synonyms"].str.contains(r"(?i)hsa_circ_\d{7}", na=False)
hum["has_region"] = hum["Region"].notna()
hum["has_region_strand"] = hum["Region"].notna() & hum["Strand"].notna()

summary = (hum.groupby("name_class")
           .agg(rows=("CRD ID", "size"), names=("circRNA Name", "nunique"),
                syn_circbase=("syn_circbase", "sum"), region=("has_region", "sum"),
                region_strand=("has_region_strand", "sum"))
           .sort_values("rows", ascending=False))
print("\n" + summary.to_string())
for k in summary.index:
    print(f"  {k:18s}", hum.loc[hum["name_class"] == k, "circRNA Name"].drop_duplicates().head(6).tolist())

# Region format among human rows
ok = hum["Region"].str.fullmatch(r"chr[\dXYM]{1,2}:\d+-\d+", na=False)
print(f"\nRegion in chrN:start-end format: {int(ok.sum())} | other formats:",
      hum.loc[hum["has_region"] & ~ok, "Region"].drop_duplicates().head(8).tolist())

# The disease written with two capitalisations
dn = e["Disease Name"]
v = dn.groupby(dn.str.lower()).unique()
print("\ndiseases differing only by case:", v[v.map(len) > 1].tolist())

missing per column:
CRD ID                        0
circRNA Name                  0
Synonyms                   2927
Gene Symbol                 699
Disease Name                  0
Expression pattern            2
PubMed ID                     2
Region                     1295
Strand                     2785
Species                       0
Experimental techniques       0
Brief description             7
Title                         0
species_clean                 0
name_pattern                  0

species after normalising:
Species
Human         3854
Mouse          248
Rat             93
Chicken          4
C. elegans       2

human rows: 3854 | non-human removed: 347

                   rows  names  syn_circbase  region  region_strand
name_class                                                         
circbase           2236   1501             4    2232           1176
gene_based          958    744           111     305            105
arraystar           262    230             0      90 

In [ ]:
# Cell 5 - Can the 967 human "other" rows be rescued, and how reliable is a text rule?
d = e                                   # <- your table with name_pattern and species_clean
MISSING = {"", "N/A", "NA", "n/a", "-", "None"}
present = lambda s: s.notna() & ~s.astype(str).str.strip().isin(MISSING)
CB = r"(?i)(?:hsa|has)[_-]?circ[_-]?(\d{7})"            # circBase number, tolerant of prefix typos
ids_in = lambda s: s.fillna("").astype(str).str.findall(CB).map(lambda x: sorted({f"hsa_circ_{n}" for n in x}))

hum = d[d["species_clean"] == "Human"].copy()
for col, src in [("ids_syn", "Synonyms"), ("ids_desc", "Brief description"), ("ids_title", "Title")]:
    hum[col] = ids_in(hum[src])

# 1. What do the "other" rows have?
oth = hum[hum["name_pattern"] == "other"]
n1 = lambda c: (oth[c].map(len) == 1)
print(f"'other' human rows: {len(oth)} | unique names: {oth['circRNA Name'].nunique()} | papers: {oth['PubMed ID'].nunique()}")
print(pd.Series({
    "Region present": int(present(oth["Region"]).sum()),
    "Region + Strand": int((present(oth["Region"]) & present(oth["Strand"])).sum()),
    "Gene Symbol present": int(present(oth["Gene Symbol"]).sum()),
    "exactly 1 ID in Synonyms": int(n1("ids_syn").sum()),
    "exactly 1 ID in Brief description": int(n1("ids_desc").sum()),
    ">1 ID in Brief description (ambiguous)": int((oth["ids_desc"].map(len) > 1).sum()),
    "exactly 1 ID in Title": int(n1("ids_title").sum()),
    "none of the above": int((~present(oth["Region"]) & (oth["ids_syn"].map(len) == 0)
                              & (oth["ids_desc"].map(len) == 0) & (oth["ids_title"].map(len) == 0)).sum()),
}).to_string())

# 2. Precision of the description rule, measured on rows whose answer is known
known = hum[hum["name_pattern"] == "circbase_full"].copy()
known["truth"] = known["circRNA Name"].str.extract(r"(?i)(hsa_circ_\d{7})")[0].str.lower()
one = known[known["ids_desc"].map(len) == 1]
hit = (one["ids_desc"].str[0] == one["truth"])
print(f"\ndescription rule on {len(known)} known rows: gives exactly one ID for {len(one)} "
      f"({100*len(one)/len(known):.1f}%) | correct {int(hit.sum())} -> precision {100*hit.mean():.1f}%")
print("wrong examples:", one.loc[~hit, ["circRNA Name", "ids_desc"]].head(5).values.tolist())

# 3. Examples to read by eye
print("\n'other' names:", oth["circRNA Name"].drop_duplicates().sample(12, random_state=0).tolist())

'other' human rows: 967 | unique names: 739 | papers: 765
Region present                            283
Region + Strand                            95
Gene Symbol present                       831
exactly 1 ID in Synonyms                  118
exactly 1 ID in Brief description           8
>1 ID in Brief description (ambiguous)      7
exactly 1 ID in Title                      12
none of the above                         672

description rule on 2317 known rows: gives exactly one ID for 418 (18.0%) | correct 413 -> precision 98.8%
wrong examples: [['hsa_circ_0079929', ['hsa_circ_0079299']], ['hsa_circ_0006461', ['hsa_circ_0074854']], ['hsa_circ_0008514', ['hsa_circ_0074854']], ['hsa_circ_0067582', ['hsa_circ_0000675']], ['hsa_circ_001730', ['hsa_circ_0007482']]]

'other' names: ['circKRT6A', 'circRNA-0067835', 'circRNA378', 'circRNA9227', 'circRNA-0006896', 'CircWAC', 'circATRNL1', 'circRNA PPP1CC', 'circ-ELF2', 'circRNA8505', 'circVRK1', 'circSETDB1']


In [ ]:
pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv").rename(columns={"circrna_id": "circbase_id"})
pairs = pairs[pairs["seq_len"].between(1, 5000)].drop_duplicates(["circbase_id", "disease"])
assert (pairs["circbase_id"].nunique(), len(pairs), pairs["disease"].nunique()) == (1_582, 2_137, 207)

dpc = pairs.groupby("circbase_id")["disease"].nunique()
print("diseases per circRNA (1,582 set):\n" + dpc.value_counts().sort_index().to_string())
print(f"one disease only: {100 * (dpc == 1).mean():.1f}%")
print("\ncircRNAs with the most diseases:\n" + dpc.sort_values(ascending=False).head().to_string())

diseases per circRNA (1,582 set):
disease
1     1267
2      218
3       60
4       15
5        6
6        6
7        2
8        1
9        1
10       1
11       1
12       1
15       1
22       1
27       1
one disease only: 80.1%

circRNAs with the most diseases:
circbase_id
hsa_circ_0000284    27
hsa_circ_0001946    22
hsa_circ_0000615    15
hsa_circ_0067934    12
hsa_circ_0001821    11
hsa_circ_0001649    10
hsa_circ_0001445     9
hsa_circ_0072088     8


In [ ]:
idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False).set_index("circbase_id")
top = dpc.sort_values(ascending=False).head(8).rename("n_diseases").to_frame()
print(top.join(idx[["gene", "seq_len"]]).to_string())

                  n_diseases            gene  seq_len
circbase_id                                          
hsa_circ_0000284          27           HIPK3     1099
hsa_circ_0001946          22            CDR1     1485
hsa_circ_0000615          15          ZNF609      874
hsa_circ_0067934          12           PRKCI      170
hsa_circ_0001821          11  TCONS_00015354      410
hsa_circ_0001649          10           SHPRH      440
hsa_circ_0001445           9         SMARCA5      269
hsa_circ_0072088           8             ZFR      693


In [ ]:
prim.groupby("circrna_id")["disease"].nunique().value_counts()


disease
1     1323
2      223
3       61
4       15
6        6
5        6
7        2
27       1
15       1
9        1
10       1
11       1
22       1
12       1
8        1
Name: count, dtype: int64

In [ ]:
dpc = prim.groupby("circrna_id")["disease"].nunique()
print("diseases per circRNA (1,582 set):\n" + dpc.value_counts().sort_index().to_string())
print(f"circRNAs {len(dpc)} | pairs {int(dpc.sum())} | one disease only: {100 * (dpc == 1).mean():.1f}%")

diseases per circRNA (1,582 set):
disease
1     1323
2      223
3       61
4       15
5        6
6        6
7        2
8        1
9        1
10       1
11       1
12       1
15       1
22       1
27       1
circRNAs 1644 | pairs 2206 | one disease only: 80.5%


In [44]:
# Agreement between CircR2Disease 'miRNA validated' and ENCORI working positives (check only, never labels)
import re
D = PROCESSED_DIR / "dataset_v1" 
cur = pd.read_excel(CDA_DIR / "The_manually_curated_associations_between_circRNAs_and_other_associations.xlsx",
                    dtype=str, keep_default_na=False)
print("examples of 'miRNA validated':", cur.loc[cur["miRNA validated"].str.strip().ne(""), "miRNA validated"].head(8).tolist())

def norm_mir(x):
    x = x.strip().lower().replace("hsa-", "")
    return re.sub(r"-(5p|3p)$", "", x)

circ_ids = set(pd.read_csv(D / "circrnas.csv")["circbase_id"])
pairs = []
for _, r in cur.iterrows():
    cid = r["circRNA Name"].strip().lower()
    if cid not in circ_ids:
        continue                                                   # only circRNAs in the 1,582 set (by circBase ID)
    for m in re.split(r"[;,/]", r["miRNA validated"]):
        if m.strip():
            pairs.append((cid, norm_mir(m)))
cur_pairs = pd.DataFrame(pairs, columns=["circbase_id", "mir"]).drop_duplicates()

enc = pd.read_csv(D / "mirna_edges.csv", dtype={"pos_working": bool})
enc["mir"] = enc["mirna_name"].map(norm_mir)
enc_all = set(zip(enc["circbase_id"], enc["mir"]))
enc_work = set(zip(enc.loc[enc["pos_working"], "circbase_id"], enc.loc[enc["pos_working"], "mir"]))
labelled = set(pd.read_csv(D / "circrnas.csv", dtype={"mirna_labelled": bool}).query("mirna_labelled")["circbase_id"])

cp = cur_pairs[cur_pairs["circbase_id"].isin(labelled)]
hit_any = cp.apply(lambda r: (r.circbase_id, r.mir) in enc_all, axis=1).mean()
hit_work = cp.apply(lambda r: (r.circbase_id, r.mir) in enc_work, axis=1).mean()
print(f"curated validated pairs on ENCORI-labelled circRNAs: {len(cp)} | "
      f"in ENCORI (any CLIP >= 1): {100*hit_any:.1f}% | in ENCORI working positives (CLIP >= 2): {100*hit_work:.1f}%")

examples of 'miRNA validated': ['miR-506', 'miR-339-5p, miR-96-3p and miR-135b-3p', 'miR-124', 'miR-124', 'miR-124', 'miR-26a', 'miR-17-3p, miR-181b-5p', 'miR-30b-3p']
curated validated pairs on ENCORI-labelled circRNAs: 813 | in ENCORI (any CLIP >= 1): 18.9% | in ENCORI working positives (CLIP >= 2): 13.9%


## CircR2Disease_v2 reading

In [47]:
# How much does circRNADisease v2.0 add to the 1,582-circRNA CircR2Disease set? (read-only)
import pandas as pd
norm = lambda s: s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

cd2 = pd.read_csv(PROJECT_ROOT / "data/raw/CircBank-circRNADisese_V2/circRNADisese_V2.csv", dtype=str,
                  keep_default_na=False, encoding_errors="replace").apply(lambda c: c.str.strip())
h = cd2[cd2["species"].str.lower().eq("human") & cd2["circrna_id"].str.fullmatch(r"hsa_circ_\d{7}")].copy()
h["d_norm"] = norm(h["disease_details"])
new_pairs = h.drop_duplicates(["circrna_id", "d_norm"]).rename(columns={"circrna_id": "circbase_id"})

ours = pd.read_csv(D / "cda_pairs.csv"); ours["d_norm"] = norm(ours["disease"])
idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False).set_index("circbase_id")["seq_len"]
ours_c, ours_p, ours_d = set(ours["circbase_id"]), set(zip(ours["circbase_id"], ours["d_norm"])), set(ours["d_norm"])

in_c = new_pairs["circbase_id"].isin(ours_c)
same = [(c, d) in ours_p for c, d in zip(new_pairs["circbase_id"], new_pairs["d_norm"])]
nc = new_pairs.loc[~in_c, "circbase_id"].drop_duplicates()
nc_ok = nc[nc.map(idx).between(1, 5000)]
print(f"circRNADisease human pairs with circBase ID: {len(new_pairs)}")
print(f"  same pair already in our set (exact disease name): {sum(same)}")
print(f"  our circRNA, different disease name: {int(in_c.sum()) - sum(same)}  (part may be name variants)")
print(f"  pairs for circRNAs not in our set: {int((~in_c).sum())}")
print(f"new circRNAs: {len(nc)} | with sequence and <= 5,000 nt: {len(nc_ok)}")
print(f"their diseases matching one of our 207 names exactly: {new_pairs['d_norm'].isin(ours_d).mean():.1%}")

circRNADisease human pairs with circBase ID: 2854
  same pair already in our set (exact disease name): 1205
  our circRNA, different disease name: 715  (part may be name variants)
  pairs for circRNAs not in our set: 934
new circRNAs: 846 | with sequence and <= 5,000 nt: 764
their diseases matching one of our 207 names exactly: 88.2%


In [48]:
# Is the GitHub CSV derived from CircR2Disease v2.0? Compare paper titles (read-only)
import re
norm_t = lambda s: s.astype(str).str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()

cr2 = pd.read_excel(CDA_DIR / "The_circRNA-disease_entries.xlsx", dtype=str, keep_default_na=False)
t_cr2 = set(norm_t(cr2["Title"])) - {""}
t_csv = set(norm_t(cd2["title"])) - {""}
both = t_cr2 & t_csv
print(f"papers: CircR2Disease {len(t_cr2)} | CSV {len(t_csv)} | shared {len(both)}")
print(f"share of CSV papers also in CircR2Disease: {len(both)/len(t_csv):.1%}")
print(f"share of CircR2Disease papers also in CSV: {len(both)/len(t_cr2):.1%}")

papers: CircR2Disease 2448 | CSV 5363 | shared 2195
share of CSV papers also in CircR2Disease: 40.9%
share of CircR2Disease papers also in CSV: 89.7%


In [49]:
mt = pd.read_csv(D / "mirna_targets.csv")
for fl in [50, 40, 30, 20, 10]:
    t = mt[mt["n_pos"] >= fl]
    print(f"floor {fl:2d}: {len(t):3d} targets | positive edges {int(t['n_pos'].sum()):6d} | "
          f"min positives per test fold ~{fl // 5}")

floor 50:  21 targets | positive edges   1605 | min positives per test fold ~10
floor 40:  49 targets | positive edges   2812 | min positives per test fold ~8
floor 30: 130 targets | positive edges   5655 | min positives per test fold ~6
floor 20: 288 targets | positive edges   9340 | min positives per test fold ~4
floor 10: 492 targets | positive edges  12278 | min positives per test fold ~2


In [53]:
mt = pd.read_csv(D / "rbp_targets.csv")
for fl in [50, 40, 30, 20, 10]:
    t = mt[mt["n_pos"] >= fl]
    print(f"floor {fl:2d}: {len(t):3d} targets | positive edges {int(t['n_pos'].sum()):6d} | "
          f"min positives per test fold ~{fl // 5}")

floor 50:  83 targets | positive edges  20763 | min positives per test fold ~10
floor 40:  90 targets | positive edges  21082 | min positives per test fold ~8
floor 30: 103 targets | positive edges  21518 | min positives per test fold ~6
floor 20: 116 targets | positive edges  21842 | min positives per test fold ~4
floor 10: 147 targets | positive edges  22259 | min positives per test fold ~2


In [50]:
# Target lists for floors 40 (primary) and 30 (sensitivity); in_head (floor 50) is kept unchanged
for fn in ["mirna_targets.csv", "rbp_targets.csv"]:
    t = pd.read_csv(OUT / fn, keep_default_na=False)
    t["in_head_f40"], t["in_head_f30"] = t["n_pos"] >= 40, t["n_pos"] >= 30
    t.to_csv(OUT / fn, index=False)
    print(f"{fn}: floor 50 {int((t['n_pos'] >= 50).sum())} | floor 40 {int(t['in_head_f40'].sum())} "
          f"| floor 30 {int(t['in_head_f30'].sum())}")

mirna_targets.csv: floor 50 21 | floor 40 49 | floor 30 130
rbp_targets.csv: floor 50 83 | floor 40 90 | floor 30 103


### The disease-degree check (comment 3), runs in seconds

In [54]:
# How many test positives involve diseases with few or no training edges? (read-only)
cda = pd.read_csv(D / "cda_pairs.csv")
fo = pd.read_csv(D / "folds.csv").set_index("circbase_id")
rows = []
for s in (0, 1, 2):
    f = cda["circbase_id"].map(fo[f"fold_seed{s}"])
    for k in range(5):
        tr, te = cda[f != k], cda[f == k]
        deg = te["disease"].map(tr["disease"].value_counts()).fillna(0)
        rows.append({"seed": s, "fold": k, "test_pos": len(te),
                     "disease_0_train": int((deg == 0).sum()), "disease_1_2": int(deg.between(1, 2).sum()),
                     "disease_3_9": int(deg.between(3, 9).sum()), "disease_10plus": int((deg >= 10).sum())})
R = pd.DataFrame(rows)
print(R.to_string(index=False))
cold = (R["disease_0_train"] + R["disease_1_2"]).sum() / R["test_pos"].sum()
print(f"\nshare of test positives on diseases with 0-2 training edges: {cold:.1%}")

 seed  fold  test_pos  disease_0_train  disease_1_2  disease_3_9  disease_10plus
    0     0       427               15           21           69             322
    0     1       428               17           28          100             283
    0     2       427               17           18           92             300
    0     3       428               19           22           64             323
    0     4       427               15           22           98             292
    1     0       428               11           15           88             314
    1     1       427               23           27           85             292
    1     2       428               17           20           78             313
    1     3       427               15           19           91             302
    1     4       427               13           25           88             301
    2     0       428               16           20           78             314
    2     1       428       

## ENCORI dataset

### 27. Structural Inspection of the Downloaded ENCORI Tables

**Objective.** Inspect the structure of the three downloaded ENCORI files (size, number of lines, header, column names, data types and the first records) without loading them completely, in order to decide which columns are needed and how each file must be read.

In [52]:
# Cell 27 - Structural inspection of the three downloaded ENCORI tables
ENC_FILES = {
    "miRNA-circRNA (binding sites)":  ENC_MIRNA_DIR / "hg38_miRcircRNA.txt",
    "RBP-circRNA (gene level)":       ENC_RBP_DIR   / "hg38_RBPcircRNA.txt",
    "RBP-circRNA (CLIP clusters)":    ENC_RBP_DIR   / "hg38_RBPcircRNA_cluster.txt",
}

def count_lines(path, chunk=1 << 24):
    """Count newline characters in binary chunks, so that the file is never held in memory."""
    n = 0
    with open(path, "rb") as f:
        while buf := f.read(chunk):
            n += buf.count(b"\n")
    return n

for name, p in ENC_FILES.items():
    print("=" * 100)
    print(f"{name} | {p.name} | {p.stat().st_size / 1e6:,.1f} MB")

    # Line count includes the header; it is one too low if the last line has no trailing newline.
    print(f"  lines (incl. header): {count_lines(p):,}")

    # The API files carried '#' comment lines; check whether the downloaded files do as well.
    with open(p, encoding="utf-8", errors="replace") as f:
        print("  first line is a comment:", f.readline().startswith("#"))

    # Only five rows are parsed; 'NA' strings are converted to real missing values.
    d = pd.read_csv(p, sep="\t", nrows=5, na_values=["NA"])
    print(f"  columns ({d.shape[1]}): {list(d.columns)}")
    print("\n  dtypes (from the 5-row sample, provisional):")
    print(d.dtypes.to_string())

    # Transposing the first two records makes every field readable at a glance.
    print("\n  first two records (transposed):")
    print(d.head(2).T.to_string(max_colwidth=60))

miRNA-circRNA (binding sites) | hg38_miRcircRNA.txt | 2,205.7 MB
  lines (incl. header): 5,598,985
  first line is a comment: False
  columns (37): ['lineID', 'chromosome', 'start', 'end', 'bindID', 'merClass', 'strand', 'miRNAid', 'miRNAname', 'miRseq', 'align', 'targetSeq', 'tdmdScore', 'clipExpNum', 'bindClipExpNum', 'clipIDnum', 'RBP', 'RBPclipNum', 'RBPclipID', 'degraExpNum', 'bindDegraExpNum', 'degraIDNum', 'degraID', 'geneID', 'geneName', 'geneType', 'txIDcat', 'txInfo', 'eclip', 'hitsclip', 'iclip', 'other', 'parclip', 'site', 'peak', 'pvalue', 'consAve']

  dtypes (from the 5-row sample, provisional):
lineID               int64
chromosome             str
start                int64
end                  int64
bindID                 str
merClass               str
strand                 str
miRNAid                str
miRNAname              str
miRseq                 str
align                  str
targetSeq              str
tdmdScore          float64
clipExpNum           int64
bind

**Interpretation.** The three files differ in size and, more importantly, in what one row represents. *(A) Format and size.* All three are tab-separated tables whose first line is the header (no comment lines), so they can be read directly with `sep="\t"`. Subtracting the header from the line counts gives 5,598,984 records for the miRNA table, 899,786 for the gene-level RBP table and 4,669,990 for the RBP cluster table (this assumes that each file ends with a newline, which was not checked). The miRNA and cluster files exceed 2 GB each and therefore have to be read in chunks with column selection. *(B) miRNA table (37 columns).* One row is one predicted binding site of a miRNA on a circRNA. In both sampled rows `txIDcat` holds a single circBase identifier (`hsa_circ_0009177`), whereas `geneName` holds the host gene (`WASH7P`), so the join key is `txIDcat` and not `geneName`. Both sampled sites have `clipExpNum = 0`, which means they are sequence-predicted matches without CLIP support, so an evidence filter will be necessary. The columns `miRseq`, `align` and `targetSeq` contain alignment strings; they are the heaviest part of the file and are not needed to define an edge. *(C) RBP table at gene level (17 columns).* This file has no circRNA identifier (there is no `txIDcat` or `txInfo` column), and its `geneID` values, such as `NM_000968`, are mRNA accessions. Its rows are therefore RBP–gene pairs and cannot provide RBP–circRNA edges (inference from the header and the two sampled rows). *(D) RBP cluster table (19 columns).* This is the only RBP table that carries `txIDcat` and `txInfo`. In the first two records `txIDcat` is a comma-joined list of circBase identifiers (the printed value is cut at 60 characters), and `clipID` is comma-joined as well. One CLIP cluster is thus linked to several circRNAs, and the table must be expanded to one row per (RBP, circRNA) pair before any counting. The relation between `clusterClipExpNum` and `clipExpNum` is not clear from the two rows shown, where the two values are equal. *(E) Data types.* In the miRNA sample, `RBP`, `RBPclipNum`, `RBPclipID` and `degraID` are entirely missing and were therefore parsed as `float64`. `pvalue` was read as `int64` only because both sampled values are 0, while it is a float in both RBP tables. All dtypes are provisional and will be set explicitly.

**Not checked.** (i) Whether `txIDcat` in the miRNA table always holds a single identifier; only two records of this table were inspected in this cell. (ii) Whether the identifiers of the three files match the circBase IDs of our FASTA index. (iii) The meaning of `bindClipExpNum`, `clipIDnum`, `degraExpNum`, `tdmdScore` and `consAve`; they are interpreted from their names only and the ENCORI documentation was not consulted.

**Decision.** (i) The miRNA table will be read in chunks with an explicit `usecols` list that excludes `miRseq`, `align` and `targetSeq`; `txIDcat` is the circRNA key. (ii) The RBP cluster table is the source for circRNA–RBP edges, after expansion of `txIDcat`; the gene-level RBP table is not used to build edges. (iii) The next cell profiles the miRNA table in chunks: missing values per column, the distributions of `merClass` and `clipExpNum`, whether `txIDcat` ever contains more than one identifier, and the numbers of distinct miRNAs and circRNAs.

### 28. Reference Genome (GRCh38, RefSeq): Location, Index and Sequence Names

**Objective.** Register the GRCh38 RefSeq genome FASTA as a raw input, build a random-access index so that any region can be read without loading the file, and inspect the sequence names and headers, so that the mapping between NCBI accessions (NC_000001.11) and ENCORI chromosome names (chr1) can be defined from the observed format.

In [57]:
# Cell 28 - Reference genome (GRCh38, RefSeq): location, index and sequence names
import pandas as pd
from pyfaidx import Fasta

# Path from the configuration file; Cell 1 is left unchanged.
NCBI_DIR  = PROJECT_ROOT / ds["ncbi_grch38"]["dir"]
GENOME_FA = NCBI_DIR / "GCF_000001405.26_GRCh38_genomic.fna"
assert GENOME_FA.exists(), f"genome FASTA not found: {GENOME_FA}"
print(f"{GENOME_FA.name} | {GENOME_FA.stat().st_size / 1e9:.2f} GB")

# The first call scans the file once and writes a .fai index next to it (this can take a minute).
# The FASTA itself is not modified; sequences are read from disk on demand.
genome = Fasta(str(GENOME_FA), as_raw=True)
FAI_FILE = GENOME_FA.with_name(GENOME_FA.name + ".fai")
print("index file:", FAI_FILE.name, "| exists:", FAI_FILE.exists())

# The .fai is a tab-separated table: name, length, byte offset, bases per line, bytes per line.
fai = pd.read_csv(FAI_FILE, sep="\t", header=None,
                  names=["name", "length", "offset", "line_bases", "line_bytes"])
print(f"\nsequences: {len(fai):,} | total length: {fai['length'].sum():,} bp")

# Accession prefix counts (NC_ = complete chromosome, NT_/NW_ = scaffolds), to see what the file contains.
print("\naccession prefixes:")
print(fai["name"].str.extract(r"^([A-Z]+)_")[0].value_counts().to_string())

# pyfaidx keeps only the first word of each header, so the description is read from the file itself.
# The index offset points to the first base, so the header line ends just before it.
def read_header(path, offset, window=512):
    with open(path, "rb") as f:
        f.seek(max(0, offset - window))
        chunk = f.read(min(window, offset)).decode("ascii", errors="replace")
    return [ln for ln in chunk.splitlines() if ln.startswith(">")][-1]

head30 = fai.head(30).copy()
head30["header"] = [read_header(GENOME_FA, o) for o in head30["offset"]]
print("\nfirst 30 sequences:")
print(head30[["name", "length", "header"]].to_string(index=False, max_colwidth=110))

# Random-access test: only the requested bytes are read, not the whole chromosome.
first = fai["name"].iloc[0]
print(f"\n{first}[10000:10040] = {genome[first][10000:10040]}")

GCF_000001405.26_GRCh38_genomic.fna | 3.25 GB
index file: GCF_000001405.26_GRCh38_genomic.fna.fai | exists: True

sequences: 455 | total length: 3,209,286,105 bp

accession prefixes:
0
NT    359
NW     71
NC     25

first 30 sequences:
        name    length                                                                                                         header
NC_000001.11 248956422                                               >NC_000001.11 Homo sapiens chromosome 1, GRCh38 Primary Assembly
 NT_187361.1    175055 >NT_187361.1 Homo sapiens chromosome 1 unlocalized genomic scaffold, GRCh38 Primary Assembly HSCHR1_CTG1_UN...
 NT_187362.1     32032 >NT_187362.1 Homo sapiens chromosome 1 unlocalized genomic scaffold, GRCh38 Primary Assembly HSCHR1_CTG2_UN...
 NT_187363.1    127682 >NT_187363.1 Homo sapiens chromosome 1 unlocalized genomic scaffold, GRCh38 Primary Assembly HSCHR1_CTG3_UN...
 NT_187364.1     66860 >NT_187364.1 Homo sapiens chromosome 1 unlocalized genomic scaffold, GR

**Interpretation.** *(A) Index and access.* The 3.25 GB FASTA was indexed without problems, and the index file exists. The file contains 455 sequences with a total length of 3,209,286,105 bp: 25 chromosome-level records (`NC_`), 359 `NT_` scaffolds and 71 `NW_` scaffolds. The random-access test returned a 40-bp slice of `NC_000001.11` without reading the chromosome. The slice is a telomeric repeat, which is consistent with the known N gap at the start of chromosome 1 (inference: the gap itself was not printed). *(B) Sequence names.* The chromosome records follow the pattern `>NC_0000NN.V Homo sapiens chromosome N, GRCh38 Primary Assembly`. The accession version differs between chromosomes (.11 for chromosome 1, .12 for chromosome 2, .14 for chromosome 7), so the mapping to `chrN` cannot be derived from the accession number and has to be read from the header text. The 25 `NC_` records fit chromosomes 1–22, X, Y and the mitochondrial genome, but only the headers of chromosomes 1–11 were printed. *(C) Scaffolds.* The scaffolds that appear among the first 30 records are unlocalized scaffolds of chromosomes 1, 2, 3, 4, 5, 9 and 11. They are interleaved with the chromosomes, so the order of the file cannot be used to separate chromosomes from scaffolds. *(D) Soft-masking.* The returned slice is in lower case, which indicates that repeats are soft-masked in this file.

**Not checked.** (i) The headers of chromosomes 12–22, X, Y and the mitochondrial genome. (ii) The nature of the remaining scaffolds beyond the 30 printed records. (iii) Whether the chromosome names used by ENCORI (including the mitochondrial one) match the names that will be derived from these headers.

**Decision.** (i) Only the `NC_` records are used for the coordinate validation, and each is mapped to `chr` + chromosome number or letter by parsing the text after "chromosome" in its header. The `NT_` and `NW_` scaffolds are not mapped. (ii) Every comparison with `targetSeq` is made after upper-casing the genome slice, because the file is soft-masked. (iii) The mapping is built in Cell 31 with explicit checks, namely 25 `NC_` records, a unique `chr` name for each, and the mitochondrial header printed and inspected, and ENCORI chromosome names are tested against it using the list printed in Cell 29. (iv) The `.fai` file stays next to the FASTA as a derived file and is not committed.

### 29. Profile of the ENCORI miRNA–circRNA Table

**Objective.** Profile the ENCORI miRNA–circRNA binding-site table in a single chunked pass, after expanding records that list several circRNAs to one row per circRNA, in order to establish (i) how many circRNAs a record lists, (ii) how many distinct circRNAs, miRNAs and circRNA–miRNA pairs the table contains, (iii) how the CLIP evidence is distributed, and (iv) whether the circBase identifiers of ENCORI refer to the same circRNAs as those of the circBase FASTA index.

In [59]:
# Cell 29 - Profile of the ENCORI miRNA-circRNA table (single chunked pass; records exploded to circRNAs)
import numpy as np

MIRNA_FILE = ENC_MIRNA_DIR / "hg38_miRcircRNA.txt"
USECOLS = ["chromosome", "start", "end", "strand", "merClass", "miRNAid", "miRNAname",
           "tdmdScore", "clipExpNum", "bindClipExpNum", "geneType", "txIDcat", "consAve"]
BASE = 10**7          # exact integer key for a pair: circRNA number * BASE + MIMAT number (no hashing, no collisions)

def add(total, new):
    """Accumulate counts across chunks."""
    return new if total is None else total.add(new, fill_value=0)

n_rows = n_ex = n_valid = n_bad_circ = n_bad_mir = n_bad_coord = k_max = 0
na_cnt = pd.Series(0, index=USECOLS, dtype="int64")
acc = dict.fromkeys(["k", "mer", "type", "chrom", "strand", "clip", "mer_x_clip", "clip_x_bind"])
bad_examples, keys_all, keys_s1, keys_s2, loc_parts, mir_parts, samples = [], [], [], [], [], [], []

reader = pd.read_csv(MIRNA_FILE, sep="\t", usecols=USECOLS, chunksize=1_000_000,
                     keep_default_na=False, na_values=["NA"])
for i, ch in enumerate(reader, 1):
    n_rows += len(ch)
    na_cnt += ch.isna().sum()
    n_bad_coord += int((ch["start"] >= ch["end"]).sum())

    # ---- record-level summaries (every record, before any explosion) -------------------------------
    ids = ch["txIDcat"].str.split(",")
    k = ids.str.len()                                    # number of circRNAs listed in the record
    k_max = max(k_max, int(k.max()))
    acc["k"] = add(acc["k"], k.clip(upper=11).value_counts())
    clip = ch["clipExpNum"]
    supp = clip.gt(0).rename("clip_supported")
    for key, col in [("mer", "merClass"), ("type", "geneType"), ("chrom", "chromosome"), ("strand", "strand")]:
        acc[key] = add(acc[key], ch[col].value_counts())
    acc["clip"] = add(acc["clip"], clip.clip(upper=3).value_counts())
    acc["mer_x_clip"] = add(acc["mer_x_clip"], ch.groupby(["merClass", supp]).size())
    acc["clip_x_bind"] = add(acc["clip_x_bind"],
                             ch.groupby([supp, ch["bindClipExpNum"].gt(0).rename("bind_supported")]).size())
    samples.append(ch.sample(frac=0.01, random_state=42))
    mir_parts.append(ch[["miRNAid", "miRNAname"]].drop_duplicates())

    # ---- explode: one row per (record, circRNA); the evidence of the site applies to each listed circRNA -
    ex = (ch[["miRNAid", "clipExpNum", "chromosome", "strand"]].assign(circ=ids)
          .explode("circ").reset_index(drop=True))
    n_ex += len(ex)
    cnum = ex["circ"].str.extract(r"^hsa_circ_(\d{7})$")[0]
    mnum = ex["miRNAid"].str.extract(r"^MIMAT(\d{7})$")[0]
    n_bad_circ += int(cnum.isna().sum())
    n_bad_mir += int(mnum.isna().sum())
    if len(bad_examples) < 10:                           # a few examples only; the counts above are exact
        bad_examples.extend(ex.loc[cnum.isna().to_numpy(), "circ"].unique()[:10 - len(bad_examples)])
    good = (cnum.notna() & mnum.notna()).to_numpy()
    ex = ex[good].reset_index(drop=True)
    c_num = pd.to_numeric(cnum[good]).to_numpy(dtype="int64")
    m_num = pd.to_numeric(mnum[good]).to_numpy(dtype="int64")
    n_valid += len(ex)

    key = c_num * BASE + m_num
    cl = ex["clipExpNum"].to_numpy()
    keys_all.append(np.unique(key))
    keys_s1.append(np.unique(key[cl >= 1]))
    keys_s2.append(np.unique(key[cl >= 2]))
    loc_parts.append(ex[["chromosome", "strand"]].assign(c=c_num).drop_duplicates())
    print(f"  chunk {i}: {n_rows:,} records", end="\r")

# ---- 1. size, missing values, coordinates ------------------------------------------------------------
print(f"records: {n_rows:,} | equals the Cell 27 line count minus the header: {n_rows == 5_598_984}")
print("missing values per column:", na_cnt[na_cnt > 0].to_dict() or "none")
print(f"records with start >= end: {n_bad_coord:,}")

# ---- 2. identifiers: how many circRNAs does one record list? ----------------------------------------
n_multi = int(acc["k"].loc[acc["k"].index > 1].sum())
print(f"\nrecords listing more than one circRNA: {n_multi:,} ({100 * n_multi / n_rows:.1f}%) | "
      f"maximum in one record: {k_max}")
print("circRNAs per record (11 = 11 or more):\n", acc["k"].astype(int).sort_index().to_string())
print(f"\nexploded (record, circRNA) rows: {n_ex:,} ({n_ex / n_rows:.2f} per record)")
print(f"tokens that are not 'hsa_circ_' + 7 digits: {n_bad_circ:,} (examples: {bad_examples})")
print(f"miRNA ids that are not 'MIMAT' + 7 digits: {n_bad_mir:,}")

# ---- 3. entities and edges (valid tokens only; an edge is a unique circRNA-miRNA pair) ---------------
pairs_all = np.unique(np.concatenate(keys_all))
pairs_s1  = np.unique(np.concatenate(keys_s1))
pairs_s2  = np.unique(np.concatenate(keys_s2))
c_all, c_s1, c_s2 = (np.unique(p // BASE) for p in (pairs_all, pairs_s1, pairs_s2))
mm = pd.concat(mir_parts).drop_duplicates()
print(f"\ndistinct circRNAs: {len(c_all):,} | with a site of clipExpNum >= 1: {len(c_s1):,} | >= 2: {len(c_s2):,}")
print(f"distinct miRNAs: {mm['miRNAid'].nunique():,} (ids) | {mm['miRNAname'].nunique():,} (names) | "
      f"ids with more than one name: {int((mm.groupby('miRNAid')['miRNAname'].nunique() > 1).sum())}")
print(f"circRNA-miRNA pairs: {len(pairs_all):,} | clipExpNum >= 1: {len(pairs_s1):,} | >= 2: {len(pairs_s2):,}")
print(f"(site, circRNA) rows per pair: {n_valid / len(pairs_all):.2f}")

_, d_circ   = np.unique(pairs_all // BASE, return_counts=True)
_, d_circ_s = np.unique(pairs_s1 // BASE, return_counts=True)
_, d_mir    = np.unique(pairs_all % BASE, return_counts=True)
deg = pd.concat({"miRNAs per circRNA (all pairs)": pd.Series(d_circ),
                 "miRNAs per circRNA (CLIP >= 1)": pd.Series(d_circ_s),
                 "circRNAs per miRNA (all pairs)": pd.Series(d_mir)}, axis=1)
print("\ndegree distributions:\n", deg.describe(percentiles=[.05, .25, .5, .75, .95]).round(1).T.to_string())

# ---- 4. distributions over records -----------------------------------------------------------------
for key, title in [("type", "geneType"), ("strand", "strand"), ("mer", "merClass")]:
    print(f"\n{title}:\n{acc[key].astype(int).to_string()}")
print("\nclipExpNum (3 = 3 or more):\n", acc["clip"].astype(int).sort_index().to_string())
mx = acc["mer_x_clip"].unstack().fillna(0).astype(int)
mx["share_supported_%"] = (100 * mx[True] / mx.sum(axis=1)).round(2)
print("\nmerClass x CLIP support (False / True):\n", mx.to_string())
print("\nCLIP support (rows) x bindClipExpNum support (columns):\n",
      acc["clip_x_bind"].unstack().fillna(0).astype(int).to_string())
print("\nchromosomes:", sorted(acc["chrom"].index.tolist()))
smp = pd.concat(samples)
smp["site_len"] = smp["end"] - smp["start"]
print("\n1% sample, quantiles:")
print(smp[["tdmdScore", "consAve", "site_len", "clipExpNum"]]
      .quantile([0, .05, .25, .5, .75, .95, 1]).round(3).to_string())

# ---- 5. consistency with the circBase FASTA index (same ID = same circRNA?) -------------------------
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
fa_ids = set(fa_idx["circbase_id"])
enc_ids = {f"hsa_circ_{n:07d}" for n in c_all}
print(f"\nENCORI circRNAs with a sequence in the circBase FASTA index: {len(enc_ids & fa_ids):,} "
      f"of {len(enc_ids):,} | FASTA records absent from ENCORI: {len(fa_ids - enc_ids):,}")

# Chromosome and strand do not depend on the assembly (hg38 here, hg19 in the FASTA), so they can be compared directly.
locs = pd.concat(loc_parts).drop_duplicates()
locs["circbase_id"] = "hsa_circ_" + locs["c"].astype(str).str.zfill(7)
print(f"circRNAs seen with more than one (chromosome, strand) combination in ENCORI: "
      f"{int(locs.groupby('circbase_id').size().gt(1).sum()):,}")
chk = locs.merge(fa_idx[["circbase_id", "chrom", "strand"]], on="circbase_id", how="inner", suffixes=("_enc", "_cb"))
chk["chrom_ok"] = chk["chromosome"] == chk["chrom"]
chk["strand_ok"] = chk["strand_enc"] == chk["strand_cb"]
print(f"(circRNA, chromosome, strand) combinations compared: {len(chk):,} | "
      f"chromosome agrees: {100 * chk['chrom_ok'].mean():.2f}% | strand agrees: {100 * chk['strand_ok'].mean():.2f}% | "
      f"both: {100 * (chk['chrom_ok'] & chk['strand_ok']).mean():.2f}%")
bad = chk[~chk["chrom_ok"]]
print(f"chromosome disagreements on alt/random/unplaced contigs (name contains '_'): "
      f"{int(bad['chromosome'].str.contains('_').sum()):,} of {len(bad):,}")
print(chk.loc[~(chk["chrom_ok"] & chk["strand_ok"]),
              ["circbase_id", "chromosome", "chrom", "strand_enc", "strand_cb"]].head(10).to_string(index=False))

records: 5,598,984 | equals the Cell 27 line count minus the header: True
missing values per column: none
records with start >= end: 0

records listing more than one circRNA: 1,910,786 (34.1%) | maximum in one record: 220
circRNAs per record (11 = 11 or more):
 txIDcat
1     3688198
2      791530
3      483110
4      185850
5      100076
6       93731
7       54426
8       44826
9       34937
10      17413
11     104887

exploded (record, circRNA) rows: 11,882,876 (2.12 per record)
tokens that are not 'hsa_circ_' + 7 digits: 0 (examples: [])
miRNA ids that are not 'MIMAT' + 7 digits: 0

distinct circRNAs: 91,050 | with a site of clipExpNum >= 1: 70,347 | >= 2: 56,491
distinct miRNAs: 642 (ids) | 642 (names) | ids with more than one name: 0
circRNA-miRNA pairs: 5,490,359 | clipExpNum >= 1: 1,866,819 | >= 2: 1,175,226
(site, circRNA) rows per pair: 2.16

degree distributions:
                                   count    mean     std    min      5%     25%     50%      75%      95%      ma

**Interpretation.** *(A) Integrity.* The 5,598,984 records match the line count of Cell 27, none of the 13 columns read has a missing value, no record has `start ≥ end`, and every record has `geneType = circRNA`. *(B) One record, several circRNAs.* 1,910,786 records (34.1%) list more than one circBase identifier, and one record lists 220. The 104,887 records that list 11 or more circRNAs (1.9%) account for at least 1.15 million of the 11,882,876 exploded rows (about 10%, a lower bound). Every exploded token matches `hsa_circ_` plus seven digits, and every miRNA accession matches `MIMAT` plus seven digits, so all identifiers can be parsed without exceptions. *(C) Entities and edges.* After expansion the table contains 91,050 distinct circRNAs and 642 miRNAs, with one name per accession. It yields 5,490,359 circRNA–miRNA pairs, which is 9.4% of the 58.5 million possible pairs, and each pair is backed by 2.16 (site, circRNA) rows on average. Of these pairs, 1,866,819 (34.0%) have a site with `clipExpNum ≥ 1` and 1,175,226 (21.4%) have one with `clipExpNum ≥ 2`. The median circRNA is linked to 34 miRNAs (13 when only CLIP-supported pairs are counted), the mean is 60.3, and the maximum is all 642. Every miRNA is linked to at least 458 circRNAs (median 8,278). *(D) Evidence.* 71.9% of the records have `clipExpNum = 0`. Seed type does not track CLIP support (28.5% for 7mer-m8 against 26.8% for 8mer), and only these two seed classes occur in the file. `bindClipExpNum > 0` never occurs without `clipExpNum > 0` (464,081 of 1,571,573 supported records), so it is a stricter subset. *(E) Support per circRNA.* 70,347 circRNAs (77.3%) have at least one supported site, whereas only 28.1% of the records are supported. A likely reason is that supported sites are shared by many overlapping circRNAs, and each circRNA collects many sites (inference, not tested). Support at the circRNA level is therefore permissive and carries less information than support at the pair level. *(F) Agreement with circBase.* All 91,050 circRNAs have an identifier in the circBase FASTA index (leaving 49,740 FASTA records without ENCORI data), and none appears with more than one chromosome–strand combination in ENCORI. Chromosome agrees for 99.98% of the circRNAs (17 disagreements, 13 of them with an ENCORI contig name), strand for 99.71% and both for 99.69%. Of the 10 printed disagreements, 8 have consecutive identifiers between `hsa_circ_0013801` and `hsa_circ_0013813`, which suggests clustered mismatches rather than random ones (inference from 10 rows). *(G) Other columns.* ENCORI uses 31 chromosome names: the 24 primary chromosomes, `chrM` and six alt/random/unplaced contigs. Site length has a median of 22 nt (5th–95th percentile 19–25 nt) in the 1% sample.

**Not checked.** (i) Which database is correct for the identifiers that disagree on chromosome or strand. (ii) The meaning of `clipExpNum`, `bindClipExpNum` and `consAve`, which are interpreted from their names and values only. (iii) Whether sharing of sites between overlapping circRNAs inflates support per circRNA. (iv) The 1% sample is used for the quantiles of `tdmdScore`, `consAve` and site length, not the full table.

**Decision.** (i) Records are expanded to one row per circRNA, and edges are defined at the level of the (circRNA, miRNA) pair, with the evidence of a pair taken as the maximum `clipExpNum` over its sites. The thresholds (≥ 1 or ≥ 2) are not fixed yet. (ii) `merClass` is not used as evidence. (iii) Identifiers with a chromosome or strand disagreement are flagged in the ENCORI tables, not silently dropped, and their handling is decided once the exact number is known. (iv) The number of circRNAs listed per site is stored with every site, so that a cap on widely shared sites can be tested later. (v) The ENCORI tables are built for all 91,050 circRNAs, and the intersection with the task circRNAs is a later, separate step.

### 30. Coverage of the CircR2Disease circRNAs in ENCORI (read-only)

**Objective.** Count how many of the circRNAs of the CircR2Disease task are present in ENCORI, and with which level of CLIP evidence, without building or saving any intersection table.

In [60]:
# Cell 30 - Coverage of the CircR2Disease circRNAs in ENCORI (read-only: nothing is saved and no intersection table is built)
circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")

# circBase identifier -> integer, in the same way as in Cell 29 ('hsa_circ_0009177' -> 9177)
circ["num"] = pd.to_numeric(circ["circrna_id"].str.extract(r"^hsa_circ_(\d{7})$")[0])
assert circ["num"].notna().all(), "unexpected identifiers in circrna_table.csv"
num = circ["num"].astype("int64").to_numpy()

# ENCORI evidence per circRNA, from the arrays built in Cell 29 (c_all, c_s1, c_s2, pairs_*)
circ["in_encori"] = np.isin(num, c_all)
circ["clip_ge1"]  = np.isin(num, c_s1)        # at least one site with clipExpNum >= 1
circ["clip_ge2"]  = np.isin(num, c_s2)        # at least one site with clipExpNum >= 2

rows = []
for lab, mask in [("CircR2Disease circRNAs", np.ones(len(circ), dtype=bool)),
                  ("model-input (<=5,000 nt)", circ["use_primary"].to_numpy())]:
    sub = circ[mask]
    rows.append((lab, len(sub), int(sub["in_encori"].sum()), int(sub["clip_ge1"].sum()), int(sub["clip_ge2"].sum())))
cov = pd.DataFrame(rows, columns=["circRNA set", "n", "in ENCORI", "site CLIP >= 1", "site CLIP >= 2"])
for c in ["in ENCORI", "site CLIP >= 1", "site CLIP >= 2"]:
    cov[c + " %"] = (100 * cov[c] / cov["n"]).round(1)
print("coverage of the task circRNAs:\n", cov.to_string(index=False))

# ENCORI pairs that involve a task circRNA (counted only; the pairs are not stored)
print("\nENCORI pairs involving a task circRNA:")
for lab, arr in [("all pairs", pairs_all), ("CLIP >= 1", pairs_s1), ("CLIP >= 2", pairs_s2)]:
    sel = arr[np.isin(arr // BASE, num)]
    print(f"  {lab}: {len(sel):,} pairs | {len(np.unique(sel // BASE)):,} circRNAs | {len(np.unique(sel % BASE)):,} miRNAs")

# Do the identifiers flagged in Cell 29 (chromosome or strand disagreement with circBase) touch the task set?
flag = set(chk.loc[~(chk["chrom_ok"] & chk["strand_ok"]), "circbase_id"])
print(f"\ntask circRNAs with a chromosome/strand disagreement: "
      f"{int(circ['circrna_id'].isin(flag).sum())} of {int(circ['in_encori'].sum())} found in ENCORI "
      f"(ENCORI-wide: {len(flag)} flagged identifiers)")

coverage of the task circRNAs:
              circRNA set    n  in ENCORI  site CLIP >= 1  site CLIP >= 2  in ENCORI %  site CLIP >= 1 %  site CLIP >= 2 %
  CircR2Disease circRNAs 1644       1485            1094             875         90.3              66.5              53.2
model-input (<=5,000 nt) 1582       1433            1048             835         90.6              66.2              52.8

ENCORI pairs involving a task circRNA:
  all pairs: 72,870 pairs | 1,485 circRNAs | 642 miRNAs
  CLIP >= 1: 23,373 pairs | 1,094 circRNAs | 638 miRNAs
  CLIP >= 2: 15,033 pairs | 875 circRNAs | 635 miRNAs

task circRNAs with a chromosome/strand disagreement: 4 of 1485 found in ENCORI (ENCORI-wide: 282 flagged identifiers)


**Interpretation.** *(A) Coverage.* Of the 1,644 CircR2Disease circRNAs, 1,485 (90.3%) occur in ENCORI. Of these, 1,094 (66.5% of 1,644) have at least one site with `clipExpNum ≥ 1` and 875 (53.2%) have at least one with `clipExpNum ≥ 2`. The 1,582 model-input circRNAs (≤ 5,000 nt) give almost the same figures (1,433, 1,048 and 835; 90.6%, 66.2% and 52.8%), so the length limit hardly changes coverage. 159 task circRNAs (9.7%) have no record in ENCORI and therefore receive no miRNA edge from this source. *(B) Comparison with ENCORI as a whole.* Among the 1,485 task circRNAs found in ENCORI, 73.7% have a site with `clipExpNum ≥ 1` and 58.9% have one with `clipExpNum ≥ 2`. Over all 91,050 ENCORI circRNAs the shares are 77.3% and 62.0% (Cell 29), so the task circRNAs are slightly less often CLIP-supported than ENCORI circRNAs in general. *(C) Pairs.* 72,870 ENCORI pairs involve a task circRNA, of which 23,373 have `clipExpNum ≥ 1` and 15,033 have `clipExpNum ≥ 2`. The corresponding means are 49.1, 21.4 and 17.2 miRNAs per task circRNA, against 60.3, 26.5 and 20.8 over ENCORI as a whole (ratios computed from the printed counts). The task circRNAs are therefore somewhat less connected than the average ENCORI circRNA. The miRNA side is not limiting: all 642 miRNAs appear among the task pairs (638 with `clipExpNum ≥ 1`, 635 with `clipExpNum ≥ 2`). *(D) Identifier disagreements.* 4 of the 1,485 task circRNAs found in ENCORI (0.3%) belong to the 282 identifiers flagged in Cell 29 for a chromosome or strand disagreement with circBase. This is the same order as the ENCORI-wide share (282 of 91,050, 0.31%), so the disagreement is not concentrated in the task set and cannot change the coverage figures by more than 4 circRNAs.

**Not checked.** (i) Why 159 task circRNAs are absent from ENCORI (identifier not used by ENCORI, retired or merged identifier, or circRNA genuinely not included); the absent identifiers were not examined. (ii) Which 4 task circRNAs are flagged, and whether their disagreement concerns the chromosome or the strand. (iii) Coverage is defined by identifier only, which assumes that the same circBase ID is the same circRNA; Cell 29 tested this for chromosome and strand but not for exact coordinates (hg19 in circBase, hg38 in ENCORI). (iv) The lower connectivity of the task circRNAs is shown by means only; medians, the number of task circRNAs with very few pairs, and the cause (for example circRNA length) were not examined. (v) The overlap between these ENCORI pairs and the curated miRNA edges of Cell 26.

**Decision.** (i) The 159 task circRNAs without an ENCORI record keep their rows in the CDA task and are treated as unlabelled for the miRNA auxiliary task, not as negatives, because absence from ENCORI is not evidence of absence of binding. (ii) The ENCORI-based miRNA labels can cover at most 1,485 of 1,644 circRNAs (1,433 of 1,582 for the model input), and at most 1,094 (1,048) when a CLIP threshold of 1 is applied; the threshold is not fixed here. (iii) The 4 flagged task circRNAs are kept and flagged, and their handling is decided after the coordinate validation of Cell 31. (iv) The flags `in_encori`, `clip_ge1` and `clip_ge2` are stored with the ENCORI edge tables (Cell 32), not in this read-only cell.

### 31. Coordinate Convention of the ENCORI Binding Sites, Checked Against GRCh38

**Objective.** Map the ENCORI chromosome names to the RefSeq chromosome records of the GRCh38 FASTA, and test on a random 2% sample of the miRNA–circRNA records which coordinate convention (0- or 1-based start, inclusive or exclusive end) and which strand orientation reproduce the `targetSeq` of a site from the genome. This establishes (i) whether the ENCORI coordinates are really GRCh38, (ii) how `start` and `end` must be read, (iii) whether the `strand` column refers to the orientation of `targetSeq`, and (iv) how many sites cannot be checked because their contig has no RefSeq chromosome record.

In [61]:
# Cell 31 - Coordinate convention of the ENCORI sites, checked against GRCh38
# (read-only: nothing is saved; the 2% sample is re-read from the file because Cell 29 did not keep targetSeq)
import re

# ---- 1. chromosome name -> RefSeq accession, parsed from the FASTA headers (decision of Cell 28) ----------
nc = fai[fai["name"].str.startswith("NC_")].copy()
nc["header"] = [read_header(GENOME_FA, o) for o in nc["offset"]]

def chr_label(header):
    """'>NC_000001.11 Homo sapiens chromosome 1, ...' -> 'chr1'; the mitochondrial record has no 'chromosome' word."""
    m = re.search(r"chromosome (\w+),", header)
    if m:
        return "chr" + m.group(1)
    return "chrM" if "mitochondrion" in header else None

nc["chr"] = nc["header"].map(chr_label)
assert len(nc) == 25 and nc["chr"].notna().all() and nc["chr"].is_unique, "unexpected NC_ headers"
assert set(nc["chr"]) == {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY", "chrM"}
CHR2NC = dict(zip(nc["chr"], nc["name"]))
print(nc.loc[nc["chr"].isin(["chr1", "chrX", "chrY", "chrM"]), ["name", "chr", "length", "header"]]
        .to_string(index=False, max_colwidth=85))

# ENCORI names versus the mapping, using the record counts per chromosome name kept from Cell 29
enc_chr  = acc["chrom"].astype(int)
unmapped = enc_chr[~enc_chr.index.isin(CHR2NC)]
print(f"\nENCORI chromosome names: {len(enc_chr)} | mapped: {int(enc_chr.index.isin(CHR2NC).sum())} | not mapped: {len(unmapped)}")
print(f"records on unmapped contigs: {int(unmapped.sum()):,} ({100 * unmapped.sum() / n_rows:.3f}% of {n_rows:,})")
print(unmapped.to_string())
print("mapped chromosomes absent from ENCORI:", sorted(set(CHR2NC) - set(enc_chr.index)) or "none")

# ---- 2. random 2% sample with the site sequence (same chunking as Cell 29) --------------------------------
COLS = ["lineID", "chromosome", "start", "end", "strand", "targetSeq", "txIDcat", "clipExpNum"]
parts = [ch.sample(frac=0.02, random_state=42)
         for ch in pd.read_csv(MIRNA_FILE, sep="\t", usecols=COLS, chunksize=1_000_000,
                               keep_default_na=False, na_values=["NA"])]
smp = pd.concat(parts, ignore_index=True)
n0 = len(smp)
# sites on unmapped contigs cannot be checked; start >= 1 is required for the 1-based variant below
smp = (smp[smp["chromosome"].isin(CHR2NC) & (smp["start"] >= 1)]
       .sort_values(["chromosome", "start"]).reset_index(drop=True))     # sorted for sequential disk access
print(f"\nsample: {n0:,} records | checkable (mapped chromosome, start >= 1): {len(smp):,}")

# ---- 3. six hypotheses: three coordinate conventions x two strand orientations ---------------------------
COMP = str.maketrans("ACGTN", "TGCAN")
def revcomp(s):
    return s.translate(COMP)[::-1]

# (a, b): python slice genome[start + a : end + b] on the 0-based genome string
CONV = {"0-based, end exclusive": (0, 0),
        "1-based, end inclusive": (-1, 0),
        "0-based, end inclusive": (0, 1)}
res = {f"{c} | {o}": [] for c in CONV for o in ("as annotated", "opposite strand")}
len_diff = []
for r in smp.itertuples(index=False):
    t = r.targetSeq.upper().replace("U", "T")                  # targetSeq is RNA; the genome is DNA
    win = genome[CHR2NC[r.chromosome]][r.start - 1 : r.end + 1].upper()   # one read per site; the file is soft-masked
    L = r.end - r.start
    len_diff.append(len(t) - L)
    for cname, (a, b) in CONV.items():
        s = win[a + 1 : L + 1 + b]                             # slice of the window that corresponds to genome[start+a : end+b]
        res[f"{cname} | as annotated"].append((s if r.strand == "+" else revcomp(s)) == t)
        res[f"{cname} | opposite strand"].append((revcomp(s) if r.strand == "+" else s) == t)
hyp = pd.DataFrame(res)
smp["len_diff"] = len_diff

print("\nlen(targetSeq) - (end - start):\n", smp["len_diff"].value_counts().head(6).to_string())
tab = hyp.groupby(smp["strand"]).mean().T * 100
tab["all"] = hyp.mean() * 100
print("\nshare of sites whose targetSeq equals the genome sequence (%), by hypothesis and strand:")
print(tab.round(2).sort_values("all", ascending=False).to_string())

# ---- 4. the best hypothesis: where does it fail? ----------------------------------------------------------
best = tab["all"].idxmax()
cname, orient = best.split(" | ")
a, b = CONV[cname]
ok = hyp[best]
print(f"\nbest hypothesis: {best} | match: {100 * ok.mean():.2f}% | sites that do not match: {int((~ok).sum()):,}")

by_chr = smp.assign(ok=ok).groupby("chromosome")["ok"].agg(n="size", match_pct=lambda x: round(100 * x.mean(), 2))
print("\nlowest match per chromosome:\n", by_chr.sort_values("match_pct").head(8).to_string())

bad = smp[~ok]
same_len = bad["len_diff"].eq(b - a)       # expected len(targetSeq) - (end - start) under the best hypothesis
print(f"\nmismatches with the expected sequence length (different content): {int(same_len.sum()):,} | "
      f"with a different length (possible spliced or discontinuous site): {int((~same_len).sum()):,}")

def genome_seq(chrom, start, end, strand):
    s = genome[CHR2NC[chrom]][start + a : end + b].upper()
    return s if (strand == "+") == (orient == "as annotated") else revcomp(s)
print("\nfirst mismatches (target | genome):")
for r in bad.head(5).itertuples(index=False):
    print(f"  line {r.lineID} {r.chromosome}:{r.start}-{r.end}({r.strand}) k={len(r.txIDcat.split(','))}\n"
          f"    {r.targetSeq.upper().replace('U', 'T')}\n    {genome_seq(r.chromosome, r.start, r.end, r.strand)}")

        name  chr    length                                                           header
NC_000001.11 chr1 248956422 >NC_000001.11 Homo sapiens chromosome 1, GRCh38 Primary Assembly
NC_000023.11 chrX 156040895 >NC_000023.11 Homo sapiens chromosome X, GRCh38 Primary Assembly
NC_000024.10 chrY  57227415 >NC_000024.10 Homo sapiens chromosome Y, GRCh38 Primary Assembly
 NC_012920.1 chrM     16569         >NC_012920.1 Homo sapiens mitochondrion, complete genome

ENCORI chromosome names: 31 | mapped: 25 | not mapped: 6
records on unmapped contigs: 450 (0.008% of 5,598,984)
chromosome
chr14_GL000009v2_random     61
chr15_KI270850v1_alt        11
chr1_KI270706v1_random      71
chr22_KI270879v1_alt        96
chr4_GL000008v2_random     160
chrUn_KI270742v1            51
mapped chromosomes absent from ENCORI: none

sample: 111,980 records | checkable (mapped chromosome, start >= 1): 111,976

len(targetSeq) - (end - start):
 len_diff
0    85074
1    17127
2     6350
3     1999
4      608
5    

In [62]:
# Cell 31 (part 2) - same test with the alignment gaps removed from targetSeq
# (uses smp and CHR2NC from the previous cell; no file is re-read; read-only)
smp["t"]     = smp["targetSeq"].str.upper().str.replace("U", "T").str.replace("-", "", regex=False)
smp["n_gap"] = smp["targetSeq"].str.count("-")

def genome_site(chrom, start, end, strand):
    """Genome sequence of a site under the convention found in part 1: 0-based, end exclusive, strand as annotated."""
    s = genome[CHR2NC[chrom]][start:end].upper()
    return s if strand == "+" else revcomp(s)

smp["g"] = [genome_site(c, s, e, st) for c, s, e, st in zip(smp["chromosome"], smp["start"], smp["end"], smp["strand"])]
smp["match"]  = smp["g"] == smp["t"]
smp["len_ok"] = smp["t"].str.len() == (smp["end"] - smp["start"])

print(f"sites with at least one gap character in targetSeq: {int((smp['n_gap'] > 0).sum()):,} "
      f"({100 * (smp['n_gap'] > 0).mean():.1f}% of {len(smp):,})")
print(f"length of the gap-free targetSeq equals end - start: {100 * smp['len_ok'].mean():.2f}%")
print(f"gap-free targetSeq equals the genome sequence: {100 * smp['match'].mean():.2f}% | "
      f"sites that still do not match: {int((~smp['match']).sum()):,}")

print("\nmatch by number of gap characters (3 = 3 or more):")
print(smp.groupby(smp["n_gap"].clip(upper=3))["match"].agg(n="size", match_pct=lambda x: round(100 * x.mean(), 3)).to_string())
print("\nmatch by strand:")
print(smp.groupby("strand")["match"].agg(n="size", match_pct=lambda x: round(100 * x.mean(), 3)).to_string())

bad = smp.loc[~smp["match"], ["lineID", "chromosome", "start", "end", "strand", "n_gap", "t", "g"]]
print("\nremaining mismatches (up to 8):")
print(bad.head(8).to_string(index=False))

sites with at least one gap character in targetSeq: 26,894 (24.0% of 111,976)
length of the gap-free targetSeq equals end - start: 99.99%
gap-free targetSeq equals the genome sequence: 99.99% | sites that still do not match: 12

match by number of gap characters (3 = 3 or more):
           n  match_pct
n_gap                  
0      85082     99.989
1      17127     99.994
2       6352     99.969
3       3415    100.000

match by strand:
            n  match_pct
strand                  
+       57645     99.998
-       54331     99.980

remaining mismatches (up to 8):
 lineID chromosome     start       end strand  n_gap                       t                                                                                                                           g
  32485       chr1  12617774  12617797      -      1  TTTTTTTTAATCTTGCACTTTG                                                                                                     TTTTTTTTTAATCTTGCACTTTG
  32488       chr1  126

**Interpretation.** *(A) Chromosome mapping.* The 25 `NC_` records map one-to-one to chr1–22, chrX, chrY and chrM, the mitochondrion being identified from its header ("mitochondrion, complete genome"). All 25 mapped names occur in ENCORI. ENCORI uses six further contig names (unlocalized or alternative scaffolds) that have no RefSeq chromosome record; they carry only 450 of the 5,598,984 records (0.008%) and were not checked. *(B) Coordinate convention.* Of six hypotheses (three coordinate conventions, each in the annotated and the opposite strand orientation), only "0-based start, end exclusive, strand as annotated" reproduced any site sequence. The other five gave 0.00%. The result is the same on both strands (76.1% and 75.8% before the gap correction), so `strand` is the orientation of `targetSeq`, and the coordinates are GRCh38 coordinates. *(C) Alignment gaps.* The 24% of sites that failed in the first test contain gap characters (`-`) inside `targetSeq`, which is therefore an aligned string and not a plain genome slice: 26,894 of 111,976 checked sites (24.0%) have at least one gap, and 3,415 have three or more. After the gaps are removed, the length of `targetSeq` equals `end − start` in 99.99% of the sites, and the sequence equals the genome sequence in 99.99% (111,964 of 111,976 sites). The match rate is the same with 0, 1, 2 and 3 or more gaps (99.97–100%) and on both strands (99.998% on `+`, 99.980% on `−`). *(D) The 12 remaining mismatches.* Eight were printed, and in each of them the gap-free `targetSeq` has a different length from the interval. (a) Two records of one site on chr1 (12,617,774–12,617,797, `−`) lack one T of a run of nine T's in the genome. (b) One site on chr1 (120,051,221–120,051,252, `−`) lacks eight nucleotides of a (GT)n repeat. (c) Five records on chr14 (start 105,867,292, `−`) have intervals of 122–123 nt, whereas `targetSeq` has 22–23 nt and equals the first 22–23 nt of the interval in the genomic direction. The interval contains two copies of a repeated unit of about 60 nt, and the maximum site length of 123 nt in Cell 29 is of the same size. By inspection of the printed strings, `start` is therefore correct and `end` too large for this site (inference; not tested in code). All eight mismatches sit in repeat sequences, and none shows a wrong chromosome, strand or assembly.

**Not checked.** (i) Only a 2% random sample was checked (111,976 of 5,598,984 records), not the full table; if the rate holds, about 600 records would be affected (extrapolation: 12 / 111,976 × 5,598,984). (ii) The four mismatches that were not printed, and whether all 12 lie in repeats. (iii) The reason for the gap characters; they presumably come from the miRNA–target alignment, but the ENCORI documentation was not consulted. (iv) The 450 records on unmapped contigs, 4 sampled records on unmapped contigs or with `start < 1` that were excluded, and the match rate of the mitochondrial sites separately. (v) Whether any site spans the back-splice junction, which a contiguous genomic interval cannot represent. (vi) Whether the circBase coordinates (hg19) agree with ENCORI for the flagged identifiers; this is the subject of Cell 32.

**Decision.** (i) ENCORI site coordinates are read as 0-based, end exclusive, on GRCh38, and `strand` is the orientation of `targetSeq`. These rules go into `src/circmtl/data.py` as named constants when the code is moved. (ii) `targetSeq` is stripped of `-` and converted to DNA before any comparison with a genome or a circBase sequence. (iii) No site is removed on the basis of this check, since edges are defined at the pair level and do not use site coordinates. To quantify inconsistent sites on the full table without genome lookups, the pass of Cell 33 will also record, per record, whether `len(targetSeq without gaps) == end − start`; this flag detects the length-inconsistent records seen here, and it will be tested on the full table, not extrapolated. (iv) Records on the six unmapped contigs are kept and flagged, not dropped. (v) Cell 32 tests the flagged identifiers using the convention found here: the gap-free `targetSeq` of a site is searched in the circBase FASTA sequence of its circRNA in both orientations, which separates a true strand conflict from an error in either database.

### 32. Do the ENCORI Sites of the Flagged Identifiers Lie in the circBase Sequence?

**Objective.** For the 282 circRNAs whose chromosome or strand differs between circBase and ENCORI (Cell 29), and for a random control of 300 circRNAs without disagreement, search the gap-free `targetSeq` of every ENCORI site in the circBase FASTA sequence of the same identifier, in the direct and in the reverse-complement orientation, in order to establish (i) how often a site is found in the sequence when nothing disagrees (the baseline), (ii) whether the flagged identifiers behave like the baseline, are found only in the opposite orientation, or are not found, and (iii) what this means for the 4 flagged circRNAs of the task set.

In [63]:
# Cell 32 - Do the ENCORI sites of the flagged identifiers lie in the circBase sequence? (read-only: nothing is saved)
# Uses from earlier cells: chk, fa_idx (Cell 29), c_all, MIRNA_FILE, circ (Cell 30), fa_path (Cell 6), revcomp (Cell 31).

# ---- 1. identifiers: all flagged ones plus a random control of 300 identifiers without disagreement ----------
flagged = chk[~(chk["chrom_ok"] & chk["strand_ok"])].set_index("circbase_id").copy()
flagged["kind"] = np.select([~flagged["chrom_ok"] & ~flagged["strand_ok"], ~flagged["chrom_ok"]],
                            ["both", "chromosome"], default="strand")
flag_ids = set(flagged.index)
pool     = sorted({f"hsa_circ_{n:07d}" for n in c_all} - flag_ids)
ctrl_ids = {str(x) for x in np.random.default_rng(42).choice(pool, size=300, replace=False)}
task_ids = set(circ["circrna_id"])
wanted   = flag_ids | ctrl_ids
print(f"flagged: {len(flag_ids)} ({flagged['kind'].value_counts().to_dict()}) | in the task set: {len(flag_ids & task_ids)} | control: {len(ctrl_ids)}")

# ---- 2. circBase sequences of these identifiers only (one streaming pass; the FASTA is 500 MB) --------------
def read_fasta_subset(path, wanted):
    out, cur, buf = {}, None, []
    with open(path) as f:
        for line in f:
            if line[0] == ">":
                if cur is not None:
                    out[cur] = "".join(buf).upper()
                cid = line[1:].split("|", 1)[0]
                cur, buf = (cid if cid in wanted else None), []
            elif cur is not None:
                buf.append(line.strip())
    if cur is not None:
        out[cur] = "".join(buf).upper()
    return out

seqs = read_fasta_subset(fa_path, wanted)
assert wanted <= set(seqs), "identifiers missing from the FASTA"
print(f"sequences read: {len(seqs)} | empty: {sum(len(s) == 0 for s in seqs.values())} | "
      f"longest: {max(map(len, seqs.values())):,} nt")

# ---- 3. all ENCORI sites of these identifiers (one pass over the table; a record may list several circRNAs) --
COLS, parts = ["lineID", "chromosome", "start", "end", "strand", "targetSeq", "txIDcat"], []
for ch in pd.read_csv(MIRNA_FILE, sep="\t", usecols=COLS, chunksize=1_000_000, keep_default_na=False, na_values=["NA"]):
    ids = ch["txIDcat"].str.split(",").explode()
    ids = ids[ids.isin(wanted)]
    parts.append(ch.loc[ids.index, COLS[:-1]].assign(cid=ids.values))
sites = pd.concat(parts, ignore_index=True)
print(f"\n(site, circRNA) rows: {len(sites):,} | flagged: {int(sites['cid'].isin(flag_ids).sum()):,} | "
      f"control: {int(sites['cid'].isin(ctrl_ids).sum()):,}")

# ---- 4. orientation test: the gap-free site sequence is searched in the circBase sequence -------------------
sites["t"] = sites["targetSeq"].str.upper().str.replace("U", "T").str.replace("-", "", regex=False)
direct, rc_only = [], []
for t, c in zip(sites["t"], sites["cid"]):
    s = seqs[c]
    d = t in s                                   # found as written (same orientation as targetSeq)
    direct.append(d)
    rc_only.append((not d) and (revcomp(t) in s))   # found only as reverse complement
sites["direct"], sites["rc_only"] = direct, rc_only
sites["none"] = ~(sites["direct"] | sites["rc_only"])
OUT = ["direct", "rc_only", "none"]

# ---- 5. baseline: control identifiers (ENCORI and circBase agree on chromosome and strand) -------------------
ctrl = sites[sites["cid"].isin(ctrl_ids)]
print("\ncontrol, share of sites (%) by ENCORI strand:")
print((ctrl.groupby("strand")[OUT].mean() * 100).round(2).assign(n=ctrl.groupby("strand").size()).to_string())
per_c = ctrl.groupby("cid")[OUT].mean()
print("\ncontrol, per-circRNA share of sites found directly (quantiles):")
print(per_c["direct"].quantile([0, .05, .25, .5, .75, 1]).round(3).to_string())
thr = per_c["direct"].quantile(0.05)             # a flagged circRNA is 'like the baseline' if it is not below 95% of the controls
print(f"threshold (5th percentile of the control): {thr:.3f}")

# ---- 6. flagged identifiers --------------------------------------------------------------------------------
fl = sites[sites["cid"].isin(flag_ids)]
pf = (fl.groupby("cid").agg(n_sites=("direct", "size"), direct=("direct", "mean"), rc_only=("rc_only", "mean"),
                            none=("none", "mean"), enc_min=("start", "min"), enc_max=("end", "max")))
pf = flagged[["kind", "chromosome", "strand_enc", "chrom", "strand_cb"]].join(pf)
pf = pf.join(fa_idx.set_index("circbase_id")[["start", "end", "seq_len"]].rename(columns={"start": "cb_start", "end": "cb_end"}))
pf["verdict"] = np.select([pf["direct"] >= thr, pf["rc_only"] >= thr],
                          ["found, same orientation", "found, opposite orientation"], default="not confirmed")
pf["task"] = pf.index.isin(task_ids)

print("\nflagged identifiers: verdict by kind of disagreement")
print(pd.crosstab(pf["kind"], pf["verdict"], margins=True).to_string())
print("\nflagged identifiers: verdict by strand pair (ENCORI -> circBase), all kinds")
print(pd.crosstab(pf["strand_enc"] + " -> " + pf["strand_cb"], pf["verdict"]).to_string())
print(f"\nflagged circRNAs with an empty circBase sequence: {int((pf['seq_len'] == 0).sum())}")

print("\nflagged circRNAs of the task set:")
print(pf.loc[pf["task"], ["kind", "chromosome", "strand_enc", "chrom", "strand_cb", "cb_start", "cb_end",
                          "enc_min", "enc_max", "seq_len", "n_sites", "direct", "rc_only", "none", "verdict"]]
        .round(3).to_string())

flagged: 282 ({'strand': 265, 'chromosome': 15, 'both': 2}) | in the task set: 4 | control: 300
sequences read: 582 | empty: 0 | longest: 275,938 nt

(site, circRNA) rows: 75,807 | flagged: 40,347 | control: 35,460

control, share of sites (%) by ENCORI strand:
        direct  rc_only  none      n
strand                              
+        99.68      0.0  0.32  27011
-        99.44      0.0  0.56   8449

control, per-circRNA share of sites found directly (quantiles):
0.00    0.667
0.05    0.989
0.25    1.000
0.50    1.000
0.75    1.000
1.00    1.000
threshold (5th percentile of the control): 0.989

flagged identifiers: verdict by kind of disagreement
verdict     found, same orientation  not confirmed  All
kind                                                   
both                              2              0    2
chromosome                       15              0   15
strand                          175             90  265
All                             192             90  282

f

In [64]:
# Cell 32 (part 2) - how unconfirmed are the 90? Distribution of the share of sites found, and threshold sensitivity
# (uses pf and thr from the previous cell; read-only)
nc90 = pf[pf["verdict"] == "not confirmed"]
print(f"not confirmed: {len(nc90)} | of which in the task set: {int(nc90['task'].sum())}")
print("\nshare of sites found directly among the 90 (quantiles):")
print(nc90["direct"].quantile([0, .1, .25, .5, .75, .9, 1]).round(3).to_string())
print("\nshare found in the opposite orientation among the 90 (max):", round(float(nc90["rc_only"].max()), 3))

print("\nflagged identifiers called 'found, same orientation' at other thresholds:")
for t in (0.5, 0.7, 0.8, 0.9, thr):
    print(f"  direct share >= {t:.3f}: {int((pf['direct'] >= t).sum())} of {len(pf)}")

print("\nnumber of sites and sequence length of the 90 (medians) versus the confirmed ones:")
print(pf.assign(conf=pf["verdict"].ne("not confirmed")).groupby("conf")[["n_sites", "seq_len"]].median().to_string())
print("\nnot confirmed with fewer than 5 sites:", int((nc90["n_sites"] < 5).sum()))

not confirmed: 90 | of which in the task set: 2

share of sites found directly among the 90 (quantiles):
0.00    0.417
0.10    0.763
0.25    0.840
0.50    0.891
0.75    0.943
0.90    0.971
1.00    0.988

share found in the opposite orientation among the 90 (max): 0.0

flagged identifiers called 'found, same orientation' at other thresholds:
  direct share >= 0.500: 281 of 282
  direct share >= 0.700: 276 of 282
  direct share >= 0.800: 272 of 282
  direct share >= 0.900: 235 of 282
  direct share >= 0.989: 192 of 282

number of sites and sequence length of the 90 (medians) versus the confirmed ones:
       n_sites  seq_len
conf                   
False     70.5   1223.0
True      25.0    551.0

not confirmed with fewer than 5 sites: 1


In [65]:
# Cell 32 (part 3) - length-matched control: is the lower share of sites found explained by circRNA length?
# Uses from earlier cells: pf, thr, flag_ids, ctrl_ids, pool, COLS (Cell 32); fa_idx, MIRNA_FILE, fa_path, revcomp,
# read_fasta_subset. Read-only: nothing is saved.
K = 5                                               # matched controls per flagged circRNA
rng = np.random.default_rng(42)

# ---- 1. matching on log10 sequence length, from the circBase index (no pass over the table needed) ----------
# candidates: ENCORI circRNAs that are not flagged and not in the first control, so the two controls are disjoint
cand = (fa_idx.set_index("circbase_id")["seq_len"]
        .reindex([i for i in pool if i not in ctrl_ids]).dropna())
cand = cand[cand > 0].sort_values()
cand_logl, cand_ids = np.log10(cand.to_numpy()), cand.index.to_numpy()
taken, match = np.zeros(len(cand), dtype=bool), {}
for fid in rng.permutation(pf.index.to_numpy()):    # random order, so no flagged circRNA is favoured
    order  = np.argsort(np.abs(cand_logl - np.log10(pf.loc[fid, "seq_len"])))
    chosen = order[~taken[order]][:K]               # the K nearest in length that are still free
    taken[chosen] = True
    match[fid] = list(cand_ids[chosen])
m_ids = {c for v in match.values() for c in v}
m_len = fa_idx.set_index("circbase_id").loc[sorted(m_ids), "seq_len"]
print(f"matched controls: {len(m_ids)} for {len(match)} flagged circRNAs | median length: "
      f"flagged {pf['seq_len'].median():,.0f} nt, matched {m_len.median():,.0f} nt")
worst = max(abs(np.log10(pf.loc[f, "seq_len"]) - np.log10(fa_idx.set_index('circbase_id').loc[c, "seq_len"]))
            for f, v in match.items() for c in v)
print(f"largest length difference within a match: a factor of {10 ** worst:.2f}")

# ---- 2. sequences and sites of the matched controls (one FASTA pass and one table pass) ----------------------
seqs_m = read_fasta_subset(fa_path, m_ids)
assert m_ids <= set(seqs_m), "identifiers missing from the FASTA"
parts = []
for ch in pd.read_csv(MIRNA_FILE, sep="\t", usecols=COLS, chunksize=1_000_000, keep_default_na=False, na_values=["NA"]):
    s = ch["txIDcat"].str.split(",").explode()
    s = s[s.isin(m_ids)]
    parts.append(ch.loc[s.index, COLS[:-1]].assign(cid=s.values))
sm = pd.concat(parts, ignore_index=True)
print(f"(site, circRNA) rows of the matched controls: {len(sm):,}")

# ---- 3. same orientation test as in Cell 32 --------------------------------------------------------------
t = sm["targetSeq"].str.upper().str.replace("U", "T").str.replace("-", "", regex=False)
d, r = [], []
for x, c in zip(t, sm["cid"]):
    hit = x in seqs_m[c]
    d.append(hit)
    r.append((not hit) and (revcomp(x) in seqs_m[c]))
sm["direct"], sm["rc_only"] = d, r
pm = (sm.groupby("cid").agg(n_sites=("direct", "size"), direct=("direct", "mean"), rc_only=("rc_only", "mean"))
        .join(m_len.rename("seq_len")))

# ---- 4. flagged versus matched controls, per circRNA -----------------------------------------------------
cols = ["seq_len", "n_sites", "direct", "rc_only"]
cmp = pd.concat({"flagged": pf[cols], "matched control": pm[cols]}, names=["group"]).reset_index(level=0)

def summ(g):
    return pd.Series({"n circRNAs": len(g), "median length (nt)": g["seq_len"].median(), "median n_sites": g["n_sites"].median(),
                      "direct q05": g["direct"].quantile(.05), "direct q25": g["direct"].quantile(.25),
                      "direct median": g["direct"].median(), "direct mean": g["direct"].mean(),
                      f"share >= {thr:.3f} (%)": 100 * (g["direct"] >= thr).mean(),
                      "share >= 0.9 (%)": 100 * (g["direct"] >= 0.9).mean(),
                      "max rc_only": g["rc_only"].max()})
print("\nper-circRNA share of sites found directly:")
print(pd.DataFrame({k: summ(g) for k, g in cmp.groupby("group")}).round(3).to_string())

# within length bins (quartiles of the flagged circRNAs): does the gap between the groups persist at equal length?
edges = np.unique([cmp["seq_len"].min() - 1, *pf["seq_len"].quantile([.25, .5, .75]).to_numpy(), cmp["seq_len"].max() + 1])
cmp["length bin (nt)"] = pd.cut(cmp["seq_len"], bins=edges)
print("\nby length bin:")
print(cmp.groupby(["length bin (nt)", "group"], observed=True)["direct"]
         .agg(n="size", median="median", mean="mean").round(3).unstack("group").to_string())

# difference in the mean share (flagged - matched), bootstrap 95% interval over circRNAs
def boot_diff(a, b, n=5000, seed=1):
    g = np.random.default_rng(seed)
    dd = g.choice(a, (n, len(a))).mean(axis=1) - g.choice(b, (n, len(b))).mean(axis=1)
    return a.mean() - b.mean(), np.percentile(dd, [2.5, 97.5])
diff, ci = boot_diff(pf["direct"].to_numpy(), pm["direct"].to_numpy())
print(f"\nmean share found, flagged - matched control: {diff:+.4f} (95% bootstrap interval {ci[0]:+.4f} to {ci[1]:+.4f})")

matched controls: 1410 for 282 flagged circRNAs | median length: flagged 670 nt, matched 670 nt
largest length difference within a match: a factor of 1.02
(site, circRNA) rows of the matched controls: 201,960

per-circRNA share of sites found directly:
                    flagged  matched control
n circRNAs          282.000         1410.000
median length (nt)  670.000          670.000
median n_sites       34.500           33.500
direct q05            0.829            1.000
direct q25            0.947            1.000
direct median         1.000            1.000
direct mean           0.959            0.995
share >= 0.989 (%)   68.085           96.312
share >= 0.9 (%)     83.333           98.156
max rc_only           0.000            0.000

by length bin:
                          n                  median                    mean                
group               flagged matched control flagged matched control flagged matched control
length bin (nt)                                     

**Interpretation.** *(A) Baseline.* When circBase and ENCORI agree (300 random control circRNAs, 35,460 sites), the gap-free `targetSeq` is found as written in the circBase sequence for about 99.6% of the sites (99.68% on `+`, 99.44% on `−`; computed from the two strand rows), and for none of the sites only as a reverse complement. The circBase sequences are therefore stored in the orientation of `targetSeq` on both strands. The 0.3–0.6% of sites that are not found probably span an exon junction, which a spliced sequence cannot contain (not tested). Per circRNA, the share of sites found is 1.000 for the median control, 0.989 at the 5th percentile and 0.667 at the minimum; 0.989 is used as the threshold below. *(B) Chromosome disagreements.* All 17 identifiers with a different chromosome (15 chromosome only, 2 together with a strand difference) have their sites in the circBase sequence in the same orientation, so the sequence agrees with ENCORI although the chromosome label differs. *(C) Strand disagreements.* Of the 267 identifiers with a different strand (265 strand only, 2 together with a chromosome difference), 177 (66.3%) are confirmed: their sites are found in the circBase sequence in the same orientation as in ENCORI. For these, the sequence follows the ENCORI strand and not the strand written in the circBase header, so the header strand is the discrepant element. The other 90 (33.7%) fall below the threshold. None of the 282 flagged circRNAs has its sites in the opposite orientation, so no case supports a true strand reversal. The two strand directions behave alike (`+`→`−`: 82 confirmed, 54 not; `−`→`+`: 95 confirmed, 36 not). *(D) Task circRNAs.* All four task circRNAs are ENCORI `+` against circBase `−` on the same chromosome. `hsa_circ_0013912` (6 sites) and `hsa_circ_0061259` (87 sites) are confirmed with 100% of their sites found. `hsa_circ_0018289` (22 sites) and `hsa_circ_0018293` (123 sites) are called "not confirmed", but 86.4% and 84.6% of their sites are found in the same orientation, none in the opposite one, and both values lie inside the range of the control (minimum 0.667), only below its 5th percentile. For all four, the span of the ENCORI sites is 8–72 nt smaller than the circBase interval, which is consistent with the same circRNA (the two spans are in different assemblies, so this is not proof). *(E) The 90 unconfirmed circRNAs and a length-matched control.* The 90 circRNAs have most, but not all, of their sites in the circBase sequence: the share of sites found directly has a median of 0.891 (quartiles 0.840–0.943, maximum 0.988, minimum 0.417), and none has sites in the opposite orientation. Counting flagged identifiers by this share gives 281 of 282 at ≥ 0.5, 276 at ≥ 0.7, 272 at ≥ 0.8, 235 at ≥ 0.9 and 192 at ≥ 0.989, so the threshold separates a gradient and not two groups. For circRNAs with up to 90 sites, a share of at least 0.989 means that every site is found, so the "not confirmed" class is partly defined by having more sites (median 70.5 against 25.0); this is a property of the cut-off. To test whether the lower share of the flagged circRNAs is explained by length, each flagged circRNA was compared with five unflagged circRNAs of matching length (1,410 controls; median 670 nt in both groups; median 34.5 and 33.5 sites). The flagged circRNAs have a lower share of sites found (mean 0.959 against 0.995; difference −0.035, 95% bootstrap interval −0.045 to −0.026), and the difference is present in all four length quartiles (flagged 0.939–0.973, controls 0.991–0.998) without a trend over length, so length does not explain it. The effect is small: the median is 1.000 in both groups, 83.3% of the flagged circRNAs reach 0.9 against 98.2% of the controls, and no circRNA has sites in the opposite orientation. The cause is not known (a different extent of the same circRNA in the two annotations is one possibility; exon structure was not examined).

**Decision.** (i) No flagged identifier is removed. The edges are keyed by the circBase identifier, the ENCORI strand is consistent with the genome (Cell 31, 99.99%), and for 192 of 282 flagged identifiers the ENCORI sites also lie in the circBase sequence that the model reads. (ii) The flag column `id_check` has four values: `consistent` (no disagreement), `header differs, sequence agrees` (direct share ≥ 0.989; 192), `header differs, sequence mostly agrees` (0.5 ≤ share < 0.989; 89) and `header differs, sequence not confirmed` (share < 0.5; 1). The share of sites found and the number of sites are stored as numeric columns as well, so that other cut-offs can be applied later without recomputation. (iii) The 4 task circRNAs are kept with this flag, and a sensitivity analysis without the flagged circRNAs can be run later. (iv) No exclusion rule is needed at this stage: no flagged circRNA has its sites in the opposite orientation, and 281 of 282 have at least half of their sites in the circBase sequence. The single circRNA below 0.5 is flagged and kept. A sensitivity analysis without the 282 flagged circRNAs (0.31% of ENCORI) can be run when the model results exist. (v) The strand written in the circBase header is not used as evidence in later steps; the sequence and the ENCORI coordinates are.(vi) The flagged circRNAs form a group that differs modestly from unflagged circRNAs of the same length in how many of their ENCORI sites lie in the circBase sequence. This supports keeping the flags and the numeric share of sites found with each edge table, and running the sensitivity analysis without the 282 flagged circRNAs later. No exclusion is made now.

### 33. Full ENCORI Tables: circRNA–miRNA Edges, circRNAs and miRNAs

**Objective.** Build, in a single chunked pass over the miRNA–circRNA table, (i) the edge table (one row per circBase identifier and miRBase accession, with the number of sites, the number of sites listed for that circRNA alone, the maximum `clipExpNum` over all sites and over those private sites, and further site statistics), (ii) a circRNA table with location and the identifier check of Cell 32, and (iii) a miRNA table, and use the same pass to (iv) quantify how much of the CLIP support depends on sites shared between overlapping circRNAs, (v) compute the quantiles of `tdmdScore`, `consAve` and site length on all records instead of the 1% sample, and (vi) count the records whose gap-free `targetSeq` is not `end − start` long.

In [67]:
# Cell 33 - Full ENCORI tables: circRNA-miRNA edges, circRNA table and miRNA table (one chunked pass)
# Uses from earlier cells: MIRNA_FILE, BASE, pairs_all, pairs_s1, pairs_s2, c_all (Cell 29); num (Cell 30);
# CHR2NC (Cell 31); pf, thr (Cell 32); PROCESSED_DIR (Cell 1).
CHUNK = 1_000_000                                      # lower this if memory is short
USE = ["lineID", "chromosome", "start", "end", "strand", "miRNAid", "miRNAname", "tdmdScore",
       "clipExpNum", "bindClipExpNum", "txIDcat", "consAve", "targetSeq"]
AGG = {"n_sites": "sum", "n_priv": "sum", "clip_max": "max", "clip_priv_max": "max", "bind_max": "max",
       "tdmd_max": "max", "cons_max": "max", "k_min": "min", "n_len_bad": "sum"}
DT  = {"n_sites": "int32", "n_priv": "int32", "clip_max": "int16", "clip_priv_max": "int16", "bind_max": "int16",
       "tdmd_max": "float32", "cons_max": "float32", "k_min": "int16", "n_len_bad": "int32"}

def combine(frames):
    """Merge partial edge tables: counts add up, maxima and minima are taken over the parts."""
    return pd.concat(frames).groupby(level=0, sort=False).agg(AGG).astype(DT)

cnt = dict(records=0, rows=0, repeats=0, len_bad=0, len_bad_unmapped=0, private=0)
q = {"tdmdScore": [], "consAve": [], "site_len": []}      # exact quantiles later (5.6 million values are small)
partial, merged, loc_parts, mir_parts = [], None, [], []

reader = pd.read_csv(MIRNA_FILE, sep="\t", usecols=USE, chunksize=CHUNK, keep_default_na=False, na_values=["NA"])
for i, ch in enumerate(reader, 1):
    cnt["records"] += len(ch)
    ids = ch["txIDcat"].str.split(",")
    k = ids.str.len().to_numpy()
    site_len = (ch["end"] - ch["start"]).to_numpy()
    gap_free = (ch["targetSeq"].str.len() - ch["targetSeq"].str.count("-")).to_numpy()
    len_bad = gap_free != site_len                       # Cell 31: the gap-free targetSeq should be end - start long
    cnt["len_bad"] += int(len_bad.sum())
    cnt["len_bad_unmapped"] += int((len_bad & ~ch["chromosome"].isin(CHR2NC).to_numpy()).sum())
    cnt["private"] += int((k == 1).sum())
    q["tdmdScore"].append(ch["tdmdScore"].to_numpy("float32"))
    q["consAve"].append(ch["consAve"].to_numpy("float32"))
    q["site_len"].append(site_len.astype("int32"))
    mir_parts.append(ch[["miRNAid", "miRNAname"]].drop_duplicates())

    # one row per (record, circRNA); the evidence of a site applies to every circRNA listed in the record
    base = pd.DataFrame({"lineID": ch["lineID"].to_numpy(),
                         "mir": ch["miRNAid"].str[5:].astype("int64").to_numpy(),      # 'MIMAT0002833' -> 2833
                         "k": k.astype("int16"),                                       # circRNAs listed in the record
                         "clip": ch["clipExpNum"].to_numpy("int16"),
                         "bind": ch["bindClipExpNum"].to_numpy("int16"),
                         "tdmd": ch["tdmdScore"].to_numpy("float32"),
                         "cons": ch["consAve"].to_numpy("float32"),
                         "bad": len_bad.astype("int16"),
                         "chromosome": ch["chromosome"].to_numpy(),
                         "strand": ch["strand"].to_numpy(),
                         "circ": ids.to_numpy()})
    ex = base.explode("circ", ignore_index=True)
    ex["circ"] = ex["circ"].str[9:].astype("int64")      # 'hsa_circ_0009177' -> 9177 (all tokens were validated in Cell 29)
    rep = ex.duplicated(["lineID", "circ"])              # the same circRNA listed twice in one record, if it occurs
    cnt["repeats"] += int(rep.sum())
    ex = ex[~rep]
    cnt["rows"] += len(ex)
    loc_parts.append(ex[["circ", "chromosome", "strand"]].drop_duplicates())

    ex["private"]   = ex["k"].eq(1)                      # site listed for this circRNA only
    ex["clip_priv"] = ex["clip"].where(ex["private"], 0)
    key = (ex["circ"] * BASE + ex["mir"]).rename("key")  # same integer key as in Cell 29
    g = ex.groupby(key, sort=False).agg(n_sites=("clip", "size"), n_priv=("private", "sum"), clip_max=("clip", "max"),
                                        clip_priv_max=("clip_priv", "max"), bind_max=("bind", "max"),
                                        tdmd_max=("tdmd", "max"), cons_max=("cons", "max"),
                                        k_min=("k", "min"), n_len_bad=("bad", "sum")).astype(DT)
    partial.append(g)
    if len(partial) == 3:                                # merge regularly, so that memory stays bounded
        merged = combine(([merged] if merged is not None else []) + partial)
        partial = []
    print(f"  chunk {i}: {cnt['records']:,} records", end="\r")
edges = combine(([merged] if merged is not None else []) + partial).sort_index()

# ---- 1. consistency with Cell 29 -----------------------------------------------------------------------
cn, mn = edges.index.to_numpy() // BASE, edges.index.to_numpy() % BASE
print(f"records: {cnt['records']:,} | exploded rows: {cnt['rows']:,} (Cell 29: 11,882,876) | "
      f"repeated (record, circRNA) tokens removed: {cnt['repeats']:,}")
assert cnt["records"] == 5_598_984
assert np.array_equal(edges.index.to_numpy(), pairs_all), "edge keys differ from Cell 29"
assert int((edges["clip_max"] >= 1).sum()) == len(pairs_s1) and int((edges["clip_max"] >= 2).sum()) == len(pairs_s2)
print(f"edges: {len(edges):,} | clipExpNum >= 1: {len(pairs_s1):,} | >= 2: {len(pairs_s2):,} | circRNAs: {len(np.unique(cn)):,} "
      f"-> identical to Cell 29")

# ---- 2. site-length consistency (decision of Cell 31), on the full table --------------------------------
print(f"\nrecords whose gap-free targetSeq is not end - start long: {cnt['len_bad']:,} "
      f"({100 * cnt['len_bad'] / cnt['records']:.4f}%) | of which on unmapped contigs: {cnt['len_bad_unmapped']:,} | "
      f"records listing a single circRNA: {cnt['private']:,} ({100 * cnt['private'] / cnt['records']:.1f}%)")

# ---- 3. exact quantiles on the full table versus the 1% sample of Cell 29 -----------------------------------
QS = [0, .05, .25, .5, .75, .95, 1]
sample_c29 = pd.DataFrame({"tdmdScore": [-2.600, -0.169, 0.463, 0.848, 1.104, 1.433, 2.652],
                           "consAve":   [-4.345, -0.536, -0.128, 0.068, 0.440, 4.767, 8.441],
                           "site_len":  [8, 19, 21, 22, 23, 25, 123]}, index=QS)      # printed in Cell 29
full = pd.DataFrame({c: np.quantile(np.concatenate(v).astype("float64"), QS) for c, v in q.items()}, index=QS)
print("\nquantiles on all records:\n", full.round(3).to_string())
print("\nfull table minus the 1% sample of Cell 29:\n", (full - sample_c29).round(3).to_string())

# ---- 4. shared sites: does attributing a site to every listed circRNA inflate the support? ----------------
rows = []
for t in (1, 2):
    a, p = (edges["clip_max"] >= t).to_numpy(), (edges["clip_priv_max"] >= t).to_numpy()
    rows.append({"clipExpNum >=": t, "pairs (all sites)": int(a.sum()), "pairs (private sites only)": int(p.sum()),
                 "pairs supported only by shared sites": int((a & ~p).sum()),
                 "circRNAs (all sites)": len(np.unique(cn[a])), "circRNAs (private sites only)": len(np.unique(cn[p])),
                 "task circRNAs (all sites)": int(np.isin(num, np.unique(cn[a])).sum()),
                 "task circRNAs (private sites only)": int(np.isin(num, np.unique(cn[p])).sum())})
print(f"\npairs with at least one private site: {int((edges['n_priv'] > 0).sum()):,} of {len(edges):,} "
      f"({100 * (edges['n_priv'] > 0).mean():.1f}%)")
print("support when every site counts versus when only private sites count:\n",
      pd.DataFrame(rows).set_index("clipExpNum >=").T.to_string())

# ---- 5. circRNA table, with location and the identifier check of Cell 32 ----------------------------------
cl = pd.DataFrame({"cn": cn, "n_site_rows": edges["n_sites"].to_numpy(), "c1": (edges["clip_max"] >= 1).to_numpy(),
                   "c2": (edges["clip_max"] >= 2).to_numpy(), "p1": (edges["clip_priv_max"] >= 1).to_numpy()})
ct = cl.groupby("cn").agg(n_mirna=("c1", "size"), n_mirna_clip1=("c1", "sum"), n_mirna_clip2=("c2", "sum"),
                          n_mirna_private_clip1=("p1", "sum"), n_site_rows=("n_site_rows", "sum"))
locs = pd.concat(loc_parts).drop_duplicates()
assert locs["circ"].is_unique, "a circRNA with more than one (chromosome, strand) combination"
ct = ct.join(locs.set_index("circ").rename(columns={"chromosome": "enc_chrom", "strand": "enc_strand"}))
ct["enc_contig_unmapped"] = ~ct["enc_chrom"].isin(CHR2NC)
ct.insert(0, "circbase_id", "hsa_circ_" + ct.index.astype(str).str.zfill(7))
idc = pd.Series(np.select([pf["direct"] >= thr, pf["direct"] >= 0.5],
                          ["header differs, sequence agrees", "header differs, sequence mostly agrees"],
                          default="header differs, sequence not confirmed"), index=pf.index)
ct["id_check"] = ct["circbase_id"].map(idc).fillna("consistent")
ct["seq_share_found"]   = ct["circbase_id"].map(pf["direct"])       # only measured for the flagged circRNAs
ct["seq_sites_checked"] = ct["circbase_id"].map(pf["n_sites"])
print("\nid_check:\n", ct["id_check"].value_counts().to_string())

# ---- 6. edge table and miRNA table ---------------------------------------------------------------------
ed = pd.DataFrame({"circbase_id": ct["circbase_id"].reindex(cn).to_numpy(),
                   "mirna_id": ("MIMAT" + pd.Series(mn).astype(str).str.zfill(7)).to_numpy()})
ed = ed.join(edges.reset_index(drop=True).rename(columns={"n_priv": "n_sites_private", "clip_priv_max": "clip_max_private",
                                                           "n_len_bad": "n_sites_len_bad"}))
ed["id_check"] = ct["id_check"].reindex(cn).to_numpy()

mt = pd.concat(mir_parts).drop_duplicates().sort_values("miRNAid").reset_index(drop=True)
mnum = mt["miRNAid"].str[5:].astype("int64")
mt["n_circ"] = mnum.map(pd.Series(mn).value_counts()).to_numpy()
mt["n_circ_clip1"] = mnum.map(pd.Series(mn[(edges["clip_max"] >= 1).to_numpy()]).value_counts()).fillna(0).astype(int).to_numpy()

# ---- 7. save ---------------------------------------------------------------------------------------------
F_EDGES, F_CIRC, F_MIR = (PROCESSED_DIR / n for n in
                          ("encori_mirna_edges_all.csv.gz", "encori_circ_table.csv", "encori_mirna_table.csv"))
ed.to_csv(F_EDGES, index=False, float_format="%.4f", compression="gzip")
ct.to_csv(F_CIRC, index=False, float_format="%.4f")
mt.to_csv(F_MIR, index=False)
for f, d in [(F_EDGES, ed), (F_CIRC, ct), (F_MIR, mt)]:
    print(f"saved {f.name}: {len(d):,} rows x {d.shape[1]} columns | {f.stat().st_size / 1e6:,.1f} MB")
print("\nedge table, first rows:\n", pd.read_csv(F_EDGES, nrows=3).to_string(index=False))

records: 5,598,984 | exploded rows: 11,882,876 (Cell 29: 11,882,876) | repeated (record, circRNA) tokens removed: 0
edges: 5,490,359 | clipExpNum >= 1: 1,866,819 | >= 2: 1,175,226 | circRNAs: 91,050 -> identical to Cell 29

records whose gap-free targetSeq is not end - start long: 353 (0.0063%) | of which on unmapped contigs: 0 | records listing a single circRNA: 3,688,198 (65.9%)

quantiles on all records:
       tdmdScore  consAve  site_len
0.00     -2.680   -5.188       8.0
0.05     -0.173   -0.537      19.0
0.25      0.468   -0.128      21.0
0.50      0.849    0.070      22.0
0.75      1.104    0.436      23.0
0.95      1.430    4.757      25.0
1.00      3.578    9.199   56279.0

full table minus the 1% sample of Cell 29:
       tdmdScore  consAve  site_len
0.00     -0.080   -0.843       0.0
0.05     -0.004   -0.001       0.0
0.25      0.005   -0.000       0.0
0.50      0.001    0.002       0.0
0.75      0.000   -0.004       0.0
0.95     -0.003   -0.010       0.0
1.00      0.926   

**Interpretation.** *(A) Consistency with Cell 29.* The pass reads 5,598,984 records and produces 11,882,876 exploded rows; no (record, circRNA) token is repeated. The edge keys are identical to those of Cell 29: 5,490,359 edges over 91,050 circRNAs, of which 1,866,819 (34.0%) have a site with `clipExpNum ≥ 1` and 1,175,226 (21.4%) one with `clipExpNum ≥ 2`. The 3,688,198 records that list a single circRNA agree with the count of Cell 29, and make up 31.0% of the exploded rows. *(B) Quantiles on the full table.* Between the 5th and 95th percentile the full-table quantiles differ from the 1% sample of Cell 29 by at most 0.005 for `tdmdScore`, 0.010 for `consAve` and 0 nt for site length, so the sample was representative. Only the extremes differ, as expected for order statistics: the `tdmdScore` maximum is 3.578 against 2.652, and the site length ranges from 8 to 56,279 nt (sample: 8 to 123 nt; median 22 nt in both). A site of 56,279 nt cannot be a miRNA binding site, and its `targetSeq` is far shorter than its interval, so such records belong to the 353 length-inconsistent ones (inference; the records are listed in Cell 33, part 2). The first run of this cell stored the lengths as 16-bit integers, which printed a minimum of −9,258 and a maximum of 145 (lengths of 32,768 nt or more wrap around); the cell was corrected and re-run, and the saved tables were not affected. *(C) Length-inconsistent records.* 353 records (0.0063%) have a gap-free `targetSeq` whose length differs from `end − start`; none lies on an unmapped contig. The extrapolation of Cell 31 (about 600) was based on 12 mismatches in a 2% sample, some of which were repeats at one locus, so the full count is of the same order and lower, which is compatible with sampling noise (not tested formally). *(D) Dependence on shared sites.* Only 211,555 of the 1,866,819 pairs with `clipExpNum ≥ 1` (11.3%) have a CLIP-supported site that lists this circRNA alone; 1,655,264 (88.7%) are supported only by shared sites. For `clipExpNum ≥ 2` the shares are 121,438 (10.3%) and 1,053,788 (89.7%). At the circRNA level the number with support falls from 70,347 to 11,166 (15.9%) at ≥ 1 and from 56,491 to 8,026 (14.2%) at ≥ 2. Among the task circRNAs it falls from 1,094 to 329 (30.1%) and from 875 to 241 (27.5%); the task set depends less on shared sites than ENCORI as a whole, but still loses about 70%. The task counts (1,094; 875) agree with Cell 30. *(E) Identifier check.* `id_check` holds 90,768 `consistent`, 192 `header differs, sequence agrees`, 89 `header differs, sequence mostly agrees` and 1 `header differs, sequence not confirmed`, i.e. 282 flagged identifiers, as in Cell 32. *(F) Files.* `encori_mirna_edges_all.csv.gz` (5,490,359 × 12, 36.9 MB), `encori_circ_table.csv` (91,050 × 12, 5.2 MB) and `encori_mirna_table.csv` (642 × 4) were written; the first rows of the edge table have the expected columns.

**Not checked.** (i) How many records are longer than 100 nt or than 32,768 nt, and whether all of them are among the 353 length-inconsistent records (listed in part 2). (ii) How many edges contain at least one length-inconsistent site, and whether their `clipExpNum` support depends on it (`n_sites_len_bad` is stored but was not tabulated). (iii) Whether the sharing is explained by overlapping isoforms of one host gene (inference). (iv) `k_min` is the minimum over all sites of a pair, so it cannot test a cap on CLIP-supported sites.

**Decision.** (i) The three tables are kept as saved; the int16 artefact affects only the printed quantiles. (ii) Site length is not used as evidence; records with an implausible length stay in the tables and are flagged through n_sites_len_bad (see iii).. (iv) The shared-site dependence is reported with every evidence threshold, and the private-site columns (`clip_max_private`, `n_sites_private`) stay in the edge table so that a "private sites only" variant can be used as a sensitivity analysis. (v) Because `k_min` cannot support a cap on CLIP-supported sites, Cell 33 (part 2) adds `k_min_clip1` and `k_min_clip2` to the edge table; the thresholds and the cap are proposed afterwards from these columns, not copied from the old Cells 41 and 43.

### 33b. Long Site Records and the Number of circRNAs Listed per CLIP-Supported Site

**Objective.** Repeat the pass of Cell 33 without the alignment strings in order to (i) list the records whose site is longer than 60 nt (by length band, and the ten longest) and confirm the site-length quantiles of Cell 33 (8 to 56,279 nt), (ii) tabulate how often CLIP support occurs by the number of circRNAs listed in a record, (iii) store for every edge the smallest number of circRNAs listed in a record with `clipExpNum ≥ 1` (`k_min_clip1`) and with `clipExpNum ≥ 2` (`k_min_clip2`), and (iv) show how the number of supported edges, circRNAs and task circRNAs falls when a site may list at most 1, 2, 3, 5, 10 or 25 circRNAs. Cell 33 stored only the minimum over all sites of an edge (`k_min`), which cannot answer (iv) for supported sites.

In [69]:
# Cell 33 (part 2) - exact site lengths (int32) and the number of circRNAs listed per CLIP-supported site
# Uses from earlier cells: MIRNA_FILE, BASE, pairs_all (Cell 29); num (Cell 30); F_EDGES (Cell 33).
# Dry run: the edge table is rewritten only if SAVE = True (set it only after approval).
SAVE  = True 
CHUNK = 1_000_000                                      # lower this if memory is short
NONE  = 9999                                           # sentinel: no site with the required support (k never exceeds 220)
USE2  = ["lineID", "chromosome", "start", "end", "strand", "miRNAid", "clipExpNum", "txIDcat"]   # no targetSeq / align
KB_BINS, KB_LAB = [2, 3, 4, 6, 11], ["1", "2", "3", "4-5", "6-10", ">= 11"]

def combine2(frames):
    """Merge partial tables: the minimum is taken over the parts."""
    return pd.concat(frames).groupby(level=0, sort=False).min().astype("int16")

lens, tail, partial, merged = [], [], [], None
kb = {"records": np.zeros(6, "int64"), "clip>=1": np.zeros(6, "int64"), "clip>=2": np.zeros(6, "int64")}
n_rec = 0
reader = pd.read_csv(MIRNA_FILE, sep="\t", usecols=USE2, chunksize=CHUNK, keep_default_na=False, na_values=["NA"])
for i, ch in enumerate(reader, 1):
    n_rec += len(ch)
    ids  = ch["txIDcat"].str.split(",")
    k    = ids.str.len().to_numpy()                                   # circRNAs listed in the record
    clip = ch["clipExpNum"].to_numpy("int16")
    site_len = (ch["end"] - ch["start"]).to_numpy("int64")            # no downcast to int16 (Cell 33 wrapped here)
    lens.append(site_len.astype("int32"))
    big = site_len > 60                                               # far above the 95th percentile (25 nt)
    if big.any():
        t = ch.loc[big, ["lineID", "chromosome", "start", "end", "strand", "miRNAid", "clipExpNum"]].copy()
        t["site_len"], t["k"] = site_len[big], k[big]
        tail.append(t)
    # records by number of circRNAs listed, and by CLIP support
    b = np.digitize(k, KB_BINS)
    kb["records"] += np.bincount(b, minlength=6)
    kb["clip>=1"] += np.bincount(b[clip >= 1], minlength=6)
    kb["clip>=2"] += np.bincount(b[clip >= 2], minlength=6)
    # one row per (record, circRNA); per edge, the smallest k among the sites that carry the required support
    base = pd.DataFrame({"mir": ch["miRNAid"].str[5:].astype("int64").to_numpy(), "k": k.astype("int16"),
                         "clip": clip, "circ": ids.to_numpy()})
    ex = base.explode("circ", ignore_index=True)
    ex["circ"] = ex["circ"].str[9:].astype("int64")
    key = (ex["circ"] * BASE + ex["mir"]).rename("key")               # same integer key as in Cells 29 and 33
    ex["k1"] = ex["k"].where(ex["clip"] >= 1, NONE)
    ex["k2"] = ex["k"].where(ex["clip"] >= 2, NONE)
    partial.append(ex.groupby(key, sort=False).agg(k_min_clip1=("k1", "min"), k_min_clip2=("k2", "min")).astype("int16"))
    if len(partial) == 3:                                             # merge regularly, so that memory stays bounded
        merged = combine2(([merged] if merged is not None else []) + partial)
        partial = []
    print(f"  chunk {i}: {n_rec:,} records", end="\r")
kmin = combine2(([merged] if merged is not None else []) + partial).sort_index()
assert n_rec == 5_598_984
assert np.array_equal(kmin.index.to_numpy(), pairs_all), "edge keys differ from Cell 29"

# ---- 1. exact site-length quantiles and extremes ----------------------------------------------------------
L, QS = np.concatenate(lens), [0, .05, .25, .5, .75, .95, 1]
print("\nsite length (end - start), all records:\n", pd.Series(np.quantile(L, QS), index=QS).to_string())
print("interior quantiles identical to Cell 33:",
      np.array_equal(np.quantile(L, QS[1:-1]), [19, 21, 22, 23, 25]))
tl = pd.concat(tail) if tail else pd.DataFrame(columns=["site_len"])
bands = pd.cut(tl["site_len"], [60, 99, 199, 999, 32767, np.inf],
               labels=["61-99", "100-199", "200-999", "1,000-32,767", ">= 32,768"])
print(f"\nrecords longer than 60 nt: {len(tl):,}\n", bands.value_counts().sort_index().to_string())
# a site of >= 32,768 nt cannot have a gap-free targetSeq of that length, so each such record is among the 353 of Cell 33
assert int((L >= 32768).sum()) <= 353, "more long records than length-inconsistent records in Cell 33"
print("\nlongest records:\n", tl.nlargest(10, "site_len").to_string(index=False))

# ---- 2. CLIP support by the number of circRNAs listed in the record ---------------------------------------
kt = pd.DataFrame(kb, index=KB_LAB)
kt["% clip>=1"] = (100 * kt["clip>=1"] / kt["records"]).round(1)
kt["% clip>=2"] = (100 * kt["clip>=2"] / kt["records"]).round(1)
print("\nrecords by number of circRNAs listed in the record:\n", kt.to_string())

# ---- 3. support when a site may list at most `cap` circRNAs ----------------------------------------------
cn, rows = kmin.index.to_numpy() // BASE, []
for t, col in ((1, "k_min_clip1"), (2, "k_min_clip2")):
    kk = kmin[col].to_numpy()
    for cap in (1, 2, 3, 5, 10, 25, 220):                             # 220 = the maximum in the file, i.e. no cap
        m = kk <= cap                                                 # the sentinel never passes
        u = np.unique(cn[m])
        rows.append({"clipExpNum >=": t, "cap": cap if cap < 220 else "none", "pairs": int(m.sum()),
                     "circRNAs": len(u), "task circRNAs": int(np.isin(num, u).sum())})
cap_tab = pd.DataFrame(rows)
print("\nsupport when a site may list at most `cap` circRNAs:\n", cap_tab.to_string(index=False))
# consistency with Cell 33: cap = 1 is the private-site variant, no cap is the all-sites variant
exp = {(1, 1): (211_555, 11_166, 329), (2, 1): (121_438, 8_026, 241),
       (1, "none"): (1_866_819, 70_347, 1_094), (2, "none"): (1_175_226, 56_491, 875)}
for (t, cap), v in exp.items():
    r = cap_tab[(cap_tab["clipExpNum >="] == t) & (cap_tab["cap"].astype(str) == str(cap))].iloc[0]
    assert (r["pairs"], r["circRNAs"], r["task circRNAs"]) == v, f"clip>={t}, cap {cap}: differs from Cell 33"

# ---- 4. append the two columns to the edge table (dry run unless SAVE = True) ------------------------------
ed = pd.read_csv(F_EDGES)
ek = ed["circbase_id"].str[9:].astype("int64").to_numpy() * BASE + ed["mirna_id"].str[5:].astype("int64").to_numpy()
assert np.array_equal(ek, kmin.index.to_numpy()), "row order of the edge table differs from the keys"
for t, col in ((1, "k_min_clip1"), (2, "k_min_clip2")):
    s = pd.Series(kmin[col].to_numpy(), dtype="Int16")
    ed[col] = s.mask(s >= NONE)                                       # empty = no site with clipExpNum >= t
    assert ed[col].notna().eq(ed["clip_max"] >= t).all(), f"{col}: support differs from clip_max"
    assert (ed[col] == 1).eq(ed["clip_max_private"] >= t).all(), f"{col} == 1 differs from clip_max_private"
print(f"\nedge table with the new columns: {ed.shape[0]:,} rows x {ed.shape[1]} columns (checks passed)")
if SAVE:
    ed.to_csv(F_EDGES, index=False, float_format="%.4f", compression="gzip")
    print(f"saved {F_EDGES.name}: {F_EDGES.stat().st_size / 1e6:,.1f} MB")
else:
    print("dry run: nothing written (SAVE = False)")

  chunk 6: 5,598,984 records
site length (end - start), all records:
 0.00        8.0
0.05       19.0
0.25       21.0
0.50       22.0
0.75       23.0
0.95       25.0
1.00    56279.0
interior quantiles identical to Cell 33: True

records longer than 60 nt: 227
 site_len
61-99             0
100-199         223
200-999           0
1,000-32,767      0
>= 32,768         4

longest records:
  lineID chromosome     start       end strand      miRNAid  clipExpNum  site_len  k
2296552       chr7  73027076  73083355      + MIMAT0014989           0     56279  1
2296553       chr7  73027076  73083355      + MIMAT0014989           0     56279  1
2296554       chr7  73027077  73083355      + MIMAT0027650           0     56278  1
2296555       chr7  73027077  73083355      + MIMAT0027650           0     56278  1
 177008       chr1 146944885 146945030      + MIMAT0019868           0       145  1
 177010       chr1 146944887 146945030      + MIMAT0019899           0       143  1
 177013       chr1 1469

**Interpretation.** *(A) Site length.* The quantiles reproduce Cell 33 (8, 19, 21, 22, 23, 25 and 56,279 nt; interior quantiles identical). Only 227 of 5,598,984 records (0.004%) have an interval longer than 60 nt: 223 of 100–199 nt and 4 of at least 32,768 nt; none lies between 61–99 nt or between 200 and 32,767 nt. The four long records are consecutive lines (2,296,552–2,296,555) at one locus on chr7 (start 73,027,076 or 73,027,077, end 73,083,355, `+`), two records for each of two miRNAs (56,279 and 56,278 nt). They are the source of the −9,258 and the 145 of the first run. The six longest records below 32,768 nt (142–145 nt) lie in a 161-nt window on chr1 (146,944,885–146,945,046). All ten printed records have `clipExpNum = 0` and list one circRNA. A 56 kb interval cannot be a miRNA site and cannot match a `targetSeq` of about 22 nt, so these four are presumably among the 353 length-inconsistent records (inference; `targetSeq` was not read in this pass). The four chr7 records give no CLIP support to any edge. *(B) Support by the number of circRNAs listed.* Single-circRNA records are 58.4% of the supported records (918,198 of 1,571,573) but yield only 11.3% of the supported pairs, which is consistent with the expansion, because a record listing k circRNAs contributes to k pairs (the number of supported rows was not counted). Every group with two or more circRNAs has a higher share of supported records than single-circRNA records (29.4–44.2% against 24.9% at `clipExpNum ≥ 1`; 17.3–29.6% against 15.0% at ≥ 2), but the relation is not monotone (peaks at 3 and at 11 or more), so the number of listed circRNAs is not a simple measure of evidence strength. *(C) Effect of a cap.* An edge passes a cap when at least one supported site lists at most that many circRNAs; `cap = 1` reproduces the private-site numbers of Cell 33 and `none` the all-site numbers (asserted).

| cap | ≥ 1: pairs | circRNAs | task | ≥ 2: pairs | circRNAs | task |
|---|---|---|---|---|---|---|
| 1 | 211,555 (11.3%) | 11,166 (15.9%) | 329 (30.1%) | 121,438 (10.3%) | 8,026 (14.2%) | 241 (27.5%) |
| 2 | 376,705 (20.2%) | 19,530 (27.8%) | 424 (38.8%) | 219,081 (18.6%) | 14,256 (25.2%) | 316 (36.1%) |
| 3 | 523,606 (28.0%) | 26,841 (38.2%) | 516 (47.2%) | 307,618 (26.2%) | 19,728 (34.9%) | 386 (44.1%) |
| 5 | 770,077 (41.3%) | 38,540 (54.8%) | 704 (64.4%) | 461,363 (39.3%) | 28,978 (51.3%) | 542 (61.9%) |
| 10 | 1,152,170 (61.7%) | 54,754 (77.8%) | 904 (82.6%) | 699,159 (59.5%) | 42,191 (74.7%) | 703 (80.3%) |
| 25 | 1,574,770 (84.4%) | 66,795 (95.0%) | 1,059 (96.8%) | 969,391 (82.5%) | 52,980 (93.8%) | 841 (96.1%) |
| none | 1,866,819 | 70,347 | 1,094 | 1,175,226 | 56,491 | 875 |

Pairs fall faster than circRNAs: a cap of 10 removes 38.3% of the pairs at ≥ 1 (40.5% at ≥ 2) but only 17.4% of the task circRNAs at ≥ 1 (19.7% at ≥ 2). A cap of 3 or lower costs about half of the task circRNAs. At every cap the task circRNAs keep a larger share than ENCORI as a whole (for example 80.3% against 74.7% at cap 10, ≥ 2). The reason was not examined. *(D) Edge table.* `encori_mirna_edges_all.csv.gz` now has 5,490,359 rows × 14 columns (37.7 MB). The checks passed: `k_min_clip1` and `k_min_clip2` are present exactly where `clip_max ≥ 1` and `≥ 2`, and equal 1 exactly where `clip_max_private` is at least 1 and 2.

**Not checked.** (i) Which edges the four chr7 records belong to (they can affect `tdmd_max` and `cons_max` of those edges), and whether any of the 223 intervals of 100–199 nt gives CLIP support (only ten records were printed). (ii) Whether all 227 long records are among the 353 length-inconsistent ones; if they are, at least 126 of the 353 have an interval of at most 60 nt and differ from `targetSeq` by a few nucleotides, as in Cell 31. (iii) Why support is not monotone in the number of listed circRNAs, and whether sites listed for many circRNAs lie in exons shared by isoforms of one host gene (inference). (iv) The number of positive pairs for the task circRNAs under each cap: the table counts pairs over all 91,050 circRNAs. (v) Whether a cap reduces sibling leakage; this is tested when the labels are rebuilt (old Cells 44 and 46).

**Decision.** (i) Long intervals are kept and flagged through `n_sites_len_bad`; no edge depends on them for CLIP support. If `tdmd_max` or `cons_max` are used as features, the edges with `n_sites_len_bad > 0` are examined in a sensitivity check. (ii) `k_min_clip1` and `k_min_clip2` stay in the edge table, and no cap is fixed here. (iii) Three evidence variants are carried forward and compared when the label tables are rebuilt (old Cell 43): permissive (`clipExpNum ≥ 2`, no cap: 1,175,226 pairs, 875 task circRNAs), capped (`≥ 2`, cap 10: 699,159 pairs, 703 task circRNAs) and strict (`≥ 2`, private sites only: 121,438 pairs, 241 task circRNAs). The choice depends on the positives per task circRNA and on the sibling test, not on the thresholds of the old Cells 41 and 43. (iv) The threshold `≥ 1` is kept as a sensitivity analysis only: it adds 691,593 pairs but only 219 task circRNAs (1,094 against 875).

### 34. Peek at the ENCORI RBP Cluster Table

**Objective.** Inspect the CLIP-cluster table on a systematic 1% sample (every 100th line, because the file is sorted by RBP) and on the first 200,000 lines, in order to establish (i) the relation between `clusterClipExpNum`, `clipExpNum`, `clusterClipSiteNum` and the number of identifiers in `clipID`, so that the evidence column is chosen correctly, (ii) whether the circBase identifiers of `txIDcat` are well formed and agree with `txInfo`, and what the region labels of `txInfo` contain, (iii) the width of the clusters and the distribution of `pvalue`, and (iv) whether one cluster (RBP and `clusterID`) is repeated on several lines, which decides how an edge must be keyed in the full pass of Cell 35.

In [70]:
# Cell 34 - Peek at the ENCORI RBP cluster table: a systematic 1% sample and the first 200,000 lines
# Uses from earlier cells: ENC_RBP_DIR (Cell 1). Read-only: nothing is saved.
import re
RBP_FILE = ENC_RBP_DIR / "hg38_RBPcircRNA_cluster.txt"
STEP = 100          # keep the header and every 100th data line; the file is sorted by RBP, so its first rows are not representative
NHEAD = 200_000     # contiguous block, to see whether a cluster is repeated on neighbouring lines

rb = pd.read_csv(RBP_FILE, sep="\t", skiprows=lambda i: i > 0 and i % STEP != 0,
                 keep_default_na=False, na_values=["NA"])

# ---- 1. size, missing values, sampling check --------------------------------------------------------------
print(f"sampled records: {len(rb):,} (expected about {4_669_990 // STEP:,}) | lineID unique: {rb['lineID'].is_unique}")
print("missing values per column:", rb.isna().sum()[lambda s: s > 0].to_dict() or "none")
print(f"distinct RBPs: {rb['RBP'].nunique()} | first letters of the RBP names: {rb['RBP'].str[0].str.upper().nunique()} different")
per_rbp = rb["RBP"].value_counts()
print("records per RBP, largest 6:", per_rbp.head(6).to_dict(), "| smallest 3:", per_rbp.tail(3).to_dict())

# ---- 2. cluster identifier and evidence columns -------------------------------------------------------------
print(f"\nclusterID starts with '<RBP>:': {rb['clusterID'].str.split(':').str[0].eq(rb['RBP']).mean():.4f}")
n_clipid = rb["clipID"].str.count(",") + 1                       # number of experiment identifiers listed
ev = {"clusterClipExpNum == clipExpNum": rb["clusterClipExpNum"].eq(rb["clipExpNum"]),
      "clusterClipExpNum >= clipExpNum": rb["clusterClipExpNum"].ge(rb["clipExpNum"]),
      "clusterClipSiteNum == clipExpNum": rb["clusterClipSiteNum"].eq(rb["clipExpNum"]),
      "number of clipID tokens == clipExpNum": n_clipid.eq(rb["clipExpNum"]),
      "number of clipID tokens == clusterClipExpNum": n_clipid.eq(rb["clusterClipExpNum"])}
print("\nrelations between the evidence columns (share of records):")
for name, v in ev.items():
    print(f"  {name:46s} {v.mean():.4f}")
print("\nclipExpNum (5 = 5 or more):\n", rb["clipExpNum"].clip(upper=5).value_counts().sort_index().to_string())
print("\npvalue quantiles (a negative value looks like a log10 p-value; not verified):\n",
      rb["pvalue"].quantile([0, .05, .25, .5, .75, .95, 1]).round(2).to_string())

# ---- 3. coordinates ---------------------------------------------------------------------------------------------
w = (rb["narrowEnd"] - rb["narrowStart"]).astype("int64")
print(f"\nnarrow start >= end: {int((rb['narrowStart'] >= rb['narrowEnd']).sum())} | narrow inside broad: "
      f"{((rb['narrowStart'] >= rb['broadStart']) & (rb['narrowEnd'] <= rb['broadEnd'])).mean():.4f}")
print("narrow width (nt) quantiles:\n", w.quantile([0, .05, .25, .5, .75, .95, 1]).to_string())
print("strand:", rb["strand"].value_counts().to_dict(), "| chromosome names with '_':", int(rb["chromosome"].str.contains("_").sum()))

# ---- 4. gene annotation -----------------------------------------------------------------------------------------
print("\ngeneType:", rb["geneType"].value_counts().to_dict())
print("geneID prefix:", rb["geneID"].str.extract(r"^([A-Za-z]+)")[0].value_counts(dropna=False).head(6).to_dict())

# ---- 5. circRNA identifiers: how many per cluster, and are all tokens well formed? ---------------------------------
ids = rb["txIDcat"].str.split(",")
k = ids.str.len()
print("\ncircRNAs per cluster: quantiles", k.quantile([.5, .9, .99, 1]).astype(int).to_dict(),
      "| single:", f"{k.eq(1).mean():.3f}")
tok = ids.explode()
bad = ~tok.str.match(r"^hsa_circ_\d{7}$")
print(f"tokens that are not 'hsa_circ_' + 7 digits: {int(bad.sum())} of {len(tok):,} (examples: {tok[bad].unique()[:5].tolist()})")

# ---- 6. txInfo: format, agreement with txIDcat, region labels -------------------------------------------------------
def info_ids(s):
    """circBase identifiers written first in each '|'-separated token of txInfo."""
    return [m.group(0) for t in s.split("|") if (m := re.search(r"hsa_circ_\d{7}", t))]

info = rb["txInfo"].str.split("|")
same_n = info.str.len().eq(k)
same_ids = pd.Series([info_ids(s) == l for s, l in zip(rb["txInfo"], ids)], index=rb.index)
print(f"\ntxInfo has as many tokens as txIDcat has circRNAs: {same_n.mean():.4f} | first identifiers identical in order: {same_ids.mean():.4f}")
parts = info.explode().str.split(":")
print("fields per txInfo token:", parts.str.len().value_counts().to_dict())
print("last field (region type):", parts.str[-1].value_counts().head(8).to_dict())
print("label before it, number removed:", parts.str[-2].str.replace(r"-?\d+$", "", regex=True).value_counts().head(8).to_dict())
print("examples:", info.iloc[0][:2], "|", info.iloc[len(info) // 2][:2])

# ---- 7. is a cluster repeated on neighbouring lines? (contiguous block at the top of the file) ----------------------
hd = pd.read_csv(RBP_FILE, sep="\t", nrows=NHEAD, usecols=["lineID", "RBP", "clusterID", "geneName", "txIDcat"], keep_default_na=False)
dup = hd.duplicated(["RBP", "clusterID"], keep=False)
print(f"\nfirst {len(hd):,} lines: {hd['RBP'].nunique()} RBPs, {hd[['RBP','clusterID']].drop_duplicates().shape[0]:,} distinct (RBP, clusterID) | "
      f"lines whose (RBP, clusterID) occurs more than once: {int(dup.sum()):,}")
print(hd[dup].sort_values(["RBP", "clusterID", "lineID"]).head(6).to_string(index=False))

sampled records: 46,699 (expected about 46,699) | lineID unique: True
missing values per column: none
distinct RBPs: 264 | first letters of the RBP names: 24 different
records per RBP, largest 6: {'ELAVL1': 2730, 'RBMX': 1884, 'HNRNPC': 1683, 'IGF2BP2': 1333, 'U2AF1': 1313, 'CSTF2T': 1202} | smallest 3: {'TRMT13': 1, 'TUT1': 1, 'WRN': 1}

clusterID starts with '<RBP>:': 1.0000

relations between the evidence columns (share of records):
  clusterClipExpNum == clipExpNum                0.2383
  clusterClipExpNum >= clipExpNum                0.2383
  clusterClipSiteNum == clipExpNum               0.2110
  number of clipID tokens == clipExpNum          0.2110
  number of clipID tokens == clusterClipExpNum   0.8581

clipExpNum (5 = 5 or more):
 clipExpNum
1     6098
2     9937
3     5035
4     5934
5    19695

pvalue quantiles (a negative value looks like a log10 p-value; not verified):
 0.00   -78841.80
0.05      -59.55
0.25      -12.72
0.50       -6.47
0.75       -4.26
0.95       -3.02
1.

**Interpretation.** *(A) Sample.* The systematic sample has 46,699 records (as expected), unique `lineID`s and no missing values. It contains 264 different RBPs, not only the six of the old API block (EIF4A3, FUS, IGF2BP1–3, QKI). The largest are ELAVL1 (2,730 records, 5.8%), RBMX (1,884), HNRNPC (1,683), IGF2BP2 (1,333), U2AF1 (1,313) and CSTF2T (1,202); the smallest have a single sampled record (TRMT13, TUT1, WRN). `clusterID` begins with the RBP name in 100% of the records. *(B) Evidence columns.* `clusterClipExpNum` is never larger than `clipExpNum` (the share of records with `≥` equals the share with `=`, 23.8%), so it is strictly smaller in 76.2%. The number of identifiers in `clipID` equals `clusterClipExpNum` in 85.8% of the records but `clipExpNum` in only 21.1%. `clipExpNum` is therefore not the number of experiments listed for the cluster itself; it is larger and may count over a wider region (inference, not tested). `clipExpNum` is 1 in 13.1% of the records, 2 in 21.3%, 3 in 10.8%, 4 in 12.7% and 5 or more in 42.2%, so 86.9% pass a threshold of 2 and that threshold hardly separates anything. This contrasts with the QKI table of the API (Cell 28), where 72.6% of the clusters rested on one experiment; the figure may correspond to `clusterClipExpNum` of this file (hypothesis; tested in part 2). *(C) `pvalue` and coordinates.* All `pvalue` values are negative (median −6.47; 5th percentile −59.55; minimum −78,841.8; maximum −1.43), a heavy-tailed score whose scale is not known. No record has `start ≥ end`, and the narrow region lies inside the broad region in 100%. The narrow width has a median of 40 nt (25th percentile 40, 75th 54, 95th 261, maximum 6,719), and at least 5% of the clusters are 1 nt wide. Strands are balanced (51.2% `+`, 48.8% `−`); the sample has no contig name with an underscore. *(D) Gene annotation and repeated clusters.* `geneType` is `circRNA` in all records. `geneID` is an NM accession in 91.3%, an NR accession in 3.9%, an Arraystar-type alias (`hsa…`) in 4.1% and `TCONS` in 0.8%. A record is not a cluster: in the first 200,000 lines (27 RBPs), 173,810 distinct (RBP, `clusterID`) pairs occupy 200,000 lines, and 47,960 lines (24.0%) belong to a cluster that occurs on more than one line. By subtraction, about 21,770 clusters (12.5%) are repeated, on 2.2 lines on average. The printed examples show the same cluster with different host genes (GAS5 and SNORD44; N4BP2L2 twice with different circRNA sets; an alias), so a line seems to be one cluster and one host transcript (inference from six printed lines). *(E) circRNAs per record.* A record lists a median of 2 circRNAs (90th percentile 10, 99th 29, maximum 217), and 38.4% list one; the 198,609 tokens of the sample (4.25 per record) are all `hsa_circ_` plus seven digits. Because a cluster can be spread over several lines, this count per line underestimates the number of circRNAs that share the cluster. *(F) `txInfo`.* All 198,609 tokens have four fields (`id:id:label:type`), and `txInfo` has as many tokens as `txIDcat` in 100% of the records. The first identifiers agree in the same order in only 70.8%, so in 29.2% the order or the identity of the circRNAs differs; which of the two is not yet known (part 2). The region type is `intron` in 49.4% of the tokens, `CDS` in 33.1%, `CDS,intron` in 7.9%, `3'UTR` in 5.8%, `5'UTR` in 1.3% and `Exon` in 1.3% (99.3% together with two further `UTR/CDS` combinations). The label before it is `intron-N` in 98,152 tokens (49.4%, the same count as the type `intron`), `Exon-N` in 84,004 (42.3%) and mixed `Exon-N,intron` labels in at least 7,804 more. About half of the (cluster, circRNA) associations therefore carry an intron label. If the label means that the cluster lies in an intron of the host transcript within the genomic interval of the circRNA (inference; the ENCORI documentation was not consulted), such a cluster is not part of the mature sequence of an exonic circRNA.

**Not checked.** (i) The documented meaning of `clipExpNum`, `clusterClipExpNum`, `clusterClipSiteNum` and of the region labels. (ii) Why 29.2% of the records differ between `txInfo` and `txIDcat`. (iii) Whether the repeated lines are one per host transcript. (iv) The sample is 1% and, for the repeats, a block of 27 RBPs, so per-RBP figures are noisy and the full-file figures may differ. (v) The reason for clusters of 1 nt and for the width of exactly 40 nt. (vi) The scale of `pvalue`. (vii) Whether the full file contains alt or unplaced contigs.

**Decision.** (i) The RBP edges are built from the full cluster file, with all 264 RBPs, in a chunked pass; the six-RBP downloads of the API block stay as reference only. (ii) An edge is a unique (RBP, circBase identifier) pair. A cluster is identified by (RBP, `clusterID`) and can span several lines, so the pass counts distinct (RBP, cluster, circRNA) triples, the repeated ones, and the number of circRNAs per cluster as the union over all its lines, not per line. (iii) The evidence column is not fixed: `clipExpNum` and `clusterClipExpNum` are both profiled in Cell 35, and the old rule `clipExpNum ≥ 2` is not carried over, because 86.9% of the records pass it. (iv) `txInfo` is parsed per token (identifier, label, type), and the support of an edge is split into exonic and intron-only tokens, in the same way as private and shared sites for the miRNA edges. Whether the identifier comes from `txInfo` or `txIDcat` is decided after part 2.

In [71]:
# Cell 34 (part 2) - txInfo versus txIDcat, what clipExpNum measures, and exonic versus intronic labels
# Uses from Cell 34: rb, ids, info, parts, same_ids, info_ids (the 1% sample in memory; no file is read; nothing is saved).

# ---- 1. do txInfo and txIDcat list the same circRNAs? ------------------------------------------------------
same_set = pd.Series([sorted(info_ids(s)) == sorted(l) for s, l in zip(rb["txInfo"], ids)], index=rb.index)
print(f"same circRNAs in the same order: {same_ids.mean():.4f} | same circRNAs, order ignored: {same_set.mean():.4f} | "
      f"records with different circRNAs: {int((~same_set).sum()):,}")
print(f"records in which txIDcat lists a circRNA twice: {int(ids.map(lambda l: len(l) != len(set(l))).sum()):,}")
for name, m in [("order only", same_set & ~same_ids), ("different circRNAs", ~same_set)]:
    for j in rb.index[m][:2]:
        print(f"  {name}: txIDcat {ids[j][:4]} | txInfo {info_ids(rb.at[j, 'txInfo'])[:4]}")

# ---- 2. what does clipExpNum measure, compared with clusterClipExpNum? --------------------------------------------
same_reg = rb["narrowStart"].eq(rb["broadStart"]) & rb["narrowEnd"].eq(rb["broadEnd"])
eq = rb["clusterClipExpNum"].eq(rb["clipExpNum"])
print(f"\nnarrow region equals broad region: {same_reg.mean():.4f}")
print(pd.crosstab(same_reg.rename("narrow == broad"), eq.rename("clusterClipExpNum == clipExpNum")).to_string())
print("\nrows clusterClipExpNum, columns clipExpNum (5 = 5 or more):")
print(pd.crosstab(rb["clusterClipExpNum"].clip(upper=5), rb["clipExpNum"].clip(upper=5)).to_string())
print(f"\nclusterClipSiteNum >= clusterClipExpNum: {rb['clusterClipSiteNum'].ge(rb['clusterClipExpNum']).mean():.4f} | "
      f"clusterClipSiteNum == clusterClipExpNum: {rb['clusterClipSiteNum'].eq(rb['clusterClipExpNum']).mean():.4f}")
q = rb[rb["RBP"] == "QKI"]
if len(q):
    print(f"QKI in the sample: {len(q)} records | single experiment by clipExpNum: {q['clipExpNum'].eq(1).mean():.3f}, "
          f"by clusterClipExpNum: {q['clusterClipExpNum'].eq(1).mean():.3f} (API table of Cell 28, clipExpNum: 0.726 of 5,571 clusters)")

# ---- 3. exonic versus intronic labels of the (cluster, circRNA) tokens -----------------------------------------------
lab = parts.str[-2]
exonic = lab.str.contains("Exon")                                  # 'Exon-3' or 'Exon-2,intron'
intr_only = ~exonic & lab.str.startswith("intron")
print(f"\ntokens: {len(lab):,} | label contains 'Exon': {exonic.mean():.4f} | intron only: {intr_only.mean():.4f} | other: {(~exonic & ~intr_only).mean():.4f}")
rec = pd.DataFrame({"any_exonic": exonic.groupby(level=0).any(), "all_intron": intr_only.groupby(level=0).all()})
print(f"records with at least one exonic token: {rec['any_exonic'].mean():.4f} | records in which every token is intron only: {rec['all_intron'].mean():.4f}")
print("region type of the exonic tokens:", parts.str[-1][exonic].value_counts().head(5).to_dict())

same circRNAs in the same order: 0.7078 | same circRNAs, order ignored: 1.0000 | records with different circRNAs: 0
records in which txIDcat lists a circRNA twice: 0
  order only: txIDcat ['hsa_circ_0058026', 'hsa_circ_0058028'] | txInfo ['hsa_circ_0058028', 'hsa_circ_0058026']
  order only: txIDcat ['hsa_circ_0008994', 'hsa_circ_0059185'] | txInfo ['hsa_circ_0059185', 'hsa_circ_0008994']

narrow region equals broad region: 1.0000
clusterClipExpNum == clipExpNum  False  True 
narrow == broad                              
True                             35572  11127

rows clusterClipExpNum, columns clipExpNum (5 = 5 or more):
clipExpNum            1     2     3     4     5
clusterClipExpNum                              
1                  6098  6738  3216  3291  9516
2                     0  3199  1173  1404  3635
3                     0     0   646   610  2027
4                     0     0     0   629  1367
5                     0     0     0     0  3150

clusterClipSiteNum >= cluster

**Interpretation.** *(A) `txInfo` versus `txIDcat`.* The two columns list the same circRNAs in all 46,699 records (same set with order ignored: 100%; records with different circRNAs: 0), and `txIDcat` never lists a circRNA twice. They differ only in order: the full lists agree in the same order in 70.8% of the records. Single-circRNA records (38.4%) agree trivially, so about 47% of the records that list two or more circRNAs are ordered differently (derived from the printed shares; the single-circRNA share is rounded). Both printed examples are pairs in reversed order. Pairing `txIDcat` with `txInfo` by position would therefore attach the label of one circRNA to another in these records whenever the labels differ. *(B) What `clipExpNum` is not.* `clusterClipExpNum` is never larger than `clipExpNum` (every cell below the diagonal of the cross-table is empty), and the two are exactly equal in 11,127 records (23.8%). The narrow region equals the broad region in 100% of the records, so the difference is not caused by two different regions, and the broad coordinates add no information in this sample. `clipExpNum` therefore appears to count experiments beyond those listed for the cluster of the line (inference; the documentation was not consulted). *(C) `clusterClipSiteNum`.* It is never smaller than `clusterClipExpNum` and equals it in 85.8% of the records, the same share as the agreement between the number of `clipID` entries and `clusterClipExpNum` (Cell 34). Against `clipExpNum` both columns also agree in the same share (21.1%). This suggests that `clusterClipSiteNum` is the number of `clipID` entries and `clusterClipExpNum` the number of distinct experiments among them, so that 14.2% of the clusters would have several sites from one experiment (inference from equal shares; not tested record by record). *(D) Distribution of the evidence.* By `clusterClipExpNum`, 61.8% of the records rest on one experiment (28,859), 20.2% on two, 7.0% on three, 4.3% on four and 6.7% on five or more. By `clipExpNum` only 13.1% rest on one (Cell 34). A threshold of 2 keeps 38.2% of the records by `clusterClipExpNum` but 86.9% by `clipExpNum`, so only `clusterClipExpNum` separates the records. For QKI (56 records) the share of single-experiment records is 71.4% by `clusterClipExpNum` and 32.1% by `clipExpNum`, against 72.6% in the API table of Cell 28. With 40 and 18 of 56 (reconstructed from the printed proportions), the Wilson intervals are about 0.59–0.82 and 0.21–0.45. The API figure is compatible with `clusterClipExpNum` and not with `clipExpNum`, which supports the hypothesis of Cell 34, but 56 records are few. *(E) Exonic and intronic labels.* Of 198,609 (cluster, circRNA) tokens, 50.6% (100,457) have a label containing `Exon` and 49.4% (98,152) are intron only; no token has another label. The two record-level shares (48.8% with at least one exonic token, 51.2% with intron-only tokens) are complementary by construction and say nothing about records that mix both kinds. The counts of the exonic region types (`CDS` 65,743; `3'UTR` 11,426; `5'UTR` 2,592; `Exon` 2,513) equal the type counts of Cell 34, so every token of these types is exonic and every token of type `intron` is intron only. A further 15,658 exonic tokens have the type `CDS,intron` (15.6% of the exonic tokens), so "exonic" here includes clusters that also touch an intron. *(F) Consequence.* About half of the associations carry an intron label only. If this label means that the cluster lies in an intron of the host transcript and not in sequence retained in the circRNA (inference; not consulted), these associations document binding to the linear transcript rather than to the mature circRNA.

**Decision.** (i) The circRNA of an association is taken from `txIDcat`. `txInfo` is parsed into a mapping from identifier to (label, type) and joined by identifier, never by position; Cell 35 asserts that the two identifier sets are equal and counts the records whose order differs. (ii) `clusterClipExpNum` is the primary support column of the RBP edges. `clipExpNum` and `clusterClipSiteNum` are stored as maxima for sensitivity analyses, not as the threshold. The threshold is not fixed: Cell 35 reports the numbers of triples, edges and task circRNAs at ≥ 1, ≥ 2 and ≥ 3, and the old rule `clipExpNum ≥ 2` is not carried over. (iii) Only the narrow coordinates are kept; the pass counts the records in which the broad region differs (expected 0). (iv) Every (RBP, cluster, circRNA) triple receives an `exonic` flag (label contains `Exon`), and every edge stores the maximum evidence over exonic triples and over all triples, together with the numbers of exonic and intron-only triples. Three evidence variants are carried forward for the label rebuild (old Cell 43): permissive (all tokens), exonic (at least one exonic token) and exonic with `clusterClipExpNum ≥ 2`. Intron-only support is not used alone as a positive label without a sensitivity test.

### 35. Profile of the ENCORI RBP Cluster Table

**Objective.** Profile the complete ENCORI RBP cluster table in a single chunked pass, with every record expanded to one token per listed circRNA, in order to establish (i) the integrity of the file and the number of records per RBP, (ii) the full-file relations between `clusterClipExpNum`, `clipExpNum`, `clusterClipSiteNum` and `clipID`, and the distributions of the evidence, `pvalue` and cluster width, (iii) whether `txIDcat` and `txInfo` list the same circRNAs in every record, and how many records differ only in order, (iv) the exonic and intronic labels of the (cluster, circRNA) tokens, (v) how many distinct (RBP, cluster, circRNA) triples the tokens represent, how many lines a cluster occupies, and whether its evidence is constant over its lines, and (vi) how many distinct circRNAs, RBPs and (RBP, circRNA) edges the table contains under five evidence variants, and whether its circRNAs are a subset of the ENCORI miRNA table.

In [73]:
# Cell 35 - Full profile of the ENCORI RBP cluster table (one chunked pass; records exploded to (cluster, circRNA) tokens)
# Uses from earlier cells: RBP_FILE (Cell 34), BASE and add (Cell 29), c_all (Cell 29). Read-only: nothing is saved.
from collections import defaultdict

CHUNK    = 500_000                                   # lower this if memory is short
EXPECTED = 4_669_990                                 # number of records used in Cell 34
USE = ["lineID", "clusterID", "RBP", "geneID", "geneType", "chromosome", "strand", "narrowStart", "narrowEnd",
       "broadStart", "broadEnd", "clipID", "clusterClipExpNum", "clipExpNum", "clusterClipSiteNum", "pvalue",
       "txIDcat", "txInfo"]
VARIANTS = ["all", "c>=2", "c>=3", "exonic", "exonic & c>=2"]   # evidence variants of an (RBP, circRNA) edge; c = clusterClipExpNum
QS = [0, .05, .25, .5, .75, .95, 1]

S = defaultdict(int)                                 # scalar counters
A = dict.fromkeys(["cexp", "clip", "xtab", "strand", "gtype", "gid", "chrom", "k", "type", "type_ex",
                   "lines_cl", "k_union", "tri_x"])  # accumulated count tables
na_cnt = pd.Series(0, index=USE, dtype="int64")
rbp_code, rbp_rows, edge_parts = {}, [], {v: [] for v in VARIANTS}
pvals, widths = [], []
last_line, last_rbp, mono = -1, None, True

def flush(rbp, buf):
    """Summarise one RBP block. The file is sorted by RBP, so a cluster never leaves its block (checked after the pass)."""
    t = pd.concat(buf, ignore_index=True)
    code = rbp_code.setdefault(rbp, len(rbp_code))                       # integer code of the RBP, used in the edge key
    cl = pd.factorize(t["cl"])[0].astype("int64")                        # cluster number within the block
    t["tkey"] = cl * BASE + t["circ"].to_numpy()                         # exact key of a (cluster, circRNA) triple

    # (cluster, circRNA) triples: repeated tokens come from clusters that occupy several lines
    g = t.groupby("tkey").agg(cexp=("cexp", "max"), ex_max=("exonic", "max"), ex_min=("exonic", "min"))
    S["tokens"] += len(t); S["triples"] += len(g); S["triples_mixed"] += int((g["ex_max"] != g["ex_min"]).sum())
    A["tri_x"] = add(A["tri_x"], g.groupby([g["ex_max"].rename("exonic"), g["cexp"].clip(upper=3).rename("cexp")]).size())

    # clusters: lines per cluster, evidence constant over the lines, circRNAs per cluster (union over all its lines)
    ln = t.drop_duplicates("line")                                       # one row per record line
    gl = ln.groupby("cl")
    lc = gl.size()
    v = gl.agg(a=("cexp", "nunique"), b=("clip", "nunique"))
    S["lines"] += len(ln); S["clusters"] += len(lc); S["clusters_rep"] += int((lc > 1).sum())
    S["cexp_varies"] += int((v["a"] > 1).sum()); S["clip_varies"] += int((v["b"] > 1).sum())
    A["lines_cl"] = add(A["lines_cl"], lc.clip(upper=5).value_counts())
    u = t.drop_duplicates("tkey").groupby("cl").size()
    A["k_union"] = add(A["k_union"], u.clip(upper=11).value_counts())
    S["k_union_max"] = max(S["k_union_max"], int(u.max()))

    # (RBP, circRNA) edges: an edge passes a variant when at least one of its tokens does
    cexp, ex, circ = t["cexp"].to_numpy(), t["exonic"].to_numpy().astype(bool), t["circ"].to_numpy()
    masks = {"all": np.ones(len(t), bool), "c>=2": cexp >= 2, "c>=3": cexp >= 3,
             "exonic": ex, "exonic & c>=2": ex & (cexp >= 2)}
    row = {"RBP": rbp, "lines": len(ln), "clusters": len(lc), "triples": len(g)}
    for name, m in masks.items():
        e = np.unique(circ[m] * BASE + code)                             # same key convention as Cell 29: circRNA * BASE + code
        edge_parts[name].append(e)
        row["circ " + name] = len(e)
    rbp_rows.append(row)

cur, buf, n_rows = None, [], 0
reader = pd.read_csv(RBP_FILE, sep="\t", usecols=USE, chunksize=CHUNK, keep_default_na=False, na_values=["NA"])
for i, ch in enumerate(reader, 1):
    n0 = n_rows; n_rows += len(ch)
    na_cnt += ch.isna().sum()

    # ---- record-level checks -------------------------------------------------------------------------------
    lid = ch["lineID"].to_numpy()
    mono = mono and bool((np.diff(lid) > 0).all()) and bool(last_line < lid[0]); last_line = lid[-1]
    r = ch["RBP"].to_numpy()
    S["rec_blocks"] += int((r[1:] != r[:-1]).sum()) + int(r[0] != last_rbp); last_rbp = r[-1]
    S["prefix"] += int(ch["clusterID"].str.split(":").str[0].eq(ch["RBP"]).sum())

    cexp, clip, site = ch["clusterClipExpNum"], ch["clipExpNum"], ch["clusterClipSiteNum"]
    ncl = ch["clipID"].str.count(",") + 1                                # number of experiment identifiers listed
    S["cexp0"] += int(cexp.lt(1).sum());      S["cexp_eq"] += int(cexp.eq(clip).sum()); S["cexp_gt"] += int(cexp.gt(clip).sum())
    S["site_ge"] += int(site.ge(cexp).sum()); S["site_eq"] += int(site.eq(cexp).sum())
    S["ncl_site"] += int(ncl.eq(site).sum()); S["ncl_cexp"] += int(ncl.eq(cexp).sum())
    c5, l5 = cexp.clip(upper=5).rename("cexp"), clip.clip(upper=5).rename("clip")
    A["cexp"] = add(A["cexp"], c5.value_counts()); A["clip"] = add(A["clip"], l5.value_counts())
    A["xtab"] = add(A["xtab"], ch.groupby([c5, l5]).size())

    width = (ch["narrowEnd"] - ch["narrowStart"]).to_numpy("int64")
    widths.append(width.astype("int32")); pvals.append(ch["pvalue"].to_numpy("float32"))
    S["narrow_bad"] += int((width <= 0).sum())
    S["in_broad"] += int((ch["narrowStart"].ge(ch["broadStart"]) & ch["narrowEnd"].le(ch["broadEnd"])).sum())
    S["narrow_eq_broad"] += int((ch["narrowStart"].eq(ch["broadStart"]) & ch["narrowEnd"].eq(ch["broadEnd"])).sum())
    for key, col in [("strand", "strand"), ("gtype", "geneType"), ("chrom", "chromosome")]:
        A[key] = add(A[key], ch[col].value_counts())
    A["gid"] = add(A["gid"], ch["geneID"].str.extract(r"^([A-Za-z]+)")[0].value_counts(dropna=False))

    # ---- tokens: txIDcat versus txInfo ---------------------------------------------------------------------
    idl = ch["txIDcat"].str.split(",");  k = idl.str.len()
    inf = ch["txInfo"].str.split("|");   k_inf = inf.str.len()
    assert (k == k_inf).all(), f"chunk {i}: txInfo and txIDcat have different numbers of tokens in some record"
    A["k"] = add(A["k"], k.clip(upper=11).value_counts())
    S["rec_multi"] += int((k >= 2).sum())
    cat, tok = idl.explode(), inf.explode()                              # same record index, same length
    pt = tok.str.split(":", expand=True)
    assert pt.shape[1] == 4, f"chunk {i}: txInfo token with an unexpected number of fields"
    cat_num = cat.str.extract(r"^hsa_circ_(\d{7})$")[0]
    inf_num = pt[0].str.extract(r"^hsa_circ_(\d{7})$")[0]
    S["tok"] += len(cat); S["tok_bad"] += int(cat_num.isna().sum()); S["info_bad"] += int(inf_num.isna().sum())
    assert S["tok_bad"] == 0 and S["info_bad"] == 0, f"chunk {i}: identifiers that are not 'hsa_circ_' + 7 digits"
    S["id_pair_diff"] += int((pt[0] != pt[1]).sum())                     # first two fields of a txInfo token differ

    recs = cat.index.to_numpy("int64")
    cn, inn = pd.to_numeric(cat_num).to_numpy("int64"), pd.to_numeric(inf_num).to_numpy("int64")
    S["set_diff_tok"] += int((np.sort(recs * BASE + cn) != np.sort(recs * BASE + inn)).sum())   # 0 = same circRNAs in every record
    S["dup_tok"] += int(pd.Series(recs * BASE + cn).duplicated().sum())  # a circRNA listed twice in one record
    ok = pd.Series(cn == inn).groupby(recs).all().reindex(ch.index).to_numpy()   # same order in this record
    S["order_diff"] += int((~ok).sum()); S["order_diff_multi"] += int((~ok & (k.to_numpy() >= 2)).sum())

    # ---- region labels (from txInfo, joined by identifier, not by position) ------------------------------------
    lab = pt[2]
    exonic = lab.str.contains("Exon")
    intr = lab.str.startswith("intron") & ~exonic
    S["tok_exonic"] += int(exonic.sum()); S["tok_intron"] += int(intr.sum()); S["tok_other"] += int((~exonic & ~intr).sum())
    A["type"] = add(A["type"], pt[3].value_counts()); A["type_ex"] = add(A["type_ex"], pt[3][exonic.to_numpy()].value_counts())

    # ---- token table, handed to the RBP block it belongs to ---------------------------------------------------
    pos = ch.index.get_indexer(tok.index)
    tk = pd.DataFrame({"circ": inn, "exonic": exonic.to_numpy().astype("int8"), "rbp": r[pos],
                       "cl": ch["clusterID"].to_numpy()[pos], "cexp": cexp.to_numpy()[pos],
                       "clip": clip.to_numpy()[pos], "line": lid[pos]})
    for rbp, gg in tk.groupby("rbp", sort=False):
        if rbp != cur:
            if cur is not None:
                flush(cur, buf)
            cur, buf = rbp, []
        buf.append(gg.drop(columns="rbp"))
    print(f"  chunk {i}: {n_rows:,} records", end="\r")
flush(cur, buf)

assert n_rows == S["lines"], "a record without a token, or a line counted twice"
assert S["rec_blocks"] == len(rbp_code), "the file is not sorted by RBP: per-RBP and per-cluster figures are not valid"

# ---- derived objects ----------------------------------------------------------------------------------------
n = n_rows
pct = lambda a, b: f"{100 * a / b:.1f}%"
rbp_df = pd.DataFrame(rbp_rows).set_index("RBP")
E = {v: np.unique(np.concatenate(edge_parts[v])) for v in VARIANTS}      # edge keys: circRNA * BASE + RBP code

# ---- 1. integrity and RBPs ----------------------------------------------------------------------------------
print(f"records: {n:,} | equals the number used in Cell 34 ({EXPECTED:,}): {n == EXPECTED} | lineID strictly increasing: {mono}")
print("missing values per column:", na_cnt[na_cnt > 0].to_dict() or "none")
print(f"records with clusterClipExpNum < 1: {S['cexp0']:,}")
print(f"RBPs: {len(rbp_code)} | blocks of consecutive records: {S['rec_blocks']} (equal = sorted by RBP)")
rr = rbp_df["lines"].sort_values(ascending=False)
print("records per RBP, largest 6:", rr.head(6).to_dict(), "| smallest 3:", rr.tail(3).to_dict(), "| median:", int(rr.median()))
print(f"clusterID starts with '<RBP>:': {S['prefix'] / n:.4f}")

# ---- 2. evidence columns --------------------------------------------------------------------------------------
print("\nrelations between the evidence columns (share of records):")
for name, v in [("clusterClipExpNum == clipExpNum", S["cexp_eq"]), ("clusterClipExpNum > clipExpNum", S["cexp_gt"]),
                ("clusterClipSiteNum >= clusterClipExpNum", S["site_ge"]), ("clusterClipSiteNum == clusterClipExpNum", S["site_eq"]),
                ("number of clipID tokens == clusterClipSiteNum", S["ncl_site"]), ("number of clipID tokens == clusterClipExpNum", S["ncl_cexp"])]:
    print(f"  {name:48s} {v / n:.4f}")
for lab_, key in [("clusterClipExpNum", "cexp"), ("clipExpNum", "clip")]:
    s = A[key].sort_index()
    print(f"\n{lab_} (5 = 5 or more), % of records:\n", (100 * s / n).round(1).to_string())
    print(f"  >= 2: {pct(s[s.index >= 2].sum(), n)} | >= 3: {pct(s[s.index >= 3].sum(), n)}")
print("\nrows clusterClipExpNum, columns clipExpNum (5 = 5 or more):\n", A["xtab"].unstack(fill_value=0).astype(int).to_string())

# ---- 3. pvalue, coordinates, annotation -----------------------------------------------------------------------
pv, w = np.concatenate(pvals), np.concatenate(widths)
print("\npvalue quantiles:\n", pd.Series(pv).quantile(QS).round(2).to_string(), f"\npvalue >= 0: {int((pv >= 0).sum())}")
print("\nnarrow width (nt) quantiles:\n", pd.Series(w).quantile(QS).to_string())
print(f"narrow start >= end: {S['narrow_bad']} | narrow inside broad: {S['in_broad'] / n:.4f} | narrow == broad: {S['narrow_eq_broad'] / n:.4f}")
print("strand:", A["strand"].astype(int).to_dict(), "| geneType:", A["gtype"].astype(int).to_dict())
print("geneID prefix:", A["gid"].sort_values(ascending=False).astype(int).head(6).to_dict())
chroms = list(A["chrom"].index)
print(f"chromosome names: {len(chroms)} | with '_' (alt/random/unplaced): {[x for x in chroms if '_' in x]}")

# ---- 4. txIDcat versus txInfo, labels ---------------------------------------------------------------------------
print(f"\nrecords listing more than one circRNA: {S['rec_multi']:,} ({pct(S['rec_multi'], n)}) | tokens: {S['tok']:,} ({S['tok'] / n:.2f} per record)")
print("circRNAs per record (11 = 11 or more), % of records:\n", (100 * A["k"].sort_index() / n).round(1).to_string())
print(f"tokens that are not 'hsa_circ_' + 7 digits: {S['tok_bad']} (txIDcat), {S['info_bad']} (txInfo) | circRNA listed twice in a record: {S['dup_tok']}")
print(f"tokens in which the first two txInfo fields differ: {S['id_pair_diff']:,}")
print(f"txInfo vs txIDcat: tokens without a counterpart (same-circRNA test): {S['set_diff_tok']:,} | records in a different order: "
      f"{S['order_diff']:,} ({pct(S['order_diff'], n)}) | among records with >= 2 circRNAs: {pct(S['order_diff_multi'], S['rec_multi'])}")
print(f"\ntokens: {S['tok']:,} | label contains 'Exon': {S['tok_exonic'] / S['tok']:.4f} | intron only: {S['tok_intron'] / S['tok']:.4f} | other: {S['tok_other'] / S['tok']:.4f}")
print("region type, all tokens:", A["type"].sort_values(ascending=False).astype(int).to_dict())
print("region type, exonic tokens:", A["type_ex"].sort_values(ascending=False).astype(int).to_dict())

# ---- 5. clusters and triples ------------------------------------------------------------------------------------
print(f"\nclusters (RBP, clusterID): {S['clusters']:,} on {S['lines']:,} lines | on more than one line: {S['clusters_rep']:,} ({pct(S['clusters_rep'], S['clusters'])})")
print("lines per cluster (5 = 5 or more), % of clusters:\n", (100 * A["lines_cl"].sort_index() / S["clusters"]).round(1).to_string())
print(f"clusters whose clusterClipExpNum differs between lines: {S['cexp_varies']:,} | whose clipExpNum differs: {S['clip_varies']:,} (of {S['clusters_rep']:,} repeated)")
print("circRNAs per cluster, union over its lines (11 = 11 or more), % of clusters:\n", (100 * A["k_union"].sort_index() / S["clusters"]).round(1).to_string(), f"\nmaximum: {S['k_union_max']}")
print(f"\ntokens: {S['tokens']:,} | distinct (RBP, cluster, circRNA) triples: {S['triples']:,} | repeated tokens: {S['tokens'] - S['triples']:,} ({pct(S['tokens'] - S['triples'], S['tokens'])})")
print(f"triples with both exonic and intron-only tokens: {S['triples_mixed']:,}")
print("triples: rows exonic (1 = yes), columns clusterClipExpNum (3 = 3 or more):\n", A["tri_x"].unstack(fill_value=0).astype(int).to_string())

# ---- 6. entities and edges -------------------------------------------------------------------------------------
rc = np.unique(E["all"] // BASE)
print(f"\ndistinct circRNAs: {len(rc):,} | RBPs: {len(rbp_code)} | in the ENCORI miRNA table (Cell 29): {int(np.isin(rc, c_all).sum()):,} "
      f"({pct(int(np.isin(rc, c_all).sum()), len(rc))}) | miRNA-table circRNAs without an RBP edge: {len(c_all) - int(np.isin(c_all, rc).sum()):,}")
ed = pd.DataFrame([(v, len(E[v]), len(np.unique(E[v] // BASE)), len(np.unique(E[v] % BASE))) for v in VARIANTS],
                  columns=["edge variant", "edges", "circRNAs", "RBPs"])
ed["% of all edges"] = (100 * ed["edges"] / ed["edges"].iloc[0]).round(1)
print("\nedges (RBP, circRNA) by evidence variant:\n", ed.to_string(index=False))
print(f"edges / possible pairs ({len(rc):,} x {len(rbp_code)}): {pct(len(E['all']), len(rc) * len(rbp_code))}")
deg = pd.DataFrame({"circRNAs per RBP (all)": rbp_df["circ all"], "circRNAs per RBP (exonic & c>=2)": rbp_df["circ exonic & c>=2"]}).describe(percentiles=[.05, .25, .5, .75, .95]).T
deg2 = pd.DataFrame({f"RBPs per circRNA ({v})": pd.Series(np.unique(E[v] // BASE, return_counts=True)[1]).describe(percentiles=[.05, .25, .5, .75, .95])
                     for v in ["all", "exonic & c>=2"]}).T
print("\ndegree distributions:\n", pd.concat([deg, deg2]).round(1).to_string())

records: 4,669,990 | equals the number used in Cell 34 (4,669,990): True | lineID strictly increasing: True
missing values per column: none
records with clusterClipExpNum < 1: 0
RBPs: 281 | blocks of consecutive records: 281 (equal = sorted by RBP)
records per RBP, largest 6: {'ELAVL1': 272955, 'RBMX': 188395, 'HNRNPC': 168301, 'IGF2BP2': 133295, 'U2AF1': 131346, 'CSTF2T': 120191} | smallest 3: {'ZC3HAV1': 12, 'ABCF1': 11, 'MEX3A': 9} | median: 3873
clusterID starts with '<RBP>:': 1.0000

relations between the evidence columns (share of records):
  clusterClipExpNum == clipExpNum                  0.2372
  clusterClipExpNum > clipExpNum                   0.0000
  clusterClipSiteNum >= clusterClipExpNum          1.0000
  clusterClipSiteNum == clusterClipExpNum          0.8564
  number of clipID tokens == clusterClipSiteNum    1.0000
  number of clipID tokens == clusterClipExpNum     0.8564

clusterClipExpNum (5 = 5 or more), % of records:
 cexp
1    61.7
2    20.5
3     6.9
4     4.1
5  

**Interpretation.** *(A) Integrity.* The pass reads 4,669,990 records (the number of Cell 34), `lineID` strictly increases, no column has a missing value, and no record has `clusterClipExpNum < 1`. The file is sorted by RBP: 281 blocks of consecutive records for 281 RBPs, which makes the per-RBP and per-cluster figures valid. `clusterID` starts with the RBP name in 100% of the records. The largest RBPs are ELAVL1 (272,955 records, 5.8%), RBMX and HNRNPC; the smallest have 9–12 records (MEX3A, ABCF1, ZC3HAV1); the median is 3,873. *(B) The 1% sample.* The 17 RBPs absent from the sample (264 against 281) are presumably small ones (inference). The sample figures of Cell 34 differ from the full file by at most 0.3 percentage points: 38.4% against 38.5% single-circRNA records, 29.2% against 29.3% records in a different order, 50.6% against 50.5% exonic tokens, 86.9% against 86.9% with `clipExpNum ≥ 2`, 61.8% against 61.7% with one experiment by `clusterClipExpNum`. The 95th percentile of the narrow width is 261 nt against 264 nt; only the maximum differs (6,719 against 17,979 nt), as expected for an extreme value. The sample was therefore representative. *(C) Evidence columns.* `clusterClipExpNum` is never larger than `clipExpNum`, and the two are equal in 23.7% of the records. The number of `clipID` entries equals `clusterClipSiteNum` in **100%** of the records, which confirms the hypothesis of Cell 34 (part 2); it equals `clusterClipExpNum` in 85.6%, so `clusterClipSiteNum` exceeds `clusterClipExpNum` in 14.4% (the sample gave 14.2%). That `clusterClipExpNum` counts the distinct experiments among the entries remains a hypothesis. The two evidence columns differ strongly in scale: 61.7% of the records rest on one experiment by `clusterClipExpNum`, 38.3% on two or more and 17.8% on three or more, whereas by `clipExpNum` 86.9% have two or more and 65.8% three or more. `clipExpNum = 1` occurs only together with `clusterClipExpNum = 1`, but 78.8% of the 2,881,510 records with `clusterClipExpNum = 1` have `clipExpNum ≥ 2`. `clusterClipExpNum` never differs between the lines of a cluster, whereas `clipExpNum` differs in 197,951 of the 497,468 repeated clusters (39.8%). `clusterClipExpNum` is therefore a property of the cluster, and `clipExpNum` a property of the line (cluster and host transcript; inference), which agrees with its larger values. *(D) `txIDcat` versus `txInfo`.* No token lacks a counterpart: the two columns list the same circRNAs in every record, none twice, and every identifier is `hsa_circ_` plus seven digits. The order differs in 1,366,745 records (29.3%), which is 47.6% of the 2,871,797 records that list two or more circRNAs. In 163,662 tokens (0.82%) the first two fields of the `txInfo` token differ, whereas the first field always equals the identifier of `txIDcat`. The meaning of the second field is not yet known (part 2). *(E) Labels.* Of 19,986,999 tokens, 50.5% carry a label with `Exon` and 49.5% are intron only (type `intron`: 9,893,173). Among the 10,093,826 exonic tokens, 1,674,636 (16.6%) have a type that also contains `intron` (for example `CDS,intron`, 1,591,917), so "exonic" includes clusters that also touch an intron. Exonic associations are better supported: 45.7% of them have `clusterClipExpNum ≥ 2` against 33.2% of the intron-only ones (21.2% against 15.3% at ≥ 3). *(F) Clusters and triples.* The 4,669,990 lines hold 4,069,242 clusters; 497,468 (12.2%) occupy more than one line (87.8% one line, 10.2% two, 1.7% three; 2.2 lines on average for the repeated ones). The 19,986,999 tokens are all distinct (RBP, cluster, circRNA) triples: no triple is repeated, so the lines of one cluster list disjoint sets of circRNAs. The count of triples with both exonic and intron-only tokens (0) is therefore trivial, because a triple has a single label. Per cluster, the union of the lines lists a median of 3 circRNAs (29.6% list one; 10.3% eleven or more; maximum 220). *(G) Entities and edges.* The table contains 89,255 circRNAs and 281 RBPs, which give 4,788,414 (RBP, circRNA) edges, 19.1% of the 25.1 million possible pairs, so an edge is backed by 4.2 clusters on average. 88,951 circRNAs (99.7%) are also in the ENCORI miRNA table; 304 are not, and 2,099 circRNAs of the miRNA table have no RBP edge. Edges by evidence variant (`c` = `clusterClipExpNum`; an edge passes when one of its tokens does):

| variant | edges | % of all | circRNAs | RBPs |
|---|---|---|---|---|
| all | 4,788,414 | 100.0 | 89,255 | 281 |
| `c ≥ 2` | 2,360,505 | 49.3 | 86,011 | 252 |
| `c ≥ 3` | 1,163,844 | 24.3 | 80,089 | 167 |
| exonic | 3,450,908 | 72.1 | 88,259 | 280 |
| exonic and `c ≥ 2` | 1,755,345 | 36.7 | 84,036 | 251 |

The graph is dense on the circRNA side: the median circRNA is linked to 48 of the 281 RBPs (mean 53.6, maximum 253), and to 15 (mean 20.9) under the strictest variant. On the RBP side the number of circRNAs per RBP is very uneven: median 8,548, 5th percentile 124, minimum 20, maximum 67,600. Under "exonic and `c ≥ 2`" 30 RBPs have no edge, and the 5th percentile is 0. A threshold of `c ≥ 3` leaves only 167 RBPs. *(H) `pvalue` and coordinates.* All values are negative (median −6.50, 5th percentile −58.84, minimum −383,934). The narrow region equals the broad region in 100% of the records, so the broad coordinates carry no information. The median narrow width is 40 nt (5th percentile 1 nt, 95th 264 nt); no record has `start ≥ end`; strands are 51.3% `+`. `geneID` is an NM accession in 91.2% of the records, an NR accession in 3.9%, an alias (`hsa…`) in 4.1% and `TCONS` in 0.8%. The 25 chromosome names contain no underscore, so the file has no alt or unplaced contigs.

**Decision.** (i) An RBP edge is a unique (circBase identifier, RBP) pair. Its main evidence is the maximum `clusterClipExpNum` over its clusters, which is well defined because the value never varies within a cluster. `clipExpNum` is not used as a threshold, because it varies between the lines of 39.8% of the repeated clusters and exceeds 1 in 86.9% of the lines; its maximum is kept for sensitivity analyses. (ii) The circRNA of a triple comes from `txIDcat`, and its label from the `txInfo` token with the same first field, never by position. Whether this is safe for the 0.82% of tokens with a different second field is decided after part 2, before Cell 38. (iii) Three evidence variants are carried forward for the label rebuild (old Cell 43): permissive (all tokens, 4,788,414 edges), exonic (3,450,908) and exonic with `c ≥ 2` (1,755,345). The variant `c ≥ 2` without the exonic condition (2,360,505) serves to separate the effect of the label from that of the evidence, and `c ≥ 3` is a sensitivity analysis only, because it leaves 167 of the 281 RBPs. The permissive variant links the median circRNA to 48 RBPs and probably carries little information as a positive label; the choice depends on the positives per task circRNA and per RBP and on the sibling test, not on the figures of the old API cells. (iv) Cell 38 stores, per edge, the number of clusters, the number of exonic clusters, the maximum `clusterClipExpNum` over all and over exonic clusters, the maximum `clipExpNum` and `clusterClipSiteNum`, and the most extreme `pvalue` (descriptive only; its scale is unknown). (v) A minimum number of positives per RBP for a multi-task head is decided after Cell 36, when the task coverage is known: 30 RBPs have no edge under the strictest variant, and the median RBP has 858 circRNAs. (vi) Cell 36 reuses the edge keys of this cell (`E`, `BASE`, `rbp_code`) and counts, read-only, the task circRNAs and the edges involving them under each variant.

In [74]:
# Cell 35 (part 2) - the second field of a txInfo token, when it differs from the first
# Uses from Cell 34: rb, parts (the 1% sample in memory; no file is read; read-only).
p = pd.DataFrame(parts.tolist(), index=parts.index, columns=["f0", "f1", "label", "type"])
d = p["f0"] != p["f1"]
print(f"sample tokens: {len(p):,} | first two fields differ: {int(d.sum()):,} ({100 * d.mean():.2f}%) | "
      f"full file: 163,662 of 19,986,999 (0.82%)")

q = p[d].copy()
rec = rb["txIDcat"].str.split(",")
q["f1_is_circ"]   = q["f1"].str.match(r"^hsa_circ_\d{7}$")                  # is the second field a circBase identifier?
q["f1_in_record"] = [f in rec[i] for i, f in zip(q.index, q["f1"])]         # does it name another circRNA of the same record?
print("\nsecond field of the differing tokens (share):\n", q[["f1_is_circ", "f1_in_record"]].mean().round(4).to_string())

print("\nregion type: differing tokens versus all tokens (%):")
print(pd.concat({"differ": q["type"].value_counts(normalize=True), "all": p["type"].value_counts(normalize=True)}, axis=1)
        .mul(100).round(1).sort_values("all", ascending=False).head(6).to_string())
print(f"\nexonic share: differing {100 * q['label'].str.contains('Exon').mean():.1f}% | all {100 * p['label'].str.contains('Exon').mean():.1f}%")

print("\nexamples:")
print(q[["f0", "f1", "label", "type"]].head(8).to_string())

sample tokens: 198,609 | first two fields differ: 1,606 (0.81%) | full file: 163,662 of 19,986,999 (0.82%)

second field of the differing tokens (share):
 f1_is_circ      0.0
f1_in_record    0.0

region type: differing tokens versus all tokens (%):
            differ   all
type                    
intron        41.0  49.4
CDS           52.3  33.1
CDS,intron     6.4   7.9
3'UTR          NaN   5.8
5'UTR          0.2   1.3
Exon           NaN   1.3

exonic share: differing 59.0% | all 50.6%

examples:
                   f0               f1     label    type
8    hsa_circ_0001155  hsa_circ_000332    Exon-1     CDS
20   hsa_circ_0000006  hsa_circ_001621    Exon-1     CDS
34   hsa_circ_0001524  hsa_circ_002123    Exon-2     CDS
43   hsa_circ_0001617  hsa_circ_000972  intron-3  intron
73   hsa_circ_0001932  hsa_circ_001452    Exon-2     CDS
88   hsa_circ_0000304  hsa_circ_000795    Exon-6     CDS
89   hsa_circ_0000319  hsa_circ_001028    Exon-1     CDS
109  hsa_circ_0000553  hsa_circ_001753   

**Interpretation.** In the sample, 1,606 of 198,609 tokens (0.81%) have a different second field, matching the full-file share (0.82%). None of the 1,606 is a circBase identifier of the form `hsa_circ_` plus seven digits, and none names another circRNA of the same record. The second test is trivial, because the circRNAs of a record all have seven-digit identifiers. The eight printed second fields have six digits (for example `hsa_circ_000332`), a format like the six-digit near-miss names of Cell 9 (inference from eight examples; not compared in code). The first field is always the identifier of `txIDcat`, so the label and type of a token belong to that circRNA, and the second field looks like an alternative name of the same circRNA, not a reference to another one. The differing tokens are not a random subset: they are `CDS` in 52.3% (all tokens: 33.1%), intron in 41.0% (49.4%) and `CDS,intron` in 6.4%, and none has the type `3'UTR` or plain `Exon` (5.8% and 1.3% overall). Their exonic share is 59.0% against 50.6%.

**Decision.** (i) The label and type of a circRNA are read from the `txInfo` token whose first field equals the identifier of `txIDcat`, and the second field is not used. This decision of Cell 35 stands. (ii) The 0.82% of tokens with a different second field stay in the tables, because their identifier is unambiguous. No flag is stored, since the field is not used. Cell 38 is unchanged.

### 36. Coverage of the CircR2Disease circRNAs in the ENCORI RBP Table (read-only)

**Objective.** Count how many circRNAs of the CircR2Disease task have an RBP edge in ENCORI under each of the five evidence variants of Cell 35, how many edges and RBPs this gives, how many task circRNAs each RBP has (which decides the RBPs usable for a multi-task head), and how the RBP coverage relates to the miRNA coverage of Cell 30 and to the identifier flags of Cell 29, without building or saving any intersection table.

In [75]:
# Cell 36 - Coverage of the task circRNAs in the ENCORI RBP table (read-only: nothing is saved, no intersection table is built)
# Uses from earlier cells: E, VARIANTS, BASE, rbp_code (Cell 35); circ, num, flag (Cell 30); c_all (Cell 29).
inv = {v: k for k, v in rbp_code.items()}                          # RBP code -> name
C   = {v: np.unique(E[v] // BASE) for v in VARIANTS}               # circRNAs with at least one edge, per variant
for v in VARIANTS:
    circ["rbp_" + v] = np.isin(num, C[v])                          # task circRNA has an RBP edge under this variant
sel = {v: E[v][np.isin(E[v] // BASE, num)] for v in VARIANTS}      # edges that involve a task circRNA (counted, not stored)

# ---- 1. coverage of the task circRNAs, per variant ---------------------------------------------------------------
sets = [("CircR2Disease circRNAs", np.ones(len(circ), dtype=bool)), ("model-input (<=5,000 nt)", circ["use_primary"].to_numpy())]
rows = []
for lab, mask in sets:
    sub = circ[mask]
    rows.append([lab, len(sub)] + [int(sub["rbp_" + v].sum()) for v in VARIANTS])
cov = pd.DataFrame(rows, columns=["circRNA set", "n"] + VARIANTS)
print("task circRNAs with at least one RBP edge, by evidence variant:\n", cov.to_string(index=False))
print("\nshare of the set (%):\n", pd.concat([cov["circRNA set"], (100 * cov[VARIANTS].div(cov["n"], axis=0)).round(1)], axis=1).to_string(index=False))

# ---- 2. edges that involve a task circRNA, compared with ENCORI as a whole ------------------------------------------
print("\nedges involving a task circRNA, and mean RBPs per circRNA (task versus all ENCORI RBP circRNAs):")
for v in VARIANTS:
    s = sel[v]; nc = len(np.unique(s // BASE))
    print(f"  {v:14s}: {len(s):>9,} edges | {nc:>5,} circRNAs | {len(np.unique(s % BASE)):>3} RBPs | "
          f"mean {len(s) / nc:5.1f} RBPs per task circRNA against {len(E[v]) / len(C[v]):5.1f} over ENCORI")
dist = pd.DataFrame({f"RBPs per task circRNA ({v})": pd.Series(np.unique(sel[v] // BASE, return_counts=True)[1]).describe(percentiles=[.05, .25, .5, .75, .95])
                     for v in ["all", "exonic & c>=2"]}).T
print("\ndegree of the task circRNAs that have an edge:\n", dist.round(1).to_string())

# ---- 3. task circRNAs per RBP: which RBPs have enough positives? ----------------------------------------------------
TH = [1, 5, 10, 20, 50, 100]
per_rbp = {v: np.bincount(sel[v] % BASE, minlength=len(rbp_code)) for v in VARIANTS}
tab = pd.DataFrame({v: [int((per_rbp[v] >= t).sum()) for t in TH] for v in VARIANTS}, index=[f">= {t}" for t in TH])
print("\nnumber of RBPs (of 281) with at least this many task circRNAs among their edges:\n", tab.to_string())
print("\ntask circRNAs per RBP, largest 6 (variant 'exonic & c>=2'):",
      {inv[i]: int(per_rbp["exonic & c>=2"][i]) for i in np.argsort(-per_rbp["exonic & c>=2"])[:6]})
named = [r for r in ["QKI", "FUS", "EIF4A3", "IGF2BP1", "IGF2BP2", "IGF2BP3", "AGO2"] if r in rbp_code]
print("\nRBPs of the old API block (task circRNAs per variant):\n",
      pd.DataFrame({v: [int(per_rbp[v][rbp_code[r]]) for r in named] for v in VARIANTS}, index=named).to_string())

# ---- 4. relation to the miRNA coverage and to the identifier flags ----------------------------------------------------
print("\ntask circRNAs: miRNA table (rows, Cell 30) versus RBP table (columns, any variant):")
print(pd.crosstab(circ["in_encori"].rename("in miRNA table"), circ["rbp_all"].rename("in RBP table"), margins=True).to_string())
absent = circ[~circ["rbp_all"]]
print(f"\nabsent from the RBP table: {len(absent)} | of which in the miRNA table: {int(absent['in_encori'].sum())} | examples: {absent['circrna_id'].head(6).tolist()}")
print(f"task circRNAs with a chromosome/strand disagreement (Cell 29) that have an RBP edge: "
      f"{int((circ['circrna_id'].isin(flag) & circ['rbp_all']).sum())} of {int(circ['circrna_id'].isin(flag).sum())} flagged")

task circRNAs with at least one RBP edge, by evidence variant:
              circRNA set    n  all  c>=2  c>=3  exonic  exonic & c>=2
  CircR2Disease circRNAs 1644 1457  1403  1294    1444           1365
model-input (<=5,000 nt) 1582 1405  1352  1245    1392           1314

share of the set (%):
              circRNA set  all  c>=2  c>=3  exonic  exonic & c>=2
  CircR2Disease circRNAs 88.6  85.3  78.7    87.8           83.0
model-input (<=5,000 nt) 88.8  85.5  78.7    88.0           83.1

edges involving a task circRNA, and mean RBPs per circRNA (task versus all ENCORI RBP circRNAs):
  all           :    67,434 edges | 1,457 circRNAs | 279 RBPs | mean  46.3 RBPs per task circRNA against  53.6 over ENCORI
  c>=2          :    32,443 edges | 1,403 circRNAs | 237 RBPs | mean  23.1 RBPs per task circRNA against  27.4 over ENCORI
  c>=3          :    15,672 edges | 1,294 circRNAs | 155 RBPs | mean  12.1 RBPs per task circRNA against  14.5 over ENCORI
  exonic        :    48,728 edges | 1,44

**Interpretation.** *(A) Coverage.* Of the 1,644 CircR2Disease circRNAs, 1,457 (88.6%) have at least one RBP edge in the cluster table. The numbers fall with the evidence variant: 1,403 (85.3%) with `clusterClipExpNum ≥ 2`, 1,294 (78.7%) with `≥ 3`, 1,444 (87.8%) with an exonic association and 1,365 (83.0%) with an exonic association and `≥ 2`. The 1,582 model-input circRNAs give the same shares (88.8%, 85.5%, 78.7%, 88.0% and 83.1%), so the length limit again does not change coverage. 187 task circRNAs (11.4%) have no RBP edge. *(B) Comparison with ENCORI as a whole.* Among the 1,457 task circRNAs that have an edge, 96.3% keep one at `c ≥ 2`, 88.8% at `c ≥ 3`, 99.1% with an exonic association and 93.7% with both exonic and `c ≥ 2`. The corresponding shares over all 89,255 circRNAs of the table are 96.4%, 89.7%, 98.9% and 94.2% (computed from Cell 35). Once a task circRNA is in the table, it is therefore supported as well as any other, and the loss comes from the 187 absent circRNAs. *(C) Edges.* 67,434 of the 4,788,414 edges (1.4%) involve a task circRNA, which are 1.6% of the circRNAs of the table. The shares of the variants among the task edges are those of the whole table (exonic 72.3% against 72.1%; `c ≥ 2` 48.1% against 49.3%; exonic and `c ≥ 2` 35.9% against 36.7%; computed from the printed counts). The task circRNAs have 14–17% fewer RBP edges than the average circRNA: 46.3 against 53.6 RBPs (permissive), 23.1 against 27.4 (`c ≥ 2`), 12.1 against 14.5 (`c ≥ 3`), 33.7 against 39.1 (exonic) and 17.7 against 20.9 (exonic and `c ≥ 2`). This is the same direction as on the miRNA side (49.1 against 60.3 miRNAs in Cell 30). The median task circRNA is linked to 41 RBPs (permissive, 5th percentile 4, maximum 207) and to 12 under the strictest variant (5th percentile 1, 25th percentile 5), so under that variant a quarter of the task circRNAs have five or fewer positives. *(D) RBPs.* 279 of the 281 RBPs have at least one task circRNA among their edges (permissive); 227 do under the strictest variant. The number of RBPs with enough task circRNAs falls quickly with the variant:

| task circRNAs per RBP | all | `c ≥ 2` | `c ≥ 3` | exonic | exonic and `c ≥ 2` |
|---|---|---|---|---|---|
| ≥ 20 | 219 | 159 | 82 | 194 | 122 |
| ≥ 50 | 189 | 115 | 54 | 151 | 87 |
| ≥ 100 | 150 | 81 | 43 | 112 | 61 |

Under the strictest variant the largest RBPs are U2AF2 (782 task circRNAs, 57% of the 1,365), SCAF8 (775), SCAF4 (766), RNPS1 and ALYREF (749 each) and IGF2BP2 (607). Several of these are known as splicing or export factors that bind exons broadly, so their edges may describe binding to the exons of the host transcript, not binding specific to the circRNA (inference; not tested). *(E) The old API block.* The RBPs differ strongly in how much of their support survives. IGF2BP2 keeps 607 of 1,010 task circRNAs (60%), FUS 261 of 692 (38%) and QKI only 16 of 182 (8.8%). EIF4A3 is almost entirely exonic (850 of 859) but only 75 of its task circRNAs reach `c ≥ 3`. QKI, the RBP of the old API trial, would be nearly unusable under the strictest variant. AGO2 does not appear in the printed block, which lists only names found in the table, so it is not among the 281 RBPs under this spelling. *(F) Relation to the miRNA side.* 1,456 task circRNAs are in both tables, 29 only in the miRNA table, 1 only in the RBP table and 158 in neither. The 158 are 9.6% of the 1,644 and nearly all of the 159 task circRNAs without an ENCORI miRNA record (Cell 30), so the absence is probably not specific to one table (inference; the identifiers were not examined). The RBP table adds only 1 circRNA to the 1,485 of the miRNA table (1,486 in the union). *(G) Identifier flags.* 3 of the 4 flagged task circRNAs of Cell 29 have an RBP edge.

**Decision.** (i) The 187 task circRNAs without an RBP edge keep their rows in the CDA task and are treated as unlabelled for the RBP auxiliary task, not as negatives, for the same reason as in Cell 30. (ii) The RBP labels can therefore cover at most 1,457 of 1,644 circRNAs (1,405 of 1,582 for the model input) under the permissive variant, and 1,365 (1,314) under the exonic variant with `c ≥ 2`. (iii) The RBPs of the multi-task head are selected by a minimum number of task circRNAs among their edges. A floor of 50 (about 10 per fold in a 5-fold split) would keep 87 RBPs under exonic with `c ≥ 2`, 115 under `c ≥ 2` and 189 under the permissive variant; the floor is fixed when the folds are rebuilt (old Cells 43–44), because host-gene grouping makes the folds uneven. (iv) Cell 38 stores, per edge, `n_clusters`, `n_clusters_exonic`, `n_clusters_private` (clusters whose circRNA set, over all their lines, contains only this circRNA), `c_max`, `c_max_exonic`, `c_max_private`, `k_min_c1` and `k_min_c2` (the smallest number of circRNAs listed by a cluster with `c ≥ 1` and `c ≥ 2`), `clip_max`, `site_max` and the most extreme `pvalue`, so that the shared-cluster dependence of Cell 33 and the cap of Cell 33b can be repeated for the RBP side. (v) Cell 38 also stores an RBP table (lines, clusters, edges and task circRNAs per variant), so that broad binders such as U2AF2 can be examined as hubs before the head is fixed. (vi) The 4 flagged task circRNAs keep their flag (`id_check` of the ENCORI circRNA table) and are not treated separately for the RBP side.

### 37a. Consistency of the Cluster–circRNA Association

**Objective.** Test, for every (cluster line, circRNA) token of the RBP cluster table, whether the association agrees with the circRNA as it is described by the ENCORI miRNA table, in order to establish (i) whether the cluster lies on the same chromosome and strand as the circRNA, overall and for the identifiers flagged in Cell 33, (ii) whether the narrow cluster interval lies within, overlaps or lies away from the genomic span covered by the miRNA sites of the circRNA (the span is a lower bound of the circRNA interval, so it is built here from all sites of at most 60 nt and stratified by its width), (iii) how these results differ between exonic and intron-only labels and between clusters that list one or several circRNAs, and (iv) what a random circRNA would give, so that the agreement can be read against a chance level. This shows whether the RBP cluster coordinates are on the same assembly as the miRNA sites (hg38) and whether the label and the position agree.

In [76]:
# Cell 37 - Consistency of the cluster-circRNA association (read-only: nothing is saved)
# Uses from earlier cells: MIRNA_FILE (Cell 29), RBP_FILE (Cell 34), PROCESSED_DIR (Cell 1).
SPAN_MAX = 60                  # sites longer than 60 nt (227 records, Cell 33b) are left out of the span of a circRNA
GAP_BINS = [0, 100, 1000]      # distance (nt) between a cluster and the span: 0 | 1-100 | 101-1000 | > 1000
W_BINS   = [200, 2000]         # width of the span (nt): < 200 | 200-1,999 | >= 2,000
CLS      = ["inside the span", "overlaps the span", "gap 1-100 nt", "gap 101-1,000 nt", "gap > 1,000 nt"]
WLAB     = ["span < 200 nt", "span 200-1,999 nt", "span >= 2,000 nt"]

# ---- 1. reference per circRNA: chromosome and strand (Cell 33 table) and the span of its ENCORI sites -------------------
ct = pd.read_csv(PROCESSED_DIR / "encori_circ_table.csv")
ct["num"] = ct["circbase_id"].str[9:].astype("int64")
NMAX = int(ct["num"].max()) + 1
chrom_cats = pd.Index(sorted(ct["enc_chrom"].unique()))
idc_cats   = pd.Index(["consistent", "header differs, sequence agrees", "header differs, sequence mostly agrees",
                       "header differs, sequence not confirmed"])
CH  = np.full(NMAX, -1, "int16"); CH[ct["num"]]  = chrom_cats.get_indexer(ct["enc_chrom"])
ST  = np.full(NMAX, -1, "int8");  ST[ct["num"]]  = (ct["enc_strand"] == "+").astype("int8")
IDC = np.full(NMAX, -1, "int8");  IDC[ct["num"]] = idc_cats.get_indexer(ct["id_check"])
assert (CH[ct["num"]] >= 0).all() and (IDC[ct["num"]] >= 0).all()

parts = []
for ch in pd.read_csv(MIRNA_FILE, sep="\t", usecols=["start", "end", "txIDcat"], chunksize=1_000_000,
                      keep_default_na=False, na_values=["NA"]):
    ch = ch[(ch["end"] - ch["start"]) <= SPAN_MAX]
    ex = (pd.DataFrame({"s": ch["start"].to_numpy("int64"), "e": ch["end"].to_numpy("int64"),
                        "c": ch["txIDcat"].str.split(",").to_numpy()}).explode("c"))
    ex["c"] = ex["c"].str[9:].astype("int64")
    parts.append(ex.groupby("c").agg(s=("s", "min"), e=("e", "max")))
sp = pd.concat(parts).groupby(level=0).agg(s=("s", "min"), e=("e", "max"))
SS = np.full(NMAX, -1, "int64"); SE = np.full(NMAX, -1, "int64")
SS[sp.index.to_numpy()] = sp["s"].to_numpy(); SE[sp.index.to_numpy()] = sp["e"].to_numpy()
wd = sp["e"] - sp["s"]
print(f"circRNAs of the table: {len(ct):,} | with a span (at least one site of <= {SPAN_MAX} nt): {len(sp):,}")
print("span width (nt) quantiles:\n", wd.quantile([0, .05, .25, .5, .75, .95, 1]).round(0).astype(int).to_string())
print("share of circRNAs by span width:", {l: round(float(x), 3) for l, x in zip(WLAB, np.bincount(np.digitize(wd, W_BINS), minlength=3) / len(wd))})

def classify(ns, ne, ss, se):
    """Position of the cluster [ns, ne] relative to the span [ss, se]: 0 inside, 1 overlaps, 2-4 gap bins."""
    gap = np.maximum(0, np.maximum(ss - ne, ns - se))
    inside = (ns >= ss) & (ne <= se)
    return np.where(inside, 0, np.where(gap == 0, 1, 2 + np.digitize(gap, GAP_BINS[1:], right=True)))

# ---- 2. one pass over the cluster table; tokens (cluster line, circRNA) as in Cell 35 -------------------------------------
rng = np.random.default_rng(42)
P  = np.zeros((3, 2, 2, 2, 5), "int64")      # span width x exonic x strand agrees x (record lists >= 2 circRNAs) x position class
PC = np.zeros((3, 5), "int64")               # shuffled control: same, for a random circRNA among the tokens of the chunk
X  = np.zeros((4, 2, 2), "int64")            # id_check x chromosome agrees x strand agrees
S  = dict.fromkeys(["records", "tokens", "no_ref", "chrom_ok", "strand_ok", "both_ok", "no_span", "c_chrom_unknown",
                    "ctl_tokens", "ctl_chrom_ok", "ctl_strand_ok"], 0)
USEB = ["chromosome", "strand", "narrowStart", "narrowEnd", "txInfo"]
for i, ch in enumerate(pd.read_csv(RBP_FILE, sep="\t", usecols=USEB, chunksize=500_000,
                                   keep_default_na=False, na_values=["NA"]), 1):
    inf = ch["txInfo"].str.split("|")
    k   = inf.str.len().to_numpy()
    pos = np.repeat(np.arange(len(ch)), k)                       # record of every token (explode keeps this order)
    pt  = inf.explode().str.split(":", expand=True)
    num = pd.to_numeric(pt[0].str[9:]).to_numpy("int64")         # circRNA of the token = first field of txInfo (= txIDcat as a set, Cell 35)
    ex_ = pt[2].str.contains("Exon").to_numpy()
    cc  = chrom_cats.get_indexer(ch["chromosome"].to_numpy())[pos]
    cs  = (ch["strand"].to_numpy() == "+").astype("int8")[pos]
    ns, ne = ch["narrowStart"].to_numpy("int64")[pos], ch["narrowEnd"].to_numpy("int64")[pos]
    multi = (k[pos] >= 2).astype(int)

    ref = num < NMAX
    ref[ref] = CH[num[ref]] >= 0                                 # circRNA present in the miRNA table
    chrom_ok  = ref & (CH[np.where(ref, num, 0)] == cc)
    strand_ok = ref & (ST[np.where(ref, num, 0)] == cs)
    S["records"] += len(ch); S["tokens"] += len(num); S["no_ref"] += int((~ref).sum())
    S["c_chrom_unknown"] += int((cc < 0).sum())
    S["chrom_ok"] += int(chrom_ok.sum()); S["strand_ok"] += int(strand_ok.sum()); S["both_ok"] += int((chrom_ok & strand_ok).sum())
    np.add.at(X, (IDC[num[ref]], chrom_ok[ref].astype(int), strand_ok[ref].astype(int)), 1)

    # position of the cluster relative to the span: only where the chromosome agrees and a span exists
    has = chrom_ok & (SS[np.where(ref, num, 0)] >= 0)
    S["no_span"] += int((chrom_ok & ~has).sum())
    n_, ss, se = num[has], SS[num[has]], SE[num[has]]
    cl = classify(ns[has], ne[has], ss, se)
    wb = np.digitize(se - ss, W_BINS)
    np.add.at(P, (wb, ex_[has].astype(int), strand_ok[has].astype(int), multi[has], cl), 1)

    # shuffled control: the circRNA of every token is replaced by that of another token of the chunk
    sh = rng.permutation(len(num)); num2 = num[sh]; ref2 = ref[sh]
    chrom2 = ref2 & (CH[np.where(ref2, num2, 0)] == cc)
    S["ctl_tokens"] += len(num); S["ctl_chrom_ok"] += int(chrom2.sum())
    S["ctl_strand_ok"] += int((ref2 & (ST[np.where(ref2, num2, 0)] == cs)).sum())
    h2 = chrom2 & (SS[np.where(ref2, num2, 0)] >= 0)
    cl2 = classify(ns[h2], ne[h2], SS[num2[h2]], SE[num2[h2]])
    np.add.at(PC, (np.digitize(SE[num2[h2]] - SS[num2[h2]], W_BINS), cl2), 1)
    print(f"  chunk {i}: {S['records']:,} records", end="\r")

# ---- 3. report ----------------------------------------------------------------------------------------------------------
pc = lambda a, b: f"{100 * a / b:.2f}%"
T = S["tokens"]; R = T - S["no_ref"]
print(f"\nrecords: {S['records']:,} | tokens: {T:,} | tokens whose circRNA is not in the miRNA table: {S['no_ref']:,} ({pc(S['no_ref'], T)})")
print(f"tokens whose cluster chromosome is not among the 31 miRNA-table names: {S['c_chrom_unknown']:,}")
print(f"\nchromosome of the cluster equals that of the circRNA: {pc(S['chrom_ok'], R)} of {R:,} | strand equal: {pc(S['strand_ok'], R)} | both: {pc(S['both_ok'], R)}")
print(f"shuffled control (random circRNA): chromosome equal {pc(S['ctl_chrom_ok'], S['ctl_tokens'])} | strand equal {pc(S['ctl_strand_ok'], S['ctl_tokens'])}")
print("\nagreement by the identifier check of Cell 33 (tokens, % chromosome agrees / % strand agrees):")
for j, name in enumerate(idc_cats):
    n_j = int(X[j].sum())
    print(f"  {name:42s} {n_j:>11,} | chromosome {pc(int(X[j, 1].sum()), n_j) if n_j else '-'} | strand {pc(int(X[j, :, 1].sum()), n_j) if n_j else '-'}")
print(f"\nchromosome agrees but the circRNA has no span: {S['no_span']:,}")

Pn = P.sum(axis=(1, 2, 3, 4)); print("\ntokens with a span, by span width:", dict(zip(WLAB, Pn.tolist())))
def show(arr, title):
    t = pd.DataFrame(arr, columns=CLS)
    print(title); print((100 * t.div(t.sum(axis=1), axis=0)).round(2).assign(n=t.sum(axis=1)).to_string())
show(P[:, :, 1].sum(axis=(1, 2)), "\nposition of the cluster (% of tokens), strand agrees, by span width:")
show(P[:, :, 0].sum(axis=(1, 2)), "\nsame, strand differs:")
show(PC, "\nshuffled control (random circRNA, chromosome agrees), by span width:")
for w in (1, 2):
    print(f"\n{WLAB[w]} | strand agrees | % of tokens by position, exonic versus intron only, record lists 1 versus >= 2 circRNAs:")
    rows = {f"{'exonic' if e else 'intron only'}, {'>= 2' if m else '1'} circRNA{'s' if m else ''}": P[w, e, 1, m] for e in (1, 0) for m in (0, 1)}
    t = pd.DataFrame(rows, index=CLS).T
    print((100 * t.div(t.sum(axis=1), axis=0)).round(2).assign(n=t.sum(axis=1)).to_string())

circRNAs of the table: 91,050 | with a span (at least one site of <= 60 nt): 91,050
span width (nt) quantiles:
 0.00          8
0.05         84
0.25       1853
0.50       7952
0.75      23851
0.95      91762
1.00    3227999
share of circRNAs by span width: {'span < 200 nt': 0.107, 'span 200-1,999 nt': 0.153, 'span >= 2,000 nt': 0.74}
  chunk 10: 4,669,990 records
records: 4,669,990 | tokens: 19,986,999 | tokens whose circRNA is not in the miRNA table: 4,025 (0.02%)
tokens whose cluster chromosome is not among the 31 miRNA-table names: 0

chromosome of the cluster equals that of the circRNA: 100.00% of 19,982,974 | strand equal: 100.00% | both: 100.00%
shuffled control (random circRNA): chromosome equal 5.62% | strand equal 50.00%

agreement by the identifier check of Cell 33 (tokens, % chromosome agrees / % strand agrees):
  consistent                                  19,962,593 | chromosome 100.00% | strand 100.00%
  header differs, sequence agrees                 10,141 | chromosome 

**Interpretation.** *(A) Coverage of the reference.* The reference is the 91,050 circRNAs of the ENCORI miRNA table, and every one of them has at least one site of at most 60 nt, so each has a span (no token falls in the "chromosome agrees but no span" class: 0). The span widths are very uneven: median 7,952 nt, 5th percentile 84 nt, maximum 3,227,999 nt; 10.7% of the circRNAs have a span under 200 nt, 15.3% one of 200–1,999 nt and 74.0% one of 2,000 nt or more. Of the 19,986,999 tokens, 4,025 (0.02%) belong to circRNAs that are absent from the miRNA table (the 304 circRNAs of Cell 35) and cannot be tested; the other 19,982,974 can. Every cluster chromosome is among the 31 miRNA-table names (0 exceptions). *(B) Chromosome and strand.* The cluster lies on the chromosome and strand of its circRNA in 100.00% of the 19,982,974 testable tokens. This is not a trivial agreement: a random circRNA gives 5.62% for the chromosome and 50.00% for the strand. The agreement is also 100.00% in the three groups of flagged identifiers (10,141 tokens "header differs, sequence agrees", 10,238 "mostly agrees", 2 "not confirmed"), as in the 19,962,593 consistent tokens. The 2 tokens of the last group are too few to say anything. The flagged identifiers are therefore no less coherent with the RBP table than the others. *(C) Position.* Because the strand always agrees, the tables for "strand differs" are empty (n = 0). The share of tokens with the cluster inside or overlapping the span grows with the span width: 62.0% for spans under 200 nt, 91.9% for 200–1,999 nt and 98.6% for 2,000 nt or more (sums of the printed columns). The share more than 1,000 nt away from the span falls in the same order: 22.1%, 3.3% and 0.8%. Since the span is a lower bound of the circRNA (it covers only the miRNA sites), a short span is expected to miss part of the circRNA, so the large distances for short spans are not evidence of a wrong association (inference; the true circRNA interval was not used). 95.9% of the tokens (19,157,089) fall in the widest class, so the overall picture is the one of that class. *(D) Chance level.* A random circRNA on the correct chromosome puts the cluster more than 1,000 nt from its span in 99.87% (span under 200 nt), 99.83% (200–1,999 nt) and 98.08% (2,000 nt or more) of the cases, and inside the span in only 1.86% of the widest class, against 97.13% for the real association. The agreement is therefore far above chance for all three widths. The control keeps the chromosome but not the strand and the real association does keep both, which makes the comparison slightly generous to the real tokens; because the real strand agreement is 100% and the control's is 50%, the effect is only to halve the control's already negligible inside shares at most (inference). *(E) Same assembly.* The miRNA sites are on GRCh38 (Cell 31) and the clusters fall in the spans of those sites in 97.1% of the tokens of the widest class, and within 100 nt of the span in 94.6% of the 200–1,999 nt class. A cluster table on hg19 would be expected to place many tokens far from the hg38 span, so the cluster coordinates appear to be hg38 as well (inference; a liftover was not tested directly, and an offset of a few nucleotides would not be detected by this test). *(F) Exonic versus intron-only.* In the 200–1,999 nt class the two labels behave differently. For exonic tokens 85.8% (1 circRNA listed) and 83.5% (2 or more) lie inside the span, 10.1% and 12.3% overlap it, and only 0.68% are more than 1,000 nt away. For intron-only tokens 61.1% and 74.2% lie inside, almost none overlaps (0.02% and 0.01%) and 29.4% and 14.9% are more than 1,000 nt away. In the widest class the intron-only tokens lie inside the span slightly more often than the exonic ones (97.8% and 98.4% against 95.6% and 95.9%), and none overlaps the span (0.00%); the share more than 1,000 nt away is 1.8% and 1.3% against 0.29% and 0.25%. The span is the genomic interval from the first to the last site, so it also contains the introns between the exons. A cluster inside the span with an intron label is therefore expected, and the test cannot show whether such a cluster lies in sequence kept in the mature circRNA (inference). The absence of overlaps for intron-only tokens is consistent with the boundary of the span lying in an exon, which an intron-only cluster by definition does not touch (inference; not tested). *(G) One versus several circRNAs.* For exonic tokens the position hardly depends on whether the record lists one circRNA or several (inside 85.8% against 83.5% in the middle class and 95.6% against 95.9% in the widest), so a cluster shared by several circRNAs is placed as well as a private one. The larger difference for intron-only tokens in the middle class (61.1% against 74.2% inside) is based on 15,794 and 92,903 tokens and is not explained by the printed data; it disappears in the widest class.

**Decision.** (i) The cluster coordinates are treated as hg38, the same assembly as the miRNA sites. No liftover is applied and no token is removed or flagged for position. (ii) Chromosome and strand are not stored per edge, because they equal those of the circRNA in 100% of the tokens (Cell 38 is unchanged). (iii) The 4,025 tokens (304 circRNAs) without a reference stay in the RBP tables, because their identifiers are valid in the cluster table; the circRNA universe table records whether a circRNA is in the miRNA table (Cell 35: 99.7% are). (iv) The identifier flags of Cell 33 are kept as they are and are not used to treat the RBP edges separately, which agrees with Decision (vi) of Cell 36. (v) No position-based filter is introduced. The span is only a lower bound of the circRNA and, being genomic, cannot separate retained from spliced-out sequence, so the exonic flag and `clusterClipExpNum` of Cell 35 remain the only evidence filters. (vi) The Decision of Cell 34 (part 2) stands: intron-only support is not used alone as a positive label without a sensitivity test, because this cell gives no argument for or against it. (vii) The shared-cluster dependence is not removed by position; it is handled by the private-cluster and `k_min` columns of Cell 38.


### 37b. Cluster Width, Evidence and the Profile of Every RBP

**Objective.** Profile the ENCORI RBP cluster table from the RBP side alone, in a single chunked pass, in order to establish (i) how the width of the narrow cluster interval relates to the evidence (`clusterClipExpNum`), to the exonic label and to the number of circRNAs listed, including whether the frequent width of 40 nt (Cell 34) marks a distinct kind of cluster, (ii) how the RBPs differ in their exonic share, evidence, cluster width and share of single-circRNA records, and whether RBPs that bind broadly (U2AF2, SCAF8, SCAF4, RNPS1, ALYREF; Cell 36) stand out from the others on these properties, and (iii) which of these properties are related across RBPs, so that the RBPs of the multi-task head can be chosen and described before the edge tables are built in Cell 38.

In [77]:
# Cell 37b - Cluster width, evidence and the profile of every RBP (one chunked pass; read-only: nothing is saved)
# Uses from earlier cells: RBP_FILE (Cell 34), EXPECTED, rbp_code, rbp_df (Cell 35), per_rbp (Cell 36, dict of arrays).
assert isinstance(per_rbp, dict), "per_rbp must be the dict of Cell 36 (Cell 34 used the same name for a Series)"
NR      = len(rbp_code)
names   = [k for k, _ in sorted(rbp_code.items(), key=lambda kv: kv[1])]   # RBP name at every code
W_EDGES = [10, 39, 40, 100, 300]                                           # width bins (nt): <= 10 | 11-39 | 40 | 41-100 | 101-300 | > 300
WL      = ["<= 10 nt", "11-39 nt", "40 nt", "41-100 nt", "101-300 nt", "> 300 nt"]
USEC    = ["RBP", "clusterClipExpNum", "narrowStart", "narrowEnd", "txInfo"]
MIN_REC = 100                                                              # RBPs with fewer records are left out of the correlations
FLOOR   = 50                                                               # task-circRNA floor discussed in Cell 36 (iii)

A_w  = np.zeros((6, 5), "int64")      # records: width bin x clusterClipExpNum (5 = 5 or more)
A_wt = np.zeros((6, 2), "int64")      # tokens: width bin x exonic
A_wk = np.zeros(6, "int64")           # tokens (circRNAs listed), per width bin
R    = {n: np.zeros(NR, "int64") for n in ["rec", "rec_c2", "rec_k1", "rec_w100", "tok"]}
T    = np.zeros((NR, 2, 2), "int64")  # tokens: RBP x exonic x (clusterClipExpNum >= 2)
bc   = lambda idx, n: np.bincount(idx, minlength=n).astype("int64")

# ---- 1. one pass over the cluster table; tokens (cluster line, circRNA) as in Cells 35 and 37 --------------------------
for i, ch in enumerate(pd.read_csv(RBP_FILE, sep="\t", usecols=USEC, chunksize=500_000,
                                   keep_default_na=False, na_values=["NA"]), 1):
    r   = ch["RBP"].map(rbp_code).to_numpy("int64")
    c   = ch["clusterClipExpNum"].to_numpy("int64")
    w   = (ch["narrowEnd"] - ch["narrowStart"]).to_numpy("int64")        # width as in Cell 34 (end - start)
    wb  = np.digitize(w, W_EDGES, right=True)
    inf = ch["txInfo"].str.split("|")
    k   = inf.str.len().to_numpy("int64")                                # circRNAs listed by the record
    pos = np.repeat(np.arange(len(ch)), k)                               # record of every token (explode keeps this order)
    ex_ = inf.explode().str.split(":").str[-2].str.contains("Exon").to_numpy().astype("int64")   # label before the type, as in Cell 34

    A_w  += bc(wb * 5 + np.minimum(c, 5) - 1, 30).reshape(6, 5)
    A_wt += bc(wb[pos] * 2 + ex_, 12).reshape(6, 2)
    A_wk += np.bincount(wb, weights=k, minlength=6).astype("int64")
    R["rec"]      += bc(r, NR)
    R["rec_c2"]   += bc(r[c >= 2], NR)
    R["rec_k1"]   += bc(r[k == 1], NR)
    R["rec_w100"] += bc(r[w <= 100], NR)
    R["tok"]      += np.bincount(r, weights=k, minlength=NR).astype("int64")
    T             += bc((r[pos] * 2 + ex_) * 2 + (c[pos] >= 2), NR * 4).reshape(NR, 2, 2)
    print(f"  chunk {i}: {R['rec'].sum():,} records", end="\r")

assert R["rec"].sum() == EXPECTED, "record count differs from Cell 35"
assert R["tok"].sum() == T.sum() == A_wt.sum()
print(f"\nrecords: {R['rec'].sum():,} (Cell 35: {EXPECTED:,}) | tokens: {T.sum():,} (Cell 35: 19,986,999)")

# ---- 2. width versus evidence, label and number of circRNAs listed -----------------------------------------------------
rec_w = A_w.sum(axis=1)
wt = pd.DataFrame({"records": rec_w, "% of records": 100 * rec_w / rec_w.sum()}, index=WL)
for j, lab in enumerate(["c=1", "c=2", "c=3", "c=4", "c>=5"]):
    wt[lab] = 100 * A_w[:, j] / rec_w                                    # c = clusterClipExpNum
wt["c>=2"]                = 100 * A_w[:, 1:].sum(axis=1) / rec_w
wt["exonic tokens %"]     = 100 * A_wt[:, 1] / A_wt.sum(axis=1)
wt["circRNAs per record"] = A_wk / rec_w
print("\nnarrow width of the cluster (nt): records, evidence (% of records), exonic share of tokens, circRNAs per record:\n",
      wt.round(1).to_string())

# ---- 3. profile of every RBP (all records of the RBP, not only those of task circRNAs) --------------------------------
prof = pd.DataFrame({
    "records":                  R["rec"],
    "circRNAs":                 rbp_df.reindex(names)["circ all"].to_numpy(),
    "task circRNAs":            per_rbp["exonic & c>=2"],                  # task circRNAs under 'exonic & c>=2' (Cell 36)
    "exonic %":                 100 * T[:, 1].sum(axis=1) / T.sum(axis=(1, 2)),
    "c>=2 % (records)":         100 * R["rec_c2"] / R["rec"],
    "single-circRNA records %": 100 * R["rec_k1"] / R["rec"],
    "circRNAs per record":      R["tok"] / R["rec"],
    "width <= 100 nt %":        100 * R["rec_w100"] / R["rec"]}, index=names)
feat = ["exonic %", "c>=2 % (records)", "single-circRNA records %", "circRNAs per record", "width <= 100 nt %"]
use  = prof[prof["task circRNAs"] >= FLOOR]
print(f"\nRBPs: {len(prof)} | with >= {FLOOR} task circRNAs under 'exonic & c>=2': {len(use)}")
print("\nquantiles of the RBP properties, all RBPs:\n", prof[feat].quantile([.05, .25, .5, .75, .95]).round(1).T.to_string())
print(f"\nsame, the {len(use)} RBPs above the floor:\n", use[feat].quantile([.05, .25, .5, .75, .95]).round(1).T.to_string())

named = [x for x in ["U2AF2", "SCAF8", "SCAF4", "RNPS1", "ALYREF", "IGF2BP2", "FUS", "QKI", "EIF4A3", "ELAVL1"] if x in prof.index]
print("\nnamed RBPs (broad binders of Cell 36 first, then the RBPs of the old API block and ELAVL1):\n", prof.loc[named].round(1).to_string())
for col in ["circRNAs per record", "exonic %"]:
    s = use[col].sort_values()
    print(f"\n{col}, RBPs above the floor: lowest 6 {s.head(6).round(1).to_dict()}\n  highest 6 {s.tail(6).round(1).to_dict()}")

# ---- 4. which properties go together across RBPs? -----------------------------------------------------------------------
big = prof[prof["records"] >= MIN_REC]
print(f"\nSpearman correlation across the {len(big)} RBPs with >= {MIN_REC} records:\n",
      big[feat + ["circRNAs"]].corr(method="spearman").round(2).to_string())

  chunk 10: 4,669,990 records
records: 4,669,990 (Cell 35: 4,669,990) | tokens: 19,986,999 (Cell 35: 19,986,999)

narrow width of the cluster (nt): records, evidence (% of records), exonic share of tokens, circRNAs per record:
             records  % of records   c=1   c=2   c=3   c=4  c>=5  c>=2  exonic tokens %  circRNAs per record
<= 10 nt     720030          15.4  92.5   7.1   0.2   0.1   0.1   7.5             43.0                  4.1
11-39 nt      41686           0.9  52.5  40.5   5.5   1.1   0.4  47.5             76.1                  4.5
40 nt       2206764          47.3  92.1   7.0   0.5   0.3   0.1   7.9             45.7                  4.3
41-100 nt    974234          20.9   6.9  52.0  18.6  10.0  12.5  93.1             59.3                  4.6
101-300 nt   555082          11.9  15.5  36.4  17.6  11.2  19.3  84.5             57.7                  4.2
> 300 nt     172194           3.7   4.7  15.7  15.1  14.7  49.7  95.3             60.0                  3.9

RBPs: 281 | wit

**Interpretation.** *(A) Integrity.* The pass reads 4,669,990 records and 19,986,999 tokens, equal to Cell 35. *(B) Width and evidence are tied together.* Two narrow classes hold most of the records: 40 nt wide (2,206,764 records, 47.3%) and 10 nt or less (720,030, 15.4%), together 62.7% (derived). In both, 92% of the records rest on one experiment (`clusterClipExpNum = 1`: 92.1% and 92.5%) and only 7.9% and 7.5% reach 2. From 41 nt upwards the picture reverses: 93.1% of the records of 41–100 nt, 84.5% of 101–300 nt and 95.3% of those over 300 nt have `c ≥ 2`, and 49.7% of the widest have `c ≥ 5`. The 11–39 nt class is small (41,686 records, 0.9%) and intermediate (47.5% with `c ≥ 2`). Of the 2,881,510 single-experiment records (Cell 35), about 93.6% are 40 nt or at most 10 nt wide, and about 86% of the records with `c ≥ 2` are wider than 40 nt (both derived from the rounded percentages). The width of a cluster therefore looks like a consequence of how many experiments it merges: a single experiment gives a window of 40 nt (or a very narrow peak), and merging several experiments gives wider intervals (inference; the documentation was not consulted and the construction of the clusters was not tested). This also answers the open point of Cell 34: the frequent width of 40 nt is the typical single-experiment cluster, not a separate class with different evidence. *(C) Label and circRNAs listed.* The exonic share of the tokens is lower in the narrow single-experiment classes (43.0% for at most 10 nt, 45.7% for 40 nt) than from 41 nt upwards (57.7–60.0%), against 50.5% overall; the small 11–39 nt class has 76.1%. Because width and evidence are tied, this cannot be separated into an effect of the label and one of the evidence in this table (Cell 35 gave 45.7% with `c ≥ 2` for exonic tokens against 33.2% for intron-only ones). The mean number of circRNAs listed per record hardly changes with the width (3.9–4.6, overall 4.3 = 19,986,999 / 4,669,990), so a wider cluster does not list more circRNAs. *(D) Spread of the RBPs.* The RBPs differ strongly. The exonic share of an RBP has a median of 55.8% and runs from 8.9% (5th percentile) to 95.8% (95th); the share of records with `c ≥ 2` has a median of 26.7% (5th percentile 0.0%, 95th 63.6%); the share of single-circRNA records is more uniform (28.7–50.8%), and circRNAs per record run from 2.9 to 6.3. The 87 RBPs with at least 50 task circRNAs under "exonic and `c ≥ 2`" are shifted towards more exonic tokens (median 67.2%), more multi-experiment records (40.1%) and wider clusters (87.1% of the records are at most 100 nt, against 93.6%). This is partly built into the selection, since the floor is counted under the exonic and `c ≥ 2` condition (inference). *(E) The broad binders.* U2AF2, SCAF8, SCAF4, RNPS1 and ALYREF each have edges to 62,990–67,102 of the 89,255 circRNAs (70.6–75.2%, derived), and they have a high share of multi-experiment records (50.5–70.1%, against a median of 26.7%) and fewer narrow clusters (76.3–86.8% at most 100 nt). Their exonic share is not a common property: 91.3% for RNPS1 and 70.5% for ALYREF, but 53.0% for U2AF2, close to the median, and 60.3% and 66.7% for SCAF8 and SCAF4. The remark of Cell 36 (D) that these RBPs bind the exons of the host transcript is therefore supported for RNPS1 and, less clearly, for ALYREF and SCAF4, and not for U2AF2 (inference; the remark was itself an inference). What the five share is coverage of most circRNAs together with well-supported clusters. IGF2BP2 (65,054 circRNAs, 72.9%) and ELAVL1 (66,385, 74.4%) cover as many circRNAs but with different profiles: IGF2BP2 has 40.0% exonic and 28.8% `c ≥ 2` (near the median), ELAVL1 only 12.8% exonic. QKI has few records (5,571), 14.0% exonic tokens and 16 task circRNAs under the strictest variant, which agrees with Cell 36 (E). *(F) Extremes among the 87 RBPs.* The fewest circRNAs per record are FAM120A (3.3), GRSF1, UPF1, ELAVL3, RBFOX1 and ZCCHC14 (3.6); the most are GRWD1 (5.8), CENPC (6.3), YBX3 (6.5), FXR2 (6.9), BCLAF1 (7.0) and FXR1 (7.9). The lowest exonic shares are HNRNPC (11.7%), ELAVL1 (12.8%), CSTF2 (13.4%), HNRNPK (15.2%), CSTF2T (17.1%) and PTBP1 (22.1%); the highest are DDX6 (96.9%), MTDH, BCLAF1, FXR2 (98.2%), YBX1 and FXR1 (99.1%). FXR1, FXR2 and BCLAF1 are at the top of both lists: almost all their clusters are exonic and list 7–8 circRNAs each, so their edges are probably supported by exons shared by many overlapping circRNAs and are little specific to one circRNA (inference; the overlap of the circRNAs was not examined). *(G) Relations across RBPs (253 RBPs with at least 100 records, Spearman).* The share of single-circRNA records and the number of circRNAs per record are almost redundant (ρ = −0.89). RBPs with a higher exonic share list more circRNAs per record (ρ = 0.49) and have fewer single-circRNA records (ρ = −0.51), as expected if exons are shared by overlapping circRNAs and introns less so (inference). The share of `c ≥ 2` records goes with fewer narrow clusters (ρ = −0.48), the RBP-level counterpart of (B), and RBPs with more circRNAs have fewer narrow clusters (ρ = −0.61), probably because RBPs with many experiments have more merged clusters (inference; not tested). The exonic share and the share of `c ≥ 2` records are nearly independent (ρ = 0.12), so the exonic label and the evidence threshold carry different information.

**Not checked.** (i) The documented meaning of the widths of 40 nt and 10 nt or less, and whether the 40 nt clusters are windows of fixed size around one site. (ii) Every figure counts records (lines), not clusters; 12.2% of the clusters occupy more than one line (Cell 35). (iii) The RBP properties use all records of an RBP, not only those of task circRNAs. (iv) The causes proposed in (D), (F) and (G) are inferences. (v) The known function of the RBPs named here was not consulted.

**Decision.** (i) The cluster width is not stored per edge and is not used as an evidence column, because it is a function of `clusterClipExpNum` (single-experiment records are 40 nt or at most 10 nt wide in about 94% of the cases, derived). (ii) The four evidence variants of Cell 35 stay. The exonic variant and the `c ≥ 2` variant are not redundant (ρ = 0.12 across RBPs), so "exonic and `c ≥ 2`" is kept as the strict variant and the other two as controls. (iii) Cell 38 also saves a table of RBPs with the properties of this cell (records, circRNAs, task circRNAs, exonic share, share of `c ≥ 2` records, circRNAs per record, share of records of at most 100 nt), so that the RBPs of the head can be described and chosen from it. The share of single-circRNA records is left out because it is redundant with circRNAs per record. (iv) The RBP floor is not fixed here. The floor of 50 task circRNAs selects RBPs with more exonic and better supported clusters than average (D), so it is reported together with the properties above. (v) RBPs that are linked to about 70% of all circRNAs (U2AF2, SCAF8, SCAF4, RNPS1, ALYREF, IGF2BP2, ELAVL1) carry little information as a label; the label rebuild (old Cell 43) reports the positive rate of every RBP among the task circRNAs, and these RBPs are examined with and without a sensitivity run before the head is fixed. (vi) Cell 36 (D) stays, with the correction of (E): broad binders are defined by coverage and evidence, and an exonic majority is shown only for some of them. (vii) The edges of RBPs that list many circRNAs per cluster (FXR1, FXR2, BCLAF1) are treated through the private-cluster and `k_min` columns of Cell 38, not by a separate rule. (viii) The next step, Cell 37c, checks what the gene-level file `hg38_RBPcircRNA.txt` adds.

### 37c. What the Gene-Level RBP Table Adds to the Cluster Table

**Objective.** Profile the gene-level ENCORI RBP table (`hg38_RBPcircRNA.txt`, 17 columns, no circRNA identifier) and compare it with the cluster table at the level of (RBP, host gene) pairs, in order to establish (i) how many RBPs, genes and (RBP, gene) pairs it contains and whether its pairs are unique, (ii) whether it is an aggregation of the cluster table (does `clusterNum` equal the number of clusters of the pair in the cluster table, and `clipExpNum` its maximum evidence), (iii) whether it contains pairs that the cluster table lacks, and (iv) whether it can therefore add anything to the circRNA-level edges of Cell 38 or only serves as a gene-level check.

In [78]:
# Cell 37c - What does the gene-level RBP table add to the cluster table? (read-only: nothing is saved)
# Uses from earlier cells: ENC_RBP_DIR (Cell 1), RBP_FILE (Cell 34), rbp_code (Cell 35).
GENE_FILE = ENC_RBP_DIR / "hg38_RBPcircRNA.txt"
pc_ = lambda a, b: f"{100 * a / b:.1f}%"

# ---- 1. the gene-level table -------------------------------------------------------------------------------------------
gl = pd.read_csv(GENE_FILE, sep="\t", keep_default_na=False, na_values=["NA"],
                 usecols=["RBP", "geneID", "geneName", "geneType", "clusterNum", "clipExpNum", "clipIDnum", "pvalue"])
print(f"records: {len(gl):,} | missing values per column:", gl.isna().sum()[lambda s: s > 0].to_dict() or "none")
rg, rc = set(gl["RBP"]), set(rbp_code)
print(f"RBPs: {len(rg)} | not in the cluster table: {sorted(rg - rc)[:5]} ({len(rg - rc)}) | in the cluster table but not here: {len(rc - rg)}")
n_pairs = len(gl.drop_duplicates(["RBP", "geneID"]))
print(f"distinct (RBP, geneID): {n_pairs:,} | repeated lines: {len(gl) - n_pairs:,} | geneIDs: {gl['geneID'].nunique():,} | geneNames: {gl['geneName'].nunique():,}")
print("geneType:", gl["geneType"].value_counts().to_dict())
print("clusterNum quantiles:", gl["clusterNum"].quantile([.05, .25, .5, .75, .95, 1]).astype(int).to_dict(), f"| clusterNum == 1: {pc_((gl['clusterNum'] == 1).sum(), len(gl))}")
print("clipExpNum quantiles:", gl["clipExpNum"].quantile([.05, .25, .5, .75, .95, 1]).astype(int).to_dict(),
      f"| clipExpNum == clipIDnum: {(gl['clipExpNum'] == gl['clipIDnum']).mean():.4f}")
gl = gl.drop_duplicates(["RBP", "geneID"]).assign(RBP=lambda d: d["RBP"].map(rbp_code).fillna(-1).astype("int64"))

# ---- 2. cluster table aggregated to (RBP, host gene) pairs (one chunked pass; clusters can span chunks, so de-duplicate at the end) ----
parts = []
for i, ch in enumerate(pd.read_csv(RBP_FILE, sep="\t", usecols=["RBP", "clusterID", "clusterClipExpNum", "clipExpNum", "geneID"],
                                   chunksize=1_000_000, keep_default_na=False, na_values=["NA"]), 1):
    ch["RBP"] = ch["RBP"].map(rbp_code).astype("int64")
    parts.append(ch.groupby(["RBP", "geneID", "clusterID"], sort=False)
                   .agg(c=("clusterClipExpNum", "max"), clip=("clipExpNum", "max")).reset_index())
    print(f"  chunk {i}", end="\r")
cl = pd.concat(parts, ignore_index=True).groupby(["RBP", "geneID", "clusterID"], sort=False).agg(c=("c", "max"), clip=("clip", "max")).reset_index()
pc = cl.groupby(["RBP", "geneID"]).agg(n_cl=("clusterID", "size"), c_max=("c", "max"), clip_max=("clip", "max")).reset_index()
print(f"\ncluster table: {len(cl):,} (RBP, gene, cluster) rows | {len(pc):,} distinct (RBP, geneID) pairs | {pc['geneID'].nunique():,} genes")

# ---- 3. comparison at the level of (RBP, gene) pairs ----------------------------------------------------------------------
m = gl.merge(pc, on=["RBP", "geneID"], how="outer", indicator=True)
vc = m["_merge"].value_counts()
both, only_g, only_c = int(vc.get("both", 0)), int(vc.get("left_only", 0)), int(vc.get("right_only", 0))
print(f"\npairs in both tables: {both:,} | only in the gene-level table: {only_g:,} | only in the cluster table: {only_c:,}")
b = m[m["_merge"] == "both"]
for lab, a, x in [("clusterNum vs number of clusters", b["clusterNum"], b["n_cl"]),
                  ("clipExpNum (gene level) vs max clipExpNum of the clusters", b["clipExpNum"], b["clip_max"]),
                  ("clipExpNum (gene level) vs max clusterClipExpNum", b["clipExpNum"], b["c_max"])]:
    print(f"  {lab:60s} equal {(a == x).mean():.4f} | gene level larger {(a > x).mean():.4f} | smaller {(a < x).mean():.4f}")

go = m[m["_merge"] == "left_only"]
if len(go):
    gi = go["geneID"].isin(pc["geneID"])
    print(f"\ngene-level pairs without cluster rows: {len(go):,} | gene occurs in the cluster table for another RBP: {int(gi.sum()):,} | gene absent from the cluster table: {int((~gi).sum()):,}")
    print("their clusterNum quantiles:", go["clusterNum"].quantile([.25, .5, .75, 1]).astype(int).to_dict(), "| clipExpNum quantiles:", go["clipExpNum"].quantile([.25, .5, .75, 1]).astype(int).to_dict())
    print("examples:", go[["RBP", "geneID", "geneName", "clusterNum", "clipExpNum"]].head(5).assign(RBP=lambda d: d["RBP"].map({v: k for k, v in rbp_code.items()})).to_string(index=False))
co = m[m["_merge"] == "right_only"]
print(f"\ncluster-table pairs absent from the gene-level table: {len(co):,} | share of the cluster-table pairs: {pc_(len(co), len(pc))}")

records: 899,786 | missing values per column: none
RBPs: 281 | not in the cluster table: [] (0) | in the cluster table but not here: 0
distinct (RBP, geneID): 899,786 | repeated lines: 0 | geneIDs: 14,867 | geneNames: 13,530
geneType: {'circRNA': 899786}
clusterNum quantiles: {0.05: 1, 0.25: 1, 0.5: 2, 0.75: 5, 0.95: 18, 1.0: 1296} | clusterNum == 1: 36.6%
clipExpNum quantiles: {0.05: 1, 0.25: 1, 0.5: 2, 0.75: 3, 0.95: 9, 1.0: 94} | clipExpNum == clipIDnum: 0.3880
  chunk 5
cluster table: 4,669,990 (RBP, gene, cluster) rows | 899,786 distinct (RBP, geneID) pairs | 14,867 genes

pairs in both tables: 899,786 | only in the gene-level table: 0 | only in the cluster table: 0
  clusterNum vs number of clusters                             equal 1.0000 | gene level larger 0.0000 | smaller 0.0000
  clipExpNum (gene level) vs max clipExpNum of the clusters    equal 1.0000 | gene level larger 0.0000 | smaller 0.0000
  clipExpNum (gene level) vs max clusterClipExpNum             equal 0.7136 | ge

**Interpretation.** *(A) The two tables hold the same pairs.* The gene-level table has 899,786 records, one per (RBP, `geneID`) pair (no pair is repeated), for all 281 RBPs of the cluster table and none outside it. The cluster table, aggregated to (RBP, `geneID`) pairs, gives exactly 899,786 pairs. The merge finds 899,786 pairs in both tables, none only in the gene-level table and none only in the cluster table. *(B) The gene-level values are aggregates of the cluster table.* `clusterNum` equals the number of distinct clusters of the pair in the cluster table in 100.00% of the pairs, and `clipExpNum` equals the maximum `clipExpNum` of the lines of the pair in 100.00% (never larger, never smaller). It equals the maximum `clusterClipExpNum` in only 71.4% and is larger in 28.6% of the pairs, as expected, since line-level `clipExpNum` is never below `clusterClipExpNum` (Cell 35). `clipExpNum` equals `clipIDnum` in 38.8% of the pairs, so the two gene-level columns are different quantities; `clipIDnum` was not compared with the cluster table. *(C) Every line is a distinct (cluster, gene) combination.* The cluster table contains 4,669,990 distinct (RBP, `geneID`, `clusterID`) rows, which equals the number of lines, so no combination occupies two lines. A cluster that occupies several lines (12.2% of the clusters, Cell 35) therefore has different `geneID` values on those lines, which means that one line is one cluster on one host transcript (derived). This answers an open point of Cell 34 (whether the repeated lines are one per host transcript) and agrees with the disjoint circRNA sets of the lines of a cluster found in Cell 35. It also makes `clusterNum` equal to the number of lines of the pair (derived). *(D) What the table contains.* All 899,786 records have `geneType = circRNA`. There are 14,867 `geneID` values (transcript accessions; Cell 35 gave NM/NR in 95.1%) and 13,530 gene names, so some names carry several accessions (about 1.10 accessions per name, derived). On average a gene is bound by 60.5 of the 281 RBPs (21.5% of the possible pairs, derived). The median pair rests on 2 clusters (5th percentile 1, 95th percentile 18, maximum 1,296), 36.6% of the pairs on a single one, and the median pair has `clipExpNum` 2 (95th percentile 9, maximum 94).

**Not checked.** (i) The `pvalue` of the gene-level table against that of the cluster table. (ii) The meaning of `clipIDnum` and its relation to the number of `clipID` entries. (iii) The experiment-type columns (`eclip`, `hitsclip`, `iclip`, `parclip`, `other`, `site`, `peak`), which are in neither edge definition. (iv) Whether a gene name with several accessions refers to one host gene in the miRNA table too.

**Decision.** (i) The gene-level table is redundant: it has no pair, no circRNA identifier and no value that cannot be reproduced from the cluster table. It is not loaded in Cell 38 or in the package, and the decision of Cell 27 (ii) now rests on data. (ii) A small test in `tests/` aggregates a synthetic cluster table to (RBP, `geneID`) pairs and checks that `clusterNum` is the number of clusters and `clipExpNum` the maximum `clipExpNum`, so that the definition of Cell 33's counterpart stays documented. (iii) A cluster is counted once per (RBP, `clusterID`) for the edge columns of Cell 38 (`n_clusters` and so on), as in Cells 35–36; a cluster that occupies lines of several host genes still counts once per edge. (iv) For any gene-level analysis (the sibling-leakage test and the grouped folds of old Cells 44 and 46), the host gene of a circRNA is taken from the circRNA table, and the choice between `geneID` (transcript accession) and `geneName` is made when those cells are rebuilt, because the two differ (14,867 against 13,530). (v) The next step is Cell 38: the full RBP edge tables with the columns listed in Cell 36 (iv), the RBP property table of Cell 37b (iii) and no gene-level input.

### 38. ENCORI RBP Tables: (RBP, circRNA) Edges with Evidence Columns and the RBP Table

**Objective.** Build, in a single chunked pass over the cluster table, the full table of (RBP, circRNA) edges with the evidence columns decided in Cells 35–37, and a table of the 281 RBPs, and save both, in order to establish (i) that the edge keys and the counts of the five evidence variants equal those of Cell 35 exactly, (ii) how many clusters support an edge, how many of them are exonic or private (list this circRNA alone over all their lines), and how strongly the support depends on shared clusters at each evidence threshold, as was done for the miRNA edges in Cell 33, (iii) how many circRNAs the smallest supporting cluster lists (`k_min_c1`, `k_min_c2`), so that a cap on shared clusters can be proposed later, and (iv) how many task circRNAs each RBP has under each variant, together with the cluster properties of Cell 37b, so that the RBPs of the multi-task head can be chosen from one table.

In [83]:
# Cell 38 - ENCORI RBP tables: (RBP, circRNA) edges with evidence columns, and the RBP table (one chunked pass; saves two files)
# Uses from earlier cells: RBP_FILE (Cell 34); BASE (Cell 29); rbp_code, E, VARIANTS, EXPECTED (Cell 35); circ (Cell 30);
# per_rbp (Cell 36, dict of arrays); prof (Cell 37b); PROCESSED_DIR (Cell 1).
assert isinstance(per_rbp, dict), "per_rbp must be the dict of Cell 36"
CHUNK = 500_000                                        # lower this if memory is short
USE   = ["RBP", "clusterID", "clusterClipExpNum", "clipExpNum", "clusterClipSiteNum", "pvalue", "txIDcat", "txInfo"]
KBIG  = 32_000                                         # sentinel for 'no cluster with c >= 2'; replaced by 0 after the minimum
NR    = len(rbp_code)
names = [k for k, _ in sorted(rbp_code.items(), key=lambda kv: kv[1])]
task_num = circ["circrna_id"].str[9:].astype("int64").to_numpy()   # task circRNAs; `num` of Cell 30 was overwritten by Cell 37
assert len(np.unique(task_num)) == 1_644, "the task circRNA list differs from Cell 30"
DT    = {"n_clusters": "int32", "n_clusters_exonic": "int32", "n_clusters_private": "int32", "c_max": "int16",
         "c_max_exonic": "int16", "c_max_private": "int16", "k_min_c1": "int16", "k_min_c2": "int16",
         "clip_max": "int16", "site_max": "int16", "pvalue_min": "float32"}
SUF   = {"all": "all", "c>=2": "c2", "c>=3": "c3", "exonic": "exonic", "exonic & c>=2": "exonic_c2"}
parts, rbp_rows, S = [], [], dict(tokens=0, blocks=0)

def flush(rbp, buf):
    """Edge table of one RBP block (the file is sorted by RBP, so a cluster never leaves its block; checked after the pass)."""
    t = pd.concat(buf, ignore_index=True)
    code = rbp_code[rbp]
    cl   = pd.factorize(t["cl"])[0].astype("int64")                       # cluster number within the block
    assert len(np.unique(cl * BASE + t["circ"].to_numpy())) == len(t), f"{rbp}: a (cluster, circRNA) triple occurs twice"   # Cell 35: all tokens are distinct triples
    k    = np.bincount(cl)[cl]                                            # circRNAs of the cluster over all its lines (triples are distinct)
    c, ex = t["cexp"].to_numpy(), t["exonic"].to_numpy()
    e = pd.DataFrame({"circ": t["circ"].to_numpy(), "ex": ex, "priv": (k == 1).astype("int8"), "k": k, "c": c,
                      "c_ex": np.where(ex == 1, c, 0), "c_priv": np.where(k == 1, c, 0),
                      "k_c2": np.where(c >= 2, k, KBIG), "clip": t["clip"].to_numpy(), "site": t["site"].to_numpy(),
                      "pv": t["pv"].to_numpy()})
    g = e.groupby("circ").agg(n_clusters=("c", "size"), n_clusters_exonic=("ex", "sum"), n_clusters_private=("priv", "sum"),
                              c_max=("c", "max"), c_max_exonic=("c_ex", "max"), c_max_private=("c_priv", "max"),
                              k_min_c1=("k", "min"), k_min_c2=("k_c2", "min"), clip_max=("clip", "max"),
                              site_max=("site", "max"), pvalue_min=("pv", "min"))
    g["k_min_c2"] = g["k_min_c2"].where(g["k_min_c2"] < KBIG, 0)          # 0 = no cluster of the edge has c >= 2
    g = g.astype(DT).reset_index()
    g.insert(1, "rbp_code", np.int16(code))
    parts.append(g)
    rbp_rows.append({"rbp": rbp, "clusters": int(cl.max()) + 1, "triples": len(t)})
    S["tokens"] += len(t); S["blocks"] += 1

# ---- 1. one pass; tokens (cluster line, circRNA) as in Cells 35 and 37; label and circRNA from the same txInfo token ------
cur, buf, n_rows = None, [], 0
for i, ch in enumerate(pd.read_csv(RBP_FILE, sep="\t", usecols=USE, chunksize=CHUNK, keep_default_na=False, na_values=["NA"]), 1):
    n_rows += len(ch)
    cat, tok = ch["txIDcat"].str.split(",").explode(), ch["txInfo"].str.split("|").explode()
    pt = tok.str.split(":", expand=True)
    assert pt.shape[1] == 4, f"chunk {i}: txInfo token with an unexpected number of fields"
    recs, cn, inn = cat.index.to_numpy("int64"), cat.str[9:].astype("int64").to_numpy(), pt[0].str[9:].astype("int64").to_numpy()
    assert np.array_equal(np.sort(recs * BASE + cn), np.sort(recs * BASE + inn)), f"chunk {i}: txInfo and txIDcat list different circRNAs"
    pos = ch.index.get_indexer(tok.index)
    tk = pd.DataFrame({"circ": inn, "exonic": pt[2].str.contains("Exon").to_numpy().astype("int8"),
                       "rbp": ch["RBP"].to_numpy()[pos], "cl": ch["clusterID"].to_numpy()[pos],
                       "cexp": ch["clusterClipExpNum"].to_numpy("int64")[pos], "clip": ch["clipExpNum"].to_numpy("int64")[pos],
                       "site": ch["clusterClipSiteNum"].to_numpy("int64")[pos], "pv": ch["pvalue"].to_numpy("float32")[pos]})
    for rbp, gg in tk.groupby("rbp", sort=False):
        if rbp != cur:
            if cur is not None:
                flush(cur, buf)
            cur, buf = rbp, []
        buf.append(gg.drop(columns="rbp"))
    print(f"  chunk {i}: {n_rows:,} records", end="\r")
flush(cur, buf)
assert n_rows == EXPECTED and S["blocks"] == NR, "records differ from Cell 35, or the file is not sorted by RBP"
edges = pd.concat(parts, ignore_index=True)
cn = edges["circ"].to_numpy("int64")
key = cn * BASE + edges["rbp_code"].to_numpy("int64")                     # same key as Cell 35: circRNA * BASE + RBP code
print(f"\ntokens: {S['tokens']:,} (Cell 35: 19,986,999) | edges: {len(edges):,} | circRNAs: {len(np.unique(cn)):,} | RBPs: {edges['rbp_code'].nunique()}")
assert S["tokens"] == 19_986_999 and len(np.unique(key)) == len(key)

# ---- 2. consistency with Cell 35 (exact equality of the edge keys of the five variants) ---------------------------------
mask = {"all": np.ones(len(edges), bool), "c>=2": (edges["c_max"] >= 2).to_numpy(), "c>=3": (edges["c_max"] >= 3).to_numpy(),
        "exonic": (edges["n_clusters_exonic"] > 0).to_numpy(), "exonic & c>=2": (edges["c_max_exonic"] >= 2).to_numpy()}
print("\nedge keys equal to those of Cell 35:", {v: bool(np.array_equal(np.sort(key[mask[v]]), E[v])) for v in VARIANTS})
assert all(np.array_equal(np.sort(key[mask[v]]), E[v]) for v in VARIANTS)
# internal consistency of the columns
assert (edges["c_max_exonic"] <= edges["c_max"]).all() and (edges["c_max_private"] <= edges["c_max"]).all()
assert (edges["n_clusters_exonic"] <= edges["n_clusters"]).all() and (edges["n_clusters_private"] <= edges["n_clusters"]).all()
assert ((edges["n_clusters_exonic"] == 0) == (edges["c_max_exonic"] == 0)).all() and ((edges["n_clusters_private"] == 0) == (edges["c_max_private"] == 0)).all()
assert ((edges["k_min_c2"] == 0) == (edges["c_max"] < 2)).all() and (edges["k_min_c1"] >= 1).all()
assert (edges["k_min_c1"][mask["c>=2"]] <= edges["k_min_c2"][mask["c>=2"]]).all()
print("column checks passed (maxima over subsets <= overall maximum; counts of subsets <= n_clusters; k_min_c2 = 0 exactly when c_max < 2)")

# ---- 3. support of an edge: clusters, exonic and private clusters ---------------------------------------------------------
QS = [.05, .25, .5, .75, .95, 1]
pcnt = lambda a, b: f"{100 * a / b:.1f}%"
print("\nclusters per edge, quantiles:", edges["n_clusters"].quantile(QS).astype(int).to_dict(),
      f"| edges with one cluster: {pcnt((edges['n_clusters'] == 1).sum(), len(edges))}")
print(f"edges with at least one private cluster: {pcnt((edges['n_clusters_private'] > 0).sum(), len(edges))} | "
      f"edges whose clusters are all private: {pcnt((edges['n_clusters_private'] == edges['n_clusters']).sum(), len(edges))} | "
      f"with at least one exonic cluster: {pcnt(mask['exonic'].sum(), len(edges))}")
rows = []
for th in (1, 2, 3):
    a, p = (edges["c_max"] >= th).to_numpy(), (edges["c_max_private"] >= th).to_numpy()
    rows.append({"c >=": th, "edges (all clusters)": int(a.sum()), "edges (private clusters only)": int(p.sum()),
                 "edges with only shared clusters": int((a & ~p).sum()),
                 "circRNAs (all)": len(np.unique(cn[a])), "circRNAs (private)": len(np.unique(cn[p])),
                 "task circRNAs (all)": int(np.isin(task_num, np.unique(cn[a])).sum()), "task circRNAs (private)": int(np.isin(task_num, np.unique(cn[p])).sum())})
    
print("\nsupport when every cluster counts versus when only private clusters count:\n", pd.DataFrame(rows).set_index("c >=").T.to_string())

# ---- 4. number of circRNAs listed by the smallest supporting cluster (basis for a later cap) -------------------------------
for col in ["k_min_c1", "k_min_c2"]:
    s = edges[col][edges[col] > 0]
    print(f"\n{col} (n = {len(s):,}): quantiles {s.quantile([.25, .5, .75, .95, 1]).astype(int).to_dict()} | "
          + " | ".join(f"<= {x}: {pcnt((s <= x).sum(), len(s))}" for x in (1, 2, 5, 10, 20)))
print("\nclip_max, site_max and pvalue_min of the edges, quantiles:\n",
      edges[["clip_max", "site_max", "pvalue_min"]].quantile([0, .05, .25, .5, .75, .95, 1]).round(2).to_string())

# ---- 5. final edge table and RBP table ---------------------------------------------------------------------------------------
ct  = pd.read_csv(PROCESSED_DIR / "encori_circ_table.csv", usecols=["circbase_id", "id_check"])
idm = pd.Series(ct["id_check"].to_numpy(), index=ct["circbase_id"].str[9:].astype("int64"))
ed = pd.DataFrame({"circbase_id": "hsa_circ_" + edges["circ"].astype(str).str.zfill(7),
                   "rbp": pd.Categorical.from_codes(edges["rbp_code"].to_numpy(), names)})
ed = pd.concat([ed, edges.drop(columns=["circ", "rbp_code"])], axis=1)
ed["id_check"] = pd.Categorical(idm.reindex(cn).fillna("not in the miRNA table").to_numpy())
print("\nid_check of the edges:", ed["id_check"].value_counts().to_dict())

rt = pd.DataFrame(rbp_rows).set_index("rbp").reindex(names)
rt.insert(0, "records", prof.reindex(names)["records"].to_numpy())
tm = np.isin(cn, task_num)                                                       # edges of task circRNAs
for v in VARIANTS:
    rt["edges_" + SUF[v]] = np.bincount(edges["rbp_code"].to_numpy()[mask[v]], minlength=NR)
for v in VARIANTS:
    rt["task_" + SUF[v]] = np.bincount(edges["rbp_code"].to_numpy()[mask[v] & tm], minlength=NR)
    assert np.array_equal(rt["task_" + SUF[v]].to_numpy(), per_rbp[v]), f"task counts differ from Cell 36 ({v})"
assert np.array_equal(rt["edges_all"].to_numpy(), rbp_df.reindex(names)["circ all"].to_numpy()), "edges per RBP differ from Cell 35"
for src, dst in [("exonic %", "exonic_pct"), ("c>=2 % (records)", "c2_pct_records"),
                 ("circRNAs per record", "circrnas_per_record"), ("width <= 100 nt %", "width_le100_pct")]:
    rt[dst] = prof.reindex(names)[src].to_numpy()
print("\nRBPs with at least 50 task circRNAs, by variant:", {v: int((rt['task_' + SUF[v]] >= 50).sum()) for v in VARIANTS}, "(Cell 36: 189 | 115 | 54 | 151 | 87)")
print("\nRBP table, largest 6 by task circRNAs under 'exonic & c>=2':\n",
      rt.sort_values("task_exonic_c2", ascending=False).head(6).round(1).to_string())

# ---- 6. save ---------------------------------------------------------------------------------------------------------------------
F_EDGES, F_RBP = PROCESSED_DIR / "encori_rbp_edges_all.csv.gz", PROCESSED_DIR / "encori_rbp_table.csv"
ed.to_csv(F_EDGES, index=False, float_format="%.5f", compression="gzip")
rt.to_csv(F_RBP, float_format="%.4f")
for f, d in [(F_EDGES, ed), (F_RBP, rt)]:
    print(f"saved {f.name}: {len(d):,} rows x {d.shape[1]} columns | {f.stat().st_size / 1e6:,.1f} MB")
print("\nedge table, first rows:\n", pd.read_csv(F_EDGES, nrows=3).to_string(index=False))

  chunk 10: 4,669,990 records
tokens: 19,986,999 (Cell 35: 19,986,999) | edges: 4,788,414 | circRNAs: 89,255 | RBPs: 281

edge keys equal to those of Cell 35: {'all': True, 'c>=2': True, 'c>=3': True, 'exonic': True, 'exonic & c>=2': True}
column checks passed (maxima over subsets <= overall maximum; counts of subsets <= n_clusters; k_min_c2 = 0 exactly when c_max < 2)

clusters per edge, quantiles: {0.05: 1, 0.25: 1, 0.5: 2, 0.75: 4, 0.95: 14, 1.0: 1209} | edges with one cluster: 41.4%
edges with at least one private cluster: 9.9% | edges whose clusters are all private: 5.7% | with at least one exonic cluster: 72.1%

support when every cluster counts versus when only private clusters count:
 c >=                                   1        2        3
edges (all clusters)             4788414  2360505  1163844
edges (private clusters only)     474468   214243   103033
edges with only shared clusters  4313946  2146262  1060811
circRNAs (all)                     89255    86011    80089
cir

**Interpretation.** *(A) Consistency and size.* The pass reads 4,669,990 records and 19,986,999 tokens, equal to Cell 35, and produces 4,788,414 edges over 89,255 circRNAs and 281 RBPs. The edge keys equal those of Cell 35 exactly for all five evidence variants, and the column checks pass (maxima over exonic or private clusters never exceed the overall maximum; `k_min_c2` is 0 exactly when `c_max < 2`). An edge is supported by a median of 2 clusters (mean 4.2 = 19,986,999 / 4,788,414, derived; 95th percentile 14, maximum 1,209), and 41.4% of the edges rest on a single cluster. 72.1% have at least one exonic cluster. *(B) Dependence on shared clusters.* Only 9.9% of the edges have a private cluster (a cluster that lists this circRNA alone over all its lines), and 5.7% have only private clusters. At `c ≥ 1`, 4,313,946 of the 4,788,414 edges (90.1%, derived) are supported only by shared clusters; at `c ≥ 2` it is 2,146,262 of 2,360,505 (90.9%) and at `c ≥ 3` 1,060,811 of 1,163,844 (91.2%, derived). These shares are close to those of the miRNA side (88.7% and 89.7%, Cell 33), so the dependence is as strong for RBPs as for miRNAs. At the circRNA level, 15,183 of 89,255 circRNAs (17.0%, derived) have private-cluster support at `c ≥ 1`, 14,009 of 86,011 (16.3%) at `c ≥ 2` and 12,190 of 80,089 (15.2%) at `c ≥ 3` (miRNA side: 15.9% and 14.2%). Among the task circRNAs, 1,457, 1,403 and 1,294 have support at the three thresholds (equal to Cell 36). Among the task circRNAs, 234 of 1,457 (16.1%) have private-cluster support at `c ≥ 1`, 212 of 1,403 (15.1%) at `c ≥ 2` and 178 of 1,294 (13.8%) at `c ≥ 3` (derived), which is 14.2%, 12.9% and 10.8% of the 1,644 task circRNAs. The task circRNAs therefore depend on shared clusters about as strongly as the circRNAs of ENCORI as a whole (16.1% against 17.0% at `c ≥ 1`, 15.1% against 16.3% at `c ≥ 2`, 13.8% against 15.2% at `c ≥ 3`; derived). This differs from the miRNA side, where the task circRNAs kept private-site support twice as often as ENCORI as a whole (30.1% against 15.9% at `clipExpNum ≥ 1`, Cell 33), so the RBP labels of the task circRNAs rest almost entirely on shared clusters.. *(C) Size of the smallest supporting cluster.* The smallest cluster of an edge lists a median of 6 circRNAs (quartiles 3 and 11, 95th percentile 32, maximum 220). A cluster listing the circRNA alone supports 9.9% of the edges (`k_min_c1 ≤ 1`), which equals the share with a private cluster, and for `c ≥ 2` it is 9.1% (`k_min_c2 ≤ 1`), which equals 214,243 of 2,360,505 edges and the private count above, so the two columns agree with each other. A cap on the number of circRNAs of a cluster would keep, among the edges with `c ≥ 2`, 19.0% at 2, 45.0% at 5, 70.4% at 10 and 88.5% at 20 circRNAs; the distribution of `k_min_c1` is nearly identical (20.4%, 47.3%, 72.3%, 89.5%). *(D) Other evidence columns.* `clip_max` (line-level `clipExpNum`) has a median of 2 (5th percentile 1, 95th percentile 12, maximum 94); `site_max` has a median of 2 (95th percentile 10, maximum 2,069). `pvalue_min`, the most extreme value over the clusters of an edge, has a median of −9.73, a 5th percentile of −153.61 and a minimum of −383,934.16; its scale is unknown (Cell 34). *(E) Identifier flags.* Of the 4,788,414 edges, 4,777,071 belong to `consistent` circRNAs, 4,369 to circRNAs whose header differs but whose sequence agrees, 3,633 to those whose sequence mostly agrees, 2 to one whose sequence is not confirmed, and 3,339 to circRNAs absent from the miRNA table (the 304 circRNAs of Cell 35). The flagged edges are 8,004 (0.17%, derived). *(F) RBP table.* The saved table holds 281 rows and 17 columns. The numbers of RBPs with at least 50 task circRNAs equal Cell 36 (189, 115, 54, 151 and 87 for the five variants), and the task counts per RBP equal Cell 36 exactly (asserted). The five hubs of Cell 36 remain hubs under the strictest variant: U2AF2, SCAF4, SCAF8, RNPS1 and ALYREF keep between 50,941 and 54,087 edges under "exonic and `c ≥ 2`" (57.1–60.6% of the 89,255 circRNAs, derived), and IGF2BP2 keeps 37,372 (41.9%, derived), so the stricter variant does not remove their hub character. *(G) Files.* `encori_rbp_edges_all.csv.gz` (4,788,414 rows × 14 columns, 33.8 MB) and `encori_rbp_table.csv` (281 × 17) were written; the first rows of the edge table have the expected columns.

**Not checked.** (i) How many edges of the strict variant "exonic and `c ≥ 2`" depend only on shared clusters; `c_max_private` is not restricted to exonic clusters, so this combination cannot be read from the table. (ii) Whether the sharing is explained by overlapping isoforms of one host gene (inference, as on the miRNA side). (iii) The scale of `pvalue_min`. (iv) The identifier check of the 3,339 edges of circRNAs that are absent from the miRNA table.

**Decision.** (i) Both tables are kept as saved. The edge table stores the evidence columns; the five evidence variants are not applied as filters but are derived from them (`c_max`, `n_clusters_exonic`, `c_max_exonic`), which was checked against Cell 35.(ii) The shared-cluster dependence is reported with every threshold, as for the miRNA edges. A "private clusters only" variant is not used as a main variant, because 90–91% of the edges have no private cluster and only 234 of the 1,644 task circRNAs (14.2%) keep an RBP edge at `c ≥ 1` (212 at `c ≥ 2`, 178 at `c ≥ 3`), which is too few for a multi-task head. It remains available as a sensitivity analysis on the columns `n_clusters_private` and `c_max_private`.(iii) A cap on the number of circRNAs of a supporting cluster is not fixed here. `k_min_c1` and `k_min_c2` are stored so that the cap can be proposed when the labels are rebuilt (old Cell 43), together with the exonic and `c ≥ 2` conditions; the figures of (C) show what a cap of 5 or 10 would cost (55% or 30% of the `c ≥ 2` edges). (iv) `clip_max` and `site_max` are not thresholds (Cell 35); `pvalue_min` is descriptive only. (v) No edge is removed for an identifier flag. The 8,004 flagged edges and the 3,339 edges of circRNAs outside the miRNA table keep their value in `id_check`, as decided for the flagged identifiers in Cell 36 (vi). (vi) The RBPs of the multi-task head are chosen from `encori_rbp_table.csv` when the folds are rebuilt (old Cell 44), with the floor of 50 task circRNAs under the chosen variant as the starting point. The hubs (U2AF2, SCAF4, SCAF8, RNPS1, ALYREF, IGF2BP2, ELAVL1) are examined with and without a sensitivity run, as decided in Cell 37b (v). (vii) Next: a table of the circRNA universe (circRNAs of CircR2Disease, the miRNA table and the RBP table, with source flags and host gene) and then the label rebuild (old Cells 43–47) on the two edge tables.

### 39. The circRNA Universe: One Node Table for CircR2Disease, the ENCORI miRNA Table and the ENCORI RBP Table

**Objective.** Combine the circRNAs of the three sources into a single table with one row per circRNA, in order to establish (i) how many circRNAs the union contains and how the three sources overlap, (ii) how many of them are in the circBase FASTA index (the sequence source of the model), (iii) whether the chromosome and strand of the FASTA header agree with those of ENCORI and how this relates to the identifier flag of Cell 33, (iv) which host gene each circRNA belongs to, using the rule of the old Cell 44 (FASTA gene field, one singleton group per circRNA with a placeholder), and how many circRNAs outside the task share a host gene with a task circRNA, which is the sibling structure that the rebuilt splits and labels must respect, and (v) to save the table with the degrees of each circRNA under every evidence variant of the two ENCORI sides.

In [85]:
# Cell 39 - The circRNA universe: one row per circRNA of CircR2Disease, the ENCORI miRNA table and the ENCORI RBP table (saves one file)
# Uses from earlier cells: PROCESSED_DIR (Cell 1) and the files saved in Cells 6, 26, 33 and 38 (everything is re-read from disk).
import re
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)   # placeholder host-gene field of the circBase header (as old Cell 44)
num_of = lambda s: s.str[9:].astype("int64")                                   # 'hsa_circ_0009177' -> 9177

# ---- 1. sources, each indexed by the circRNA number ------------------------------------------------------------------------
task = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")                                   # CircR2Disease circRNAs (Cell 26)
fa   = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)      # circBase FASTA index (Cell 6)
mi   = pd.read_csv(PROCESSED_DIR / "encori_circ_table.csv")                               # ENCORI miRNA table, one row per circRNA (Cell 33)
rb   = pd.read_csv(PROCESSED_DIR / "encori_rbp_edges_all.csv.gz", usecols=["circbase_id", "c_max", "n_clusters_exonic", "c_max_exonic"])   # RBP edges (Cell 38)
for name, d, col in [("task", task, "circrna_id"), ("FASTA index", fa, "circbase_id"), ("miRNA table", mi, "circbase_id"), ("RBP edges", rb, "circbase_id")]:
    bad = int((~d[col].str.match(r"^hsa_circ_\d{7}$")).sum())
    print(f"{name:12s}: {len(d):,} rows | identifiers that are not 'hsa_circ_' + 7 digits: {bad}")
    assert bad == 0

T = task.assign(num=num_of(task["circrna_id"])).set_index("num")[["n_diseases", "length_band", "use_primary"]]
F = (fa.assign(num=num_of(fa["circbase_id"])).set_index("num")
       .rename(columns={"chrom": "fa_chrom", "strand": "fa_strand", "gene": "fa_gene"})[["fa_chrom", "fa_strand", "fa_gene", "seq_len"]])
M = mi.assign(num=num_of(mi["circbase_id"])).set_index("num")[["n_mirna", "n_mirna_clip1", "n_mirna_clip2", "n_mirna_private_clip1",
                                                               "enc_chrom", "enc_strand", "id_check"]]
R = pd.DataFrame({"num": num_of(rb["circbase_id"]), "n_rbp": 1, "n_rbp_c2": rb["c_max"] >= 2, "n_rbp_c3": rb["c_max"] >= 3,
                  "n_rbp_exonic": rb["n_clusters_exonic"] > 0, "n_rbp_exonic_c2": rb["c_max_exonic"] >= 2}).groupby("num").sum().astype("int32")
del rb

# ---- 2. the union and the source flags ---------------------------------------------------------------------------------------
ids = np.union1d(np.union1d(T.index.to_numpy(), M.index.to_numpy()), R.index.to_numpy())
u = pd.DataFrame(index=pd.Index(ids, name="num"))
u.insert(0, "circbase_id", "hsa_circ_" + u.index.astype(str).str.zfill(7))
for lab, src in [("in_task", T), ("in_fasta", F), ("in_mirna", M), ("in_rbp", R)]:
    u[lab] = u.index.isin(src.index)
u = u.join([T, F, M, R])
for c in ["n_mirna", "n_mirna_clip1", "n_mirna_clip2", "n_mirna_private_clip1", "n_rbp", "n_rbp_c2", "n_rbp_c3", "n_rbp_exonic", "n_rbp_exonic_c2"]:
    u[c] = u[c].fillna(0).astype("int32")
u["id_check"]    = u["id_check"].fillna("not in the miRNA table")
u["use_primary"] = u["use_primary"].fillna(False).astype(bool)
u["n_diseases"]  = u["n_diseases"].astype("Int64"); u["seq_len"] = u["seq_len"].astype("Int64")

print(f"\nuniverse: {len(u):,} circRNAs | circBase FASTA index: {len(F):,} | FASTA circRNAs outside the universe: {len(F) - int(u['in_fasta'].sum()):,}")
print("\noverlap of the sources (n circRNAs, of which in the FASTA index):\n",
      u.groupby(["in_task", "in_mirna", "in_rbp"]).agg(n=("circbase_id", "size"), in_fasta=("in_fasta", "sum")).to_string())
for lab, m in [("task", u["in_task"]), ("ENCORI miRNA table", u["in_mirna"]), ("ENCORI RBP table", u["in_rbp"])]:
    print(f"  {lab:20s} {int(m.sum()):>7,} circRNAs | in the FASTA index: {int((m & u['in_fasta']).sum()):>7,} ({100 * (m & u['in_fasta']).sum() / m.sum():.1f}%)")
tk = u[u["in_task"]]
cross = pd.crosstab(tk["in_mirna"], tk["in_rbp"])
print("\ntask circRNAs: rows in the miRNA table, columns in the RBP table:\n", cross.to_string())
assert (cross.loc[True, True], cross.loc[True, False], cross.loc[False, True], cross.loc[False, False]) == (1456, 29, 1, 158), "differs from Cell 36 (F)"

# ---- 3. FASTA header versus ENCORI: chromosome and strand, and the identifier flag of Cell 33 -----------------------------
b = u[u["in_fasta"] & u["in_mirna"]]
ok_c, ok_s = b["fa_chrom"].eq(b["enc_chrom"]), b["fa_strand"].eq(b["enc_strand"])
print(f"\ncircRNAs in the FASTA index and the miRNA table: {len(b):,} | chromosome equal {ok_c.mean():.4f} | strand equal {ok_s.mean():.4f} | "
      f"chromosome or strand differs: {int((~ok_c | ~ok_s).sum()):,} (Cell 30: 282 flagged identifiers)")
print(pd.crosstab(b["id_check"], (~ok_c | ~ok_s).rename("chromosome or strand differs")).to_string())

# ---- 4. host gene (rule of the old Cell 44) and siblings ------------------------------------------------------------------
for c in ["in_task", "in_fasta", "in_mirna", "in_rbp", "use_primary"]:           # object columns after the join; '~' needs real bool
    u[c] = u[c].astype(bool)
u["host_placeholder"] = u["fa_gene"].isna() | u["fa_gene"].astype(str).map(lambda x: bool(PLACE.match(x)))
u["host_group"] = np.where(u["host_placeholder"], "solo:" + u["circbase_id"], u["fa_gene"].astype(str))
tk = u[u["in_task"]]; mdl = tk[tk["use_primary"]]
print(f"\nplaceholder or missing host gene: universe {100 * u['host_placeholder'].mean():.1f}% | task {100 * tk['host_placeholder'].mean():.1f}% | "
      f"circRNAs outside the FASTA index {int((~u['in_fasta']).sum()):,}")
print(f"model-input circRNAs: {len(mdl):,} | groups: {mdl['host_group'].nunique():,} | singleton placeholder groups: "
      f"{int(mdl['host_placeholder'].sum()):,} (old Cell 44: 1,251 groups, 26 singletons)")

real    = ~u["host_placeholder"]
n_grp   = u[real].groupby("host_group").size()                              # circRNAs of the universe per host gene
n_task  = u[real & u["in_task"]].groupby("host_group").size()               # of which in the task
tk_real = tk[~tk["host_placeholder"]]
sib_all = tk_real["host_group"].map(n_grp) - 1                              # all other circRNAs of the host gene
sib_tk  = tk_real["host_group"].map(n_task) - 1                             # other task circRNAs of the host gene
sib_out = sib_all - sib_tk                                                  # siblings outside the task
q = [.25, .5, .75, .95, 1]
print(f"\ntask circRNAs with a real host gene: {len(tk_real):,} | host genes: {tk_real['host_group'].nunique():,}")
print("siblings in the universe, quantiles:      ", sib_all.quantile(q).astype(int).to_dict())
print("siblings that are task circRNAs:          ", sib_tk.quantile(q).astype(int).to_dict(), f"| with at least one: {100 * (sib_tk > 0).mean():.1f}%")
print("siblings outside the task, quantiles:     ", sib_out.quantile(q).astype(int).to_dict(), f"| with at least one: {100 * (sib_out > 0).mean():.1f}%")
assert (sib_out >= 0).all() and (sib_out + sib_tk == sib_all).all()
print("hubs among host genes of the task (circRNAs in the universe):", n_grp[tk_real["host_group"].unique()].sort_values(ascending=False).head(5).to_dict())

# do the siblings outside the task carry ENCORI labels? (relevant for any use of non-task circRNAs)
out = u[real & ~u["in_task"] & u["host_group"].isin(set(tk_real["host_group"]))]
print(f"\nnon-task circRNAs in host genes of the task: {len(out):,} | with a miRNA edge: {100 * out['in_mirna'].mean():.1f}% | "
      f"with an RBP edge: {100 * out['in_rbp'].mean():.1f}% | with an RBP edge at 'exonic & c>=2': {100 * (out['n_rbp_exonic_c2'] > 0).mean():.1f}%")
# ---- 5. save -------------------------------------------------------------------------------------------------------------------
cols = ["circbase_id", "in_task", "in_fasta", "in_mirna", "in_rbp", "n_diseases", "length_band", "use_primary", "seq_len",
        "fa_chrom", "fa_strand", "fa_gene", "host_placeholder", "host_group", "enc_chrom", "enc_strand", "id_check",
        "n_mirna", "n_mirna_clip1", "n_mirna_clip2", "n_mirna_private_clip1", "n_rbp", "n_rbp_c2", "n_rbp_c3", "n_rbp_exonic", "n_rbp_exonic_c2"]
F_UNI = PROCESSED_DIR / "circrna_universe.csv"
u[cols].to_csv(F_UNI, index=False)
print(f"\nsaved {F_UNI.name}: {len(u):,} rows x {len(cols)} columns | {F_UNI.stat().st_size / 1e6:,.1f} MB")
print(pd.read_csv(F_UNI, nrows=3).T.to_string())

task        : 1,644 rows | identifiers that are not 'hsa_circ_' + 7 digits: 0
FASTA index : 140,790 rows | identifiers that are not 'hsa_circ_' + 7 digits: 0
miRNA table : 91,050 rows | identifiers that are not 'hsa_circ_' + 7 digits: 0
RBP edges   : 4,788,414 rows | identifiers that are not 'hsa_circ_' + 7 digits: 0

universe: 91,512 circRNAs | circBase FASTA index: 140,790 | FASTA circRNAs outside the universe: 49,278

overlap of the sources (n circRNAs, of which in the FASTA index):
                              n in_fasta
in_task in_mirna in_rbp                
False   False    True      303      303
        True     False    2070     2070
                 True    87495    87495
True    False    False     158      158
                 True        1     True
        True     False      29       29
                 True     1456     1456
  task                   1,644 circRNAs | in the FASTA index:   1,644 (100.0%)
  ENCORI miRNA table    91,050 circRNAs | in the FASTA index:  91,050

**Interpretation.** *(A) The union.* The universe holds 91,512 circRNAs: 1,644 of the task, 91,050 of the ENCORI miRNA table and 89,255 of the ENCORI RBP table. The overlap of the sources is consistent with Cells 35 and 36. 88,951 circRNAs are in both ENCORI tables (87,495 + 1,456), 2,099 only in the miRNA table (2,070 + 29) and 304 only in the RBP table (303 + 1), as in Cell 35. The task circRNAs split as in Cell 36 (F): 1,456 in both tables, 29 only in the miRNA table, 1 only in the RBP table and 158 in neither (asserted). Beyond the task, 89,868 circRNAs (87,495 in both tables, 2,070 miRNA only, 303 RBP only) are in at least one ENCORI table. *(B) Sequence source.* Every circRNA of every source is in the circBase FASTA index (100.0% of the task, the miRNA table and the RBP table), so no circRNA of the universe lacks a sequence. The 158 task circRNAs absent from ENCORI have valid circBase identifiers of the FASTA, so their absence is not due to an unknown identifier; whether ENCORI covers a different circBase release was not tested. The index holds 140,790 circRNAs, of which 49,278 (35.0%, derived) are outside the universe. *(C) Chromosome and strand.* For the 91,050 circRNAs in both the FASTA index and the miRNA table, the chromosome of the FASTA header equals that of ENCORI in 99.98% and the strand in 99.71%. 282 circRNAs differ in at least one of the two, which corresponds to about 18 chromosome and 264 strand differences if the two sets do not overlap (derived from the rounded shares). The 282 are exactly the flagged identifiers of Cell 33: all 192 + 89 + 1 circRNAs of the three "header differs" classes differ, and none of the 90,768 `consistent` circRNAs does. The identifier flag of Cell 33 is therefore the chromosome/strand disagreement, and its three classes only subdivide it by the sequence test of Cell 32. The FASTA headers are hg19 and the ENCORI coordinates hg38, but chromosome and strand agree for 99.69% (90,768 of 91,050, derived), so the identifiers refer to the same loci for all but 0.31%. *(D) Host genes.* The host gene of the FASTA header is a placeholder or missing for 2.2% of the universe and 2.1% of the task. The rule of the old Cell 44 gives 1,251 groups for the 1,582 model-input circRNAs, 26 of them singleton placeholder groups, exactly as before, so the rebuilt folds can use the same grouping. The host genes of the task include very large ones: among the groups of the task, BIRC6 has 440 circRNAs in the universe, HSPG2 246, FLNA 205, DYNC1H1 185 and PRKDC 173. The 1,610 task circRNAs with a real host gene fall into 1,262 host genes (this set includes circRNAs that are not model input, hence the difference from 1,251 groups). A task circRNA has a median of 12 siblings in the universe (quartiles 5 and 23, 95th percentile 69, maximum 439). Few of these siblings belong to the task: the median is 0, the 95th percentile 3 and the maximum 8, and only 37.0% of the task circRNAs have at least one task sibling. Almost all siblings lie outside the task: the median is 11 (quartiles 5 and 22, 95th percentile 65, maximum 438), and 95.2% of the task circRNAs have at least one sibling outside the task. These siblings are numerous and well labelled: 20,155 non-task circRNAs sit in host genes of the task, 99.7% of them with a miRNA edge, 99.2% with an RBP edge and 95.2% with an RBP edge under "exonic & c>=2" (the two values of 95.2% are a coincidence of different quantities). *(E) File.* `circrna_universe.csv` was written (91,512 rows × 26 columns, 11.0 MB); the first rows have the expected columns (for example `hsa_circ_0000001` has a placeholder host gene, 3 miRNA edges and no RBP edge, and is not a task circRNA).

**Not checked.** (i) Whether the 158 task circRNAs absent from both ENCORI tables are absent because of a different circBase release in ENCORI (inference). (ii) The agreement of the FASTA host gene with the gene names of ENCORI (the ENCORI `geneName` of the old API trial held circRNA aliases, so it was not used). (iii) Whether circRNAs of one host gene that differ in their position (isoforms with disjoint exons) are comparable siblings. (iv) Whether sharing a host gene actually produces leakage between a task circRNA and its labelled non-task siblings (for example through similar miRNA or RBP profiles); the counts show that the opportunity is large, not that the effect exists.

**Decision.** (i) `circrna_universe.csv` is the node table of the graph: one row per circRNA of the union, with the source flags, the identifier flag, the host group and the degrees of both ENCORI sides under every evidence variant. (ii) The flags `in_task`, `in_fasta`, `in_mirna`, `in_rbp` and `use_primary` are cast to `bool` at the start of Cell 39, because the join turned them into object columns (this had corrupted the first counts of outside-the-index circRNAs and of siblings outside the task). Every later cell that reads the table must apply the same cast. (iii) No circRNA is removed for a missing sequence, since all 91,512 are in the FASTA index. The length policy of Cell 26 (`use_primary`, at most 5,000 nt) exists only for the task circRNAs; a length rule for non-task circRNAs, if they are used at all, is set when the evaluation negatives are rebuilt (old Cell 45). (iv) The 158 task circRNAs with no ENCORI record stay in the CDA task and are unlabelled, not negative, for both auxiliary tasks (Cells 30 and 36). (v) The 282 flagged identifiers keep their `id_check` value; the flag equals the chromosome/strand disagreement and nothing else is added. (vi) The host-group rule of the old Cell 44 stands (FASTA gene field, singleton group for a placeholder). Because 95.2% of the task circRNAs share their host gene with at least one non-task circRNA, and these siblings are almost all labelled (20,155 circRNAs, 99.2% with RBP edges), any use of non-task circRNAs (extra labels, negatives, pretraining) must exclude their siblings or assign them to the same fold as the task circRNAs of their host gene. This is decided in old Cells 45–46. (vii) Next: the label rebuild (old Cell 43) on the two edge tables and the universe table.

### 40. Label Candidates: Positives of the Two Auxiliary Tasks under the Evidence Variants

**Objective.** Count, for the task circRNAs, the positives that each evidence variant of the ENCORI miRNA and RBP edge tables yields (edges, circRNAs with at least one positive, positives per circRNA, and the number of miRNAs or RBPs above a floor of positive circRNAs), and the positive rate of the individual targets, in order to choose the variants and the target lists of the two auxiliary heads before any label table is saved. The cell is read-only; nothing is written.

In [86]:
# Cell 40 - Label candidates: positives of the two auxiliary tasks under the evidence variants (read-only: nothing is saved)
# Uses from earlier cells: PROCESSED_DIR (Cell 1); everything else is re-read from the files of Cells 33, 33 (part 2), 38 and 39.
uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv", usecols=["circbase_id", "in_task", "use_primary", "in_mirna", "in_rbp"])
for c in ["in_task", "use_primary", "in_mirna", "in_rbp"]:
    uni[c] = uni[c].astype(bool)                                            # as decided in Cell 39 (ii)
tk    = uni[uni["in_task"]]
TASK  = set(tk["circbase_id"])
MODEL = set(tk.loc[tk["use_primary"], "circbase_id"])                       # model input: at most 5,000 nt (Cell 26)
assert len(TASK) == 1_644 and len(MODEL) == 1_582
# circRNAs that can be labelled at all: those present in the source table (the others are unlabelled, not negative; Cells 30 and 36)
SRC = {"miRNA": set(tk.loc[tk["in_mirna"] & tk["use_primary"], "circbase_id"]),
       "RBP":   set(tk.loc[tk["in_rbp"] & tk["use_primary"], "circbase_id"])}
print(f"model-input circRNAs in the miRNA table: {len(SRC['miRNA']):,} | in the RBP table: {len(SRC['RBP']):,} (of {len(MODEL):,})")

# ---- 1. edges of the task circRNAs, with the columns that define the variants ------------------------------------------------
me = pd.read_csv(PROCESSED_DIR / "encori_mirna_edges_all.csv.gz",
                 usecols=["circbase_id", "mirna_id", "clip_max", "clip_max_private", "k_min_clip2"])
me = me[me["circbase_id"].isin(TASK)].reset_index(drop=True)
re_ = pd.read_csv(PROCESSED_DIR / "encori_rbp_edges_all.csv.gz",
                  usecols=["circbase_id", "rbp", "c_max", "n_clusters_exonic", "c_max_exonic", "c_max_private", "k_min_c2"])
re_ = re_[re_["circbase_id"].isin(TASK)].reset_index(drop=True)
print(f"edges of the task circRNAs: miRNA {len(me):,} | RBP {len(re_):,}")

MV = {"clip>=1 (sensitivity)": me["clip_max"] >= 1,
      "clip>=2":               me["clip_max"] >= 2,
      "clip>=2, cap 10":       me["k_min_clip2"] <= 10,                     # empty (no site with clip >= 2) compares False
      "clip>=2, private":      me["clip_max_private"] >= 2}
RV = {"c>=1":                  pd.Series(True, index=re_.index),
      "c>=2":                  re_["c_max"] >= 2,
      "c>=3":                  re_["c_max"] >= 3,
      "exonic":                re_["n_clusters_exonic"] > 0,
      "exonic & c>=2":         re_["c_max_exonic"] >= 2,
      "c>=2, cap 10":          (re_["k_min_c2"] >= 1) & (re_["k_min_c2"] <= 10),   # k_min_c2 = 0 means no cluster with c >= 2 (Cell 38)
      "c>=2, private":         re_["c_max_private"] >= 2}

# consistency with Cells 33 (part 2), 36, 37a and 38: task circRNAs with support under each variant
exp_m = {"clip>=1 (sensitivity)": 1_094, "clip>=2": 875, "clip>=2, cap 10": 703, "clip>=2, private": 241}
exp_r = {"c>=1": 1_457, "c>=2": 1_403, "c>=3": 1_294, "exonic & c>=2": 1_365}
for lab, masks, d, exp in [("miRNA", MV, me, exp_m), ("RBP", RV, re_, exp_r)]:
    for v, e in exp.items():
        got = d.loc[masks[v], "circbase_id"].nunique()
        assert got == e, f"{lab} {v}: {got} task circRNAs, expected {e}"
assert re_.loc[re_["c_max_private"] >= 1, "circbase_id"].nunique() == 234 and RV["c>=2, private"].pipe(lambda m: re_.loc[m, "circbase_id"].nunique()) == 212
print("task circRNAs with support per variant equal Cells 33b, 36 and 38 (asserted)")

# ---- 2. positives per variant: model-input circRNAs, positives per circRNA, targets above a floor ------------------------------
FLOORS = (30, 50, 100)
def summarise(d, masks, tcol, src):
    rows, per_target = [], {}
    for v, m in masks.items():
        p  = d[m]
        pm = p[p["circbase_id"].isin(src)]                                  # positives on model-input circRNAs
        k  = pm.groupby("circbase_id").size()                               # positives per labelled circRNA
        t  = pm.groupby(tcol)["circbase_id"].nunique()                      # positive circRNAs per target
        per_target[v] = t
        r  = {"variant": v, "edges (task)": len(p), "task circRNAs": p["circbase_id"].nunique(),
              "model circRNAs": len(k), "% of source": round(100 * len(k) / len(src), 1),
              "positives per circRNA (median)": int(k.median()), "(mean)": round(k.mean(), 1), "targets with >= 1": len(t)}
        for fl in FLOORS:
            r[f"targets >= {fl}"] = int((t >= fl).sum())
        r["rate > 50% (of targets >= 50)"] = int(((t >= 50) & (t / len(src) > .5)).sum())
        rows.append(r)
    return pd.DataFrame(rows).set_index("variant"), per_target

sum_m, pt_m = summarise(me,  MV, "mirna_id", SRC["miRNA"])
sum_r, pt_r = summarise(re_, RV, "rbp",      SRC["RBP"])
print("\nmiRNA labels (positives counted on the model-input circRNAs of the miRNA table):\n", sum_m.T.to_string())
print("\nRBP labels (positives counted on the model-input circRNAs of the RBP table):\n", sum_r.T.to_string())

# ---- 3. positive rate of the individual targets ----------------------------------------------------------------------------------
mt   = pd.read_csv(PROCESSED_DIR / "encori_mirna_table.csv")
name = mt.drop_duplicates("miRNAid").set_index("miRNAid")["miRNAname"]
HUB  = [("miRNA", "clip>=2", pt_m, len(SRC["miRNA"])), ("miRNA", "clip>=2, cap 10", pt_m, len(SRC["miRNA"])),
        ("RBP", "exonic & c>=2", pt_r, len(SRC["RBP"])), ("RBP", "c>=2, cap 10", pt_r, len(SRC["RBP"]))]
for lab, v, pt, n in HUB:
    t = pt[v][pt[v] >= 50].sort_values(ascending=False)
    rate = (100 * t / n).round(1)
    if lab == "miRNA":
        rate.index = [f"{name.get(i, i)}" for i in rate.index]
    print(f"\n{lab} '{v}': {len(t)} targets with >= 50 positive circRNAs | positive rate quantiles (%) "
          f"{rate.quantile([.05, .25, .5, .75, .95]).round(1).to_dict()} | highest: {rate.head(5).to_dict()}")

# ---- 4. circRNAs with positives in both tasks (basis for the multi-task head) ----------------------------------------------------
sets_m = {v: set(me.loc[m & me["circbase_id"].isin(MODEL), "circbase_id"]) for v, m in MV.items()}
sets_r = {v: set(re_.loc[m & re_["circbase_id"].isin(MODEL), "circbase_id"]) for v, m in RV.items()}
both = pd.DataFrame({rv: {mv: len(sm & sr) for mv, sm in sets_m.items()} for rv, sr in sets_r.items()})
print("\nmodel-input circRNAs with at least one positive in both tasks (rows: miRNA variant, columns: RBP variant):\n", both.to_string())

model-input circRNAs in the miRNA table: 1,433 | in the RBP table: 1,405 (of 1,582)
edges of the task circRNAs: miRNA 72,870 | RBP 67,434
task circRNAs with support per variant equal Cells 33b, 36 and 38 (asserted)

miRNA labels (positives counted on the model-input circRNAs of the miRNA table):
 variant                         clip>=1 (sensitivity)  clip>=2  clip>=2, cap 10  clip>=2, private
edges (task)                                  23373.0  15033.0          10988.0            3344.0
task circRNAs                                  1094.0    875.0            703.0             241.0
model circRNAs                                 1048.0    835.0            666.0             217.0
% of source                                      73.1     58.3             46.5              15.1
positives per circRNA (median)                   11.0      9.0              8.0               7.0
(mean)                                           19.1     15.7             14.3              12.9
targets with >= 

**Interpretation.** *(A) Consistency and scope.* The task circRNAs with support under each variant equal the earlier cells (asserted): 1,094, 875, 703 and 241 on the miRNA side (Cells 33 and 33b) and 1,457, 1,403, 1,294 and 1,365 on the RBP side (Cells 36 and 38), and the private-cluster counts of Cell 38 (234 at `c ≥ 1`, 212 at `c ≥ 2`). Of the 1,582 model-input circRNAs, 1,433 occur in the miRNA table and 1,405 in the RBP table, so 149 and 177 have no record in the respective table. The edge counts (72,870 miRNA, 67,434 RBP) cover all 1,644 task circRNAs; every other count in the tables is restricted to the model input, which is why, for example, 835 circRNAs appear under `clip ≥ 2` against 875 task circRNAs. *(B) miRNA labels.* The share of labelled circRNAs with at least one positive falls from 73.1% (`clip ≥ 1`, 1,048 circRNAs) to 58.3% (`≥ 2`, 835), 46.5% (`≥ 2`, cap 10, 666) and 15.1% (private, 217). The median circRNA has 11, 9, 8 and 7 positive miRNAs (means 19.1, 15.7, 14.3 and 12.9). The labels are spread thinly over many miRNAs: 635 miRNAs have a positive at `≥ 2`, but only 130 reach 30 positive circRNAs and 21 reach 50 (3.3% of 635, derived); with the cap of 10 the numbers are 54 and 10. Under the private variant no miRNA reaches 30. No miRNA is positive for more than 7.4% of the labelled circRNAs (median 4.5% among the 21 with at least 50). The five highest are hsa-miR-195-5p, miR-15b-5p, miR-16-5p, miR-15a-5p and miR-424-5p (7.3–7.4%), which look like members of one family and may share many sites (inference from the names; seeds and site overlap were not compared). *(C) RBP labels.* The picture is denser. At `c ≥ 1` every labelled circRNA is positive for a median of 40 RBPs, 25 RBPs have a positive rate above 50%, and the label carries little information. The strict variant "exonic and `c ≥ 2`" keeps 1,314 of 1,405 circRNAs (93.5%; 91 have no positive, derived), a median of 12 positives (mean 17.2), 225 RBPs with a positive, 103 with at least 30 and 83 with at least 50 positive circRNAs; the positive rate of these 83 has a median of 12.7% and a 95th percentile of 50.0%, and five RBPs exceed 50% (U2AF2 53.2%, SCAF8 52.6%, SCAF4 52.1%, RNPS1 51.7%, ALYREF 51.0%). The cap of 10 (`c ≥ 2`) removes the hub character (no RBP above 50%, highest 44.5%) but costs coverage: 1,120 circRNAs (79.7%) against 1,352 (96.2%) at `c ≥ 2` without cap, i.e. 232 circRNAs (17.2%, derived). The exonic restriction costs little by comparison (1,314 against 1,352 circRNAs, 2.8%, derived). Private clusters leave 194 circRNAs, 9 RBPs with at least 50 positives and none with 100, as expected from Cell 38. *(D) Both tasks.* Under the working variants (`clip ≥ 2` and exonic with `c ≥ 2`), 823 model-input circRNAs (52.0% of 1,582, derived) have a positive in both tasks. This is 98.6% of the 835 circRNAs with a miRNA positive and 62.6% of the 1,314 with an RBP positive (derived), so miRNA-labelled circRNAs are almost all RBP-labelled as well, and the reverse does not hold. The same pattern appears in every cell of the table (for example 629 of 666 for the two capped variants, 94.4%, derived). *(E) Unlabelled versus negative.* Under `clip ≥ 2`, 598 of the 1,433 model-input circRNAs of the miRNA table (41.7%, derived) have no positive miRNA, and under the strict RBP variant 91 of 1,405 have no positive RBP. Absence of a CLIP-supported edge is not evidence that no binding occurs, because CLIP covers only the cell types and experiments that were run.

**Not checked.** (i) Whether the miRNA positives are dominated by one seed family, and how correlated the label columns of different targets are; both determine how many independent labels the heads contain. (ii) A cap combined with the exonic condition: `k_min_c2` is not restricted to exonic clusters, so "exonic and `c ≥ 2` and cap 10" cannot be read from the table. (iii) Positives per fold after host-gene grouping. A floor of 50 positive circRNAs means about 10 per fold on average (derived), but the grouped folds are uneven. (iv) Whether circRNA length or the number of edges alone predicts the labels, and whether it drives the overlap between the two tasks (D); the old shortcut check (old Cell 42) was done on 6 RBPs and not on these tables. (v) Whether circRNAs with no positive differ systematically from the others (for example in length or expression), which would make them a biased set of negatives. (vi) The sibling test of the variants (the choice between them depends on it, Cell 33b (iii)).

**Decision.** (i) The label tables keep every edge that is positive under at least one variant, one row per (circRNA, target), with one boolean column per variant, and apply no floor or cap as a deletion (as in old Cell 43). The miRNA table therefore holds the 23,373 edges with `clip_max ≥ 1` of the task circRNAs; the other 49,497 edges (derived) have no CLIP support and stay only in `encori_mirna_edges_all.csv.gz`. The RBP table holds all 67,434 edges. (ii) Working variants, fixed here: `clip ≥ 2` for miRNAs and "exonic and `c ≥ 2`" for RBPs. Controls: `clip ≥ 2` with cap 10 and `clip ≥ 1` (sensitivity) for miRNAs, `c ≥ 2` with cap 10 and `c ≥ 2` for RBPs. The variants are stored as flag columns, so the sibling test (Cell 43) and the positives per fold (Cell 42) can still revise the choice; a revision is recorded as such in the cell that makes it.(iii). (iii) The private-only variant is not used as a label set: no miRNA reaches 30 positive circRNAs and only 9 RBPs reach 50. It remains a sensitivity analysis on the stored columns. `c ≥ 1` is not carried for RBPs (median 40 positives, 25 targets above 50%); `c ≥ 3` and "exonic" stay as flag columns only.  (iv) circRNAs of the source table are labelled for that head. The main run keeps those without a positive as all-negative rows, and the masked run (only circRNAs with at least one positive under the working variant) is stored next to it as a second loss mask, so that both are available without rebuilding. circRNAs absent from the source table stay unlabelled (Cells 30 and 36). (v) Floor, fixed here: at least 50 positive model-input circRNAs under the working variant, which selects 21 miRNAs and 83 RBPs; a floor of 30 (130 and 103) is stored as the alternative. Both are columns of the target tables, not deletions. The selection uses label counts over all model-input circRNAs, including the later test folds; it depends only on how frequent a target is, not on any circRNA's features, but it is a mild selection effect (inference) and Cell 42 reports the positives per fold. (vi) The five RBPs above 50% (U2AF2, SCAF8, SCAF4, RNPS1, ALYREF) stay in the table and are examined with and without a sensitivity run, as decided in Cell 37b (v). (vii) Next: Cell 41 builds and saves the two label tables; then the host-gene folds (Cell 42), negatives with the popularity and sibling tests (Cell 43), and the frozen folds (Cell 44).


### 41. ENCORI Label Tables with Evidence Flags, Target Tables and Loss Masks

**Objective.** Build, for the 1,644 task circRNAs, the miRNA and RBP label tables with one boolean flag per evidence variant (Cell 40), the two target tables (positive circRNAs per target and variant, with the floors of 50 and 30), and a circRNA-level table with the number of positives per variant and the two loss masks of each head (all-negative rows kept, or masked), to verify them against the counts of Cell 40 and to save them.

In [87]:
# Cell 41 - ENCORI label tables with evidence flags, target tables and loss masks (saves five files)
# Uses from earlier cells: PROCESSED_DIR (Cell 1); everything else is re-read from the files of Cells 33, 33 (part 2), 38 and 39.
SAVE   = True
WORK_M, WORK_R = "pos_clip2", "pos_exonic_c2"                     # working variants (Cell 40 Decision ii)
MF = ["pos_clip1", "pos_clip2", "pos_clip2_cap10", "pos_clip2_private"]
RF = ["pos_c1", "pos_c2", "pos_c3", "pos_exonic", "pos_exonic_c2", "pos_c2_cap10", "pos_c2_private"]

uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv",
                  usecols=["circbase_id", "in_task", "use_primary", "in_mirna", "in_rbp", "n_mirna_clip1", "n_mirna_clip2",
                           "n_rbp", "n_rbp_c2", "n_rbp_c3", "n_rbp_exonic", "n_rbp_exonic_c2"])
for c in ["in_task", "use_primary", "in_mirna", "in_rbp"]:
    uni[c] = uni[c].astype(bool)                                    # as decided in Cell 39 (ii)
tk = uni[uni["in_task"]].set_index("circbase_id")
TASK, model = set(tk.index), tk["use_primary"]
src_m, src_r = model & tk["in_mirna"], model & tk["in_rbp"]
assert (len(tk), int(model.sum()), int(src_m.sum()), int(src_r.sum())) == (1_644, 1_582, 1_433, 1_405)

# ---- 1. miRNA labels: every edge of a task circRNA with CLIP support (clip >= 1), one flag per variant -----------------------
me = pd.read_csv(PROCESSED_DIR / "encori_mirna_edges_all.csv.gz",
                 usecols=["circbase_id", "mirna_id", "clip_max", "clip_max_private", "k_min_clip2", "id_check"])
me = me[me["circbase_id"].isin(TASK) & (me["clip_max"] >= 1)].reset_index(drop=True)
mname = pd.read_csv(PROCESSED_DIR / "encori_mirna_table.csv").drop_duplicates("miRNAid").set_index("miRNAid")["miRNAname"]
me.insert(2, "mirna_name", me["mirna_id"].map(mname))
assert me["mirna_name"].notna().all(), "a miRNA id without a name in the miRNA table"
me["pos_clip1"]         = True
me["pos_clip2"]         = me["clip_max"] >= 2
me["pos_clip2_cap10"]   = me["k_min_clip2"] <= 10                    # empty (no site with clip >= 2) compares False
me["pos_clip2_private"] = me["clip_max_private"] >= 2
me["model_input"]       = me["circbase_id"].map(model).astype(bool)

# ---- 2. RBP labels: every edge of a task circRNA (c >= 1), one flag per variant ---------------------------------------------------
rl = pd.read_csv(PROCESSED_DIR / "encori_rbp_edges_all.csv.gz",
                 usecols=["circbase_id", "rbp", "n_clusters", "n_clusters_exonic", "c_max", "c_max_exonic", "c_max_private",
                          "k_min_c2", "clip_max", "id_check"])
rl = rl[rl["circbase_id"].isin(TASK)].reset_index(drop=True)
rl["pos_c1"]         = True
rl["pos_c2"]         = rl["c_max"] >= 2
rl["pos_c3"]         = rl["c_max"] >= 3
rl["pos_exonic"]     = rl["n_clusters_exonic"] > 0
rl["pos_exonic_c2"]  = rl["c_max_exonic"] >= 2
rl["pos_c2_cap10"]   = (rl["k_min_c2"] >= 1) & (rl["k_min_c2"] <= 10)    # k_min_c2 = 0: no cluster with c >= 2 (Cell 38)
rl["pos_c2_private"] = rl["c_max_private"] >= 2
rl["model_input"]    = rl["circbase_id"].map(model).astype(bool)
print(f"label tables: miRNA {len(me):,} edges over {me['circbase_id'].nunique():,} task circRNAs | "
      f"RBP {len(rl):,} edges over {rl['circbase_id'].nunique():,} task circRNAs")

# ---- 3. integrity: unique keys, nested variants, counts of Cell 40 ----------------------------------------------------------------
assert not me.duplicated(["circbase_id", "mirna_id"]).any() and not rl.duplicated(["circbase_id", "rbp"]).any()
implies = lambda d, a, b: bool((~d[a] | d[b]).all())                     # every positive of variant a is a positive of variant b
assert implies(me, "pos_clip2_cap10", "pos_clip2") and implies(me, "pos_clip2_private", "pos_clip2") and implies(me, "pos_clip2", "pos_clip1")
assert all(implies(rl, a, b) for a, b in [("pos_exonic_c2", "pos_c2"), ("pos_exonic_c2", "pos_exonic"), ("pos_c3", "pos_c2"), ("pos_c2", "pos_c1"),
                                          ("pos_c2_cap10", "pos_c2"), ("pos_c2_private", "pos_c2")])
print("keys are unique and the variants are nested as expected (asserted)")

def circ_counts(d, flags):
    return {f: d.loc[d["model_input"] & d[f], "circbase_id"].nunique() for f in flags}
got_m, got_r = circ_counts(me, MF), circ_counts(rl, RF)
exp_m = dict(zip(MF, (1_048, 835, 666, 217)))
exp_r = dict(zip(RF, (1_405, 1_352, 1_245, 1_392, 1_314, 1_120, 194)))
print("model-input circRNAs with a positive, miRNA:", got_m, "| RBP:", got_r)
assert got_m == exp_m and got_r == exp_r, "differs from Cell 40"
pm = set(me.loc[me["model_input"] & me[WORK_M], "circbase_id"]); pr = set(rl.loc[rl["model_input"] & rl[WORK_R], "circbase_id"])
print(f"model-input circRNAs with a positive in both tasks (working variants): {len(pm & pr):,} (Cell 40: 823)")
assert len(pm & pr) == 823

# ---- 4. target tables: positive model-input circRNAs per target and variant, and the floors -----------------------------------------
def target_table(d, key, flags, work, n_src):
    t = d[d["model_input"]].groupby(key)[flags].sum().astype(int)        # keys are unique, so a sum counts circRNAs
    t = t.reindex(sorted(d[key].unique()), fill_value=0)
    t.columns = ["n_" + f[4:] for f in flags]
    t["rate_working"]   = (t["n_" + work[4:]] / n_src).round(4)
    t["head_floor50"]   = t["n_" + work[4:]] >= 50
    t["head_floor30"]   = t["n_" + work[4:]] >= 30
    return t
tm = target_table(me, "mirna_id", MF, WORK_M, int(src_m.sum())); tm.insert(0, "mirna_name", tm.index.map(mname))
tr = target_table(rl, "rbp",      RF, WORK_R, int(src_r.sum()))
tr["hub_over_50pct"] = tr["rate_working"] > 0.5                             # broad binders, examined with and without (Cell 37b v)
print(f"\ntarget tables: miRNAs {len(tm):,} | RBPs {len(tr):,}")
print("targets in the head, working variant (floor 50 | floor 30): miRNA", int(tm["head_floor50"].sum()), "|", int(tm["head_floor30"].sum()),
      "(Cell 40: 21 | 130) -- RBP", int(tr["head_floor50"].sum()), "|", int(tr["head_floor30"].sum()), "(Cell 40: 83 | 103)")
assert (tm["head_floor50"].sum(), tm["head_floor30"].sum(), tr["head_floor50"].sum(), tr["head_floor30"].sum()) == (21, 130, 83, 103)
assert (tm["n_clip2_cap10"] >= 50).sum() == 10 and (tr["n_c2_cap10"] >= 50).sum() == 96, "cap-10 targets differ from Cell 40"
print("RBPs positive for more than 50% of the labelled circRNAs:", tr.loc[tr["hub_over_50pct"], "rate_working"].sort_values(ascending=False).to_dict())

# ---- 5. circRNA table: positives per variant and the two loss masks of each head -------------------------------------------------------
cs = pd.DataFrame({"model_input": model, "in_mirna_table": tk["in_mirna"], "in_rbp_table": tk["in_rbp"]})
for f in MF:
    cs["n_mirna_" + f[4:]] = me.groupby("circbase_id")[f].sum().reindex(cs.index).fillna(0).astype("int32")
for f in RF:
    cs["n_rbp_" + f[4:]]   = rl.groupby("circbase_id")[f].sum().reindex(cs.index).fillna(0).astype("int32")
# consistency with the node table of Cell 39 (degrees of the task circRNAs)
for a, b in [("n_mirna_clip1", "n_mirna_clip1"), ("n_mirna_clip2", "n_mirna_clip2"), ("n_rbp_c1", "n_rbp"), ("n_rbp_c2", "n_rbp_c2"),
             ("n_rbp_c3", "n_rbp_c3"), ("n_rbp_exonic", "n_rbp_exonic"), ("n_rbp_exonic_c2", "n_rbp_exonic_c2")]:
    assert (cs[a].to_numpy() == tk.loc[cs.index, b].to_numpy()).all(), f"{a} differs from the universe table (Cell 39)"
cs["use_mirna_main"]   = src_m.reindex(cs.index)                              # labelled; circRNAs without a positive are all-negative rows
cs["use_mirna_masked"] = cs["use_mirna_main"] & (cs["n_mirna_" + WORK_M[4:]] > 0)
cs["use_rbp_main"]     = src_r.reindex(cs.index)
cs["use_rbp_masked"]   = cs["use_rbp_main"] & (cs["n_rbp_" + WORK_R[4:]] > 0)
masks = cs[["use_mirna_main", "use_mirna_masked", "use_rbp_main", "use_rbp_masked"]].sum().astype(int).to_dict()
print("\nloss masks (model-input circRNAs):", masks, "(expected 1,433 | 835 | 1,405 | 1,314)")
assert (masks["use_mirna_main"], masks["use_mirna_masked"], masks["use_rbp_main"], masks["use_rbp_masked"]) == (1_433, 835, 1_405, 1_314)
cs["use_mirna_main"], cs["use_mirna_masked"] = cs["use_mirna_main"].astype(bool), cs["use_mirna_masked"].astype(bool)

# ---- 6. save and verify by reloading -----------------------------------------------------------------------------------------------------
OUT = {"encori_mirna_labels.csv": (me, MF), "encori_rbp_labels.csv": (rl, RF),
       "encori_mirna_targets.csv": (tm.reset_index(), ["head_floor50", "head_floor30"]),
       "encori_rbp_targets.csv": (tr.reset_index(), ["head_floor50", "head_floor30", "hub_over_50pct"]),
       "encori_label_status.csv": (cs.reset_index(), ["model_input", "use_mirna_main", "use_mirna_masked", "use_rbp_main", "use_rbp_masked"])}
for fn, (d, flags) in OUT.items():
    if SAVE:
        d.to_csv(PROCESSED_DIR / fn, index=False)
        r = pd.read_csv(PROCESSED_DIR / fn)
        same = r.shape == d.shape and all(int(r[f].sum()) == int(d[f].sum()) for f in flags)
        print(f"saved {fn}: {len(d):,} rows x {d.shape[1]} columns | {(PROCESSED_DIR / fn).stat().st_size / 1e6:,.2f} MB | reload equal: {same}")
        assert same
    else:
        print(f"dry run: {fn} ({len(d):,} rows x {d.shape[1]} columns) not written")
print("\nmiRNA label table, first rows:\n", me.head(3).T.to_string())
print("\ncircRNA table, first rows:\n", cs.head(3).T.to_string())

label tables: miRNA 23,373 edges over 1,094 task circRNAs | RBP 67,434 edges over 1,457 task circRNAs
keys are unique and the variants are nested as expected (asserted)
model-input circRNAs with a positive, miRNA: {'pos_clip1': 1048, 'pos_clip2': 835, 'pos_clip2_cap10': 666, 'pos_clip2_private': 217} | RBP: {'pos_c1': 1405, 'pos_c2': 1352, 'pos_c3': 1245, 'pos_exonic': 1392, 'pos_exonic_c2': 1314, 'pos_c2_cap10': 1120, 'pos_c2_private': 194}
model-input circRNAs with a positive in both tasks (working variants): 823 (Cell 40: 823)

target tables: miRNAs 638 | RBPs 279
targets in the head, working variant (floor 50 | floor 30): miRNA 21 | 130 (Cell 40: 21 | 130) -- RBP 83 | 103 (Cell 40: 83 | 103)
RBPs positive for more than 50% of the labelled circRNAs: {'U2AF2': 0.5324, 'SCAF8': 0.526, 'SCAF4': 0.521, 'RNPS1': 0.5174, 'ALYREF': 0.5103}

loss masks (model-input circRNAs): {'use_mirna_main': 1433, 'use_mirna_masked': 835, 'use_rbp_main': 1405, 'use_rbp_masked': 1314} (expected 1,433 | 83

**Interpretation.** *(A) Consistency.* The two label tables hold 23,373 miRNA edges (over 1,094 task circRNAs) and 67,434 RBP edges (over 1,457), the edge counts of Cell 40 for `clip ≥ 1` and `c ≥ 1`. The keys (circRNA, miRNA) and (circRNA, RBP) are unique, and the variants are nested as expected (asserted: for example every `clip ≥ 2` with cap 10 or private positive is a `clip ≥ 2` positive, and every "exonic and `c ≥ 2`" positive is a `c ≥ 2` positive). The number of model-input circRNAs with a positive equals Cell 40 in all 11 variants (miRNA 1,048, 835, 666, 217; RBP 1,405, 1,352, 1,245, 1,392, 1,314, 1,120, 194), and the degrees of the task circRNAs equal those of the node table of Cell 39 (asserted for the counts of `clip ≥ 1` and `≥ 2` and the five RBP degrees). 823 model-input circRNAs have a positive in both tasks under the working variants, as in Cell 40. *(B) Target tables.* 638 of the 642 miRNAs of the ENCORI miRNA table and 279 of the 281 RBPs of the RBP table have an edge on a task circRNA. The RBP table has two more rows than the 277 RBPs with a positive on the model input at `c ≥ 1` (Cell 40), so two RBPs occur only on task circRNAs that are not model input (derived; not looked at). Floors of 50 and 30 positive model-input circRNAs under the working variants select 21 and 130 miRNAs and 83 and 103 RBPs, as in Cell 40; the cap-10 variants give 10 miRNAs and 96 RBPs with at least 50 (asserted). Five RBPs are positive for more than 50% of the 1,405 labelled circRNAs (U2AF2 53.2%, SCAF8 52.6%, SCAF4 52.1%, RNPS1 51.7%, ALYREF 51.0%) and are flagged in `hub_over_50pct`. *(C) Loss masks.* The main masks keep all labelled circRNAs (1,433 for the miRNA head, 1,405 for the RBP head); the masked versions keep only those with a positive under the working variant: 835 (58.3% of 1,433) and 1,314 (93.5% of 1,405). The masked run therefore removes 598 circRNAs (41.7%) from the miRNA loss and 91 (6.5%) from the RBP loss. A circRNA with no record in the respective table is in neither mask (149 for miRNAs and 177 for RBPs among the 1,582). *(D) Files.* Five files were written and reloaded, and the flag sums of the reloaded files equal those of the tables: `encori_mirna_labels.csv` (23,373 × 12, 2.13 MB), `encori_rbp_labels.csv` (67,434 × 18, 6.30 MB), `encori_mirna_targets.csv` (638 × 9), `encori_rbp_targets.csv` (279 × 12) and `encori_label_status.csv` (1,644 × 19). The tables cover all 1,644 task circRNAs, so rows of circRNAs above 5,000 nt are present and marked by `model_input = False` (two of the three circRNAs printed at the top of the circRNA table are of this kind; the first printed circRNA has 29,313 nt in Cell 39). The files of the API trial (`rbp_labels_encori.csv`, `mirna_edges_encori.csv`) were not overwritten.

**Not checked.** (i) The positive counts per target (`n_*`) and the rates are computed over all model-input circRNAs, including those that later fall in test folds; the number of positives per target and fold is measured in Cell 42. (ii) Whether the 598 circRNAs without a positive miRNA, or the 91 without a positive RBP, differ from the others in length or degree (Cell 40, Not checked v). (iii) The number of positive edges that belong to circRNAs with a flagged identifier (`id_check` is stored in both label tables but was not tabulated). (iv) The masked version is only built for the working variants; for the control variants it can be derived from the `n_mirna_*` and `n_rbp_*` columns of `encori_label_status.csv` but was not written. (v) The two RBPs without a positive on the model input (see B) were not identified.

**Decision.** (i) The five files are kept as saved. The tables carry the edges of all 1,644 task circRNAs; training and evaluation use only rows with `model_input = True` (the length policy of Cell 26). (ii) The working variants and the floor of 50 (Cell 40 Decision ii and v) stand: `head_floor50` defines the working head targets, 21 miRNAs and 83 RBPs. The floor of 30, the control variants and the sensitivity variants stay as columns and are revised only after the per-fold counts (Cell 42) and the sibling test (Cell 43). (iii) The main run uses `use_mirna_main` and `use_rbp_main`; the sensitivity run uses `use_mirna_masked` and `use_rbp_masked`. (iv) The five hub RBPs stay in the table and are examined with and without a sensitivity run (Cell 37b (v)). (v) The target tables are used to select targets and to describe them, not as features: the rates were computed on all model-input circRNAs and must be recomputed from training folds if a positive rate or a count enters a model. (vi) Next: Cell 42, the host-gene-grouped 5-fold split on the groups of the node table (1,251 groups for the 1,582 model-input circRNAs, 26 of them singleton placeholder groups), with the number of positives of each of the 21 miRNAs and 83 RBPs per fold, and, as before, pairs, circRNAs and length per fold.

### 42. Host-Gene-Grouped 5-Fold Split and Positives per Target and Fold

**Objective.** Rebuild the host-gene-grouped 5-fold split (greedy balancing by pairs, three seeds) on the host groups of the node table, verify that it equals the split of the old Cell 44, and measure for the working variants how many positive circRNAs each target of the two heads has in every fold under the floors of 50 and 30, in order to decide whether the floors hold under grouped folds. The cell is a dry run; nothing is written.

In [88]:
# Cell 42 - Host-gene-grouped 5-fold split and positives per target and fold (dry run: nothing is written)
# Uses from earlier cells: PROCESSED_DIR (Cell 1), PLACE (Cell 39); everything else is re-read from the files of Cells 6, 26, 39 and 41.
SEEDS = (0, 1, 2)
uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv", keep_default_na=False,
                  usecols=["circbase_id", "in_task", "use_primary", "host_placeholder", "host_group", "seq_len"],
                  dtype={"in_task": bool, "use_primary": bool, "host_placeholder": bool})
mi = uni[uni["in_task"] & uni["use_primary"]].set_index("circbase_id")
assert len(mi) == 1_582

pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
print("pairs on model-input circRNAs:", len(pairs), "(expected 2,137) | circRNAs:", pairs["circrna_id"].nunique(), "(expected 1,582)")
assert (len(pairs), pairs["circrna_id"].nunique()) == (2_137, 1_582)
pairs["group"] = pairs["circrna_id"].map(mi["host_group"])

# the groups of the node table equal those of the old Cell 44 (rule applied to the FASTA index again)
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
gene_s = fa_idx.set_index("circbase_id")["gene"]
old = pairs["circrna_id"].map(lambda c: f"solo:{c}" if PLACE.match(str(gene_s.get(c, ""))) else str(gene_s.get(c, "")))
print("groups equal to the old rule (Cell 44):", bool((old == pairs["group"]).all()))
assert (old == pairs["group"]).all()

grp = pairs.groupby("group").agg(n_pairs=("circrna_id", "size"), n_circ=("circrna_id", "nunique"))
print("groups:", len(grp), "(expected 1,251) | singleton placeholder groups:", int(grp.index.str.startswith("solo:").sum()),
      "(expected 26) | circRNAs per group max:", grp["n_circ"].max())
assert (len(grp), int(grp.index.str.startswith("solo:").sum())) == (1_251, 26)

def assign(seed, k=5):                                       # same greedy balancing as the old Cell 44
    rng = np.random.default_rng(seed)
    order = grp.sample(frac=1, random_state=seed).sort_values("n_pairs", ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for g, n in order["n_pairs"].items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[g] = f; load[f] += n
    return fold
GF = {s: assign(s) for s in SEEDS}                           # group -> fold, per seed
FM = {s: pairs.drop_duplicates("circrna_id").set_index("circrna_id")["group"].map(GF[s]) for s in SEEDS}   # circRNA -> fold

# ---- 1. fold sizes and balance (as old Cell 44, with the ENCORI heads instead of the curated miRNA edges) ---------------------
st  = pd.read_csv(PROCESSED_DIR / "encori_label_status.csv", usecols=["circbase_id", "use_mirna_main", "use_rbp_main", "n_mirna_clip2", "n_rbp_exonic_c2"],
                  dtype={"use_mirna_main": bool, "use_rbp_main": bool}).set_index("circbase_id")
cur_ids = set(pd.read_csv(PROCESSED_DIR / "mirna_edges_curated.csv")["circrna_id"])
for seed in SEEDS:
    pairs["fold"] = pairs["group"].map(GF[seed])
    f = FM[seed]
    per = pairs.groupby("fold").agg(pairs=("circrna_id", "size"), circRNAs=("circrna_id", "nunique"))
    per["gt1022nt_share"] = [round(float((mi.loc[f.index[f == k], "seq_len"] > 1022).mean()), 3) for k in range(5)]
    per["curated_miRNA"] = [len(set(f.index[f == k]) & cur_ids) for k in range(5)]
    per["miRNA_labelled"] = [int(st.loc[f.index[f == k], "use_mirna_main"].sum()) for k in range(5)]
    per["miRNA_positive"] = [int((st.loc[f.index[f == k], "n_mirna_clip2"] > 0).sum()) for k in range(5)]
    per["RBP_labelled"] = [int(st.loc[f.index[f == k], "use_rbp_main"].sum()) for k in range(5)]
    per["RBP_positive"] = [int((st.loc[f.index[f == k], "n_rbp_exonic_c2"] > 0).sum()) for k in range(5)]
    print(f"\nseed {seed}: groups in >1 fold: {int((pairs.groupby('group')['fold'].nunique() > 1).sum())} | "
          f"circRNAs in >1 fold: {int((pairs.groupby('circrna_id')['fold'].nunique() > 1).sum())}")
    print(per.to_string())
    if seed == 0:
        assert per["pairs"].tolist() == [428, 427, 428, 427, 427], "seed-0 fold sizes differ from the old Cell 44"
        assert per["circRNAs"].tolist() == [320, 315, 315, 317, 315]
print("\nseed-0 pairs and circRNAs per fold equal the old Cell 44 (asserted)")

# ---- 2. positives per target and fold (working variants), floors 50 and 30 ---------------------------------------------------------
tm = pd.read_csv(PROCESSED_DIR / "encori_mirna_targets.csv", usecols=["mirna_id", "mirna_name", "head_floor50", "head_floor30"])
tr = pd.read_csv(PROCESSED_DIR / "encori_rbp_targets.csv", keep_default_na=False, usecols=["rbp", "head_floor50", "head_floor30", "hub_over_50pct"])
ml = pd.read_csv(PROCESSED_DIR / "encori_mirna_labels.csv", usecols=["circbase_id", "mirna_id", "pos_clip2", "model_input"])
rl = pd.read_csv(PROCESSED_DIR / "encori_rbp_labels.csv", keep_default_na=False, usecols=["circbase_id", "rbp", "pos_exonic_c2", "model_input"])
HEAD = {"miRNA": (ml, "mirna_id", "pos_clip2", tm), "RBP": (rl, "rbp", "pos_exonic_c2", tr)}

def per_fold(lab, key, flag, targets, fmap):
    d = lab[lab["model_input"] & lab[flag] & lab[key].isin(targets)].copy()
    d["fold"] = d["circbase_id"].map(fmap)
    assert d["fold"].notna().all(), "a model-input circRNA without a fold"
    return d.groupby([key, "fold"])["circbase_id"].nunique().unstack(fill_value=0).reindex(index=sorted(targets), columns=range(5), fill_value=0)

rows, keep = [], {}
for seed in SEEDS:
    for head, (lab, key, flag, tt) in HEAD.items():
        for fl in (50, 30):
            tg = tt.loc[tt[f"head_floor{fl}"], key].tolist()
            m = per_fold(lab, key, flag, tg, FM[seed])
            assert (m.sum(axis=1) >= fl).all()
            mn = m.min(axis=1)
            rows.append({"seed": seed, "head": head, "floor": fl, "targets": len(tg), "fold min (median)": int(mn.median()),
                         "fold min (lowest)": int(mn.min()), "targets with a fold < 5": int((mn < 5).sum()),
                         "targets with a fold < 3": int((mn < 3).sum()), "targets with an empty fold": int((mn == 0).sum()),
                         "max/min fold ratio (median)": round(float((m.max(axis=1) / m.min(axis=1).clip(lower=1)).median()), 1)})
            keep[(seed, head, fl)] = m
res = pd.DataFrame(rows)
print("\npositive circRNAs per target and fold, by floor (working variants; 'fold min' = fewest positives of a target in any fold):\n", res.to_string(index=False))

# the hubs and the targets with the thinnest fold, seed 0
m = keep[(0, "RBP", 50)]
print("\nRBP, floor 50, seed 0: positives per fold of the five hubs:\n", m.loc[tr.loc[tr["hub_over_50pct"], "rbp"]].to_string())
for head, key in (("miRNA", "mirna_id"), ("RBP", "rbp")):
    m = keep[(0, head, 50)]
    thin = m.assign(min=m.min(axis=1)).sort_values("min").head(3)
    if head == "miRNA":
        thin.index = thin.index.map(tm.set_index("mirna_id")["mirna_name"])
    print(f"\n{head}, floor 50, seed 0: three targets with the thinnest fold:\n", thin.to_string())

# ---- 3. siblings outside the task that the folds would take along (seed 0; basis for any use of non-task circRNAs) ---------------------------------
nt = uni[~uni["in_task"] & ~uni["host_placeholder"]].copy()
nt["fold"] = nt["host_group"].map(GF[0])
print(f"\nnon-task circRNAs in a host group of the model input (seed 0): {int(nt['fold'].notna().sum()):,} "
      f"| per fold: {nt['fold'].dropna().astype(int).value_counts().sort_index().tolist()}")
print("\nDry run: nothing written.")

pairs on model-input circRNAs: 2137 (expected 2,137) | circRNAs: 1582 (expected 1,582)
groups equal to the old rule (Cell 44): True
groups: 1251 (expected 1,251) | singleton placeholder groups: 26 (expected 26) | circRNAs per group max: 9

seed 0: groups in >1 fold: 0 | circRNAs in >1 fold: 0
      pairs  circRNAs  gt1022nt_share  curated_miRNA  miRNA_labelled  miRNA_positive  RBP_labelled  RBP_positive
fold                                                                                                            
0       428       320           0.234            131             288             175           286           275
1       427       315           0.210            140             278             165           269           252
2       428       315           0.235            133             296             164           289           262
3       427       317           0.211            149             283             155           279           264
4       427       315       

**Interpretation.** *(A) Equality with the old split.* The cell reproduces the input of old Cell 44: 2,137 pairs over 1,582 model-input circRNAs, 1,251 groups of which 26 are singleton placeholder groups, and at most 9 circRNAs per group. The groups taken from the node table equal those of the old rule applied to the FASTA index (asserted). For seed 0, the pairs per fold (428, 427, 428, 427, 427) and the circRNAs per fold (320, 315, 315, 317, 315) equal the old Cell 44 (asserted). For seeds 1 and 2, the pairs, circRNAs, share of circRNAs above 1,022 nt and curated-miRNA circRNAs per fold equal the printed output of the old Cell 44 (compared with that output, not asserted). The circRNAs per fold are 315–320 (seed 0), 300–328 (seed 1) and 311–326 (seed 2), and the share above 1,022 nt is 0.199–0.256 per fold (overall 350/1,582 = 0.221). No group and no circRNA appears in more than one fold in any of the three seeds. As in the old cell, this checks the code, since the fold is derived from the group by construction; it does not show an absence of biological leakage. *(B) Labels per fold.* The folds are not stratified by label. The labelled circRNAs per fold sum to the totals of Cell 41 (miRNA head 1,433, e.g. 288, 278, 296, 283, 288 in seed 0; RBP head 1,405, e.g. 286, 269, 289, 279, 282), and so do the circRNAs with a positive under the working variants (835 and 1,314). The share of labelled circRNAs with a positive miRNA is 54.8–63.6% per fold over the three seeds (overall 58.3%), and the share with a positive RBP is 90.6–96.2% (overall 93.5%; both ranges derived). The spread is the one expected from folds that were not stratified (inferred). *(C) Positives per target and fold.* With the floor of 50 positive circRNAs, the thinnest fold of a miRNA target has a median of 8, 8 and 9 positives in the three seeds (21 targets), and that of an RBP target 28, 27 and 25 (83 targets). The lowest single fold count is 2, 3 and 5 for miRNAs and 2, 4 and 5 for RBPs, so the number of targets with a fold below 5 positives is 2, 1 and 0 (miRNA) and 1, 1 and 0 (RBP); no target has an empty fold. The largest fold holds about 2 times (miRNA, median ratio 1.7–2.3) and 1.5–1.7 times (RBP) as many positives as the smallest. With the floor of 30 the miRNA head does not hold: the median thinnest fold is 4 in every seed, 70, 70 and 65 of the 130 targets (53.8%, 53.8% and 50.0%, derived) have a fold below 5 positives, 13–17 below 3, and one target has an empty fold in seeds 0 and 1. For RBPs the floor of 30 gives a median thinnest fold of 19–21 and 11, 17 and 11 of 103 targets below 5, none empty. *(D) Thin targets and hubs.* In seed 0 the thinnest miRNA targets are hsa-miR-107 and hsa-miR-103a-3p, which have identical counts in all five folds (16, 2, 9, 13, 15); I expect nearly identical positive sets, since the two mature sequences are almost the same (inference from the sequences, not checked in the data). The thinnest RBP is HNRNPK (17, 21, 11, 6, 2), followed by FAM120A (minimum 5) and PRPF4 (minimum 7). Because the thin targets differ between seeds (the lowest fold count of the miRNA targets is 2, 3 and 5), the thinness comes from where the groups fall and not from the target alone. The five RBP hubs have 124–165 positives in every fold of seed 0, which is 42.9–57.7% of the labelled circRNAs of a fold (derived), against 51.0–53.2% overall. *(E) Siblings outside the task.* In seed 0, 19,573 non-task circRNAs with a real host gene lie in a host group of the model input, 3,503–4,372 per fold. These circRNAs would have to follow their group into its fold if they were ever used. The figure is 582 lower than the 20,155 of Cell 39, which counted the host genes of all 1,644 task circRNAs (derived); I attribute the difference to host genes whose task circRNAs are all above 5,000 nt (inference; not identified).

**Not checked.** (i) Whether the positive sets of hsa-miR-107 and hsa-miR-103a-3p are identical, and more generally how redundant the labels of the 21 miRNAs and 83 RBPs are (as in Cell 40, Not checked i). (ii) Positives per target and fold were counted for the working variants only; the cap-10 variants (10 miRNAs and 96 RBPs with at least 50 positives) were not tabulated per fold. (iii) Whether siblings among the 26 placeholder-gene circRNAs fall in different folds, and whether circRNAs from the same paper or the same disease are grouped (not controlled, as in the old Cell 44). (iv) Which host genes account for the 582 non-task siblings that are not in a group of the model input. (v) The effect of the choice of seed on individual thin targets beyond the three seeds shown.

**Decision.** (i) The host-gene-grouped split of the old Cell 44 stands unchanged (greedy balancing by pairs, three seeds, groups from the node table). Nothing is frozen yet; the folds are written in the freezing cell. (ii) The floor of 50 positive circRNAs is kept for both heads (21 miRNAs, 83 RBPs). The floor of 30 is dropped for the miRNA head, since half of its 130 targets have a fold below 5 positives. For the RBP head it stays as a sensitivity set (103 targets). (iii) The five-fold structure keeps its three seeds, and results are reported as the spread over seeds, because the thin targets change with the seed. (iv) Per-target metrics are computed on the out-of-fold predictions pooled over the five folds, then averaged over targets, so that a target with a thin fold still contributes at least 50 positives; per-fold values are reported for the CDA task and for the heads at the level of all targets. Targets with fewer than 5 positives in a fold are listed for each seed. (v) The non-task siblings follow their host group into its fold. None are used so far, and their use (extra labels, negatives, pretraining) is decided with the sibling test. (vi) Next: Cell 43 builds the evaluation negatives for the CDA task and the popularity baseline (old Cell 45) on these groups; then the sibling-leakage test (old Cell 46), extended to the auxiliary labels; then a length and degree shortcut check of the ENCORI labels (Cell 40, Not checked iv); then the freezing cell.

### 43. Evaluation Negatives and Disease-Popularity Shortcut

**Objective.** Rebuild the evaluation negatives of the CDA task (random and degree-matched) on the host-gene folds of Cell 42, for all three seeds, and test a disease-popularity baseline as a shortcut check, in order to verify that the result of the old Cell 45 is reproduced for seed 0 and to measure how much it varies with the seed. The cell is a dry run; nothing is written.

In [89]:
# Cell 43 - Evaluation negatives for the CDA task and a disease-popularity shortcut check (dry run: nothing is written)
# Self-contained: rebuilds groups and folds from the node table (Cell 39) as in Cell 42.
from sklearn.metrics import roc_auc_score
SEEDS = (0, 1, 2)

# ---- 1. pairs, groups and folds (as Cell 42) ----------------------------------------------------------------------------------
uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv", keep_default_na=False,
                  usecols=["circbase_id", "in_task", "use_primary", "host_group"], dtype={"in_task": bool, "use_primary": bool})
mi = uni[uni["in_task"] & uni["use_primary"]].set_index("circbase_id")
pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
pairs["group"] = pairs["circrna_id"].map(mi["host_group"])
assert (len(pairs), pairs["circrna_id"].nunique(), pairs["group"].nunique()) == (2_137, 1_582, 1_251)
grp = pairs.groupby("group").agg(n_pairs=("circrna_id", "size"))

def assign(seed, k=5):
    rng = np.random.default_rng(seed)
    order = grp.sample(frac=1, random_state=seed).sort_values("n_pairs", ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for g, n in order["n_pairs"].items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[g] = f; load[f] += n
    return fold
folds = {s: pairs["group"].map(assign(s)) for s in SEEDS}
assert folds[0].value_counts().sort_index().tolist() == [428, 427, 428, 427, 427], "seed-0 folds differ from Cell 42"

# ---- 2. negative sampling (as the old Cell 45) --------------------------------------------------------------------------------------
dis = np.array(sorted(pairs["disease"].unique()))
print("diseases:", len(dis), "(expected 207)")
assert len(dis) == 207
pos_all = set(zip(pairs["circrna_id"], pairs["disease"]))
pos_by_c = pairs.groupby("circrna_id")["disease"].apply(set).to_dict()
dcount = pairs["disease"].value_counts().reindex(dis).astype(float)        # positives per disease, all folds (used for matching)

def sample_negatives(test_pos, scheme, seed):
    """Same circRNAs as the positives, with a disease that is not a known positive of that circRNA."""
    rng = np.random.default_rng(seed); out = []
    for c, g in test_pos.groupby("circrna_id", sort=True):
        cand = np.array([d for d in dis if d not in pos_by_c[c]])
        w = dcount.reindex(cand).values
        p = None if scheme == "random" else w / w.sum()
        out += [(c, d) for d in rng.choice(cand, size=len(g), replace=False, p=p)]
    return pd.DataFrame(out, columns=["circrna_id", "disease"])

# ---- 3. checks and the popularity baseline per seed, fold and scheme ------------------------------------------------------------------
rows = []
for seed in SEEDS:
    fold = folds[seed]
    for f in range(5):
        test = pairs[fold == f]
        train_cnt = pairs.loc[fold != f, "disease"].value_counts()          # popularity from the training folds only
        cold = float((~test["disease"].isin(train_cnt.index)).mean())
        for scheme in ("random", "degree_matched"):
            neg, again = (sample_negatives(test, scheme, seed=1000 * seed + f) for _ in range(2))
            y = np.r_[np.ones(len(test)), np.zeros(len(neg))]
            s = np.r_[test["disease"].map(train_cnt).fillna(0).values, neg["disease"].map(train_cnt).fillna(0).values]
            rows.append(dict(seed=seed, fold=f, scheme=scheme, n_pos=len(test), n_neg=len(neg),
                             pop_auroc=round(roc_auc_score(y, s), 3), cold_disease_share=round(cold, 3),
                             pos_mean_dis_count=round(test["disease"].map(dcount).mean(), 1),
                             neg_mean_dis_count=round(neg["disease"].map(dcount).mean(), 1),
                             collisions=sum((c, d) in pos_all for c, d in zip(neg["circrna_id"], neg["disease"])),
                             wrong_fold=int((~neg["circrna_id"].isin(set(test["circrna_id"]))).sum()),
                             dup=int(neg.duplicated().sum()), reproducible=neg.equals(again)))
res = pd.DataFrame(rows)
print(res[res["seed"] == 0].drop(columns="seed").to_string(index=False))

# seed 0 must reproduce the old Cell 45 (per-fold popularity AUROC) and every integrity check must be clean
old = {"random": [0.811, 0.821, 0.807, 0.846, 0.818], "degree_matched": [0.506, 0.517, 0.496, 0.530, 0.514]}
for sc, v in old.items():
    got = res[(res["seed"] == 0) & (res["scheme"] == sc)].sort_values("fold")["pop_auroc"].tolist()
    print(f"seed 0, {sc}: per-fold AUROC {got} | old Cell 45 {v} -> {'OK' if got == v else 'MISMATCH'}")
    assert got == v
assert (res["collisions"] == 0).all() and (res["wrong_fold"] == 0).all() and (res["dup"] == 0).all() and res["reproducible"].all()
assert (res["n_pos"] == res["n_neg"]).all()
print("all seeds: no collisions with positives, no negatives outside the circRNA's fold, no duplicates, reproducible, as many negatives as positives")

print("\nPopularity AUROC by scheme and seed (mean / min / max over folds):")
print(res.groupby(["scheme", "seed"])["pop_auroc"].agg(["mean", "min", "max"]).round(3).to_string())
print("\nBy scheme over all seeds and folds:")
print(res.groupby("scheme")["pop_auroc"].agg(["mean", "min", "max"]).round(3).to_string())
print("\ncold-disease share (test positives whose disease has no training positive), by seed (min / mean / max over folds):")
print(res[res["scheme"] == "random"].groupby("seed")["cold_disease_share"].agg(["min", "mean", "max"]).round(3).to_string())
print("\nDry run: nothing written.")

diseases: 207 (expected 207)
 fold         scheme  n_pos  n_neg  pop_auroc  cold_disease_share  pos_mean_dis_count  neg_mean_dis_count  collisions  wrong_fold  dup  reproducible
    0         random    428    428      0.811               0.035                49.4                11.1           0           0    0          True
    0 degree_matched    428    428      0.506               0.035                49.4                43.2           0           0    0          True
    1         random    427    427      0.821               0.035                49.4                 9.4           0           0    0          True
    1 degree_matched    427    427      0.517               0.035                49.4                45.7           0           0    0          True
    2         random    428    428      0.807               0.035                47.7                 9.8           0           0    0          True
    2 degree_matched    428    428      0.496               0.035            

**Interpretation.** *(A) Reproduction and integrity.* The rebuilt folds equal those of Cell 42 for seed 0, and the per-fold popularity AUROCs of seed 0 equal the old Cell 45 in both schemes (random 0.811, 0.821, 0.807, 0.846, 0.818; degree-matched 0.506, 0.517, 0.496, 0.530, 0.514; asserted). The same holds for the cold-disease shares and the mean disease counts printed for seed 0. For all three seeds, every fold has as many negatives as positives (2,137 per scheme and seed, i.e. 4,274 per seed), no negative equals a known positive pair, none lies outside the fold of its circRNA, none is duplicated, and drawing the negatives twice with the same seed gives the same table. The 207 diseases are those of the model-input pairs. *(B) Random negatives.* A baseline that scores a pair by the number of training positives of its disease reaches a mean AUROC of 0.820 over the 15 folds (per seed 0.821, 0.816 and 0.825; range 0.801–0.858). The reason is visible in the disease counts: the diseases of random negatives have a mean of 9.4–11.1 positives (seed 0), close to the 10.3 expected for a uniform draw (2,137/207, derived), whereas those of the positives have 47.7–54.8. With random negatives, disease popularity alone therefore separates the classes. *(C) Degree-matched negatives.* When the disease of a negative is drawn with probability proportional to its number of positives, the same baseline falls to a mean of 0.509 (per seed 0.513, 0.504 and 0.510; range 0.479–0.547). The spread between folds is about what chance alone gives for 427 positives and 427 negatives (standard error about 0.02, hand calculation). A small gap remains in the matching: the mean disease count of the negatives is 2.1–6.2 lower than that of the positives in the folds of seed 0 (43.2–48.7 against 47.7–54.8). A likely cause is that the draw excludes the diseases a circRNA already has, which tend to be the popular ones (inference; not tested). The residual does not produce an AUROC above chance. *(D) Cold diseases.* The share of test positives whose disease has no positive in the training folds is 0.021–0.061 per fold (means 0.039, 0.037 and 0.036 for the three seeds), so the popularity baseline scores about 4% of the positives with zero. *(E) Seeds.* The mean AUROC differs by at most 0.009 between seeds in the random scheme and 0.009 in the degree-matched scheme, so the conclusion does not depend on the seed.

**Not checked.** (i) The negatives are unobserved pairs, not verified negatives; the collision check only excludes known positives, so some negatives may be true associations that are not in CircR2Disease. (ii) Shortcuts other than disease popularity, for example circRNA length, the number of diseases per circRNA, or similarity between diseases. (iii) The matching uses disease counts over all folds, which is acceptable for evaluation negatives (no model sees them), but the baseline uses training-fold counts, so the two are not the same quantity. (iv) The sibling shortcut (old Cell 46) and the corresponding test for the ENCORI labels, which come next. (v) Negatives for the two auxiliary heads: none are sampled, since a target that is not positive for a circRNA is a negative in a multi-label head (Cell 41 (iv) fixes which circRNAs are labelled).

**Decision.** (i) Degree-matched negatives are the primary evaluation negatives for the CDA task, because popularity alone gives 0.509 on them and 0.820 on random negatives. Random negatives stay as a secondary row that shows how much the choice matters. Both schemes use one negative per positive. (ii) The sampling code of the old Cell 45 is kept unchanged, with the seed of fold `f` in run `s` equal to `1000 * s + f`. For seed 0 this is the seed of the old freezing cell (Cell 47: `seed = f`), so the seed-0 negatives are the same as before. (iii) Negatives depend on the fold of their circRNA, and the folds differ between seeds, so negatives are frozen for all three seeds, not for seed 0 only as in the old Cell 47. (iv) The popularity baseline is reported with the sibling baseline in the leakage section: 0.509 (range 0.479–0.547) on degree-matched and 0.820 (0.801–0.858) on random negatives, as the mean and range over 15 folds. (v) The 2.1–6.1% of positives with a cold disease keep a score of zero in the baseline and are not removed. (vi) Next: Cell 44, the sibling-leakage test of the old Cell 46 under the three split schemes, repeated for all three seeds and extended to the ENCORI labels, using the labelled non-task siblings; then Cell 45, a length and degree shortcut check of the ENCORI labels; then Cell 46, which freezes the folds, the negatives and the label tables.

### 44. Sibling Leakage: Disease Associations and ENCORI Labels

**Objective.** Repeat the host-gene sibling baseline of the old Cell 46 under three split schemes (host-gene grouped, circRNA grouped, random edge) for the three seeds, and extend it to the ENCORI labels of the two auxiliary heads: for each target of the working heads, the fraction of the labelled siblings of a circRNA (same real host gene) that are positive is used as a score, with siblings taken (i) from the task circRNAs of the other folds and (ii) from the non-task circRNAs of the same host gene. The aim is to measure how much label information a circRNA's siblings carry and whether the host-gene grouping removes it. The cell is a dry run; nothing is written.

In [90]:
# Cell 44 - Sibling leakage: disease associations (old Cell 46, three seeds) and ENCORI labels (dry run: nothing is written)
# Uses from earlier cells: PROCESSED_DIR (Cell 1); everything else is re-read from the files of Cells 26, 39 and 41.
from sklearn.metrics import roc_auc_score
SEEDS = (0, 1, 2)

# ---- 0. pairs, groups, node table (as Cells 42 and 43) --------------------------------------------------------------------------
uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv", keep_default_na=False,
                  usecols=["circbase_id", "in_task", "use_primary", "in_mirna", "in_rbp", "host_placeholder", "host_group"],
                  dtype={"in_task": bool, "use_primary": bool, "in_mirna": bool, "in_rbp": bool, "host_placeholder": bool})
mi = uni[uni["in_task"] & uni["use_primary"]].set_index("circbase_id")
pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
pairs["group"] = pairs["circrna_id"].map(mi["host_group"])
assert (len(pairs), pairs["circrna_id"].nunique(), pairs["group"].nunique()) == (2_137, 1_582, 1_251)

def greedy(units, seed, k=5):                                     # as the old Cell 46
    rng = np.random.default_rng(seed)
    order = units.sample(frac=1, random_state=seed).sort_values(ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for u, n in order.items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[u] = f; load[f] += n
    return fold

# ---- 1. disease associations: sibling baseline under three split schemes, three seeds ---------------------------------------------------
dis = np.array(sorted(pairs["disease"].unique()))
pos_by_c = pairs.groupby("circrna_id")["disease"].apply(set).to_dict()
dcount = pairs["disease"].value_counts().reindex(dis).astype(float)
def sample_negatives(test_pos, seed):                             # degree-matched, as Cell 43
    rng = np.random.default_rng(seed); out = []
    for c, g in test_pos.groupby("circrna_id", sort=True):
        cand = np.array([d for d in dis if d not in pos_by_c[c]])
        w = dcount.reindex(cand).values
        out += [(c, d) for d in rng.choice(cand, size=len(g), replace=False, p=w / w.sum())]
    return pd.DataFrame(out, columns=["circrna_id", "disease"])

cg = pairs.drop_duplicates("circrna_id").set_index("circrna_id")["group"]
gene_circ = pairs[~pairs["group"].str.startswith("solo:")].groupby("group")["circrna_id"].unique().to_dict()
def sib_score(c, d, f, pf):                                       # other circRNAs of the same real host gene, positive for d, outside fold f
    g = cg[c]
    if g.startswith("solo:"):
        return 0
    return sum(1 for s in gene_circ[g] if s != c and (s, d) in pf and pf[(s, d)] != f)

rows, FOLD_C = [], {}
for seed in SEEDS:
    pairs["fold_gene"] = pairs["group"].map(greedy(pairs.groupby("group").size(), seed))
    pairs["fold_circ"] = pairs["circrna_id"].map(greedy(pairs.groupby("circrna_id").size(), seed))
    pairs["fold_edge"] = np.random.default_rng(seed).permutation(len(pairs)) % 5
    if seed == 0:
        assert pairs.groupby("fold_gene").size().tolist() == [428, 427, 428, 427, 427]
    FOLD_C[seed] = {"host-gene grouped": pairs.groupby("circrna_id")["fold_gene"].first(),
                    "circRNA grouped": pairs.groupby("circrna_id")["fold_circ"].first()}
    for scheme, col in [("host-gene grouped", "fold_gene"), ("circRNA grouped", "fold_circ"), ("random edge", "fold_edge")]:
        pf = dict(zip(zip(pairs["circrna_id"], pairs["disease"]), pairs[col]))
        for f in range(5):
            test = pairs[pairs[col] == f]
            neg = sample_negatives(test, seed=1000 * seed + f)
            sp = np.array([sib_score(c, d, f, pf) for c, d in zip(test["circrna_id"], test["disease"])])
            sn = np.array([sib_score(c, d, f, pf) for c, d in zip(neg["circrna_id"], neg["disease"])])
            y = np.r_[np.ones(len(sp)), np.zeros(len(sn))]
            rows.append(dict(seed=seed, scheme=scheme, fold=f, n_pos=len(sp), auroc=round(roc_auc_score(y, np.r_[sp, sn]), 3),
                             pos_with_sibling=round(float((sp > 0).mean()), 3), neg_with_sibling=round(float((sn > 0).mean()), 3)))
res = pd.DataFrame(rows)
print("seed 0, per fold:\n", res[res["seed"] == 0].drop(columns="seed").to_string(index=False))
old = {"circRNA grouped": ([0.515, 0.531, 0.532, 0.527, 0.532], [0.042, 0.077, 0.075, 0.070, 0.072], [0.012, 0.016, 0.012, 0.016, 0.009]),
       "random edge":     ([0.532, 0.521, 0.520, 0.529, 0.526], [0.070, 0.056, 0.052, 0.063, 0.080], [0.007, 0.014, 0.012, 0.005, 0.028]),
       "host-gene grouped": ([0.5] * 5, [0.0] * 5, [0.0] * 5)}
for sc, (a, p, n) in old.items():
    r = res[(res["seed"] == 0) & (res["scheme"] == sc)].sort_values("fold")
    ok = r["auroc"].tolist() == a and r["pos_with_sibling"].tolist() == p and r["neg_with_sibling"].tolist() == n
    print(f"seed 0, {sc}: equal to the old Cell 46 -> {'OK' if ok else 'MISMATCH'}")
    assert ok
print("\nsibling baseline, mean over folds, by scheme and seed:\n",
      res.groupby(["scheme", "seed"])[["auroc", "pos_with_sibling", "neg_with_sibling"]].mean().round(3).to_string())
print("\nby scheme over all seeds and folds (mean / min / max AUROC):\n", res.groupby("scheme")["auroc"].agg(["mean", "min", "max"]).round(3).to_string())

# ---- 2. ENCORI labels: targets, label matrices and the siblings ---------------------------------------------------------------------
st  = pd.read_csv(PROCESSED_DIR / "encori_label_status.csv", usecols=["circbase_id", "use_mirna_main", "use_rbp_main"],
                  dtype={"use_mirna_main": bool, "use_rbp_main": bool}).set_index("circbase_id")
tmi = pd.read_csv(PROCESSED_DIR / "encori_mirna_targets.csv", usecols=["mirna_id", "mirna_name", "n_clip2", "head_floor50"])
tri = pd.read_csv(PROCESSED_DIR / "encori_rbp_targets.csv", keep_default_na=False, usecols=["rbp", "n_exonic_c2", "head_floor50", "hub_over_50pct"])
tmi, tri = tmi[tmi["head_floor50"]].reset_index(drop=True), tri[tri["head_floor50"]].reset_index(drop=True)
assert (len(tmi), len(tri)) == (21, 83)
circs = {"miRNA": st.index[st["use_mirna_main"]], "RBP": st.index[st["use_rbp_main"]]}
assert (len(circs["miRNA"]), len(circs["RBP"])) == (1_433, 1_405)

ml = pd.read_csv(PROCESSED_DIR / "encori_mirna_labels.csv", usecols=["circbase_id", "mirna_id", "pos_clip2", "model_input"])
rl = pd.read_csv(PROCESSED_DIR / "encori_rbp_labels.csv", keep_default_na=False, usecols=["circbase_id", "rbp", "pos_exonic_c2", "model_input"])
def label_matrix(lab, key, flag, targets, rows_):
    d = lab[lab["model_input"] & lab[flag] & lab[key].isin(targets)]
    return (pd.crosstab(d["circbase_id"], d[key]).reindex(index=rows_, columns=targets, fill_value=0) > 0).astype("int8")
HEADS = {"miRNA": dict(Y=label_matrix(ml, "mirna_id", "pos_clip2", tmi["mirna_id"].tolist(), circs["miRNA"]), key="mirna_id",
                       targets=tmi["mirna_id"].tolist(), names=tmi["mirna_name"].tolist(), src="in_mirna"),
         "RBP":   dict(Y=label_matrix(rl, "rbp", "pos_exonic_c2", tri["rbp"].tolist(), circs["RBP"]), key="rbp",
                       targets=tri["rbp"].tolist(), names=tri["rbp"].tolist(), src="in_rbp")}
assert (HEADS["miRNA"]["Y"].sum().values == tmi["n_clip2"].values).all() and (HEADS["RBP"]["Y"].sum().values == tri["n_exonic_c2"].values).all()
print("\nlabel matrices: miRNA", HEADS["miRNA"]["Y"].shape, "| RBP", HEADS["RBP"]["Y"].shape, "(column sums equal the target tables of Cell 41, asserted)")

real_groups = set(mi.loc[~mi["host_placeholder"], "host_group"])
sib = uni[~uni["in_task"] & ~uni["host_placeholder"] & uni["host_group"].isin(real_groups)]
print(f"non-task circRNAs in a host group of the model input: {len(sib):,} (Cell 42: 19,573) | labelled in the miRNA table: "
      f"{int(sib['in_mirna'].sum()):,} | in the RBP table: {int(sib['in_rbp'].sum()):,}")
assert len(sib) == 19_573
sib_ids = set(sib["circbase_id"])
E = {"miRNA": pd.read_csv(PROCESSED_DIR / "encori_mirna_edges_all.csv.gz", usecols=["circbase_id", "mirna_id", "clip_max"]),
     "RBP":   pd.read_csv(PROCESSED_DIR / "encori_rbp_edges_all.csv.gz", keep_default_na=False, usecols=["circbase_id", "rbp", "c_max_exonic"])}
E["miRNA"] = E["miRNA"][(E["miRNA"]["clip_max"] >= 2) & E["miRNA"]["circbase_id"].isin(sib_ids) & E["miRNA"]["mirna_id"].isin(HEADS["miRNA"]["targets"])]
E["RBP"]   = E["RBP"][(E["RBP"]["c_max_exonic"] >= 2) & E["RBP"]["circbase_id"].isin(sib_ids) & E["RBP"]["rbp"].isin(HEADS["RBP"]["targets"])]

def per_target_auc(Y, S, ok):
    out = []
    for j in range(Y.shape[1]):
        y, s = Y.values[ok, j], S[ok, j]
        out.append(np.nan if y.min() == y.max() else roc_auc_score(y, s))
    return pd.Series(out, index=Y.columns)

# ---- 3. (ii) non-task siblings: fraction of the labelled siblings that are positive for the target ---------------------------------------
def nontask_scores(h):
    Y, key = h["Y"], h["key"]
    lab = sib[sib[h["src"]]]                                      # labelled non-task siblings (all-negative rows count as negatives)
    n_lab = lab.groupby("host_group").size()
    e = E[h["head"]].merge(lab[["circbase_id", "host_group"]], on="circbase_id")
    cnt = e.groupby(["host_group", key]).size().unstack(fill_value=0).reindex(columns=h["targets"], fill_value=0)
    g = mi.loc[Y.index, "host_group"]
    n = g.map(n_lab).fillna(0).values
    S = cnt.reindex(g.values).fillna(0).values / np.maximum(n, 1)[:, None]
    return S, n > 0
for k, h in HEADS.items():
    h["head"] = k

rows_b, AUC = [], {}
for k, h in HEADS.items():
    S, ok = nontask_scores(h)
    Y = h["Y"]
    auc = per_target_auc(Y, S, ok)
    rng = np.random.default_rng(0); idx = np.flatnonzero(ok); S2 = S.copy(); S2[idx] = S[rng.permutation(idx)]
    ctl = per_target_auc(Y, S2, ok)                               # negative control: scores shuffled among the covered circRNAs
    AUC[k] = auc
    rows_b.append({"head": k, "labelled circRNAs": len(Y), "with >= 1 labelled non-task sibling": int(ok.sum()),
                   "% covered": round(100 * ok.mean(), 1), "targets": len(auc),
                   "positive rate, all (mean over targets)": round(float(Y.values.mean()), 3),
                   "positive rate, covered": round(float(Y.values[ok].mean()), 3),
                   "AUROC median": round(auc.median(), 3), "AUROC q25": round(auc.quantile(.25), 3), "AUROC q75": round(auc.quantile(.75), 3),
                   "AUROC min": round(auc.min(), 3), "AUROC max": round(auc.max(), 3),
                   "targets AUROC > 0.6": int((auc > .6).sum()), "control median": round(ctl.median(), 3)})
print("\nENCORI labels, siblings outside the task (same host gene), per-target AUROC of the sibling positive fraction:\n",
      pd.DataFrame(rows_b).set_index("head").T.to_string())
hubs = tri.loc[tri["hub_over_50pct"], "rbp"].tolist()
print("\nRBP hubs, AUROC:", AUC["RBP"].loc[hubs].round(3).to_dict())
for k, h in HEADS.items():
    a = AUC[k].copy(); a.index = h["names"]
    print(f"{k}: highest", a.sort_values(ascending=False).head(3).round(3).to_dict(), "| lowest", a.sort_values().head(3).round(3).to_dict())

# ---- 4. (i) task siblings in the other folds, circRNA-grouped versus host-gene-grouped folds -------------------------------------------
def task_scores(Y, grp, fold):
    S = np.zeros(Y.shape); cov = np.zeros(len(Y), bool)
    g_all, f_all = grp.reindex(Y.index).values, fold.reindex(Y.index).values
    for f in range(5):
        t, p = f_all == f, f_all != f
        pool = Y[p]
        cnt = pool.groupby(g_all[p]).sum(); n = pool.groupby(g_all[p]).size()
        nn = pd.Series(g_all[t]).map(n).fillna(0).values
        S[t] = cnt.reindex(g_all[t]).fillna(0).values / np.maximum(nn, 1)[:, None]
        cov[t] = nn > 0
    return S, cov
rows_c = []
for seed in SEEDS:
    for scheme in ("host-gene grouped", "circRNA grouped"):
        for k, h in HEADS.items():
            S, cov = task_scores(h["Y"], mi["host_group"], FOLD_C[seed][scheme])
            if scheme == "host-gene grouped":
                assert not cov.any(), "a same-gene sibling in another fold under host-gene grouping"
            auc = per_target_auc(h["Y"], S, cov) if cov.any() else pd.Series([np.nan])
            rows_c.append({"seed": seed, "head": k, "scheme": scheme, "% with a labelled task sibling in the other folds": round(100 * cov.mean(), 1),
                           "AUROC median": round(auc.median(), 3), "AUROC q25": round(auc.quantile(.25), 3), "AUROC q75": round(auc.quantile(.75), 3)})
print("\nENCORI labels, siblings among the task circRNAs of the other folds:\n", pd.DataFrame(rows_c).to_string(index=False))
print("\nDry run: nothing written.")

seed 0, per fold:
            scheme  fold  n_pos  auroc  pos_with_sibling  neg_with_sibling
host-gene grouped     0    428  0.500             0.000             0.000
host-gene grouped     1    427  0.500             0.000             0.000
host-gene grouped     2    428  0.500             0.000             0.000
host-gene grouped     3    427  0.500             0.000             0.000
host-gene grouped     4    427  0.500             0.000             0.000
  circRNA grouped     0    427  0.515             0.042             0.012
  circRNA grouped     1    427  0.531             0.077             0.016
  circRNA grouped     2    428  0.532             0.075             0.012
  circRNA grouped     3    427  0.527             0.070             0.016
  circRNA grouped     4    428  0.532             0.072             0.009
      random edge     0    428  0.532             0.070             0.007
      random edge     1    428  0.521             0.056             0.014
      random edge  

**Interpretation.** *(A) Host-gene grouping and the old Cell 46.* Under host-gene grouping no circRNA has a same-gene sibling in another fold: the share of positives and negatives with a sibling is 0.000 in all five folds of seed 0 and in the seed means (0.000 for seeds 1 and 2), and the sibling baseline for the disease associations gives an AUROC of 0.500 in all 15 seed-by-fold runs. For the ENCORI labels the check of the task siblings in the other folds finds 0.0% coverage for both heads under this split in all three seeds, and its assertion passed. For seed 0 the per-fold AUROC, the share of positives with a sibling and the share of negatives with a sibling equal the old Cell 46 in all three schemes (asserted). As in the old cell, the 0.500 follows from the construction of the folds (the group determines the fold) and does not show an absence of biological leakage.

*(B) Disease associations under the other schemes.* The baseline gains little when the folds do not follow the host gene: the mean AUROC is 0.528 for circRNA-grouped folds (per seed 0.527, 0.530 and 0.528; range 0.515–0.534) and 0.526 for random-edge folds (0.526, 0.525 and 0.527; range 0.519–0.535). A sibling in another fold exists for 6.4–7.2% of the positives and for 1.0–1.5% of the negatives (means per seed and scheme). The disease task is therefore only mildly sensitive to the split scheme.

| Split scheme | Mean AUROC, seeds 0 / 1 / 2 | Mean over 15 folds | Range over 15 folds | Positives with a sibling | Negatives with a sibling |
|:---|:---:|:---:|:---:|:---:|:---:|
| **Host-gene grouped** | 0.500 / 0.500 / 0.500 | **0.500** | 0.500 – 0.500 | 0.0% | 0.0% |
| circRNA grouped | 0.527 / 0.530 / 0.528 | 0.528 | 0.515 – 0.534 | 6.5 – 7.2% | 1.0 – 1.3% |
| Random edge | 0.526 / 0.525 / 0.527 | 0.526 | 0.519 – 0.535 | 6.4 – 6.7% | 1.3 – 1.5% |

*(C) ENCORI labels, siblings outside the task.* 19,573 non-task circRNAs lie in a host group of the model input (as in Cell 42), of which 19,518 are in the miRNA table and 19,416 in the RBP table. 1,354 of the 1,433 labelled circRNAs of the miRNA head (94.5%) and 1,335 of the 1,405 of the RBP head (95.0%) have at least one labelled non-task sibling; 79 and 70 have none (derived). For these circRNAs, the fraction of labelled siblings that are positive for a target separates positives from negatives almost perfectly. The median per-target AUROC is 0.967 for the 21 miRNAs (q25 0.953, q75 0.971, range 0.952–0.981) and 0.928 for the 83 RBPs (q25 0.908, q75 0.949, range 0.805–0.991), and all 104 targets are above 0.6. The control with the scores shuffled among the covered circRNAs gives a median of 0.478 and 0.500. The positive rate among the covered circRNAs (mean over targets, 0.055 and 0.182) is close to that of all labelled circRNAs (0.053 and 0.178), so the covered set is not a different population. The five RBP hubs reach 0.807–0.902 (ALYREF 0.872, RNPS1 0.902, SCAF4 0.866, SCAF8 0.866, U2AF2 0.807). The highest miRNAs are hsa-miR-103a-3p, hsa-miR-107 and hsa-miR-665 (0.981) and the lowest hsa-miR-6838-5p (0.952), hsa-miR-195-5p and hsa-miR-16-5p (0.953); the highest RBPs are MTDH (0.991), BCLAF1 and TRA2A (0.976) and the lowest PRPF8 (0.805), U2AF2 (0.807) and CSTF2T (0.861). The identical AUROC of hsa-miR-103a-3p and hsa-miR-107 agrees with the nearly identical positive sets expected for these two miRNAs (Cell 42 D; inference, not checked).

| | miRNA head | RBP head |
|:---|---:|---:|
| Labelled circRNAs | 1,433 | 1,405 |
| With at least one labelled non-task sibling | 1,354 (94.5%) | 1,335 (95.0%) |
| Without (derived) | 79 | 70 |
| Targets | 21 | 83 |
| Positive rate, all / covered (mean over targets) | 0.053 / 0.055 | 0.178 / 0.182 |
| **Median AUROC per target** | **0.967** | **0.928** |
| Interquartile range | 0.953 – 0.971 | 0.908 – 0.949 |
| Range | 0.952 – 0.981 | 0.805 – 0.991 |
| Targets above 0.6 | 21 of 21 | 83 of 83 |
| Control, scores shuffled (median) | 0.478 | 0.500 |
| Highest | hsa-miR-103a-3p, hsa-miR-107, hsa-miR-665 (0.981) | MTDH (0.991), BCLAF1, TRA2A (0.976) |
| Lowest | hsa-miR-6838-5p (0.952), hsa-miR-195-5p, hsa-miR-16-5p (0.953) | PRPF8 (0.805), U2AF2 (0.807), CSTF2T (0.861) |

| RBP hub | ALYREF | RNPS1 | SCAF4 | SCAF8 | U2AF2 |
|:---|:---:|:---:|:---:|:---:|:---:|
| AUROC | 0.872 | 0.902 | 0.866 | 0.866 | 0.807 |

*(D) ENCORI labels, task siblings in the other folds.* Under circRNA-grouped folds, 29.6–32.1% of the labelled circRNAs have a labelled task sibling in another fold (miRNA 29.6%, 31.1%, 31.6%; RBP 29.9%, 31.5%, 32.1% for seeds 0, 1 and 2). On these circRNAs the median AUROC is 0.823–0.860 for both heads in all three seeds (interquartile range 0.799–0.866), so circRNA-grouped folds would pass host-gene information on to the auxiliary heads, and host-gene grouping removes it.

| Split scheme | Seed | Head | Labelled circRNAs with a task sibling in another fold | Median AUROC (interquartile range) |
|:---|:---:|:---|:---:|:---:|
| **Host-gene grouped** | 0, 1, 2 | miRNA and RBP | **0.0%** | not defined (no such circRNA) |
| circRNA grouped | 0 | miRNA | 29.6% | 0.860 (0.852 – 0.866) |
| circRNA grouped | 0 | RBP | 29.9% | 0.836 (0.809 – 0.861) |
| circRNA grouped | 1 | miRNA | 31.1% | 0.823 (0.813 – 0.861) |
| circRNA grouped | 1 | RBP | 31.5% | 0.836 (0.805 – 0.860) |
| circRNA grouped | 2 | miRNA | 31.6% | 0.847 (0.799 – 0.861) |
| circRNA grouped | 2 | RBP | 32.1% | 0.827 (0.801 – 0.863) |

*(E) Reading.* The disease labels carry little sibling information, whereas the ENCORI labels carry a great deal: the median AUROC is 0.93–0.97 for the non-task siblings and 0.82–0.86 for the task siblings in other folds. The two sibling pools differ in size and coverage, so I do not interpret the difference in strength. A likely cause of the signal is that ENCORI attributes the CLIP evidence of a binding site to every circRNA listed for it (Cell 29), so overlapping circRNAs of one gene share many of their labels (inference; not tested). The signal then describes the labelling and not a biological similarity of siblings.

**Not checked.** (i) Whether the sibling signal comes from shared exons and binding sites or from the host gene as such, and how it changes with the number of siblings and their overlap with the circRNA. (ii) Precision and recall: the AUROC does not show the precision of the sibling score at the low positive rates of the miRNA head (about 5%). (iii) The 79 miRNA and 70 RBP circRNAs without a labelled non-task sibling; their labels were not scored. (iv) The quality of the host groups: 26 singleton placeholder groups, and whether the FASTA gene field agrees with the gene names of ENCORI (Cell 39, Not checked ii). (v) The shuffled control was run once (seed 0 of the generator); the scores of the non-task siblings do not depend on the folds, so they were not repeated per seed. (vi) The random-edge scheme was evaluated for the disease associations only, not for the ENCORI labels.

**Decision.** (i) Host-gene-grouped folds are the main protocol for all three heads, because they remove sibling information completely (0.500 for the disease task, no sibling in another fold for the ENCORI labels), which circRNA-grouped and random-edge folds do not (0.526–0.528 for the disease task, 0.82–0.86 for the ENCORI labels under circRNA grouping). (ii) circRNA-grouped and random-edge splits are kept only as comparisons for the disease task, where the leakage is weak; they are not used for the two auxiliary heads, where it is large. (iii) The sibling AUROC of the non-task circRNAs (median 0.967 for miRNAs and 0.928 for RBPs) is reported as a reference level of host-gene information, not as a baseline for the model, because it uses labels that are unavailable under the grouped split. The sibling baseline of the disease task (0.500) is reported together with the popularity baseline of Cell 43. (iv) No host-gene identifier, no non-task ENCORI label and no feature aggregated over siblings enters the model. Any non-task circRNA that is used without labels (for example for pretraining) follows its host group into the fold of its task circRNAs (Cell 42 v) and is excluded from the training of that fold. (v) The three seeds of the folds and of the negatives stand, as decided in Cell 43. Nothing is written in this cell. (vi) Next: Cell 45, a length and degree shortcut check of the ENCORI labels (Cell 40, Not checked iv); then Cell 46, which freezes the folds (three seeds), the evaluation negatives (three seeds) and the label tables.

### 45. Length and Degree Shortcuts in the ENCORI Labels

**Objective.** Test, for the 21 miRNA and 83 RBP targets of the working heads, how far two circRNA-level properties alone predict the ENCORI labels under the host-gene-grouped folds of Cell 42 (three seeds): (i) circRNA length, which a sequence model can read directly, and (ii) the number of positives that the same circRNA has for the other targets of the head (leave-target-out degree), which is a ceiling for a degree shortcut because it uses labels that are unavailable at test time. Each score is a logistic regression fitted on the training folds and evaluated on pooled out-of-fold predictions, with length shuffled among the circRNAs as a negative control. In addition, establish whether the circRNAs without any positive differ in length from the others, and whether length and degree drive the overlap between the two heads, in order to decide whether the shortcuts need to be controlled in the model evaluation and whether the all-negative rows of the main run are a biased set. The cell is a dry run; nothing is written.

In [91]:
# Cell 45 - Length and degree shortcuts in the ENCORI labels (dry run: nothing is written)
# Uses from earlier cells: PROCESSED_DIR (Cell 1); everything else is re-read from the files of Cells 26, 39 and 41.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
SEEDS = (0, 1, 2)

# ---- 0. node table, pairs, host groups and folds (as Cells 42-44) -------------------------------------------------------------------
uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv", keep_default_na=False,
                  usecols=["circbase_id", "in_task", "use_primary", "host_group", "seq_len"],
                  dtype={"in_task": bool, "use_primary": bool})
mi = uni[uni["in_task"] & uni["use_primary"]].set_index("circbase_id")
assert len(mi) == 1_582
pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
pairs["group"] = pairs["circrna_id"].map(mi["host_group"])
assert (len(pairs), pairs["circrna_id"].nunique(), pairs["group"].nunique()) == (2_137, 1_582, 1_251)
grp = pairs.groupby("group").agg(n_pairs=("circrna_id", "size"))

def assign(seed, k=5):                                            # same greedy balancing as Cells 42-44
    rng = np.random.default_rng(seed)
    order = grp.sample(frac=1, random_state=seed).sort_values("n_pairs", ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for g, n in order["n_pairs"].items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[g] = f; load[f] += n
    return fold
FM = {s: pairs.drop_duplicates("circrna_id").set_index("circrna_id")["group"].map(assign(s)) for s in SEEDS}   # circRNA -> fold
assert pairs["group"].map(assign(0)).value_counts().sort_index().tolist() == [428, 427, 428, 427, 427], "seed-0 folds differ from Cell 42"

# ---- 1. label matrices of the two working heads (as Cell 44) ---------------------------------------------------------------------
st  = pd.read_csv(PROCESSED_DIR / "encori_label_status.csv", usecols=["circbase_id", "use_mirna_main", "use_rbp_main", "n_mirna_clip2", "n_rbp_exonic_c2"],
                  dtype={"use_mirna_main": bool, "use_rbp_main": bool}).set_index("circbase_id")
tmi = pd.read_csv(PROCESSED_DIR / "encori_mirna_targets.csv", usecols=["mirna_id", "mirna_name", "n_clip2", "head_floor50"])
tri = pd.read_csv(PROCESSED_DIR / "encori_rbp_targets.csv", keep_default_na=False, usecols=["rbp", "n_exonic_c2", "head_floor50", "hub_over_50pct"])
tmi, tri = tmi[tmi["head_floor50"]].reset_index(drop=True), tri[tri["head_floor50"]].reset_index(drop=True)
assert (len(tmi), len(tri)) == (21, 83)
circs = {"miRNA": st.index[st["use_mirna_main"]], "RBP": st.index[st["use_rbp_main"]]}
assert (len(circs["miRNA"]), len(circs["RBP"])) == (1_433, 1_405)
ml = pd.read_csv(PROCESSED_DIR / "encori_mirna_labels.csv", usecols=["circbase_id", "mirna_id", "pos_clip2", "model_input"])
rl = pd.read_csv(PROCESSED_DIR / "encori_rbp_labels.csv", keep_default_na=False, usecols=["circbase_id", "rbp", "pos_exonic_c2", "model_input"])
def label_matrix(lab, key, flag, targets, rows_):
    d = lab[lab["model_input"] & lab[flag] & lab[key].isin(targets)]
    return (pd.crosstab(d["circbase_id"], d[key]).reindex(index=rows_, columns=targets, fill_value=0) > 0).astype("int8")
HEADS = {"miRNA": dict(Y=label_matrix(ml, "mirna_id", "pos_clip2", tmi["mirna_id"].tolist(), circs["miRNA"]), deg="n_mirna_clip2", names=tmi["mirna_name"].tolist()),
         "RBP":   dict(Y=label_matrix(rl, "rbp", "pos_exonic_c2", tri["rbp"].tolist(), circs["RBP"]), deg="n_rbp_exonic_c2", names=tri["rbp"].tolist())}
assert (HEADS["miRNA"]["Y"].sum().values == tmi["n_clip2"].values).all() and (HEADS["RBP"]["Y"].sum().values == tri["n_exonic_c2"].values).all()
for h in HEADS.values():
    idx = h["Y"].index
    h["loglen"] = np.log10(mi.loc[idx, "seq_len"].astype(float).values)       # circRNA length in nt, log10
    h["deg_all"] = st.loc[idx, h["deg"]].values.astype(float)                 # positives of the circRNA over all targets of the working variant
print("lengths of the labelled circRNAs (nt), median / min / max: miRNA",
      [int(10 ** f(HEADS["miRNA"]["loglen"])) for f in (np.median, np.min, np.max)], "| RBP",
      [int(10 ** f(HEADS["RBP"]["loglen"])) for f in (np.median, np.min, np.max)])

# ---- 2. out-of-fold AUROC per target: length, leave-target-out degree, both, and a shuffled-length control ---------------------------
def oof_scores(y, X, fold):
    s = np.zeros(len(y))
    for f in range(5):
        tr_, te = fold != f, fold == f
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])
        s[te] = m.decision_function(X[te])
    return s
AUC = {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    for k, h in HEADS.items():
        Y, L = h["Y"], h["loglen"]
        fold = FM[seed].reindex(Y.index).values
        Lsh = rng.permutation(L)
        for j, t in enumerate(Y.columns):
            y = Y.values[:, j]
            dg = np.log1p(h["deg_all"] - y)                                   # positives for the other targets only
            X = {"length": L[:, None], "other-target degree": dg[:, None], "length + degree": np.c_[L, dg], "length shuffled (control)": Lsh[:, None]}
            for mname, Xm in X.items():
                AUC[(seed, k, mname, t)] = roc_auc_score(y, oof_scores(y, Xm, fold))
A = pd.Series(AUC, name="auroc"); A.index.names = ["seed", "head", "model", "target"]
summ = A.groupby(["head", "model", "seed"]).agg(targets="size", median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75),
                                                 maximum="max", above_06=lambda x: int((x > .6).sum()), above_07=lambda x: int((x > .7).sum())).round(3)
print("\nout-of-fold AUROC per target (host-gene-grouped folds), by head, model and seed:\n", summ.to_string())

# the targets that length predicts best (seed 0) and the RBP hubs
for k, h in HEADS.items():
    Y, L = h["Y"], h["loglen"]; nm = dict(zip(Y.columns, h["names"]))
    a = A.loc[(0, k, "length")].sort_values(ascending=False)
    top = []
    for t in a.index[:5]:
        y = Y[t].values
        top.append((nm[t], round(float(a[t]), 3), round(float(L[y == 1].mean() - L[y == 0].mean()), 2)))
    print(f"\n{k}: five targets with the highest length AUROC (name, AUROC, mean log10 length of positives minus negatives):", top)
hubs = tri.loc[tri["hub_over_50pct"], "rbp"].tolist()
print("\nRBP hubs, seed 0, AUROC by model:\n", pd.DataFrame({m: A.loc[(0, "RBP", m)].reindex(hubs) for m in ("length", "other-target degree", "length + degree")}).round(3).to_string())

# ---- 3. circRNAs without a positive, length quartiles and the overlap of the two heads -------------------------------------------------
rows3 = []
for k, h in HEADS.items():
    Y, L = h["Y"], h["loglen"]
    has = h["deg_all"] > 0                                                    # as the masked loss mask of Cell 41
    q, bins = pd.qcut(pd.Series(L, index=Y.index), 4, labels=["Q1", "Q2", "Q3", "Q4"], retbins=True)
    rate = Y.groupby(q, observed=True).mean().mean(axis=1).round(3).to_dict()
    rows3.append({"head": k, "labelled": len(Y), "with a positive": int(has.sum()), "without": int((~has).sum()),
                  "median length with (nt)": int(10 ** np.median(L[has])), "median length without (nt)": int(10 ** np.median(L[~has])),
                  "AUROC of length for 'has a positive'": round(roc_auc_score(has, L), 3),
                  "Spearman length vs positives per circRNA": round(float(spearmanr(L, h["deg_all"])[0]), 3),
                  "quartile bounds (nt)": (10 ** bins).round().astype(int).tolist(), "positive rate by length quartile": rate})
print("\ncircRNAs without a positive and the effect of length:\n", pd.DataFrame(rows3).set_index("head").T.to_string())
both = circs["miRNA"].intersection(circs["RBP"])
Lb = np.log10(mi.loc[both, "seq_len"].astype(float).values)
dm, dr = st.loc[both, "n_mirna_clip2"].values, st.loc[both, "n_rbp_exonic_c2"].values
print(f"\ncircRNAs labelled in both heads: {len(both):,} | Spearman: positives miRNA vs RBP {spearmanr(dm, dr)[0]:.3f} | "
      f"length vs miRNA positives {spearmanr(Lb, dm)[0]:.3f} | length vs RBP positives {spearmanr(Lb, dr)[0]:.3f}")
print("\nDry run: nothing written.")

lengths of the labelled circRNAs (nt), median / min / max: miRNA [501, 75, 4864] | RBP [509, 75, 4864]

out-of-fold AUROC per target (host-gene-grouped folds), by head, model and seed:
                                       targets  median    q25    q75  maximum  above_06  above_07
head  model                     seed                                                            
RBP   length                    0          83   0.652  0.620  0.686    0.808        71        14
                                1          83   0.654  0.622  0.683    0.817        71        14
                                2          83   0.652  0.618  0.683    0.820        70        12
      length + degree           0          83   0.851  0.814  0.884    0.937        83        83
                                1          83   0.849  0.814  0.885    0.938        83        83
                                2          83   0.850  0.815  0.882    0.940        83        83
      length shuffled (control) 0     

C:\Users\aeasa\AppData\Local\Temp\ipykernel_36368\2661607556.py:86: PerformanceWarning: indexing past lexsort depth may impact performance.
  a = A.loc[(0, k, "length")].sort_values(ascending=False)
C:\Users\aeasa\AppData\Local\Temp\ipykernel_36368\2661607556.py:93: PerformanceWarning: indexing past lexsort depth may impact performance.
  print("\nRBP hubs, seed 0, AUROC by model:\n", pd.DataFrame({m: A.loc[(0, "RBP", m)].reindex(hubs) for m in ("length", "other-target degree", "length + degree")}).round(3).to_string())


**Interpretation.** *(A) Setup and control.* The 1,433 labelled circRNAs of the miRNA head and the 1,405 of the RBP head have a median length of 501 and 509 nt (range 75–4,864 nt). The setup equals Cells 42–44 (asserted: fold sizes of seed 0, 1,251 groups, label matrices equal to the target tables of Cell 41). The control with length shuffled among the circRNAs gives a median per-target AUROC of 0.453–0.492 and no target above 0.6 (highest 0.594) in any seed and head, so the AUROCs below are not an artefact of the procedure. The control medians lie below 0.5; I attribute this to pooling predictions of five fold-specific models (inference; not tested), as in the shuffled control of Cell 44 (0.478). *(B) Length alone.* Under host-gene-grouped folds, circRNA length predicts the labels of every miRNA and of most RBPs. For the 21 miRNAs the median out-of-fold AUROC is 0.739–0.742 over the three seeds (q25 0.674–0.678, q75 0.752–0.764, highest 0.784–0.786), all 21 targets are above 0.6 and 14 above 0.7. For the 83 RBPs the median is 0.652–0.654 (q25 0.618–0.622, q75 0.683–0.686, highest 0.808–0.820), with 70–71 targets above 0.6 and 12–14 above 0.7. The positive rate rises steeply with length: by length quartile (miRNA bounds 75, 307, 502, 900 and 4,865 nt) the mean positive rate over targets is 0.016, 0.031, 0.039 and 0.127 for the miRNA head, about 8 times higher in the longest quartile than in the shortest (derived), and 0.095, 0.150, 0.188 and 0.280 for the RBP head (RBP bounds 75, 311, 509, 906 and 4,865 nt), about 3 times higher (derived). The five miRNAs with the highest length AUROC are hsa-miR-3918 (0.786), hsa-miR-29b-3p, hsa-miR-29a-3p and hsa-miR-29c-3p (0.785) and hsa-miR-4731-5p (0.764), and their positives are longer than their negatives by 0.37–0.41 in log10 length, a factor of about 2.3–2.6 (derived). The three miR-29 family members have identical values, which agrees with nearly identical positive sets (inference; not checked). For the RBPs the highest are FAM120A (0.808; difference 0.48, a factor of about 3), ELAVL3 (0.774), YTHDF3 (0.749), CSTF2T (0.748) and CHTOP (0.745). The five hubs are less dependent on length (seed 0: ALYREF 0.691, RNPS1 0.608, SCAF4 0.641, SCAF8 0.625, U2AF2 0.650). *(C) Degree.* The number of positives that a circRNA has for the other targets of the head is a far stronger predictor. The median AUROC is 0.876–0.877 for the miRNAs (highest 0.895–0.897) and 0.850–0.851 for the RBPs (highest 0.927–0.929), and every target of both heads is above 0.7 in all three seeds. Adding length to the degree changes the median by at most 0.003 (miRNA 0.873–0.875, RBP 0.849–0.851), so length adds no information beyond the degree. The five RBP hubs reach 0.754–0.854 with the degree (seed 0), against 0.608–0.691 with length. The degree uses the labels of the same circRNA, which are not available at test time, so it is a ceiling for a degree shortcut, not a baseline that a sequence model can read directly. *(D) circRNAs without a positive.* The 598 circRNAs of the miRNA head and the 91 of the RBP head without a positive under the working variant are shorter than the others: the median length is 386 nt without and 626 nt with a positive for miRNAs, and 329 and 527 nt for RBPs. Length alone separates the two groups with an AUROC of 0.680 (miRNA) and 0.682 (RBP). The Spearman correlation between length and the number of positives per circRNA is 0.429 (miRNA) and 0.404 (RBP). *(E) Overlap of the heads.* 1,404 circRNAs are labelled in both heads. Their numbers of positives are correlated across the heads (Spearman 0.630) and both correlate with length (miRNA 0.426, RBP 0.404). *(F) Reading.* The labels contain a circRNA-level factor: some circRNAs are positive for many targets and others for few, and this factor is shared by the two heads and partly explained by length (more sequence, more sites; inference). A model can reach an AUROC of about 0.65–0.74 from the length alone, and a ceiling of about 0.85–0.88 from the degree. The AUROC of a model on these heads therefore overstates what it has learned about individual targets unless it is compared with these two levels.

**Not checked.** (i) The corresponding length shortcut for the disease task (Cell 43, Not checked ii) was not tested. (ii) The scores are logistic regressions on log10 length and log1p degree; a nonlinear model or further covariates (GC content, number of exons, number of isoforms of the host gene, expression, k-mers) may reach higher values, so the figures are lower bounds for what such covariates give. (iii) Whether the length effect reflects the number of binding sites on longer sequences or the way ENCORI attributes the CLIP evidence of a site to every overlapping circRNA (Cell 29), for example a sites-per-nucleotide normalisation. (iv) Whether the shared factor of the two heads (Spearman 0.630) is explained by length alone; only the marginal correlations were computed, not a partial correlation. (v) AUROC within length strata, which the Decision below adds, was not computed. (vi) The control medians below 0.5 (0.453–0.492) were attributed to the pooling of fold-specific models but this was not tested. (vii) The label counts per target and fold of Cell 42 and the sibling signal of Cell 44 were not combined with length (for example whether the sibling AUROC is higher for long circRNAs).

**Decision.** (i) Nothing is removed from the label tables or the data because of length or degree: they are properties of the labels, not an error, and the tables of Cell 41 stand. (ii) Every result of the two ENCORI heads is reported next to two reference levels computed under the same folds and seeds: the length-only logistic regression (median per-target AUROC 0.74 for miRNAs and 0.65 for RBPs) and the other-target degree ceiling (0.88 and 0.85). A model is said to learn more than length only for a target if it exceeds the length-only level of that target, and its distance to the ceiling is reported. (iii) The primary metric stays the per-target AUROC on out-of-fold predictions pooled over the five folds (Cell 42 iv). As a secondary row, the AUROC is also computed within the four length quartiles of this cell (bounds as printed for each head), so that a result that holds only through length is visible. (iv) Both loss masks stay, as decided in Cell 41: the main run keeps the circRNAs without a positive as all-negative rows, and the masked run drops them. Since the circRNAs without a positive are mostly short (median 386 and 329 nt, against 626 and 527 nt), the masked run also removes short circRNAs, and the two runs are compared with this in mind. (v) No length filter is added for the auxiliary heads; the length policy of Cell 26 (at most 5,000 nt) stands. (vi) Because the two heads share a circRNA-level factor (Spearman 0.630), a gain of the joint model over the single-task models must be judged against the length-only and degree levels above; the design of this comparison is fixed when the multi-task model is specified. (vii) Nothing is written in this cell. Next: Cell 46, which freezes the folds (three seeds), the evaluation negatives (three seeds) and the label tables.

### 46. Working Dataset: One Common circRNA Set for the Three Heads

**Objective.** Build the working tables from the 1,582 model-input circRNAs (use_primary): one node table (length, host group, labelled and positive flags per head), the CDA pairs, and the positive edges of the two ENCORI heads under the working variants and the floor of 50. Verify every count against Cells 26, 41 and 42 before saving.

In [93]:
# Cell 46 - Working dataset: one common circRNA set for CDA, miRNA and RBP (dry run unless SAVE = True)
# Uses from earlier cells: PROCESSED_DIR (Cell 1); everything is re-read from the files of Cells 26, 39 and 41.
SAVE = True

def pick(df, cands, what):
    """Return the first candidate column present in df; fail with the real column list otherwise."""
    for c in cands:
        if c in df.columns:
            return c
    raise KeyError(f"{what}: none of {cands} found; columns are {list(df.columns)}")

# ---- 1. read the frozen files -----------------------------------------------------------------------------
uni = pd.read_csv(PROCESSED_DIR / "circrna_universe.csv", keep_default_na=False,
                  usecols=["circbase_id", "in_task", "use_primary", "in_mirna", "in_rbp",
                           "host_placeholder", "host_group", "seq_len"],
                  dtype={"in_task": bool, "use_primary": bool, "in_mirna": bool,
                         "in_rbp": bool, "host_placeholder": bool})
pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
mir   = pd.read_csv(PROCESSED_DIR / "encori_mirna_labels.csv")
rbp   = pd.read_csv(PROCESSED_DIR / "encori_rbp_labels.csv")
mt    = pd.read_csv(PROCESSED_DIR / "encori_mirna_targets.csv")
rt    = pd.read_csv(PROCESSED_DIR / "encori_rbp_targets.csv")
st    = pd.read_csv(PROCESSED_DIR / "encori_label_status.csv")

# ---- 2. the common circRNA set: task circRNAs of at most 5,000 nt --------------------------------------------
node = uni[uni["in_task"] & uni["use_primary"]].copy()
MODEL = set(node["circbase_id"])
assert len(node) == 1_582

# ---- 3. CDA pairs on the common set ------------------------------------------------------------------------
c_id = pick(pairs, ["circrna_id", "circbase_id"], "pairs id")
cda = (pairs.rename(columns={c_id: "circbase_id"})[["circbase_id", "disease"]]
       .query("circbase_id in @MODEL").drop_duplicates().reset_index(drop=True))
assert len(cda) == 2_137 and cda["disease"].nunique() == 207 and cda["circbase_id"].nunique() == 1_582

# ---- 4. working targets (floor of 50 positive model-input circRNAs, Cell 41) ----------------------------------
m_id, m_t  = pick(mir, ["circbase_id", "circrna_id"], "miRNA label id"), pick(mir, ["mirna", "mirna_id", "target"], "miRNA label target")
r_id, r_t  = pick(rbp, ["circbase_id", "circrna_id"], "RBP label id"),   pick(rbp, ["rbp", "rbp_name", "target"], "RBP label target")
mt_t, rt_t = pick(mt, [m_t, "mirna", "mirna_id", "target"], "miRNA target table"), pick(rt, [r_t, "rbp", "rbp_name", "target"], "RBP target table")
M_TARGETS = set(mt.loc[mt["head_floor50"].astype(bool), mt_t])
R_TARGETS = set(rt.loc[rt["head_floor50"].astype(bool), rt_t])
assert len(M_TARGETS) == 21 and len(R_TARGETS) == 83

# ---- 5. working positive edges: model-input circRNAs, working variant, working targets ------------------------
mir_pos_all = mir[mir[m_id].isin(MODEL) & mir["pos_clip2"].astype(bool)]            # any target
rbp_pos_all = rbp[rbp[r_id].isin(MODEL) & rbp["pos_exonic_c2"].astype(bool)]
mir_w = mir_pos_all[mir_pos_all[m_t].isin(M_TARGETS)].rename(columns={m_id: "circbase_id", m_t: "mirna"})
rbp_w = rbp_pos_all[rbp_pos_all[r_t].isin(R_TARGETS)].rename(columns={r_id: "circbase_id", r_t: "rbp"})

# ---- 6. node table: one row per circRNA, flags for the three heads ------------------------------------------
deg = cda.groupby("circbase_id")["disease"].nunique().rename("n_diseases")
node = node[["circbase_id", "seq_len", "host_group", "host_placeholder", "in_mirna", "in_rbp"]].merge(
    deg, left_on="circbase_id", right_index=True, how="left")
node = node.rename(columns={"in_mirna": "mirna_labelled", "in_rbp": "rbp_labelled"})   # present in the source table = labelled for that head
node["mirna_has_pos"]    = node["circbase_id"].isin(set(mir_pos_all[m_id]))             # a positive under the working variant, any target
node["rbp_has_pos"]      = node["circbase_id"].isin(set(rbp_pos_all[r_id]))
node["mirna_has_pos_t"]  = node["circbase_id"].isin(set(mir_w["circbase_id"]))          # a positive among the 21 working targets
node["rbp_has_pos_t"]    = node["circbase_id"].isin(set(rbp_w["circbase_id"]))

# ---- 7. cross-check against Cell 41 (loss masks) and the earlier counts --------------------------------------
s_id = pick(st, ["circbase_id", "circrna_id"], "status id")
chk = node.merge(st.rename(columns={s_id: "circbase_id"})[["circbase_id", "use_mirna_main", "use_mirna_masked",
                                                           "use_rbp_main", "use_rbp_masked"]], on="circbase_id", how="left")
assert (chk["use_mirna_main"].astype(bool) == chk["mirna_labelled"]).all()
assert (chk["use_rbp_main"].astype(bool)   == chk["rbp_labelled"]).all()
assert (chk["use_mirna_masked"].astype(bool) == chk["mirna_has_pos"]).all()
assert (chk["use_rbp_masked"].astype(bool)   == chk["rbp_has_pos"]).all()
assert node["mirna_labelled"].sum() == 1_433 and node["rbp_labelled"].sum() == 1_405
assert node["mirna_has_pos"].sum() == 835 and node["rbp_has_pos"].sum() == 1_314
assert (node["mirna_has_pos"] & node["rbp_has_pos"]).sum() == 823

# ---- 8. report ---------------------------------------------------------------------------------------------
print(f"common set: {len(node):,} circRNAs | host groups: {node['host_group'].nunique():,} | "
      f"singleton placeholder circRNAs: {int(node['host_placeholder'].sum())}")
print(f"CDA: {len(cda):,} pairs | {cda['disease'].nunique()} diseases | circRNAs with one disease: "
      f"{(node['n_diseases'] == 1).mean():.3f}")
print(pd.DataFrame({
    "labelled (in source table)": [int(node["mirna_labelled"].sum()), int(node["rbp_labelled"].sum())],
    "unlabelled (not in table)":  [int((~node["mirna_labelled"]).sum()), int((~node["rbp_labelled"]).sum())],
    "positive, any target":       [int(node["mirna_has_pos"].sum()), int(node["rbp_has_pos"].sum())],
    "positive, working targets":  [int(node["mirna_has_pos_t"].sum()), int(node["rbp_has_pos_t"].sum())],
    "working targets":            [len(M_TARGETS), len(R_TARGETS)],
    "positive edges (working)":   [len(mir_w), len(rbp_w)]}, index=["miRNA head", "RBP head"]).T)
print("positive in both heads (any target):", int((node["mirna_has_pos"] & node["rbp_has_pos"]).sum()))

# ---- 9. save (only when SAVE = True) -----------------------------------------------------------------------
if SAVE:
    node.to_csv(PROCESSED_DIR / "working_circrnas.csv", index=False)
    cda.to_csv(PROCESSED_DIR / "working_cda_pairs.csv", index=False)
    mir_w.to_csv(PROCESSED_DIR / "working_mirna_edges.csv", index=False)
    rbp_w.to_csv(PROCESSED_DIR / "working_rbp_edges.csv", index=False)
    print("saved: working_circrnas.csv, working_cda_pairs.csv, working_mirna_edges.csv, working_rbp_edges.csv")
else:
    print("dry run: nothing written (set SAVE = True after checking the counts)")

common set: 1,582 circRNAs | host groups: 1,251 | singleton placeholder circRNAs: 26
CDA: 2,137 pairs | 207 diseases | circRNAs with one disease: 0.801
                            miRNA head  RBP head
labelled (in source table)        1433      1405
unlabelled (not in table)          149       177
positive, any target               835      1314
positive, working targets          324      1308
working targets                     21        83
positive edges (working)          1605     20763
positive in both heads (any target): 823
saved: working_circrnas.csv, working_cda_pairs.csv, working_mirna_edges.csv, working_rbp_edges.csv


----------------------------------------------------------------------------------------------------------

## dataset final 

In [2]:
# Cell - Validate the chosen label rules on the 1,582 model-input circRNAs (read-only)
import pandas as pd

P = PROCESSED_DIR

st = pd.read_csv(P / "encori_label_status.csv", dtype={"use_mirna_main": bool, "use_rbp_main": bool})
ml = pd.read_csv(P / "encori_mirna_labels.csv")
rl = pd.read_csv(P / "encori_rbp_labels.csv", keep_default_na=False)
ml, rl = ml[ml["model_input"]], rl[rl["model_input"]]

RULES = {
    "miRNA": (ml, "pos_clip2",      ["pos_clip1", "pos_clip2", "pos_clip2_cap10", "pos_clip2_private"], int(st["use_mirna_main"].sum())),
    "RBP":   (rl, "pos_exonic_c2",  ["pos_exonic", "pos_c2", "pos_exonic_c2", "pos_c2_cap10"],          int(st["use_rbp_main"].sum())),
}
rows = []
for head, (d, work, flags, n_lab) in RULES.items():
    for f in flags:
        p = d[d[f]]
        per = p.groupby("circbase_id").size()
        rows.append({"head": head, "rule": f + (" (WORKING)" if f == work else ""),
                     "positive edges": len(p), "circRNAs with a positive": len(per),
                     "% of labelled": round(100 * len(per) / n_lab, 1),
                     "median positives per circRNA": int(per.median()),
                     "% edges on flagged IDs": round(100 * (p["id_check"] != "consistent").mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nlabelled circRNAs (unknown otherwise, masked):", {h: v[3] for h, v in RULES.items()})
assert len(set(ml["circbase_id"]) | set(rl["circbase_id"])) <= 1_582

 head                    rule  positive edges  circRNAs with a positive  % of labelled  median positives per circRNA  % edges on flagged IDs
miRNA               pos_clip1           20044                      1048           73.1                            11                    0.02
miRNA     pos_clip2 (WORKING)           13077                       835           58.3                             9                    0.00
miRNA         pos_clip2_cap10            9534                       666           46.5                             8                    0.00
miRNA       pos_clip2_private            2797                       217           15.1                             7                    0.00
  RBP              pos_exonic           45499                      1392           99.1                            26                    0.12
  RBP                  pos_c2           30580                      1352           96.2                            17                    0.09
  RBP pos_exo

In [3]:
# Cell - Apply the three filtering criteria and the sensitivity variants
# Reads: cda_pairs_primary.csv, circrna_universe.csv, encori_mirna_labels.csv, encori_rbp_labels.csv
# Writes (SAVE = True): filtered_circrnas.csv, filtered_mirna_edges.csv, filtered_rbp_edges.csv, filtered_rules.json
import json
import numpy as np
import pandas as pd

P = PROCESSED_DIR
OUT = P / "Filtered_file_v1"; OUT.mkdir(exist_ok=True)
SAVE, MAX_LEN, FLOOR = True, 5000, 50

# ---- Criterion 1: circRNAs ------------------------------------------------------------------------
# Human only (the CDA table was built from the human subset in Cell 8), in CircR2Disease,
# length 1..5,000 nt, keyed by circBase ID. Tier 1/2 ID resolution already happened in Cells 8-15.
pairs = pd.read_csv(P / "cda_pairs_primary.csv")
pairs = pairs[pairs["seq_len"].between(1, MAX_LEN)].copy()
MODEL = set(pairs["circrna_id"])
assert (len(MODEL), len(pairs), pairs["disease"].nunique()) == (1_582, 2_137, 207)

uni = pd.read_csv(P / "circrna_universe.csv", keep_default_na=False,
                  usecols=["circbase_id", "in_task", "use_primary", "in_mirna", "in_rbp", "id_check", "host_group"],
                  dtype={"in_task": bool, "use_primary": bool, "in_mirna": bool, "in_rbp": bool})
node = uni[uni["circbase_id"].isin(MODEL)].set_index("circbase_id")
assert len(node) == 1_582 and set(node.index) == set(uni.loc[uni["in_task"] & uni["use_primary"], "circbase_id"])

# ---- Missing in ENCORI: unknown, masked (not negative) ----------------------------------------------
node["mirna_labelled"], node["rbp_labelled"] = node["in_mirna"], node["in_rbp"]
node["mirna_masked"], node["rbp_masked"] = ~node["in_mirna"], ~node["in_rbp"]
assert (node["mirna_labelled"].sum(), node["rbp_labelled"].sum()) == (1_433, 1_405)
node["flagged_id"] = node["id_check"].str.startswith("header differs")
node["in_all_three"] = node["mirna_labelled"] & node["rbp_labelled"]     # strict-intersection robustness set
assert int(node["in_all_three"].sum()) == 1_404

# ---- Criteria 2 and 3: edge rules, recomputed from the evidence columns of the two ENCORI tables -----
ml = pd.read_csv(P / "encori_mirna_labels.csv", usecols=["circbase_id", "mirna_id", "mirna_name", "clip_max",
                 "k_min_clip2", "id_check", "pos_clip1", "pos_clip2", "pos_clip2_cap10"])
rl = pd.read_csv(P / "encori_rbp_labels.csv", keep_default_na=False, usecols=["circbase_id", "rbp", "c_max",
                 "c_max_exonic", "k_min_c2", "id_check", "pos_c2", "pos_exonic_c2", "pos_c2_cap10"])
ml, rl = ml[ml["circbase_id"].isin(MODEL)].copy(), rl[rl["circbase_id"].isin(MODEL)].copy()

# miRNA: CLIP-supported site (clipExpNum >= 2 working, >= 1 sensitivity, cap: supporting site lists <= 10 circRNAs)
ml["w_clip2"] = ml["clip_max"] >= 2
ml["s_clip1"] = ml["clip_max"] >= 1
ml["s_clip2_cap10"] = ml["k_min_clip2"] <= 10                     # NaN compares False
ml["s_clip2_noflag"] = ml["w_clip2"] & ~ml["id_check"].str.startswith("header differs")
# RBP: exonic AND clusterClipExpNum >= 2 working; c >= 2 alone, cap 10 and no flagged IDs as sensitivity
rl["w_exonic_c2"] = rl["c_max_exonic"] >= 2
rl["s_c2"] = rl["c_max"] >= 2
rl["s_c2_cap10"] = (rl["k_min_c2"] >= 1) & (rl["k_min_c2"] <= 10)
rl["s_exonic_c2_noflag"] = rl["w_exonic_c2"] & ~rl["id_check"].str.startswith("header differs")

# the recomputed rules must equal the flags saved in Cell 41
for d, pairs_ in [(ml, [("w_clip2", "pos_clip2"), ("s_clip1", "pos_clip1"), ("s_clip2_cap10", "pos_clip2_cap10")]),
                  (rl, [("w_exonic_c2", "pos_exonic_c2"), ("s_c2", "pos_c2"), ("s_c2_cap10", "pos_c2_cap10")])]:
    for a, b in pairs_:
        assert (d[a] == d[b].astype(bool)).all(), f"{a} differs from the saved flag {b}"

# ---- Summary and checks against the printed counts ---------------------------------------------------
def summary(d, col, flags, n_lab, head):
    rows = []
    for f in flags:
        p = d[d[f]]
        per = p.groupby("circbase_id").size()
        rows.append({"head": head, "rule": f, "positive edges": len(p), "circRNAs with a positive": len(per),
                     "% of labelled": round(100 * len(per) / n_lab, 1), "median per circRNA": int(per.median())})
    return pd.DataFrame(rows)
S = pd.concat([summary(ml, "mirna_id", ["w_clip2", "s_clip1", "s_clip2_cap10", "s_clip2_noflag"], 1_433, "miRNA"),
               summary(rl, "rbp", ["w_exonic_c2", "s_c2", "s_c2_cap10", "s_exonic_c2_noflag"], 1_405, "RBP")])
print(S.to_string(index=False))
exp = {("miRNA", "w_clip2"): (13_077, 835), ("miRNA", "s_clip1"): (20_044, 1_048), ("miRNA", "s_clip2_cap10"): (9_534, 666),
       ("RBP", "w_exonic_c2"): (22_578, 1_314), ("RBP", "s_c2"): (30_580, 1_352), ("RBP", "s_c2_cap10"): (22_606, 1_120)}
for (h, r), (e, c) in exp.items():
    got = S[(S["head"] == h) & (S["rule"] == r)].iloc[0]
    assert (got["positive edges"], got["circRNAs with a positive"]) == (e, c), f"{h} {r} differs from the earlier output"

# ---- Targets: working rule + floor of 50 positive circRNAs --------------------------------------------
nm = ml[ml["w_clip2"]].groupby("mirna_id")["circbase_id"].nunique()
nr = rl[rl["w_exonic_c2"]].groupby("rbp")["circbase_id"].nunique()
M_T, R_T = set(nm[nm >= FLOOR].index), set(nr[nr >= FLOOR].index)
ml["in_head_targets"], rl["in_head_targets"] = ml["mirna_id"].isin(M_T), rl["rbp"].isin(R_T)
hubs = set(nr[nr / 1_405 > 0.5].index)                              # positive for > 50% of labelled circRNAs
rl["hub_rbp"] = rl["rbp"].isin(hubs)
assert (len(M_T), len(R_T)) == (21, 83)
assert (int((ml["w_clip2"] & ml["in_head_targets"]).sum()), int((rl["w_exonic_c2"] & rl["in_head_targets"]).sum())) == (1_605, 20_763)
print(f"\ntargets: {len(M_T)} miRNAs, {len(R_T)} RBPs | hub RBPs: {sorted(hubs)}")

# ---- circRNA status table and save --------------------------------------------------------------------------
node["n_diseases"] = pairs.groupby("circrna_id").size().reindex(node.index)
node["seq_len"] = pairs.drop_duplicates("circrna_id").set_index("circrna_id")["seq_len"].reindex(node.index)
node["mirna_has_pos"] = node.index.isin(set(ml.loc[ml["w_clip2"], "circbase_id"]))
node["rbp_has_pos"] = node.index.isin(set(rl.loc[rl["w_exonic_c2"], "circbase_id"]))
assert (int(node["mirna_has_pos"].sum()), int(node["rbp_has_pos"].sum())) == (835, 1_314)

RULES = {"circRNAs": f"human, in CircR2Disease, 1..{MAX_LEN} nt, circBase ID", "miRNA_positive": "clipExpNum >= 2",
         "RBP_positive": "exonic and clusterClipExpNum >= 2", "missing_in_ENCORI": "unknown, masked",
         "target_floor": FLOOR, "sensitivity": ["miRNA clip>=1", "RBP c>=2", "cap 10", "without flagged IDs"],
         "counts": {"circRNAs": 1_582, "pairs": 2_137, "diseases": 207, "miRNA_targets": 21, "RBP_targets": 83}}
if SAVE:
    node.reset_index().to_csv(OUT / "filtered_circrnas.csv", index=False)
    ml.to_csv(OUT / "filtered_mirna_edges.csv", index=False)
    rl.to_csv(OUT / "filtered_rbp_edges.csv", index=False)
    (OUT / "filtered_rules.json").write_text(json.dumps(RULES, indent=2))
    print("saved: filtered_circrnas.csv, filtered_mirna_edges.csv, filtered_rbp_edges.csv, filtered_rules.json")

 head               rule  positive edges  circRNAs with a positive  % of labelled  median per circRNA
miRNA            w_clip2           13077                       835           58.3                   9
miRNA            s_clip1           20044                      1048           73.1                  11
miRNA      s_clip2_cap10            9534                       666           46.5                   8
miRNA     s_clip2_noflag           13077                       835           58.3                   9
  RBP        w_exonic_c2           22578                      1314           93.5                  12
  RBP               s_c2           30580                      1352           96.2                  17
  RBP         s_c2_cap10           22606                      1120           79.7                  15
  RBP s_exonic_c2_noflag           22563                      1311           93.3                  12

targets: 21 miRNAs, 83 RBPs | hub RBPs: ['ALYREF', 'RNPS1', 'SCAF4', 'SCAF8', 'U2

### setup and the six data tables

In [24]:
# Cell 1 - setup and data tables -> dataset_v1/
import json, datetime, yaml, numpy as np, pandas as pd

P = PROCESSED_DIR
OUT = P / "dataset_v1"; OUT.mkdir(exist_ok=True)
SAVE, MAX_LEN, CROP, SEEDS, K, FLOOR = True, 5000, 1022, (0, 1, 2), 5, 50

B = lambda cols: {c: bool for c in cols}
node = pd.read_csv(P / "filtered_circrnas.csv", keep_default_na=False,
                   dtype=B(["mirna_labelled", "rbp_labelled", "flagged_id", "in_all_three", "mirna_has_pos", "rbp_has_pos"]))
ml = pd.read_csv(P / "filtered_mirna_edges.csv",
                 dtype=B(["w_clip2", "s_clip1", "s_clip2_cap10", "s_clip2_noflag", "in_head_targets"]))
rl = pd.read_csv(P / "filtered_rbp_edges.csv", keep_default_na=False,
                 dtype=B(["w_exonic_c2", "s_c2", "s_c2_cap10", "s_exonic_c2_noflag", "in_head_targets", "hub_rbp"]))
cda = (pd.read_csv(P / "cda_pairs_primary.csv").rename(columns={"circrna_id": "circbase_id"})
       .query("seq_len >= 1 and seq_len <= @MAX_LEN")[["circbase_id", "disease"]].drop_duplicates().reset_index(drop=True))

MODEL = set(node["circbase_id"])
assert len(MODEL) == 1_582 and set(cda["circbase_id"]) == MODEL
assert (len(cda), cda["disease"].nunique()) == (2_137, 207)

# file 1: circRNAs (loss masks live here)
circ = node[["circbase_id", "host_group", "seq_len", "n_diseases", "mirna_labelled", "rbp_labelled",
             "mirna_has_pos", "rbp_has_pos", "flagged_id", "in_all_three"]].copy()

# flag the popular disease
HUB_MIN = 10                                             
circ["is_hub"] = circ["n_diseases"] >= HUB_MIN
assert int(circ["is_hub"].sum()) == 6                    # 27, 22, 15, 12, 11, 10 diseases
print("hub circRNAs:", circ.loc[circ["is_hub"], ["circbase_id", "n_diseases"]].values.tolist())

# Host groups: merge solo circRNAs that overlap a named host gene on the same strand (overlap check)
MERGE = {"solo:hsa_circ_0000326": "MALAT1",     # overlaps hsa_circ_0096157 (MALAT1)
         "solo:hsa_circ_0001821": "PVT1"}       # overlaps hsa_circ_0009143 (PVT1); hub, 11 diseases
circ["host_group"] = circ["host_group"].replace(MERGE)
print("host groups:", circ["host_group"].nunique(),
      "| solo:", int(circ["host_group"].str.startswith("solo:").sum()))

assert (int(circ["mirna_labelled"].sum()), int(circ["rbp_labelled"].sum()), int(circ["in_all_three"].sum())) == (1_433, 1_405, 1_404)

# file 3: diseases   |   file 4: positive CDA pairs
dis = pd.DataFrame({"disease_id": range(207), "disease": sorted(cda["disease"].unique())})

# files 5 and 6: edges with one boolean column per rule (names shortened)
mir = ml.rename(columns={"w_clip2": "pos_working", "s_clip1": "pos_clip1", "s_clip2_cap10": "pos_cap10",
                         "s_clip2_noflag": "pos_noflag"})[
    ["circbase_id", "mirna_id", "mirna_name", "clip_max", "pos_working", "pos_clip1", "pos_cap10", "pos_noflag", "in_head_targets"]]
rbp = rl.rename(columns={"w_exonic_c2": "pos_working", "s_c2": "pos_c2", "s_c2_cap10": "pos_cap10",
                         "s_exonic_c2_noflag": "pos_noflag"})[
    ["circbase_id", "rbp", "c_max", "c_max_exonic", "pos_working", "pos_c2", "pos_cap10", "pos_noflag", "in_head_targets", "hub_rbp"]]
assert (int(mir["pos_working"].sum()), int(rbp["pos_working"].sum())) == (13_077, 22_578)

# files 7 and 8: target lists (positives counted on the working rule; floor of 50)
n_m = mir[mir["pos_working"]].groupby("mirna_id")["circbase_id"].nunique()
names = mir.drop_duplicates("mirna_id").set_index("mirna_id")["mirna_name"]
mt = pd.DataFrame({"mirna_id": names.index, "mirna_name": names.values})
mt["n_pos"] = mt["mirna_id"].map(n_m).fillna(0).astype(int); mt["in_head"] = mt["n_pos"] >= FLOOR
n_r = rbp[rbp["pos_working"]].groupby("rbp")["circbase_id"].nunique()
rt = pd.DataFrame({"rbp": sorted(rbp["rbp"].unique())})
rt["n_pos"] = rt["rbp"].map(n_r).fillna(0).astype(int); rt["in_head"] = rt["n_pos"] >= FLOOR
rt["hub"] = rt["n_pos"] / int(circ["rbp_labelled"].sum()) > 0.5
assert (int(mt["in_head"].sum()), int(rt["in_head"].sum()), int(rt["hub"].sum())) == (21, 83, 5)

FILES = {"circrnas.csv": circ, "diseases.csv": dis, "cda_pairs.csv": cda, "mirna_edges.csv": mir,
         "rbp_edges.csv": rbp, "mirna_targets.csv": mt, "rbp_targets.csv": rt}
if SAVE:
    for fn, d in FILES.items():
        d.to_csv(OUT / fn, index=False)
print({fn: d.shape for fn, d in FILES.items()})

hub circRNAs: [['hsa_circ_0000284', 27], ['hsa_circ_0000615', 15], ['hsa_circ_0001649', 10], ['hsa_circ_0001821', 11], ['hsa_circ_0001946', 22], ['hsa_circ_0067934', 12]]
host groups: 1249 | solo: 24
{'circrnas.csv': (1582, 11), 'diseases.csv': (207, 2), 'cda_pairs.csv': (2137, 2), 'mirna_edges.csv': (20044, 10), 'rbp_edges.csv': (63747, 11), 'mirna_targets.csv': (638, 4), 'rbp_targets.csv': (277, 4)}


In [42]:
dis[dis["disease"] == "breast carcinoma"]

,disease_id,disease


### Convert DNA to RNA sequences

In [25]:
# Cell 2 - sequences: DNA -> RNA, full length (no crop); long circRNAs are handled by windows in 06_embeddings
fa_path = next(CDA_DIR.glob("*.fa"))
W, S = 1022, 511                                   # RNA-FM window (1,024 tokens - start/end) and stride

def read_fasta_subset(path, wanted):
    out, cur, buf = {}, None, []
    with open(path) as f:
        for line in f:
            if line[0] == ">":
                if cur is not None: out[cur] = "".join(buf).upper()
                cid = line[1:].split("|", 1)[0].strip()
                cur, buf = (cid if cid in wanted else None), []
            elif cur is not None:
                buf.append(line.strip())
    if cur is not None: out[cur] = "".join(buf).upper()
    return out

def n_windows_overlap(L):
    """Overlapping 1,022-nt windows, stride 511; the last window ends at L """
    if L <= W: return 1
    st = list(range(0, L - W + 1, S))
    return len(st) + (st[-1] != L - W)

def n_chunks_nonoverlap(L):
    """Non-overlapping 1,022-nt chunks; the last chunk may be shorter."""
    return -(-L // W)

raw = read_fasta_subset(fa_path, MODEL)
assert set(raw) == MODEL

seq = pd.DataFrame({"circbase_id": circ["circbase_id"]})
seq["sequence"] = seq["circbase_id"].map(raw).str.replace("T", "U", regex=False)
seq["seq_len"] = seq["sequence"].str.len()
seq["n_windows_overlap"] = seq["seq_len"].map(n_windows_overlap)
seq["n_chunks_nonoverlap"] = seq["seq_len"].map(n_chunks_nonoverlap)
seq = seq[["circbase_id", "seq_len", "n_windows_overlap", "n_chunks_nonoverlap", "sequence"]]

bad = seq["sequence"].str.contains(r"[^ACGU]")
print("sequences with a non-ACGU character:", int(bad.sum()),
      "| need windows (>1,022 nt):", int((seq["n_windows_overlap"] > 1).sum()),
      "| max overlapping windows:", int(seq["n_windows_overlap"].max()),
      "| max non-overlapping chunks:", int(seq["n_chunks_nonoverlap"].max()))
assert (seq["seq_len"].to_numpy() == circ["seq_len"].to_numpy()).all()     # same length as the circBase index
assert seq["seq_len"].between(1, 5000).all()
assert int((seq["n_windows_overlap"] > 1).sum()) == int((seq["n_chunks_nonoverlap"] > 1).sum()) == 350         # the 1,023-5,000 nt band
if SAVE:
    seq.to_csv(OUT / "sequences.csv", index=False)

sequences with a non-ACGU character: 0 | need windows (>1,022 nt): 350 | max overlapping windows: 9 | max non-overlapping chunks: 5


### Folds

In [35]:
# Cell 3 - host-gene-grouped folds, greedy balancing by pairs (same algorithm as Cell 42)
hg = circ.set_index("circbase_id")["host_group"]
pp = cda.assign(group=cda["circbase_id"].map(hg))
grp = pp.groupby("group").agg(n_pairs=("circbase_id", "size"))
n_groups, n_solo = len(grp), int(grp.index.str.startswith("solo:").sum())
print(f"host groups: {n_groups} | solo groups: {n_solo}")
assert (n_groups, n_solo) == (1_249, 24), "expected 1,249 groups / 24 solo after the MALAT1 and PVT1 merges (Cell 1)"

def assign(seed, k=K):
    rng = np.random.default_rng(seed)
    order = grp.sample(frac=1, random_state=seed).sort_values("n_pairs", ascending=False, kind="stable")
    load, fold = np.zeros(k), {}
    for g, n in order["n_pairs"].items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[g] = f; load[f] += n
    return fold

folds = circ[["circbase_id"]].copy()
for s in SEEDS:
    folds[f"fold_seed{s}"] = circ["host_group"].map(assign(s)).to_numpy()
    assert folds[f"fold_seed{s}"].between(0, K - 1).all()
    assert (circ.assign(f=folds[f"fold_seed{s}"]).groupby("host_group")["f"].nunique() == 1).all()   # no host gene in two folds

# Fold sizes (new values: record them in the manifest; the old 428/427... came from folds with the overlap leak)
FOLD_SIZES = {}
for s in SEEDS:
    pf = pp["circbase_id"].map(folds.set_index("circbase_id")[f"fold_seed{s}"])
    FOLD_SIZES[f"seed{s}"] = {"pairs": pf.value_counts().sort_index().tolist(),
                              "circRNAs": folds[f"fold_seed{s}"].value_counts().sort_index().tolist()}
    print(f"seed {s}: pairs per fold {FOLD_SIZES[f'seed{s}']['pairs']} | circRNAs per fold {FOLD_SIZES[f'seed{s}']['circRNAs']}")
    assert sum(FOLD_SIZES[f"seed{s}"]["pairs"]) == len(cda) and sum(FOLD_SIZES[f"seed{s}"]["circRNAs"]) == len(circ)

# Hub circRNAs by fold (printed once, after all seeds exist)
hubs = circ.loc[circ["is_hub"], ["circbase_id", "n_diseases"]].set_index("circbase_id")
print("\nhub circRNAs (>=10 diseases) by fold:\n" + hubs.join(folds.set_index("circbase_id")).to_string())

if SAVE:
    folds.to_csv(OUT / "folds.csv", index=False)

host groups: 1249 | solo groups: 24
seed 0: pairs per fold [427, 428, 427, 428, 427] | circRNAs per fold [315, 315, 312, 323, 317]
seed 1: pairs per fold [428, 427, 428, 427, 427] | circRNAs per fold [320, 322, 311, 306, 323]
seed 2: pairs per fold [428, 428, 427, 427, 427] | circRNAs per fold [324, 307, 317, 323, 311]

hub circRNAs (>=10 diseases) by fold:
                  n_diseases  fold_seed0  fold_seed1  fold_seed2
circbase_id                                                     
hsa_circ_0000284          27           4           2           4
hsa_circ_0000615          15           0           1           2
hsa_circ_0001649          10           1           4           0
hsa_circ_0001821          11           0           0           2
hsa_circ_0001946          22           2           3           1
hsa_circ_0067934          12           3           1           3


### Evaluation negatives (downregulated )

In [38]:
# Cell 4 - frozen evaluation negatives for the CDA task, 2 schemes x 3 seeds (same sampler as Cell 43)
from sklearn.metrics import roc_auc_score

dis_arr = np.array(sorted(cda["disease"].unique()))
pos_all = set(zip(cda["circbase_id"], cda["disease"]))
pos_by_c = cda.groupby("circbase_id")["disease"].apply(set).to_dict()
dcount = cda["disease"].value_counts().reindex(dis_arr).astype(float)

def sample_negatives(test_pos, scheme, seed):
    rng = np.random.default_rng(seed); out = []
    for c, g in test_pos.groupby("circbase_id", sort=True):
        cand = np.array([d for d in dis_arr if d not in pos_by_c[c]])
        w = dcount.reindex(cand).values
        p = None if scheme == "random" else w / w.sum()
        out += [(c, d) for d in rng.choice(cand, size=len(g), replace=False, p=p)]
    return pd.DataFrame(out, columns=["circbase_id", "disease"])

fmap = folds.set_index("circbase_id")
neg_parts, auc_rows = [], []
for s in SEEDS:
    fold = cda["circbase_id"].map(fmap[f"fold_seed{s}"])
    for f in range(K):
        test = cda[fold == f]
        train_cnt = cda.loc[fold != f, "disease"].value_counts()
        for scheme in ("degree_matched", "random"):
            neg = sample_negatives(test, scheme, seed=1000 * s + f)
            neg.insert(0, "scheme", scheme); neg.insert(0, "fold", f); neg.insert(0, "seed", s)
            neg_parts.append(neg)
            y = np.r_[np.ones(len(test)), np.zeros(len(neg))]
            sc = np.r_[test["disease"].map(train_cnt).fillna(0).values, neg["disease"].map(train_cnt).fillna(0).values]
            auc_rows.append({"seed": s, "fold": f, "scheme": scheme, "pop_auroc": round(roc_auc_score(y, sc), 3)})
negs = pd.concat(neg_parts, ignore_index=True)[["seed", "fold", "scheme", "circbase_id", "disease"]]

# integrity: sizes, no collision with a positive, no duplicate, equal to Cell 43 for seed 0
assert (negs.groupby(["seed", "scheme"]).size() == len(cda)).all()
assert not any((c, d) in pos_all for c, d in zip(negs["circbase_id"], negs["disease"]))
assert not negs.duplicated().any()
A = pd.DataFrame(auc_rows)
# Popularity baseline: new values (folds changed after the host-group merges); record, don't force old ones
A = pd.DataFrame(auc_rows)
POP_AUROC = (A.groupby(["seed", "scheme"])["pop_auroc"].mean().round(3).unstack().to_dict(orient="index"))
print("popularity AUROC per fold, seed 0:\n" +
      A[A["seed"] == 0].pivot(index="fold", columns="scheme", values="pop_auroc").to_string())
print("\nmean per seed:", POP_AUROC)

# Sanity checks that must hold for any valid folds
dm = A.loc[A["scheme"] == "degree_matched", "pop_auroc"].mean()
rd = A.loc[A["scheme"] == "random", "pop_auroc"].mean()
assert 0.45 < dm < 0.56, f"degree-matched negatives should neutralise popularity (got {dm:.3f})"
assert rd > dm + 0.15, f"random negatives should be easy for popularity (got {rd:.3f} vs {dm:.3f})"
fmap_s = {s: fmap[f"fold_seed{s}"] for s in SEEDS}
assert all((negs.loc[negs["seed"] == s, "circbase_id"].map(fmap_s[s]) == negs.loc[negs["seed"] == s, "fold"]).all()
           for s in SEEDS), "a negative sits in the wrong fold"

if SAVE:
    negs.to_csv(OUT / "negatives_cda.csv", index=False)
print(negs.shape, f"| popularity AUROC: degree_matched {dm:.3f}, random {rd:.3f}")

popularity AUROC per fold, seed 0:
scheme  degree_matched  random
fold                          
0                0.541   0.842
1                0.500   0.809
2                0.478   0.813
3                0.526   0.834
4                0.504   0.823

mean per seed: {0: {'degree_matched': 0.51, 'random': 0.824}, 1: {'degree_matched': 0.508, 'random': 0.819}, 2: {'degree_matched': 0.512, 'random': 0.822}}
(12822, 5) | popularity AUROC: degree_matched 0.510, random 0.822


### Rules, manifest and final check

In [39]:
# Cell 5 - rules.yaml, manifest.json and a final check of every file
RULES = {"circRNAs": f"human, in CircR2Disease, 1..{MAX_LEN} nt, keyed by circBase ID",
         "miRNA_positive": "clipExpNum >= 2", "RBP_positive": "exonic and clusterClipExpNum >= 2",
         "multi_window_circRNAs": 350, "max_input_nt": CROP,"host_group_merges": MERGE,"fold_sizes": FOLD_SIZES,
         "hub_definition": f"circRNA with >= {HUB_MIN} diseases (robustness check only)",  "popularity_auroc": POP_AUROC,
         "folds": {"k": K, "seeds": list(SEEDS), "grouping": "host gene; placeholder genes are singletons"},
         "negatives": {"primary": "degree_matched", "secondary": "random", "per_positive": 1},
         "sensitivity": ["miRNA clip>=1", "RBP c>=2", "cap 10", "without flagged IDs", "in_all_three set"]}
EXPECT = {"circrnas.csv": (1_582, ["circbase_id", "host_group", "seq_len", "n_diseases", "mirna_labelled", "rbp_labelled"]),
          "sequences.csv": (1_582, ["circbase_id", "seq_len", "n_windows_overlap", "n_chunks_nonoverlap", "sequence"]),
          "diseases.csv": (207, ["disease_id", "disease"]), "cda_pairs.csv": (2_137, ["circbase_id", "disease"]),
          "folds.csv": (1_582, ["circbase_id", "fold_seed0", "fold_seed1", "fold_seed2"]),
          "negatives_cda.csv": (2_137 * 2 * len(SEEDS), ["seed", "fold", "scheme", "circbase_id", "disease"])}
if SAVE:
    for fn, (n, cols) in EXPECT.items():
        d = pd.read_csv(OUT / fn, keep_default_na=False)
        assert len(d) == n and set(cols) <= set(d.columns), fn
    sizes = {fn.name: int(len(pd.read_csv(fn, keep_default_na=False))) for fn in sorted(OUT.glob("*.csv"))}
    manifest = {"date": str(datetime.date.today()), "files": sizes,
                "counts": {"circRNAs": 1_582, "pairs": 2_137, "diseases": 207, "miRNA_targets": 21, "RBP_targets": 83,
                           "miRNA_positive_edges_working": int(mir["pos_working"].sum()),
                           "RBP_positive_edges_working": int(rbp["pos_working"].sum()),
                           "mirna_labelled": 1_433, "rbp_labelled": 1_405, "in_all_three": 1_404}, "rules": RULES}
    (OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
    (OUT / "rules.yaml").write_text(yaml.safe_dump(RULES, sort_keys=False))
    print(json.dumps(manifest["files"], indent=2))

{
  "cda_pairs.csv": 2137,
  "circrnas.csv": 1582,
  "diseases.csv": 207,
  "folds.csv": 1582,
  "mirna_edges.csv": 20044,
  "mirna_targets.csv": 638,
  "negatives_cda.csv": 12822,
  "rbp_edges.csv": 63747,
  "rbp_targets.csv": 277,
  "sequences.csv": 1582
}


### genomic overlap across host groups test check

In [29]:
# Do any two model circRNAs from DIFFERENT host groups overlap on the same strand? (leakage across folds)
idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
need = ["circbase_id", "chrom", "start", "end", "strand"]
assert set(need) <= set(idx.columns), f"index lacks {set(need) - set(idx.columns)}: rebuild it from the FASTA headers"
c = circ[["circbase_id", "host_group"]].merge(idx[need], on="circbase_id")
assert len(c) == len(circ)

hits = []
for _, g in c.sort_values("start").groupby(["chrom", "strand"]):           # opposite strands share no sequence
    s, e = g["start"].to_numpy(), g["end"].to_numpy()
    h, ids = g["host_group"].to_numpy(), g["circbase_id"].to_numpy()
    for i in range(len(g)):
        j = i + 1
        while j < len(g) and s[j] <= e[i]:                                  # sorted by start: overlaps are contiguous
            if h[j] != h[i]:
                hits.append((ids[i], ids[j], h[i], h[j]))
            j += 1
ov = pd.DataFrame(hits, columns=["circ_a", "circ_b", "group_a", "group_b"])
print(f"overlapping pairs across host groups: {len(ov)} | groups involved: {len(set(ov['group_a']) | set(ov['group_b']))}")
print(ov.head(15).to_string(index=False))

overlapping pairs across host groups: 0 | groups involved: 0
Empty DataFrame
Columns: [circ_a, circ_b, group_a, group_b]
Index: []


## trial ENCORI dataset using api

### 27. Probe of the ENCORI API

**Objective.** Probe the ENCORI API with three small requests (RBPTarget and miRNATarget on hg38, RBPTarget on hg19) to establish which endpoint and assembly return usable tables.

In [28]:
# Cell 27 - Probe the ENCORI API with three small requests (RBPTarget and miRNATarget on hg38, RBPTarget on hg19)
import io, urllib.request, urllib.error

# Base address of the public ENCORI REST API (no authentication is required)
ENCORI = "https://rnasysu.com/encori/api"

# Interim helper: print the beginning of a raw response without assuming its structure
def probe(endpoint, params, n=6, timeout=180):
    """Send one request and print the beginning of the raw response, without assuming its structure."""
    qs = "&".join(f"{k}={v}" for k, v in params.items())
    url = f"{ENCORI}/{endpoint}/?{qs}"
    print("URL:", url)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            status, txt = r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.URLError as e:
        print("request failed:", e)
        return None
    # Separate metadata lines (starting with '#') from the table itself (header + records)
    lines = [l for l in txt.splitlines() if l.strip()]
    meta = [l for l in lines if l.startswith("#")]          # comment lines (metadata)
    data = [l for l in lines if not l.startswith("#")]      # header + records
    print("HTTP", status, "| comment lines:", len(meta), "| data lines incl. header:", len(data))
    for l in meta[:3]:
        print("  #", l[:160])
    for l in data[:n]:
        print("  ", l[:260])
    print()
    return data

# Interim helper: parse a tab-separated response; None indicates a server error page
def to_df(data):
    """Convert the tab-separated response to a DataFrame; None if the response is not a table."""
    if not data or len(data) < 2 or "\t" not in data[0]:
        return None
    return pd.read_csv(io.StringIO("\n".join(data)), sep="\t")

# One request per (endpoint, assembly) pair; hg19 is included to test whether the server error is assembly-specific
tests = {
    ("RBPTarget", "hg38"): ("RBPTarget", {"assembly": "hg38", "geneType": "circRNA", "RBP": "QKI",
                            "clipExpNum": 1, "pancancerNum": 0, "target": "all", "cellType": "all"}),
    ("miRNATarget", "hg38"): ("miRNATarget", {"assembly": "hg38", "geneType": "circRNA",
                            "miRNA": "hsa-miR-145-5p", "clipExpNum": 1, "degraExpNum": 0,
                            "pancancerNum": 0, "programNum": 1, "program": "None",
                            "target": "all", "cellType": "all"}),
    ("RBPTarget", "hg19"): ("RBPTarget", {"assembly": "hg19", "geneType": "circRNA", "RBP": "QKI",
                            "clipExpNum": 1, "pancancerNum": 0, "target": "all", "cellType": "all"}),
}

# Execute every test and store the parsed table under its own key, so that outputs cannot be mislabelled
results = {}
for key, (endpoint, params) in tests.items():
    print("#" * 20, key)
    results[key] = to_df(probe(endpoint, params, n=2))

# Overview: row and column counts per request, or FAILED when no table was returned
print("\nSUMMARY")
for key, d in results.items():
    print(f"{key[0]:12s} {key[1]}: ", "FAILED" if d is None else f"{d.shape[0]} rows x {d.shape[1]} cols")

# Keep the hg38 tables for the validation cells that follow
d_rbp = results[("RBPTarget", "hg38")]      # kept for the validation cells that follow (Cells 29-32)
d_mir = results[("miRNATarget", "hg38")]
if d_mir is not None:
    print("\nmiRNATarget columns:", d_mir.columns.tolist())
    print(d_mir.head(3).T.to_string())

#################### ('RBPTarget', 'hg38')
URL: https://rnasysu.com/encori/api/RBPTarget/?assembly=hg38&geneType=circRNA&RBP=QKI&clipExpNum=1&pancancerNum=0&target=all&cellType=all
HTTP 200 | comment lines: 2 | data lines incl. header: 5572
  # #please cite:
  # #Keren Zhou, Junhong Huang, Shurong Liu, Wujian Zheng, Shun Liu, Yinghao Xing, Yinan Cao, Yibo Chen, Xinping Hu, Li Cai, Jie Zhou, Penghui Lin, Jiajia Xuan, Fan
   RBP	geneID	geneName	txIDcat	geneType	clusterNum	totalClipExpNum	totalClipSiteNum	clusterID	chromosome	narrowStart	narrowEnd	broadStart	broadEnd	strand	clipExpNum	cellline/tissue
   QKI	NM_000028	AGL	hsa_circ_0013297	circRNA	1	1	1	QKI:CH435	chr1	99923788	99923788	99923788	99923788	+	1	HEK293

#################### ('miRNATarget', 'hg38')
URL: https://rnasysu.com/encori/api/miRNATarget/?assembly=hg38&geneType=circRNA&miRNA=hsa-miR-145-5p&clipExpNum=1&degraExpNum=0&pancancerNum=0&programNum=1&program=None&target=all&cellType=all
HTTP 200 | comment lines: 2 | data lines i

**Interpretation.** On genome assembly hg38 both endpoints return tables: `RBPTarget` for QKI yields 5,571 rows (CLIP *clusters*, whose `txIDcat` field holds comma-joined circBase IDs) and `miRNATarget` for hsa-miR-145-5p yields 2,461 rows (individual sites, with `circID`, seed class `merClass`, `clipExpNum`, `TDMDScore` and `phyloP`). The hg19 request returned HTTP 200 but no table: the body is a PHP warning from the server (`mysqli_error() ... baseFunc.php`). Because the request differs from the working hg38 one only in the assembly, this is consistent with a server-side problem (inference). The first miRNA rows show `clipExpNum = 0` although a minimum of 1 was requested, which suggests that the CLIP filter is not enforced.

**Caveat.** ENCORI maps CLIP sites onto annotated circRNA spans; a site inside the span does not prove that the circRNA itself is bound unless reads cross the back-splice junction. ENCORI-derived labels should therefore be described as *CLIP-supported site overlaps*, and an evidence threshold must be applied.

**Decision.** hg38 is used for all further requests, and tables are joined through circBase IDs rather than coordinates (our FASTA is hg19, but the IDs are assembly-independent; this is verified in Cell 28). hg19 may be retried later. Each result is stored under its own key in the `results` dictionary, so outputs cannot be mislabelled.

**Note on practice.** `probe` and `to_df` are interim notebook functions; the stable client is intended to live in `src/circmtl/encori.py` (not checked in this notebook).

### 28. Cache of the QKI Table, ID Consistency, Coverage and Sibling Structure

**Objective.** Cache the QKI table on disk, expand its comma-joined identifier lists, check identifier consistency with the FASTA index, measure the coverage of the model-input circRNAs, describe the host-gene sibling structure and test the miRNATarget endpoint once more.

In [29]:
# Cell 28 - Cache the QKI result, check ID consistency, measure coverage and siblings
import io, urllib.request

ENCORI = "https://rnasysu.com/encori/api"
ENC_DIR = RAW_DIR / "encori"
ENC_DIR.mkdir(exist_ok=True)

# Download once and cache on disk; an HTML error page from the server is rejected instead of being cached
def fetch_encori(endpoint, params, name, timeout=300):
    path = ENC_DIR / f"{name}.tsv"
    if path.exists():
        return path.read_text(encoding="utf-8")
    qs = "&".join(f"{k}={v}" for k, v in params.items())
    url = f"{ENCORI}/{endpoint}/?{qs}"
    with urllib.request.urlopen(url, timeout=timeout) as r:
        txt = r.read().decode("utf-8", errors="replace")
    if txt.lstrip().startswith("<br") or "Fail to qurey" in txt:      # server-side error page
        raise RuntimeError("ENCORI returned an error page:\n" + txt[:300])
    path.write_text(txt, encoding="utf-8")
    return txt

# Drop the '#' metadata lines and read the remaining tab-separated table
def parse_encori(txt):
    lines = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    return pd.read_csv(io.StringIO("\n".join(lines)), sep="\t")

# ---- our side, from disk ------------------------------------------------------------
circ   = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
fa_gene_s  = fa_idx.set_index("circbase_id")["gene"]
fa_chrom_s = fa_idx.set_index("circbase_id")["chrom"]
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)

model_ids = set(circ.loc[circ["use_primary"], "circrna_id"])
cold_ids  = set(circ.loc[circ["use_primary"] & (circ["n_diseases"] == 1), "circrna_id"])

# ---- QKI, hg38 -------------------------------------------------------------------------
txt = fetch_encori("RBPTarget",
                   {"assembly": "hg38", "geneType": "circRNA", "RBP": "QKI",
                    "clipExpNum": 1, "pancancerNum": 0, "target": "all", "cellType": "all"},
                   "RBPTarget_hg38_circRNA_QKI_clip1")
q = parse_encori(txt)
# One CLIP cluster lists several circBase IDs (comma-joined): explode to one row per (cluster, circRNA)
q["circ_list"] = q["txIDcat"].astype(str).str.split(",")
q["n_circ"] = q["circ_list"].map(len)
pairs = q.explode("circ_list").rename(columns={"circ_list": "circrna_id"})
pairs["circrna_id"] = pairs["circrna_id"].str.strip()

print("A. clusters:", len(q), "| (cluster, circRNA) pairs:", len(pairs),
      "| distinct circRNAs:", pairs["circrna_id"].nunique())
print("circRNAs per cluster:\n", q["n_circ"].describe(percentiles=[.5, .9, .99]).round(1).to_string())
print("\nclipExpNum:\n", q["clipExpNum"].value_counts().sort_index().head(8).to_string())
print("cell lines / tissues:", q["cellline/tissue"].nunique(), "| top:",
      q["cellline/tissue"].value_counts().head(4).to_dict())

# ---- ID consistency between ENCORI hg38 and our circBase-based FASTA -----------------------
chk = pairs.drop_duplicates("circrna_id").copy()
chk = chk[chk["circrna_id"].isin(fa_gene_s.index)]
chk["fa_gene"]  = chk["circrna_id"].map(fa_gene_s)
chk["fa_chrom"] = chk["circrna_id"].map(fa_chrom_s)
print("\nB. ENCORI circRNAs found in our FASTA index:", len(chk), "of", pairs["circrna_id"].nunique())
print("chromosome agrees:", round((chk["chromosome"] == chk["fa_chrom"]).mean(), 3))
real = chk[~chk["fa_gene"].map(lambda g: bool(PLACE.match(str(g))))]
print("host gene agrees (FASTA gene is a real symbol; n=%d):" % len(real),
      round((real["geneName"].str.casefold() == real["fa_gene"].str.casefold()).mean(), 3))
print(real.loc[real["geneName"].str.casefold() != real["fa_gene"].str.casefold(),
               ["circrna_id", "geneName", "fa_gene"]].head(6).to_string())

# ---- coverage of our circRNAs ---------------------------------------------------------------
hit = set(pairs["circrna_id"])
print("\nC. QKI alone covers", len(hit & model_ids), "of", len(model_ids), "model-input circRNAs;",
      len(hit & cold_ids), "of", len(cold_ids), "cold-start ones")

# ---- sibling structure of OUR circRNAs (host gene from the FASTA header) ----------------------
g = pd.Series({i: fa_gene_s.get(i) for i in model_ids})
g = g[~g.map(lambda x: bool(PLACE.match(str(x))))]
size = g.map(g.value_counts())
print("\nD. model-input circRNAs:", len(model_ids), "| with a usable host-gene label:", len(g),
      "| distinct host genes:", g.nunique())
print("share having >=1 other circRNA from the same host gene:", round((size >= 2).mean(), 3))
print("circRNAs per host gene (mean / max):", round(g.value_counts().mean(), 2), "/", g.value_counts().max())

# ---- miRNATarget on hg38: does the endpoint and this parameter set work? ------------------------
try:
    mtxt = fetch_encori("miRNATarget",
                        {"assembly": "hg38", "geneType": "circRNA", "miRNA": "hsa-miR-145-5p",
                         "clipExpNum": 1, "degraExpNum": 0, "pancancerNum": 0,
                         "programNum": 1, "program": "None", "target": "all", "cellType": "all"},
                        "miRNATarget_hg38_circRNA_miR145-5p")
    m = parse_encori(mtxt)
    print("\nE. miRNATarget rows:", len(m))
    print("columns:", m.columns.tolist())
    print(m.head(3).T.to_string())
except Exception as e:
    print("\nE. miRNATarget probe failed:", e)

A. clusters: 5571 | (cluster, circRNA) pairs: 21848 | distinct circRNAs: 11668
circRNAs per cluster:
 count    5571.0
mean        3.9
std         5.6
min         1.0
50%         2.0
90%         9.0
99%        25.0
max       128.0

clipExpNum:
 clipExpNum
1    4043
2    1369
3     105
4      40
5       9
6       4
7       1
cell lines / tissues: 6 | top: {'HME': 4416, 'HEK293': 929, 'HEK293,HME': 121, 'HME,HMLE': 55}

B. ENCORI circRNAs found in our FASTA index: 11668 of 11668
chromosome agrees: 1.0
host gene agrees (FASTA gene is a real symbol; n=11590): 0.991
            circrna_id         geneName   fa_gene
5367  hsa_circ_0000070  hsa_circ_000597      FAF1
5368  hsa_circ_0000072  hsa_circ_001068      OMA1
5369  hsa_circ_0000073  hsa_circ_001069      OMA1
5370  hsa_circ_0000083  hsa_circ_001078    SRSF11
5371  hsa_circ_0000084  hsa_circ_000622  ANKRD13C
5372  hsa_circ_0000085  hsa_circ_001079  ANKRD13C

C. QKI alone covers 169 of 1582 model-input circRNAs; 142 of 1267 cold-start ones


**Interpretation.** *(A) QKI table.* The 5,571 CLIP clusters expand to 21,848 (cluster, circRNA) pairs over 11,668 distinct circRNAs. A cluster lists a median of 2 circRNAs (mean 3.9, 90th percentile 9, 99th percentile 25, maximum 128), so one binding site creates positive pairs for several overlapping isoforms. Evidence is thin: 4,043 clusters (72.6%) rest on a single CLIP experiment, 1,369 (24.6%) on two, and only 159 (2.9%) on three or more. The clusters carry 6 distinct cell-line/tissue labels, and 4,416 (79%) are labelled `HME`, so the evidence comes mostly from one cell-line label (what `HME` stands for in ENCORI has not been checked). *(B) ID consistency.* All 11,668 ENCORI identifiers are present in our circBase FASTA index (100%) and the chromosome agrees in 100%. The host gene agrees in 99.1% (n = 11,590 with a real FASTA symbol). The six mismatches printed all have an Arraystar-style alias (`hsa_circ_000597`) in ENCORI's `geneName` instead of a gene symbol, so they are naming noise and not real conflicts (inferred from these six rows only, because the other mismatches were not printed). Cell 29 reports 99.3% on the exploded RBP table; the difference is probably due to the filter for "real" symbols (Cell 29 excludes only the literal `None`, this cell also excludes `TCONS`/`XLOC`-type placeholders; inference). *(C) Coverage.* QKI alone reaches 169 of the 1,582 model-input circRNAs (10.7%) and 142 of the 1,267 cold-start ones (11.2%), so the label is not concentrated on well-studied circRNAs. This is a lower bound for the RBP arm, because only one RBP is counted. *(D) Sibling structure.* 1,556 of 1,582 circRNAs have a usable host-gene label, spread over 1,225 genes (mean 1.27, maximum 9 per gene). 36.4% have at least one sibling from the same host gene, and the other 63.6% have none. *(E) miRNATarget.* The endpoint works on hg38: 2,461 rows × 20 columns for hsa-miR-145-5p. The three rows shown have `clipExpNum = 0`, `degraExpNum = 0` and seed class `7mer-m8`, so they are sequence-predicted matches without CLIP support. Evidence is judged from `clipExpNum`, `degraExpNum`, `merClass`, `TDMDScore` and `phyloP`.

**Decision.** (1) Join through circBase IDs and filter `clipExpNum` locally; report coverage at both ≥1 and ≥2. (2) Use host-gene-grouped splits: groups are small (maximum 9), so folds stay balanced. The 26 circRNAs with a placeholder gene need an explicit rule (singleton groups or one separate group). (3) Report RBP coverage as a lower bound until several RBPs have been downloaded and checked. (4) For miRNA, no CLIP-less row counts as a positive label.

### 29. Validation of the ENCORI Identifiers and the CLIP Filter

**Objective.** Validate the ENCORI circBase identifiers against the FASTA index, measure the coverage of the primary circRNAs and test whether the server-side CLIP filter of the miRNA endpoint is honoured.

In [30]:
# Cell 29 - Validate ENCORI circBase IDs against the FASTA index and measure coverage of our circRNAs
# ---- A. miRNA table: is the CLIP filter real? ------------------------------------
# The request asked for clipExpNum >= 1; check whether the server honoured this filter
print("A. miRNATarget (miR-145-5p), hg38")
print("rows:", len(d_mir), "| distinct circRNAs:", d_mir["circID"].nunique())
print("clipExpNum:\n", d_mir["clipExpNum"].value_counts().sort_index().head(8).to_string())
print("rows with clipExpNum>=1:", int((d_mir["clipExpNum"] >= 1).sum()),
      "| distinct circRNAs among them:", d_mir.loc[d_mir["clipExpNum"] >= 1, "circID"].nunique())
print("merClass:\n", d_mir["merClass"].value_counts().head(5).to_string(), "\n")

# ---- B. do ENCORI circBase IDs match our FASTA? -------------------------------------
# Chromosome of every circBase ID in our hg19 FASTA index (circBase IDs are shared across assemblies)
fa_chr = fa_index.set_index("circbase_id")["chrom"].to_dict()

# NOTE: circID is used as delivered; comma-joined ID lists are NOT yet exploded here (diagnosed in Cell 31, corrected in Cell 32)
mir_ids = d_mir.assign(circ_id=d_mir["circID"])
# RBP table: one row is a CLIP cluster that may list several circRNAs, hence txIDcat is exploded
rbp_ids = (d_rbp.assign(circ_id=d_rbp["txIDcat"].str.split(","))
                .explode("circ_id").assign(circ_id=lambda x: x["circ_id"].str.strip()))

# Share of IDs found in the FASTA, and chromosome agreement among the IDs that were found
for name, t in [("miRNA table", mir_ids), ("RBP table (exploded)", rbp_ids)]:
    ok = t["circ_id"].isin(fa_ids)
    chr_ok = (t["chromosome"] == t["circ_id"].map(fa_chr))[ok].mean()
    print(f"B. {name}: rows {len(t)} | distinct IDs {t['circ_id'].nunique()} | "
          f"in FASTA {ok.mean():.3f} | chromosome agrees {chr_ok:.3f}")

# gene check on the RBP table (geneName is a real gene symbol there)
# Gene agreement is only meaningful where the FASTA holds a real gene symbol (not 'None')
g = rbp_ids.assign(gene_fa=rbp_ids["circ_id"].map(fa_gene))
g = g[g["gene_fa"].notna() & (g["gene_fa"] != "None")]
print("   RBP table gene agrees with FASTA:",
      round((g["geneName"].str.casefold() == g["gene_fa"].str.casefold()).mean(), 3), "\n")

# ---- C. coverage of OUR circRNAs ------------------------------------------------------
# Our primary circRNAs from the frozen table of Cell 26
ours = set(prim["circrna_id"])
print("C. our circRNAs with a QKI edge:", len(ours & set(rbp_ids["circ_id"])), "of", len(ours))
print("   our circRNAs with a miR-145-5p site (any):", len(ours & set(mir_ids["circ_id"])),
      "| CLIP-supported:", len(ours & set(mir_ids.loc[mir_ids["clipExpNum"] >= 1, "circ_id"])))
print("   RBP circRNAs per cluster:\n",
      d_rbp["txIDcat"].str.count(",").add(1).describe().round(1).to_string())

A. miRNATarget (miR-145-5p), hg38
rows: 2461 | distinct circRNAs: 1043
clipExpNum:
 clipExpNum
0    1607
1     407
2     170
3      97
4      53
5      29
6      24
7      12
rows with clipExpNum>=1: 854 | distinct circRNAs among them: 795
merClass:
 merClass
7mer-m8    1529
8mer        932 

B. miRNA table: rows 2461 | distinct IDs 1043 | in FASTA 0.515 | chromosome agrees 1.000
B. RBP table (exploded): rows 21848 | distinct IDs 11668 | in FASTA 1.000 | chromosome agrees 1.000
   RBP table gene agrees with FASTA: 0.993 

C. our circRNAs with a QKI edge: 182 of 1644
   our circRNAs with a miR-145-5p site (any): 14 | CLIP-supported: 12
   RBP circRNAs per cluster:
 count    5571.0
mean        3.9
std         5.6
min         1.0
25%         1.0
50%         2.0
75%         4.0
max       128.0


**Interpretation.** *(A) CLIP filter.* Of 2,461 miR-145-5p rows, 1,607 (65%) have `clipExpNum = 0`; these are sequence-predicted seed matches (`7mer-m8`, `8mer`). Only 854 rows (795 distinct circRNAs) have at least one supporting CLIP experiment, so the server-side filter is **not reliable** and must be applied locally. *(B) ID consistency.* In the exploded RBP table, 100% of the 11,668 distinct IDs are present in our FASTA, the chromosome agrees in 100% and the host gene in 99.3%; ENCORI therefore uses the same circBase numbering as our FASTA. The 51.5% found for the miRNA table is an **artefact of this cell**: `circID` was not exploded, so comma-joined lists were compared as single strings (diagnosed in Cell 31, corrected in Cell 32). *(C) Coverage.* QKI alone reaches 182 of our 1,644 circRNAs; RBP clusters list a median of 2 (mean 3.9, maximum 128) circRNAs, i.e. edges are often shared between circRNAs.

**Decision.** Join through circBase IDs; filter on `clipExpNum` locally; explode all list-valued ID columns before any join. Whether the RBP label is merely host-gene-level (and therefore a source of sibling leakage) is examined in Cell 31.

### 30. Availability of the miRNATarget Endpoint on hg38

**Objective.** Run the miRNATarget probe separately, so that a failure cannot hide the RBP result, and inspect the returned columns.

In [31]:
# Cell 30 - Does the miRNATarget endpoint work on hg38? (run separately so that a failure cannot hide the RBP result)
mir_test = probe("miRNATarget", {"assembly": "hg38", "geneType": "circRNA",
                                 "miRNA": "hsa-miR-145-5p", "clipExpNum": 1,
                                 "degraExpNum": 0, "pancancerNum": 0,
                                 "programNum": 1, "program": "None",
                                 "target": "all", "cellType": "all"})
# Parse the response; None would indicate an error page instead of a table
dm = to_df(mir_test)
print("miRNATarget hg38 ->", None if dm is None else dm.shape)
if dm is not None:
    print(dm.columns.tolist())
    print(dm.head(3).T.to_string())

URL: https://rnasysu.com/encori/api/miRNATarget/?assembly=hg38&geneType=circRNA&miRNA=hsa-miR-145-5p&clipExpNum=1&degraExpNum=0&pancancerNum=0&programNum=1&program=None&target=all&cellType=all
HTTP 200 | comment lines: 2 | data lines incl. header: 2462
  # #please cite:
  # #Keren Zhou, Junhong Huang, Shurong Liu, Wujian Zheng, Shun Liu, Yinghao Xing, Yinan Cao, Yibo Chen, Xinping Hu, Li Cai, Jie Zhou, Penghui Lin, Jiajia Xuan, Fan
   miRNAid	miRNAname	geneID	geneName	circID	geneType	chromosome	start	end	strand	clipExpNum	degraExpNum	RBP	merClass	miRseq	align	targetSeq	TDMDScore	phyloP	cellline/tissue
   MIMAT0000437	hsa-miR-145-5p	hsa_circ_0000005	hsa_circ_002136	hsa_circ_0000005	circRNA	chr1	1657106	1657129	-	0	0	NA	7mer-m8	ucccUAAG-GACCCUUUUGACCUg	    | || :| | ||||||||| 	guuaAGUCAUUAGCAAAACUGGAc	0.8557	0.204	
   MIMAT0000437	hsa-miR-145-5p	hsa_circ_0000005	hsa_circ_002136	hsa_circ_0000005	circRNA	chr1	1661214	1661236	-	0	0	NA	7mer-m8	ucccuaaggaCCCUUUUGACCUg	    |     ||| |||||||| 	

**Interpretation.** The `miRNATarget` endpoint works on hg38 with the parameter set used (2,461 rows × 20 columns for hsa-miR-145-5p). The first three rows again have `clipExpNum = 0`. The field `geneName` contains an Arraystar-style alias (`hsa_circ_002136`), whereas `circID` contains the circBase ID (`hsa_circ_0000005`). If this holds in general, ENCORI could act as an alias lookup for some unresolved entries; this is a hypothesis that has not been tested.

**Decision.** The endpoint and parameters are accepted. Evidence is judged from the columns `clipExpNum`, `merClass` and `TDMDScore` after local filtering. The alias idea is noted but is not pursued before the auxiliary labels are settled.

### 31. Unmatched miRNA IDs, Label Granularity and Curated-versus-ENCORI Agreement

**Objective.** Find out why part of the miRNA IDs do not match the FASTA index, test whether the RBP label is merely host-gene-level, and compare the curated miR-145 edges with ENCORI.

In [32]:
# Cell 31 - Unmatched miRNA IDs, granularity of the RBP label and curated-vs-ENCORI agreement
# ---- A. why are 48.5% of miRNA-table rows not in the FASTA? -------------------------
# Rows of the miRNA table whose circID is absent from the FASTA index
miss = mir_ids[~mir_ids["circ_id"].isin(fa_ids)]
# Does the unmatched string have the single-ID shape hsa_circ_<digits>?
num = pd.to_numeric(miss["circ_id"].str.extract(r"^hsa_circ_(\d+)$")[0], errors="coerce")
print("A. rows not in FASTA:", len(miss), "| distinct IDs:", miss["circ_id"].nunique())
print("   not shaped like hsa_circ_<digits>:", int(num.isna().sum()))
print("   numeric part of the others:\n", num.describe().round(0).to_string())
print("   examples:", miss["circ_id"].drop_duplicates().head(8).tolist(), "\n")

# ---- B. is the RBP label gene-level? --------------------------------------------------
print("B. RBP table clipExpNum:\n", d_rbp["clipExpNum"].value_counts().sort_index().head(6).to_string())
# Positive set: every circRNA that appears in a QKI cluster
pos = set(rbp_ids["circ_id"])
# Restrict the FASTA index to circRNAs with a real host-gene annotation
fg = fa_index[fa_index["gene"].notna() & (fa_index["gene"] != "None")]
# Per host gene: number of circRNAs (n) and fraction that is QKI-positive (frac)
gg = (fg.assign(pos=fg["circbase_id"].isin(pos)).groupby("gene")["pos"].agg(n="size", frac="mean"))
gg = gg[(gg["n"] >= 3) & (gg["frac"] > 0)]
print(f"\n   genes with >=3 circRNAs and >=1 QKI-positive: {len(gg)}")
print("   share of those genes where >=80% of their circRNAs are positive:",
      round((gg["frac"] >= 0.8).mean(), 3))
print("   fraction of circRNAs positive within such genes:\n", gg["frac"].describe().round(2).to_string())
# Host-gene diversity of our own QKI-positive circRNAs
ours_pos = prim[prim["circrna_id"].isin(pos)]["circrna_id"]
print("\n   our QKI-positive circRNAs:", ours_pos.nunique(),
      "| distinct host genes:", ours_pos.map(fa_index.set_index("circbase_id")["gene"]).nunique())

# ---- C. curated miR-145 edges vs ENCORI -------------------------------------------------
# Curated miR-145 edges (5p arm or no arm label); compared with the UNEXPLODED ENCORI table, hence unreliable (see Cell 32)
cur = mir_edges[(mir_edges["mirna"] == "miR-145") & (mir_edges["arm"].isin(["5p", None]))]
cur_ids = set(cur["circrna_id"]) & set(prim["circrna_id"])
# ENCORI circRNAs with any site, and with CLIP support (list rows not yet exploded)
enc_any  = set(mir_ids["circ_id"])
enc_clip = set(mir_ids.loc[mir_ids["clipExpNum"] >= 1, "circ_id"])
print("\nC. curated miR-145 (5p or no arm) circRNAs in our set:", len(cur_ids))
print("   also in ENCORI miR-145-5p, any site:", len(cur_ids & enc_any),
      "| CLIP-supported:", len(cur_ids & enc_clip))

A. rows not in FASTA: 1194 | distinct IDs: 672
   not shaped like hsa_circ_<digits>: 1194
   numeric part of the others:
 count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
   examples: ['hsa_circ_0009659,hsa_circ_0009660,hsa_circ_0009661,hsa_circ_0009665,hsa_circ_0009666,hsa_circ_0009667,hsa_circ_0009668,hsa_circ_0009669', 'hsa_circ_0009661,hsa_circ_0009666,hsa_circ_0009667,hsa_circ_0009669,hsa_circ_0009670,hsa_circ_0009671', 'hsa_circ_0009683,hsa_circ_0009684', 'hsa_circ_0009722,hsa_circ_0009723,hsa_circ_0009724,hsa_circ_0009725,hsa_circ_0009726,hsa_circ_0009727,hsa_circ_0009728,hsa_circ_0009729,hsa_circ_0009730,hsa_circ_0009731', 'hsa_circ_0009723,hsa_circ_0009724,hsa_circ_0009725,hsa_circ_0009727,hsa_circ_0009730,hsa_circ_0009731,hsa_circ_0009733,hsa_circ_0009734,hsa_circ_0009735,hsa_circ_0009736,hsa_circ_0009737,hsa_circ_0009738', 'hsa_circ_0009776,hsa_circ_0009777', 'hsa_circ_0009776,hsa_circ_0009777,hsa_circ_0009778,hsa_circ_0

**Interpretation.** *(A)* All 1,194 "unmatched" miRNA rows are comma-joined ID lists (none has the single-ID shape), confirming that the 51.5% in-FASTA figure of Cell 29 was a parsing artefact and not a data problem. *(B)* The RBP label is mostly **not** gene-level: among 2,073 host genes with at least three circRNAs and one QKI-positive circRNA, only 1.7% have 80% or more of their circRNAs positive (median fraction 0.23); our 182 QKI-positive circRNAs lie in 143 different genes. Evidence is thin: 4,043 of 5,571 clusters (73%) rest on a single CLIP experiment, and only about 160 on three or more. *(C)* The reported overlap of 0 of 25 curated miR-145 circRNAs is **unreliable**, because it was computed on the unexploded table and with an arm filter.

**Decision.** List columns are exploded before every join (Cell 32). The threshold on `clipExpNum` is a genuine trade-off between label quality and coverage and is to be chosen after coverage is measured. A gene-grouped check is retained in the evaluation as a precaution against mild sibling leakage.

### 32. Explosion of the Comma-Joined ID Lists

**Objective.** Explode the comma-joined miRNA ID lists, then repeat the join and coverage checks.

In [33]:
# Cell 32 - Explode the comma-joined miRNA ID lists, then repeat the join and coverage checks
# Explode the miRNA table's comma-joined circID lists, then redo the checks.
# One row per circRNA instead of one row per site cluster
m2 = (d_mir.assign(circ_id=d_mir["circID"].str.split(","))
           .explode("circ_id").assign(circ_id=lambda x: x["circ_id"].str.strip()))
# Number of circRNAs that shared the original row: wide rows are weak evidence for any single circRNA
m2["n_ids_in_row"] = m2.groupby(level=0)["circ_id"].transform("size")

print("rows:", len(d_mir), "-> exploded:", len(m2), "| distinct IDs:", m2["circ_id"].nunique())
print("IDs per row:\n", m2.groupby(level=0)["n_ids_in_row"].first().describe().round(1).to_string())

# Join check on the exploded IDs
ok = m2["circ_id"].isin(fa_ids)
print("\nin FASTA:", ok.mean().round(3),
      "| chromosome agrees:", (m2["chromosome"] == m2["circ_id"].map(fa_chr))[ok].mean().round(3))

# Coverage of our primary circRNAs, with and without CLIP support
ours = set(prim["circrna_id"])
clip = m2[m2["clipExpNum"] >= 1]
print("\nour circRNAs with a miR-145-5p site, any:", len(ours & set(m2["circ_id"])),
      "| CLIP-supported:", len(ours & set(clip["circ_id"])))

# Curated miR-145 edges; the arm filter is deliberately omitted because the curated arm labels were inconsistent
cur145 = mir_edges[mir_edges["mirna"] == "miR-145"]
print("\ncurated miR-145 edges, arm values:", cur145["arm"].value_counts(dropna=False).to_dict())
cur_ids = set(cur145["circrna_id"]) & ours
print("curated miR-145 circRNAs in our set:", len(cur_ids),
      "| in ENCORI any:", len(cur_ids & set(m2["circ_id"])),
      "| CLIP-supported:", len(cur_ids & set(clip["circ_id"])))

rows: 2461 -> exploded: 7535 | distinct IDs: 3903
IDs per row:
 count    2461.0
mean        3.1
std         4.6
min         1.0
25%         1.0
50%         1.0
75%         3.0
max        60.0

in FASTA: 1.0 | chromosome agrees: 1.0

our circRNAs with a miR-145-5p site, any: 59 | CLIP-supported: 53

curated miR-145 edges, arm values: {'5p': 17, nan: 9}
curated miR-145 circRNAs in our set: 25 | in ENCORI any: 3 | CLIP-supported: 3


**Interpretation.** After exploding, 2,461 rows expand to 7,535 (circRNA, site) rows over 3,903 distinct circRNAs; **100%** are present in the FASTA and **100%** agree on chromosome, so the miRNA join is as safe as the RBP join. The earlier coverage figure was wrong: miR-145-5p reaches **59** of our circRNAs (53 CLIP-supported), not 14. Rows are frequently shared (median 1, mean 3.1, maximum 60 circRNAs per row), and a site shared by many circRNAs is weak evidence for any one of them. The overlap with the curated edges is small: of 25 curated miR-145 circRNAs in our set, only 3 appear in ENCORI (all CLIP-supported). This concerns one miRNA only and must not be generalised.

**Decision.** Join by circBase ID and record `n_ids` as an edge attribute (a cap on row width will be evaluated). Curated and ENCORI miRNA edges are kept as separate sources until the overlap has been measured across many miRNAs (Cell 40).

### 33. Interim Download Helper

**Objective.** Define an interim download helper with disk caching, retries and explicit handling of truncated responses.

In [34]:
# Cell 33 - Interim download helper (to be superseded by the circmtl.encori module)
import time, urllib.parse, urllib.request, urllib.error

# Raw ENCORI tables are cached on disk, so that every query is requested only once
ENC_DIR = RAW_DIR / "encori"
ENC_DIR.mkdir(parents=True, exist_ok=True)

# A truncated response raises IncompleteRead, which is not an OSError and must be caught explicitly
import http.client

# Download one table (kind: 'rbp' or 'mirna'); existing files are skipped and failures are retried
def enc_download(kind, name, clip, timeout=900, retries=2, wait=20):
    f = ENC_DIR / f"{kind}_{name}_hg38_clip{clip}.tsv"
    # Resume capability: a file that already exists is never downloaded again
    if f.exists():
        return f, "exists"
    # Endpoint and parameter set depend on the interaction type
    if kind == "rbp":
        ep, p = "RBPTarget", dict(assembly="hg38", geneType="circRNA", RBP=name, clipExpNum=clip,
                                  pancancerNum=0, target="all", cellType="all")
    else:
        ep, p = "miRNATarget", dict(assembly="hg38", geneType="circRNA", miRNA=name, clipExpNum=clip,
                                    degraExpNum=0, pancancerNum=0, programNum=1, program="None",
                                    target="all", cellType="all")
    # URL-encode the query string
    url = f"{ENCORI}/{ep}/?{urllib.parse.urlencode(p)}"
    msg = "unknown"
    for attempt in range(1, retries + 1):
        # A response is accepted as a table when its first non-comment line is tab-separated
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                txt = r.read().decode("utf-8", errors="replace")
            body = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
            if body and "\t" in body[0]:
                f.write_text(txt, encoding="utf-8")
                return f, f"{len(body) - 1} rows"
            msg = "not a table: " + (body[0][:70] if body else "(empty)")
        # The server may cut a long response; the reason is recorded in the status message
        except http.client.IncompleteRead as e:
            msg = f"connection cut after {len(e.partial)/1e6:.1f} MB"
        except (urllib.error.URLError, TimeoutError, OSError, http.client.HTTPException) as e:
            msg = f"error: {e}"
        # Linear back-off between attempts
        time.sleep(wait * attempt)
    return None, msg

# sanity check on a query we know works (QKI returned 5,571 rows at clip>=1)
f, status = enc_download("rbp", "QKI", clip=2)
print("QKI, clip>=2 ->", status)

QKI, clip>=2 -> exists


**Interpretation.** In this run the helper reports only `exists` for the QKI request at `clip >= 2`, because the file is cached; no request is sent. An earlier run (output not kept) is reported to have failed on `RBP=all`: the server cut the connection and `http.client.IncompleteRead`, which is not an `OSError`, escaped the retry logic. The helper now catches it explicitly. The cached QKI file at `clip >= 2` holds 3,910 rows (5,571 rows at `clip >= 1`), but only 1,528 of its clusters have `clipExpNum >= 2` in the column (Cell 41). The server parameter therefore does not correspond exactly to the `clipExpNum` column.

**Decision.** Tables are always re-filtered locally on the `clipExpNum` column, and downloads proceed one RBP per request. **Note on practice:** this in-notebook function is interim; the same logic (with config-driven parameters, tests and logging) is intended for `src/circmtl/encori.py`, after which notebook cells only import and call it.

### 34. Names Contained in an RBP=all Response

**Objective.** Establish which RBP names a (truncated) `RBP=all` response contains.

In [42]:
# Cell 34 - Which RBP names does a (truncated) RBP=all response contain?
# Read the possibly truncated response of RBP=all, keeping whatever arrived before the connection was cut
def partial_names(clip=2, timeout=900):
    p = dict(assembly="hg38", geneType="circRNA", RBP="all", clipExpNum=clip,
             pancancerNum=0, target="all", cellType="all")
    url = f"{ENCORI}/RBPTarget/?{urllib.parse.urlencode(p)}"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            txt = r.read().decode("utf-8", errors="replace")
        note = "complete response"
    # Keep the bytes received before the cut
    except http.client.IncompleteRead as e:
        txt, note = e.partial.decode("utf-8", errors="replace"), "partial response"
    lines = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    d = pd.read_csv(io.StringIO("\n".join(lines[:-1])), sep="\t")   # drop the possibly cut last line
    return d, note

# A partial response is not a random sample and must never be used as a label source
dall, note = partial_names()
print(note, "| rows:", len(dall), "| distinct RBPs:", dall["RBP"].nunique())
print(dall["RBP"].value_counts().head(15).to_string())

partial response | rows: 33392 | distinct RBPs: 10
RBP
ALYREF    19339
ACIN1     11872
AKAP1      1325
ALKBH1      371
AATF        295
ALKBH3       63
ALKBH5       41
AIFM1        38
A1CF         27
ADAR         21


**Interpretation.** The truncated `RBP=all` response contains 58,189 rows but only 10 distinct RBPs (ALYREF 44,136 rows; ACIN1 11,872; AKAP1 1,325; …; ADAR 21). All names begin with "A", which suggests that the response is sorted alphabetically (inferred, not verified), so the full response would be enormous and the partial one is not a sample of RBPs. A single `RBP=all` request is therefore not a feasible route, and RBP names cannot be discovered in this way.

**Decision.** `RBP=all` is abandoned. RBPs are downloaded one at a time from a shortlist of circRNA-relevant proteins, and coverage computed from a subset is reported as a lower bound ("at least this much").

### 35. Download of an RBP Shortlist

**Objective.** Download a shortlist of circRNA-relevant RBPs, one request per RBP, at `clipExpNum >= 2`.

In [36]:
# Cell 35 - Download a short list of circRNA-relevant RBPs, one request per RBP
rbps = ["QKI", "FUS", "EIF4A3", "IGF2BP1", "IGF2BP2", "IGF2BP3", "AGO2"]   # spellings unverified
# Existing files are skipped; the status shows the number of rows or the failure reason
for r in rbps:
    f, status = enc_download("rbp", r, clip=2)
    print(f"{r:10s} {status}")

QKI        exists
FUS        exists
EIF4A3     exists
IGF2BP1    exists
IGF2BP2    exists
IGF2BP3    exists
AGO2       exists


**Interpretation.** Downloads at `clip >= 2` are heavy and uneven (row counts are those of the cached files, listed in Cells 37 and 41): IGF2BP2 129,780 rows, IGF2BP3 56,793, FUS 49,416, EIF4A3 41,687, IGF2BP1 40,371 and QKI 3,910. The AGO2 file is only 647 bytes; Cell 38 shows that it holds a server rejection message and not data, so the spelling `AGO2` is not accepted by `RBPTarget` (reason not verified). The helper accepts a response as a table when its first line contains a tab, and the rejection message follows a valid header line, so it was stored as a success.

**Decision.** The large RBPs require a cap on the number of circRNAs per cluster (`n_ids`) in addition to the local `clipExpNum` filter (Cells 41 and 43). The one-row files are inspected in Cells 37 and 38, and the RBP spellings accepted by ENCORI are not guessed further.

### 36. Selection and Download of About 30 miRNAs

**Objective.** Select the 30 curated miRNAs shared by the largest number of primary circRNAs (at least three) and download their ENCORI tables for both arms.

In [37]:
# Cell 36 - Select about 30 curated miRNAs and download their ENCORI tables
# Number of our circRNAs linked to each curated (arm-stripped) miRNA
cnt = (mir_edges[mir_edges["circrna_id"].isin(set(prim["circrna_id"]))]
       .groupby("mirna")["circrna_id"].nunique())
# The 30 most widely shared miRNAs, each linked to at least 3 circRNAs
top = cnt[cnt >= 3].sort_values(ascending=False).head(30).index
# ENCORI uses arm-specific names, therefore both arms are queried for every miRNA
names = [f"hsa-{m}-{arm}" for m in top for arm in ("5p", "3p")]
print(len(names), "queries; first:", names[:4])

# Download all queries; the status text reports the number of lines returned
ok, bad = [], []
for i, n in enumerate(names, 1):
    f, status = enc_download("mirna", n, clip=1)
    (ok if f else bad).append(n)
    print(f"[{i}/{len(names)}] {n:22s} {status}")
    # Courtesy pause to avoid overloading the server
    time.sleep(2)
print("\nsucceeded:", len(ok), "| failed:", len(bad), bad[:10])

60 queries; first: ['hsa-miR-145-5p', 'hsa-miR-145-3p', 'hsa-miR-149-5p', 'hsa-miR-149-3p']
[1/60] hsa-miR-145-5p         exists
[2/60] hsa-miR-145-3p         exists
[3/60] hsa-miR-149-5p         exists
[4/60] hsa-miR-149-3p         exists
[5/60] hsa-miR-338-5p         exists
[6/60] hsa-miR-338-3p         exists
[7/60] hsa-miR-326-5p         exists
[8/60] hsa-miR-326-3p         exists
[9/60] hsa-miR-130a-5p        exists
[10/60] hsa-miR-130a-3p        exists
[11/60] hsa-miR-182-5p         exists
[12/60] hsa-miR-182-3p         exists
[13/60] hsa-miR-330-5p         exists
[14/60] hsa-miR-330-3p         exists
[15/60] hsa-miR-370-5p         exists
[16/60] hsa-miR-370-3p         exists
[17/60] hsa-miR-142-5p         exists
[18/60] hsa-miR-142-3p         exists
[19/60] hsa-miR-136-5p         exists
[20/60] hsa-miR-136-3p         exists
[21/60] hsa-miR-138-5p         exists
[22/60] hsa-miR-138-3p         exists
[23/60] hsa-miR-637-5p         exists
[24/60] hsa-miR-637-3p         exists
[25/6

**Interpretation.** Sixty queries (30 curated miRNAs × both arms) were issued and all were stored as successes, but **32 of the 60** returned a single line only. Many returned thousands of rows (row counts per file are listed in Cell 37; for example hsa-miR-149-5p 4,067; hsa-miR-330-5p 4,700; hsa-miR-370-3p 4,348). For several miRNAs both arms returned one line (hsa-miR-326, -637, -1236, -217, -1205, -1299, -375 and -421). One hypothesis is that ENCORI uses older miRBase names without an arm suffix (for example `hsa-miR-375`); another is that the response contains a "no data" message. Neither has been verified, because the files were not opened.

**Decision.** Neither the curated-versus-ENCORI overlap nor the coverage is computed until the one-line files have been inspected and the arm-less names have been tested. Otherwise the ENCORI side would be biased downwards for exactly those miRNAs, and the overlap would be under-estimated. The download helper should also treat a header-only response as a failed download.

### 37. Offline Inspection of the ENCORI Cache

**Objective.** Inspect every cached ENCORI file offline, without a network request, to see which downloads contain a real table.

In [38]:
# Cell 37 - Inspect every cached ENCORI file without any network request
import pandas as pd

rows = []
for f in sorted(ENC_DIR.glob("*.tsv")):
    txt = f.read_text(encoding="utf-8", errors="replace")
    lines = [l for l in txt.splitlines() if l.strip()]
    data = [l for l in lines if not l.startswith("#")]
    rows.append({
        "file": f.name, "bytes": f.stat().st_size,
        "comment_lines": len(lines) - len(data),
        "data_lines": len(data),
        "rows": max(len(data) - 1, 0),                  # data lines minus the header
        "tab_header": bool(data) and "\t" in data[0],
        "head": txt[:300],
    })
files = pd.DataFrame(rows)

print("files:", len(files), "| with rows == 0:", int((files["rows"] == 0).sum()),
      "| without a tab-separated header:", int((~files["tab_header"]).sum()))
print(files[["file", "bytes", "comment_lines", "data_lines", "rows", "tab_header"]]
      .to_string(index=False, max_colwidth=60))

# Distinct raw contents of the empty / one-line files
empty = files[(files["rows"] == 0) | (~files["tab_header"])]
print("\nDistinct raw contents of empty or one-line files (count, repr of first 300 chars):")
for head, n in empty["head"].value_counts().items():
    print(n, "x", repr(head))

files: 73 | with rows == 0: 0 | without a tab-separated header: 0
                                  file    bytes  comment_lines  data_lines   rows  tab_header
   mirna_hsa-miR-101-3p_hg38_clip1.tsv   217864              2         915    914        True
   mirna_hsa-miR-101-5p_hg38_clip1.tsv      643              2           2      1        True
  mirna_hsa-miR-1205-3p_hg38_clip1.tsv      643              2           2      1        True
  mirna_hsa-miR-1205-5p_hg38_clip1.tsv      643              2           2      1        True
  mirna_hsa-miR-1236-3p_hg38_clip1.tsv      643              2           2      1        True
  mirna_hsa-miR-1236-5p_hg38_clip1.tsv      643              2           2      1        True
  mirna_hsa-miR-125a-3p_hg38_clip1.tsv      643              2           2      1        True
  mirna_hsa-miR-125a-5p_hg38_clip1.tsv   791371              2        3318   3317        True
  mirna_hsa-miR-1299-3p_hg38_clip1.tsv      643              2           2      1       

**Interpretation.** The cache holds 69 files: 60 miRNA queries (30 miRNAs × both arms), 7 RBP queries, the miR-145-5p probe and the QKI clip1 probe. All 69 carry two comment lines and a tab-separated header line, so the helper's check ("first line contains a tab") could not tell valid tables from rejections. 28 of the 60 miRNA files are real tables (507 to 4,700 rows, 128 KB to 1.09 MB). The other 32 are exactly 643 bytes with one data line each; Cell 38 showed that this line is a server message ("the miRNA parameter hasn't been set correctly or is not available"), so these files contain **0 real rows**. By arm, 8 miRNAs have no table for either arm, 16 have a table for one arm only, and 6 have tables for both arms. The RBP downloads at `clip >= 2` are large and uneven: IGF2BP2 129,780 rows, IGF2BP3 56,793, FUS 49,416, EIF4A3 41,687, IGF2BP1 40,371 and QKI 3,910. AGO2 is a 647-byte file with the same kind of rejection message (for the `RBP` parameter), so the spelling `AGO2` is not accepted by `RBPTarget` (reason not verified). The QKI file at `clip >= 2` has 3,910 rows, whereas the clip1 file has only 1,528 clusters whose `clipExpNum` column is >= 2, which confirms that the server parameter does not match the column and must be re-applied locally.

**Limitation of this cell.** The filter meant to list empty files tested `rows == 0`, but a rejected file has `rows == 1` (the message line), so that section printed nothing. This was fixed in Cell 38, which reads the raw text. The row counts in the "rows" column therefore overstate the content of the 33 rejected files by one each.

**Decision.** (1) The 33 rejected files (32 miRNA, 1 RBP) are excluded from all counts and stay on disk. (2) A download counts only if its first data row contains tabs and its `miRNAname` (or `RBP`) column matches the query. (3) No miRNA or RBP coverage figure is computed from these files until the failed names have been retested (Cell 39).

### 38. Content of the One-Row Files and Arm Table

**Objective.** Read the raw text of the one-row files and tabulate which miRNA/arm combinations returned data.

In [41]:
# Cell 38 - What do the one-row ENCORI files contain? And which miRNA/arm combinations have data?
import re
import pandas as pd

one = files[files["rows"] <= 1].copy()
print("one-row (or empty) files:", len(one), "| of which miRNA:", int(one["file"].str.startswith("mirna_").sum()),
      "| RBP:", int(one["file"].str.startswith("rbp_").sum()))

# 1) The raw text of three examples (full text, not truncated)
for name in ["mirna_hsa-miR-101-5p_hg38_clip1.tsv", "mirna_hsa-miR-375-3p_hg38_clip1.tsv", "rbp_AGO2_hg38_clip2.tsv"]:
    print("\n---", name)
    print(repr((ENC_DIR / name).read_text(encoding="utf-8", errors="replace")))

# 2) Are all one-row miRNA files identical, apart from the miRNA name?
def full(f): return (ENC_DIR / f).read_text(encoding="utf-8", errors="replace")
mir_one = one[one["file"].str.startswith("mirna_")]["file"].tolist()
texts = pd.Series({f: full(f) for f in mir_one})
print("\ndistinct raw texts among the", len(texts), "one-row miRNA files:", texts.nunique())

# 3) Arm table for the 30 miRNAs: rows per arm (1 = one-row response)
m = files[files["file"].str.startswith("mirna_hsa-")].copy()
m["mirna"] = m["file"].str.extract(r"mirna_hsa-(.+)-(?:5p|3p)_hg38")[0]
m["arm"] = m["file"].str.extract(r"-(5p|3p)_hg38")[0]
arms = m.pivot(index="mirna", columns="arm", values="rows").fillna(-1).astype(int)
arms["pattern"] = arms.apply(lambda r: "both" if (r["5p"] > 1 and r["3p"] > 1)
                             else "none" if (r["5p"] <= 1 and r["3p"] <= 1) else "one arm", axis=1)
print("\npattern counts:\n", arms["pattern"].value_counts().to_string())
print("\nmiRNAs with no table for either arm:", sorted(arms.index[arms["pattern"] == "none"]))

one-row (or empty) files: 33 | of which miRNA: 32 | RBP: 1

--- mirna_hsa-miR-101-5p_hg38_clip1.tsv
'#please cite:\n#Keren Zhou, Junhong Huang, Shurong Liu, Wujian Zheng, Shun Liu, Yinghao Xing, Yinan Cao, Yibo Chen, Xinping Hu, Li Cai, Jie Zhou, Penghui Lin, Jiajia Xuan, Fangzhou Xie, Lahong Xu, Junhao Li, Lingling Zheng, Lianghu Qu, Jianjun Chen, Bin Li, and Jianhua Yang, An Encyclopedic Regulatory and Functional Atlas of RNA Interactomes, Nature Methods (2026).\nmiRNAid\tmiRNAname\tgeneID\tgeneName\tcircID\tgeneType\tchromosome\tstart\tend\tstrand\tclipExpNum\tdegraExpNum\tRBP\tmerClass\tmiRseq\talign\ttargetSeq\tTDMDScore\tphyloP\tcellline/tissue\nThe "miRNA" parameter haven\'t been set correctly! Or the input of "miRNA" parameter is not available!\n'

--- mirna_hsa-miR-375-3p_hg38_clip1.tsv
'#please cite:\n#Keren Zhou, Junhong Huang, Shurong Liu, Wujian Zheng, Shun Liu, Yinghao Xing, Yinan Cao, Yibo Chen, Xinping Hu, Li Cai, Jie Zhou, Penghui Lin, Jiajia Xuan, Fangzhou Xie, Lahong

ValueError: Index contains duplicate entries, cannot reshape

**Interpretation.** The 33 one-row files (32 miRNA, 1 RBP) are not "no data" results. They are server rejections of the query parameter. The single data line in each one reads `The "miRNA" parameter haven't been set correctly! Or the input of "miRNA" parameter is not available!` (for the RBP file, the same message for the `"RBP"` parameter). All 32 miRNA files contain the identical text (1 distinct text), so the failure does not depend on the miRNA. The "1 row" counted in Cell 37 is this message line, so these files contain **0 real rows**. The arm pattern is reproduced: 16 miRNAs have a table for exactly one arm, 6 for both arms and 8 for neither (miR-1205, -1236, -1299, -217, -326, -375, -421, -637). In total 28 of the 60 miRNA requests returned tables and 32 were rejected. The AGO2 RBP request was rejected the same way, so the spelling `AGO2` is not accepted by `RBPTarget` (reason not verified). The cache helper accepted these files because it only checks for a tab in the first line, and the header line has tabs.

**Decision.** (1) A response is a valid table only if its first data row contains tabs and its `miRNAname` (or `RBP`) column matches the query. (2) The 33 rejected files are excluded from every count; they are left on disk and are not deleted. (3) The 8 miRNAs with no table for either arm are tested with arm-less names (Cell 39). (4) For the 16 one-arm miRNAs, the missing arm is not assumed to be absent from ENCORI. It may simply be an unaccepted name, so only the arms that returned tables count, and any coverage figure is a lower bound.

### 39. Arm-less miRNA Names

**Objective.** Test whether arm-less names work for the 8 miRNAs that returned no table under either arm (10 requests: the 8 miRNAs and 2 controls).

In [ ]:
# Cell 39 - Do arm-less names work for the 8 miRNAs with no table for either arm? (10 requests)
import io, time, urllib.request, urllib.parse
import pandas as pd

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
model_ids = set(circ.loc[circ["use_primary"], "circrna_id"])

def get_mirna_table(name, timeout=300):
    """Return (DataFrame or None, message). Saves the file only if the response is a real table."""
    p = dict(assembly="hg38", geneType="circRNA", miRNA=name, clipExpNum=1, degraExpNum=0,
             pancancerNum=0, programNum=1, program="None", target="all", cellType="all")
    url = "https://rnasysu.com/encori/api/miRNATarget/?" + urllib.parse.urlencode(p)
    with urllib.request.urlopen(url, timeout=timeout) as r:
        txt = r.read().decode("utf-8", errors="replace")
    data = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    if len(data) < 2 or "\t" not in data[1]:
        return None, (data[1][:90] if len(data) > 1 else "empty response")
    d = pd.read_csv(io.StringIO("\n".join(data)), sep="\t")
    (ENC_DIR / f"mirna_{name}_hg38_clip1.tsv").write_text(txt, encoding="utf-8")
    return d, "table"

none8 = ["miR-1205", "miR-1236", "miR-1299", "miR-217", "miR-326", "miR-375", "miR-421", "miR-637"]
tests = [f"hsa-{m}" for m in none8] + ["hsa-miR-145", "hsa-miR-149"]      # last two are controls

out = []
for n in tests:
    d, msg = get_mirna_table(n)
    if d is None:
        out.append(dict(query=n, ok=False, rows=0, note=msg))
    else:
        ids = d["circID"].astype(str).str.split(",").explode().str.strip()
        out.append(dict(query=n, ok=True, rows=len(d),
                        returned_names=",".join(sorted(d["miRNAname"].unique()))[:60],
                        clip_ge1=int((d["clipExpNum"] >= 1).sum()),
                        circRNAs=ids.nunique(), in_model_input=len(set(ids) & model_ids), note="table"))
    time.sleep(2)
print(pd.DataFrame(out).to_string(index=False, max_colwidth=62))

       query    ok  rows                                                           note returned_names  clip_ge1  circRNAs  in_model_input
hsa-miR-1205 False     0 The "miRNA" parameter haven't been set correctly! Or the in...            NaN       NaN       NaN             NaN
hsa-miR-1236 False     0 The "miRNA" parameter haven't been set correctly! Or the in...            NaN       NaN       NaN             NaN
hsa-miR-1299 False     0 The "miRNA" parameter haven't been set correctly! Or the in...            NaN       NaN       NaN             NaN
 hsa-miR-217  True  2002                                                          table    hsa-miR-217     717.0    3445.0            28.0
 hsa-miR-326  True  4559                                                          table    hsa-miR-326    1162.0    5068.0            51.0
 hsa-miR-375  True   132                                                          table    hsa-miR-375     100.0     475.0             6.0
 hsa-miR-421  True  1724   

**Interpretation.** Of the 8 miRNAs that returned no table under `-5p` or `-3p`, four are accepted by ENCORI **without an arm suffix**: `hsa-miR-217` (2,002 rows), `hsa-miR-326` (4,559), `hsa-miR-375` (132) and `hsa-miR-421` (1,724). In each case the returned `miRNAname` equals the query, so ENCORI stores these under the arm-less name. The other four (`hsa-miR-1205`, `-1236`, `-1299`, `-637`) are rejected with the same "parameter not available" message under all three spellings tried (`-5p`, `-3p`, arm-less). Both controls were rejected without an arm: `hsa-miR-145` and `hsa-miR-149`, although `hsa-miR-145-5p` and `hsa-miR-149-5p` return tables. So an arm-less name is not a general fallback; for each miRNA ENCORI accepts a particular spelling. The four new tables give a first, unfiltered coverage figure of the model-input circRNAs: 28 (miR-217), 51 (miR-326), 6 (miR-375) and 30 (miR-421). These counts include every circRNA in the table, including rows with `clipExpNum = 0` (sequence-predicted matches), so they are upper bounds on CLIP-supported coverage.

**Inference (not verified).** ENCORI seems to accept one naming per miRNA, probably following one miRBase annotation version. This is consistent with the four hits and two controls, but ENCORI's name list was not checked. If it holds, a missing arm (e.g. miR-145-3p) is not in ENCORI's list at all, which is different from a spelling problem.

**Tally.** Of the 30 curated miRNAs, 26 now have at least one valid table (16 one arm only, 6 both arms, 4 arm-less); 4 have none (miR-1205, -1236, -1299, -637). In total 32 valid tables from 70 requests (60 + 10), of which 4 are new.

**Decision.** (1) Accept the four arm-less tables; the files `mirna_hsa-miR-217/326/375/421_hg38_clip1.tsv` were written by Cell 39. (2) Do not chase the four rejected miRNAs with more guessed spellings: they are low-priority curated miRNAs. They are recorded as "not retrievable under the tested names". (3) Merge arms when comparing with curated edges, because curated node IDs are arm-stripped (the existing project decision). (4) The 30 miRNAs are the most widely shared in our data, so coverage computed from them is not representative of all miRNAs. It is a statement about popular miRNAs only.

### 40. miRNA Coverage from ENCORI and Overlap with the Curated Edges

**Objective.** Measure the miRNA coverage of the model-input circRNAs from the valid ENCORI tables and the overlap of those tables with the curated edges.

In [ ]:
# Cell 40 - miRNA coverage of our circRNAs from valid ENCORI tables, and overlap with the curated edges
import io, re
import pandas as pd

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
model_ids = set(circ.loc[circ["use_primary"], "circrna_id"])
cold_ids  = set(circ.loc[circ["use_primary"] & (circ["n_diseases"] == 1), "circrna_id"])

cur = pd.read_csv(PROCESSED_DIR / "mirna_edges_curated.csv")
print("curated columns:", cur.columns.tolist())
assert {"circrna_id", "mirna"} <= set(cur.columns), "adapt the column names above"

def read_valid(path):
    txt = path.read_text(encoding="utf-8", errors="replace")
    data = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    if len(data) < 2 or "\t" not in data[1]:
        return None                                   # rejected response, not a table
    return pd.read_csv(io.StringIO("\n".join(data)), sep="\t")

frames = []
for f in sorted(ENC_DIR.glob("mirna_hsa-*_hg38_clip1.tsv")):
    d = read_valid(f)
    if d is None:
        continue
    q = re.match(r"mirna_(hsa-.+)_hg38_clip1\.tsv", f.name).group(1)
    d = d[["miRNAname", "circID", "clipExpNum", "degraExpNum", "merClass"]].copy()
    d["query"] = q
    d["mirna"] = re.sub(r"-(3p|5p)$", "", q.replace("hsa-", "", 1))   # arm-stripped, as in the curated table
    d["ids"] = d["circID"].astype(str).str.split(",")
    d["n_ids"] = d["ids"].map(len)
    frames.append(d)
enc = pd.concat(frames, ignore_index=True)
enc["clipExpNum"] = pd.to_numeric(enc["clipExpNum"], errors="coerce").fillna(0)
print("valid tables:", enc["query"].nunique(), "| miRNAs:", enc["mirna"].nunique(),
      "| rows:", len(enc), "| name mismatch (miRNAname != query):", int((enc["miRNAname"] != enc["query"]).sum()))

e = enc.explode("ids").rename(columns={"ids": "circrna_id"})
e["circrna_id"] = e["circrna_id"].str.strip()
print("(row, circRNA) pairs:", len(e), "| distinct circRNAs:", e["circrna_id"].nunique())

defs = {"clip>=1": e["clipExpNum"] >= 1,
        "clip>=2": e["clipExpNum"] >= 2,
        "clip>=1 & n_ids<=10": (e["clipExpNum"] >= 1) & (e["n_ids"] <= 10)}
E = {k: e.loc[m, ["mirna", "circrna_id"]].drop_duplicates() for k, m in defs.items()}

cur_m = cur.loc[cur["circrna_id"].isin(model_ids), ["mirna", "circrna_id"]].drop_duplicates()

rows = []
for m, g in enc.groupby("mirna"):
    c_set = set(cur_m.loc[cur_m["mirna"] == m, "circrna_id"])
    r = {"mirna": m, "queries": ",".join(sorted(g["query"].str.replace("hsa-" + m, "", regex=False).replace("", "armless").unique())),
         "curated_in_model": len(c_set)}
    for k in ("clip>=1", "clip>=2"):
        s = set(E[k].loc[E[k]["mirna"] == m, "circrna_id"]) & model_ids
        r[f"enc_{k}"] = len(s); r[f"overlap_{k}"] = len(s & c_set)
    rows.append(r)
per = pd.DataFrame(rows).sort_values("curated_in_model", ascending=False)
print("\nPer miRNA (circRNAs restricted to the 1,582 model-input set):")
print(per.to_string(index=False))

print("\nCurated miRNAs with no valid ENCORI table:",
      sorted(set(cur_m["mirna"]) - set(enc["mirna"]))[:15], "...")

print("\nTotals over the", enc["mirna"].nunique(), "miRNAs:")
for k, x in E.items():
    s = set(x["circrna_id"]) & model_ids
    pairs_m = x[x["circrna_id"].isin(model_ids)]
    print(f"  {k:22s} edges on model-input circRNAs: {len(pairs_m):6d} | distinct circRNAs: {len(s):4d} "
          f"({len(s)/len(model_ids):.1%}) | cold-start: {len(s & cold_ids):4d} ({len(s & cold_ids)/len(cold_ids):.1%})")
cur_sub = cur_m[cur_m["mirna"].isin(set(enc["mirna"]))]
print("  curated edges for these miRNAs (model-input circRNAs):", len(cur_sub),
      "| distinct circRNAs:", cur_sub["circrna_id"].nunique())

curated columns: ['circrna_id', 'mirna', 'arm']
valid tables: 32 | miRNAs: 26 | rows: 79116 | name mismatch (miRNAname != query): 0
(row, circRNA) pairs: 225189 | distinct circRNAs: 40373

Per miRNA (circRNAs restricted to the 1,582 model-input set):
   mirna queries  curated_in_model  enc_clip>=1  overlap_clip>=1  enc_clip>=2  overlap_clip>=2
 miR-145     -5p                24           48                3           23                2
 miR-149     -5p                17           30                3           20                2
 miR-338     -3p                17           51                3           31                2
 miR-326 armless                17           44                7           25                5
miR-130a -3p,-5p                13           78                3           53                2
 miR-182     -5p                13           42                3           24                2
 miR-370 -3p,-5p                12           79                3           44       

**Interpretation.** *(A) Tables.* 32 valid tables cover 26 miRNAs (79,116 rows; 225,189 (row, circRNA) pairs over 40,373 distinct circRNAs). The `miRNAname` equals the query in every row (0 mismatches), so every response belongs to the miRNA it was requested for. *(B) Coverage of our circRNAs.* Restricted to the 1,582 model-input circRNAs, ENCORI gives 1,107 edges over 508 circRNAs (32.1%) at `clipExpNum >= 1`, and 704 edges over 352 circRNAs (22.3%) at `>= 2`. Cold-start circRNAs are covered at nearly the same rate (30.7% and 21.4%), so the labels are not concentrated on well-studied circRNAs. A cap of 10 circRNAs per site (`n_ids <= 10`) leaves 818 edges over 395 circRNAs (25.0%), i.e. 26% of the clip>=1 edges come from sites listing more than 10 circRNAs. *(C) Agreement with the curated edges.* For these 26 miRNAs the curated file has 260 edges over 227 model-input circRNAs. Summed from the per-miRNA table, 57 of the 260 (21.9%) also appear in ENCORI at clip>=1 and 37 (14.2%) at clip>=2. The best case is miR-326 (7 of 17 at clip>=1); miR-502, -486, -615, -328, -382 and -375 overlap 0 at clip>=1, and miR-217, -502, -486, -375, -615, -328 and -382 overlap 0 at clip>=2. ENCORI adds many edges that are not curated (for miR-145-5p, 48 ENCORI circRNAs versus 24 curated, 3 shared). *(D) Chance level.* Across the 26 miRNAs the density of ENCORI edges among all (circRNA, miRNA) pairs of the model-input set is 1,107 / (26 × 1,582) = 2.7%. The curated edges fall into ENCORI at 21.9%, about 8 times that density (clip>=2: 1.7% against 14.2%, also about 8 times). This ratio is not degree-controlled, because both sources favour highly expressed or popular circRNAs, so it should be read as an upper bound on the true lift.

**Correction to the printed line.** "Curated miRNAs with no valid ENCORI table" lists every curated miRNA that was not downloaded, and most of them (let-7, miR-100, miR-103a, ...) were never queried. Only the 4 miRNAs rejected in Cell 39 (miR-1205, -1236, -1299, -637) failed. The list is therefore not a list of failures.

**Decision.** (1) Curated and ENCORI miRNA edges stay separate sources: they overlap weakly in both directions, so each is a different label, not a duplicate. (2) The overlap table above is the evidence for the evaluation design that tests curated-only, ENCORI-only and union label sets. (3) Edges from sites with many circRNAs (`n_ids`) are down-weighted or capped, and the cap is chosen later with the coverage numbers. (4) These are 26 miRNAs chosen as the most widely shared curated miRNAs, so the figures describe popular miRNAs only and must not be generalised to all miRNAs.

### 41. RBP Coverage from ENCORI

**Objective.** Measure the RBP coverage of the model-input circRNAs from the valid ENCORI RBP tables (offline; a lower bound, because only six RBPs are included).

In [ ]:
# Cell 41 - RBP coverage of our circRNAs from the valid ENCORI RBP tables (offline, lower bound)
import io, re
import pandas as pd

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
gene_s = fa_idx.set_index("circbase_id")["gene"]
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)
model_ids = set(circ.loc[circ["use_primary"], "circrna_id"])
cold_ids  = set(circ.loc[circ["use_primary"] & (circ["n_diseases"] == 1), "circrna_id"])

def read_valid(path):
    txt = path.read_text(encoding="utf-8", errors="replace")
    data = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    if len(data) < 2 or "\t" not in data[1]:
        return None
    return pd.read_csv(io.StringIO("\n".join(data)), sep="\t")

edges = {}                       # (rbp, threshold, cap) -> DataFrame(rbp, circrna_id)
summary = []
for f in sorted(ENC_DIR.glob("rbp_*_hg38_clip2.tsv")):
    rbp = re.match(r"rbp_(.+)_hg38_clip2\.tsv", f.name).group(1)
    d = read_valid(f)
    if d is None:
        print("rejected (no table):", f.name)
        continue
    d["clipExpNum"] = pd.to_numeric(d["clipExpNum"], errors="coerce").fillna(0)
    d["ids"] = d["txIDcat"].astype(str).str.split(",")
    d["n_ids"] = d["ids"].map(len)
    row = {"rbp": rbp, "rows_in_file": len(d), "name_mismatch": int((d["RBP"] != rbp).sum())}
    for thr in (2, 3):
        sub = d[d["clipExpNum"] >= thr]
        e = sub.explode("ids").rename(columns={"ids": "circrna_id"})
        e["circrna_id"] = e["circrna_id"].str.strip()
        for cap in (None, 10):
            ee = e if cap is None else e[e["n_ids"] <= cap]
            ee = ee[["circrna_id"]].drop_duplicates()
            ee["rbp"] = rbp
            edges[(rbp, thr, cap)] = ee
            s = set(ee["circrna_id"]) & model_ids
            tag = f"clip>={thr}" + ("" if cap is None else f"&n<={cap}")
            row[f"{tag}: model"] = len(s)
        row[f"clusters clip>={thr}"] = len(sub)
    summary.append(row)
per = pd.DataFrame(summary)
print(per.to_string(index=False))

print("\nUnion over the valid RBPs (circRNAs in the 1,582 model-input set):")
for thr in (2, 3):
    for cap in (None, 10):
        u = pd.concat([v for (r, t, c), v in edges.items() if t == thr and c == cap])
        s = set(u["circrna_id"]) & model_ids
        usable = [i for i in s if not PLACE.match(str(gene_s.get(i, "")))]
        genes = {gene_s.get(i) for i in usable}
        print(f"  clip>={thr}, cap={cap}: circRNAs {len(s):4d} ({len(s)/len(model_ids):.1%}) | "
              f"cold-start {len(s & cold_ids):4d} ({len(s & cold_ids)/len(cold_ids):.1%}) | "
              f"distinct host genes among them {len(genes)}")

u2 = pd.concat([v for (r, t, c), v in edges.items() if t == 2 and c == 10])
u2 = u2[u2["circrna_id"].isin(model_ids)]
k = u2.groupby("circrna_id")["rbp"].nunique()
print("\nNumber of RBPs per positive model-input circRNA (clip>=2, cap 10):")
print(k.value_counts().sort_index().to_string())

rejected (no table): rbp_AGO2_hg38_clip2.tsv
    rbp  rows_in_file  name_mismatch  clip>=2: model  clip>=2&n<=10: model  clusters clip>=2  clip>=3: model  clip>=3&n<=10: model  clusters clip>=3
 EIF4A3         41687              0             381                   274             13414              70                    44              2019
    FUS         49416              0             277                   238             11843             169                   144              6651
IGF2BP1         40371              0             441                   351             18154             267                   203              9196
IGF2BP2        129780              0             658                   531             38337             406                   325             14298
IGF2BP3         56793              0             543                   427             24415             241                   185              8956
    QKI          3910              0              65         

**Interpretation.** *(A) Validity.* Six RBP tables are valid (EIF4A3, FUS, IGF2BP1, IGF2BP2, IGF2BP3, QKI); `name_mismatch` is 0 for all of them, so every row belongs to its query. AGO2 stays rejected (no table) and is excluded, consistent with the earlier decision to treat the AGO family as miRNA-related. The local filter reproduces the earlier QKI numbers: the file requested at `clip2` gives 1,528 clusters with `clipExpNum >= 2` and 159 with `>= 3`, the same values as the clip1 probe (Cell 28), so the server parameter is not needed and the column is the reference. *(B) Coverage per RBP.* At `clipExpNum >= 2`, the number of model-input circRNAs (of 1,582) with a site is 658 for IGF2BP2 (41.6%), 543 for IGF2BP3 (34.3%), 441 for IGF2BP1 (27.9%), 381 for EIF4A3 (24.1%), 277 for FUS (17.5%) and 65 for QKI (4.1%). *(C) Union over the six RBPs.* 949 circRNAs (60.0%; cold-start 747, 59.0%) at `>= 2`; 782 (49.4%; cold-start 615, 48.5%) when only clusters listing at most 10 circRNAs count (`n_ids <= 10`); 591 (37.4%) at `>= 3`; 473 (29.9%) at `>= 3` with the cap. The cap removes 167 circRNAs (17.6%) at `>= 2`. Coverage is almost identical for cold-start circRNAs, so it is not concentrated on well-studied ones. The threshold matters a lot: for QKI, raising it from 1 to 2 to 3 leaves 169, 65 and 13 circRNAs. *(D) Redundancy.* Of the 782 positives (`>= 2`, cap 10), 258 (33.0%) are positive for one RBP, 524 (67.0%) for two or more and 321 (41.0%) for three or more (mean 2.39 RBPs per positive circRNA). *(E) Comparison with miRNA.* The six-RBP union (49.4%) is higher than the ENCORI miRNA coverage over 26 miRNAs (22.3% at `clip >= 2`, Cell 40). The two figures are not like-for-like: different protein and miRNA sets; the six RBPs were a shortlist (Cell 35), not a random sample; and the cap of 10 circRNAs per site is applied to the RBP figure but not to the 22.3%.

**Inferences (not verified).** (i) Positive rates of up to 41.6% for one RBP, and two-thirds of positives carrying two or more RBPs, suggest that part of the signal is shared across RBPs, for example host-gene expression, CLIP detectability or circRNA length. None of these has been tested yet; Cell 42 tests length. (ii) Because CLIP reads cannot separate a circRNA from its linear host unless they span the junction (how ENCORI handles this is unconfirmed), the labels may partly describe the host locus. (iii) The 60% union is not an estimate of "bound by any RBP", since only six proteins are included.

**Decision.** (1) Coverage is no longer the bottleneck for the RBP arm; label quality is. (2) Provisional working definition: `clipExpNum >= 2` and `n_ids <= 10` (782 positive circRNAs, 49.4%), with `>= 3` as a sensitivity analysis. Both columns are kept in the frozen table so the threshold stays a choice and not a deletion. (3) The head formulation (six per-RBP binary outputs or one any-RBP output) is decided after the shortcut check in Cell 42. (4) An absent site remains unlabelled, not negative.

### 42. Shortcut Check for the RBP Labels

**Objective.** Test whether circRNA length and RBP co-occurrence explain the RBP labels (shortcut check).

In [ ]:
# Cell 42 - Shortcut check: do circRNA length and RBP co-occurrence explain the RBP labels?
import io, re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
mi = circ[circ["use_primary"]].set_index("circrna_id")            # the 1,582 model-input circRNAs

def read_valid(path):
    txt = path.read_text(encoding="utf-8", errors="replace")
    data = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    if len(data) < 2 or "\t" not in data[1]:
        return None
    return pd.read_csv(io.StringIO("\n".join(data)), sep="\t")

pos = {}
for f in sorted(ENC_DIR.glob("rbp_*_hg38_clip2.tsv")):
    rbp = re.match(r"rbp_(.+)_hg38_clip2\.tsv", f.name).group(1)
    d = read_valid(f)
    if d is None:
        continue
    d["clipExpNum"] = pd.to_numeric(d["clipExpNum"], errors="coerce").fillna(0)
    d = d[d["clipExpNum"] >= 2].copy()
    d["ids"] = d["txIDcat"].astype(str).str.split(",")
    d = d[d["ids"].map(len) <= 10]
    pos[rbp] = {i.strip() for lst in d["ids"] for i in lst} & set(mi.index)

Y = pd.DataFrame({r: mi.index.isin(list(s)) for r, s in pos.items()}, index=mi.index)
rbps = list(Y.columns)
Y["n_rbp"] = Y[rbps].sum(axis=1)
Y["any"] = Y[rbps].any(axis=1)
print("RBPs:", rbps, "| any-RBP positives:", int(Y["any"].sum()), "(Cell 41: 782)")

# 1) Positive rate by length band
Y["length_band"] = mi["length_band"]
rate = Y.groupby("length_band")[rbps + ["any"]].mean().round(3)
rate["n"] = Y.groupby("length_band").size()
print("\n1) Positive rate by length band:\n", rate.to_string())

# 2) Length alone as a predictor (AUROC of log10 sequence length; 0.5 = no information)
L = np.log10(mi["seq_len"].clip(lower=1))
print("\n2) AUROC of log10(seq_len) for each label:")
print({c: round(roc_auc_score(Y[c], L), 3) for c in rbps + ["any"]})
rho, p = spearmanr(L, Y["n_rbp"])
print("Spearman(log length, number of RBPs per circRNA):", round(rho, 3), "| p =", f"{p:.2g}")

# 3) Co-occurrence between RBPs: Jaccard and lift (observed overlap / overlap expected if independent)
N = len(Y)
J = pd.DataFrame(index=rbps, columns=rbps, dtype=float)
LF = pd.DataFrame(index=rbps, columns=rbps, dtype=float)
for a in rbps:
    for b in rbps:
        inter = int((Y[a] & Y[b]).sum()); union = int((Y[a] | Y[b]).sum())
        J.loc[a, b] = inter / union
        LF.loc[a, b] = inter * N / (Y[a].sum() * Y[b].sum())
print("\n3a) Jaccard overlap between RBP label sets:\n", J.round(2).to_string())
print("\n3b) Lift over independence (1.0 = as expected by chance):\n", LF.round(2).to_string())


RBPs: ['EIF4A3', 'FUS', 'IGF2BP1', 'IGF2BP2', 'IGF2BP3', 'QKI'] | any-RBP positives: 782 (Cell 41: 782)

1) Positive rate by length band:
              EIF4A3    FUS  IGF2BP1  IGF2BP2  IGF2BP3    QKI    any     n
length_band                                                              
1023-5000     0.240  0.249    0.337    0.500    0.423  0.057  0.626   350
<=1022        0.154  0.123    0.189    0.289    0.226  0.024  0.457  1232

2) AUROC of log10(seq_len) for each label:
{'EIF4A3': 0.614, 'FUS': 0.642, 'IGF2BP1': 0.648, 'IGF2BP2': 0.629, 'IGF2BP3': 0.653, 'QKI': 0.616, 'any': 0.623}
Spearman(log length, number of RBPs per circRNA): 0.267 | p = 3.9e-27

3a) Jaccard overlap between RBP label sets:
          EIF4A3   FUS  IGF2BP1  IGF2BP2  IGF2BP3   QKI
EIF4A3     1.00  0.21     0.23     0.33     0.24  0.07
FUS        0.21  1.00     0.30     0.32     0.27  0.09
IGF2BP1    0.23  0.30     1.00     0.40     0.56  0.08
IGF2BP2    0.33  0.32     0.40     1.00     0.44  0.08
IGF2BP3    0.24 

**Interpretation.** *(A) Rebuild.* The cell reproduces the Cell 41 labels exactly: 782 any-RBP positives at `clipExpNum >= 2` with clusters of at most 10 circRNAs. *(B) Length band.* Positive rates are higher for longer circRNAs. For any-RBP the rate is 62.6% in the 1,023-5,000 nt band (219 of 350) against 45.7% for ≤1,022 nt (563 of 1,232), a ratio of 1.37. Per RBP the ratio is 1.6 to 2.0 for EIF4A3, FUS, IGF2BP1, IGF2BP2 and IGF2BP3. For QKI it is about 2.4 (5.7% against 2.4%), but its counts are small (49 positives). *(C) Length as a predictor.* The AUROC of log10(sequence length) is 0.614 to 0.653 for the six RBPs and 0.623 for any-RBP (0.5 = no information). The Spearman correlation between log length and the number of RBPs per circRNA is 0.267 (p = 3.9e-27). Length therefore carries a modest, clearly non-zero signal. No published threshold for a "strong shortcut" is used here, and the AUROCs are descriptive values on the full label set (not cross-validated). *(D) Co-occurrence.* The RBP label sets overlap more than chance would predict. Off-diagonal lifts over independence are 1.8 to 3.3. The overlap is strongest within the IGF2BP family: Jaccard 0.56 for IGF2BP1 and IGF2BP3, 0.44 for IGF2BP2 and IGF2BP3, and 0.40 for IGF2BP1 and IGF2BP2. QKI has low Jaccard with every RBP (0.07 to 0.09) only because it has few positives, while its lift is 2.6 to 3.3. The diagonal of the lift matrix is just N divided by the number of positives, so it carries no information.

**Inference (not verified).** The length effect is probably partly mechanical, since a longer circRNA has more sequence that a CLIP cluster can overlap. The shared signal across RBPs is likely a mix of length, expression and CLIP detectability, but none of these was separated. The lift values are not controlled for length, so they overstate co-occurrence that is independent of length.

**Decision.** (1) The RBP labels are kept; the length effect is moderate. (2) The auxiliary RBP head is always reported against a length-only baseline (AUROC about 0.62 to 0.65) and stratified by length band. Only AUROC above this baseline counts as evidence that the head learns more than length. (3) Head formulation (proposal, to be confirmed with the supervisor): an any-RBP head (782 positives) as the primary RBP head, and the six-RBP multi-label output as a secondary one. IGF2BP1, 2 and 3 are largely redundant, and QKI (49 positives, about 10 per fold) is too small for its own fold-wise evaluation, so it is reported pooled only. (4) Absent site stays unlabelled, not negative.

### 43. ENCORI Label Tables with Evidence Flags

**Objective.** Build the RBP and miRNA label tables with explicit evidence flags, verify them against the counts of Cells 40 and 41 and save them.

In [ ]:
# Cell 43 - ENCORI label tables with flag columns (dry run unless SAVE = True)
import io, re
import pandas as pd

SAVE = True                                           # set True only after approval

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
ours = set(circ["circrna_id"])
model_ids = set(circ.loc[circ["use_primary"], "circrna_id"])

def read_valid(path):
    txt = path.read_text(encoding="utf-8", errors="replace")
    data = [l for l in txt.splitlines() if l.strip() and not l.startswith("#")]
    if len(data) < 2 or "\t" not in data[1]:
        return None
    return pd.read_csv(io.StringIO("\n".join(data)), sep="\t")

def explode_ids(d, col):
    d = d.copy()
    d["ids"] = d[col].astype(str).str.split(",")
    d["n_ids"] = d["ids"].map(len)
    d = d.explode("ids").rename(columns={"ids": "circrna_id"})
    d["circrna_id"] = d["circrna_id"].str.strip()
    return d[d["circrna_id"].isin(ours)]

# ---------------- RBP labels ----------------
parts = []
for f in sorted(ENC_DIR.glob("rbp_*_hg38_clip2.tsv")):
    rbp = re.match(r"rbp_(.+)_hg38_clip2\.tsv", f.name).group(1)
    d = read_valid(f)
    if d is None:
        continue
    d["clipExpNum"] = pd.to_numeric(d["clipExpNum"], errors="coerce").fillna(0)
    d = d[(d["clipExpNum"] >= 2) & (d["RBP"] == rbp)]
    e = explode_ids(d, "txIDcat")
    e["rbp"] = rbp
    parts.append(e[["circrna_id", "rbp", "clipExpNum", "n_ids"]])
rb = pd.concat(parts, ignore_index=True)
rb["pos_clip2"] = True
rb["pos_clip2_cap10"] = rb["n_ids"] <= 10
rb["pos_clip3"] = rb["clipExpNum"] >= 3
rb["pos_clip3_cap10"] = rb["pos_clip3"] & (rb["n_ids"] <= 10)
rbp_lab = (rb.groupby(["circrna_id", "rbp"])
             .agg(clip_max=("clipExpNum", "max"), n_clusters=("clipExpNum", "size"),
                  pos_clip2=("pos_clip2", "any"), pos_clip2_cap10=("pos_clip2_cap10", "any"),
                  pos_clip3=("pos_clip3", "any"), pos_clip3_cap10=("pos_clip3_cap10", "any"))
             .reset_index())

# ---------------- miRNA edges (arm-stripped node, as in the curated table) ----------------
parts = []
for f in sorted(ENC_DIR.glob("mirna_hsa-*_hg38_clip1.tsv")):
    q = re.match(r"mirna_(hsa-.+)_hg38_clip1\.tsv", f.name).group(1)
    d = read_valid(f)
    if d is None:
        continue
    d["clipExpNum"] = pd.to_numeric(d["clipExpNum"], errors="coerce").fillna(0)
    d = d[(d["clipExpNum"] >= 1) & (d["miRNAname"] == q)]
    e = explode_ids(d, "circID")
    arm = re.search(r"-(5p|3p)$", q)
    e["mirna"] = re.sub(r"-(3p|5p)$", "", q.replace("hsa-", "", 1))
    e["arm"] = arm.group(1) if arm else "armless"
    parts.append(e[["circrna_id", "mirna", "arm", "clipExpNum", "n_ids"]])
mb = pd.concat(parts, ignore_index=True)
mb["pos_clip1"] = True
mb["pos_clip2"] = mb["clipExpNum"] >= 2
mb["pos_clip1_cap10"] = mb["n_ids"] <= 10
mb["pos_clip2_cap10"] = mb["pos_clip2"] & (mb["n_ids"] <= 10)
mir_lab = (mb.groupby(["circrna_id", "mirna"])
             .agg(clip_max=("clipExpNum", "max"), n_rows=("clipExpNum", "size"),
                  arms=("arm", lambda s: ",".join(sorted(set(s)))),
                  pos_clip1=("pos_clip1", "any"), pos_clip2=("pos_clip2", "any"),
                  pos_clip1_cap10=("pos_clip1_cap10", "any"), pos_clip2_cap10=("pos_clip2_cap10", "any"))
             .reset_index())

print("rbp_lab:", rbp_lab.shape, "| mir_lab:", mir_lab.shape)

# ---------------- checks against Cells 40-42 (model-input circRNAs only) ----------------
def ok(a, b): return "OK" if a == b else f"MISMATCH (expected {b})"

R = rbp_lab[rbp_lab["circrna_id"].isin(model_ids)]
exp_rbp = {"EIF4A3": 274, "FUS": 238, "IGF2BP1": 351, "IGF2BP2": 531, "IGF2BP3": 427, "QKI": 49}
got = R[R["pos_clip2_cap10"]].groupby("rbp")["circrna_id"].nunique().to_dict()
print("\nRBP circRNAs at clip>=2 & cap10 per RBP:", got, "->", ok(got, exp_rbp))
print("any-RBP clip>=2 & cap10:", R.loc[R["pos_clip2_cap10"], "circrna_id"].nunique(),
      "->", ok(R.loc[R["pos_clip2_cap10"], "circrna_id"].nunique(), 782))
print("any-RBP clip>=2:", R["circrna_id"].nunique(), "->", ok(R["circrna_id"].nunique(), 949))
print("any-RBP clip>=3 & cap10:", R.loc[R["pos_clip3_cap10"], "circrna_id"].nunique(),
      "->", ok(R.loc[R["pos_clip3_cap10"], "circrna_id"].nunique(), 473))

M = mir_lab[mir_lab["circrna_id"].isin(model_ids)]
for flag, e_edges, e_circ in [("pos_clip1", 1107, 508), ("pos_clip2", 704, 352), ("pos_clip1_cap10", 818, 395)]:
    s = M[M[flag]]
    print(f"miRNA {flag:16s} edges {len(s):5d} -> {ok(len(s), e_edges)} | circRNAs {s['circrna_id'].nunique():4d} -> {ok(s['circrna_id'].nunique(), e_circ)}")

print("\nduplicate keys: rbp", int(rbp_lab.duplicated(["circrna_id", "rbp"]).sum()),
      "| mirna", int(mir_lab.duplicated(["circrna_id", "mirna"]).sum()))

if SAVE:
    rbp_lab.to_csv(PROCESSED_DIR / "rbp_labels_encori.csv", index=False)
    mir_lab.to_csv(PROCESSED_DIR / "mirna_edges_encori.csv", index=False)
    print("saved: rbp_labels_encori.csv, mirna_edges_encori.csv")
else:
    print("\nDry run: nothing written (SAVE = False).")

rbp_lab: (2495, 8) | mir_lab: (1259, 9)

RBP circRNAs at clip>=2 & cap10 per RBP: {'EIF4A3': 274, 'FUS': 238, 'IGF2BP1': 351, 'IGF2BP2': 531, 'IGF2BP3': 427, 'QKI': 49} -> OK
any-RBP clip>=2 & cap10: 782 -> OK
any-RBP clip>=2: 949 -> OK
any-RBP clip>=3 & cap10: 473 -> OK
miRNA pos_clip1        edges  1107 -> OK | circRNAs  508 -> OK
miRNA pos_clip2        edges   704 -> OK | circRNAs  352 -> OK
miRNA pos_clip1_cap10  edges   818 -> OK | circRNAs  395 -> OK

duplicate keys: rbp 0 | mirna 0
saved: rbp_labels_encori.csv, mirna_edges_encori.csv


**Interpretation.** Every check reproduces the earlier counts, so the label tables rebuild exactly from the cached files. *(A) RBP.* Per-RBP circRNAs at `clipExpNum >= 2` with clusters of at most 10 circRNAs match Cell 41 (EIF4A3 274, FUS 238, IGF2BP1 351, IGF2BP2 531, IGF2BP3 427, QKI 49). The any-RBP totals also match: 782 (`>= 2`, cap 10), 949 (`>= 2`) and 473 (`>= 3`, cap 10). *(B) miRNA.* The flagged edges on the 1,582 model-input circRNAs match Cell 40: `pos_clip1` 1,107 edges / 508 circRNAs, `pos_clip2` 704 / 352, `pos_clip1_cap10` 818 / 395. *(C) Table sizes.* `rbp_lab` has 2,495 (circRNA, RBP) rows and `mir_lab` has 1,259 (circRNA, miRNA) rows. These counts cover all 1,644 primary circRNAs, not only the 1,582 model-input ones, which is why they are larger than the model-input edge counts above (the same convention as the 722 circRNAs with a curated miRNA edge in Cell 26 versus 695 in Cell 22). *(D) Integrity.* There are 0 duplicate keys in both tables, so each (circRNA, RBP) and (circRNA, miRNA) pair is a single row with its evidence stored in flag columns (`pos_clip2`, `pos_clip2_cap10`, `pos_clip3`, `pos_clip3_cap10` for RBP; `pos_clip1`, `pos_clip2`, `pos_clip1_cap10`, `pos_clip2_cap10` for miRNA). No threshold is applied as a deletion.

**Not checked.** The RBP rows exist only for `clipExpNum >= 2`, so a clip1 comparison across RBPs is not possible from this table (the five large RBPs were downloaded at that threshold). The miRNA table covers 26 miRNAs, not all miRNAs.

**Decision.** Both tables were written to `data\processed` (`SAVE = True` in this run): `rbp_labels_encori.csv` and `mirna_edges_encori.csv`.

### 44. Host-Gene-Grouped 5-Fold Split

**Objective.** Construct a host-gene-grouped 5-fold split (greedy balancing by pairs, three seeds) and check fold sizes, balance and leakage. The cell is a dry run; nothing is written.

In [ ]:
# Cell 44 - Host-gene-grouped 5-fold split: sizes and balance (dry run, nothing is written)
import re
import numpy as np
import pandas as pd

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
gene_s = fa_idx.set_index("circbase_id")["gene"]
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)
mi = circ[circ["use_primary"]].set_index("circrna_id")

pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
print("pair columns:", pairs.columns.tolist())
assert "circrna_id" in pairs.columns, "adapt the circRNA column name"
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
print("pairs on model-input circRNAs:", len(pairs), "(expected 2,137) | circRNAs:", pairs["circrna_id"].nunique())

# group = host gene if it is a real symbol, otherwise one singleton group per circRNA
def group_of(cid):
    g = gene_s.get(cid, "")
    return f"solo:{cid}" if PLACE.match(str(g)) else str(g)
pairs["group"] = pairs["circrna_id"].map(group_of)
grp = pairs.groupby("group").agg(n_pairs=("circrna_id", "size"), n_circ=("circrna_id", "nunique"))
print("groups:", len(grp), "| singleton placeholder groups:", int(grp.index.str.startswith("solo:").sum()),
      "| circRNAs per group max:", grp["n_circ"].max())
print("largest groups by pairs:\n", grp.sort_values("n_pairs", ascending=False).head(5).to_string())

cur = pd.read_csv(PROCESSED_DIR / "mirna_edges_curated.csv")
cur_ids = set(cur["circrna_id"])

def assign(seed, k=5):
    rng = np.random.default_rng(seed)
    order = grp.sample(frac=1, random_state=seed).sort_values("n_pairs", ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for g, n in order["n_pairs"].items():
        f = int(rng.choice(np.flatnonzero(load == load.min())))
        fold[g] = f; load[f] += n
    return fold

for seed in (0, 1, 2):
    pairs["fold"] = pairs["group"].map(assign(seed))
    per = pairs.groupby("fold").agg(pairs=("circrna_id", "size"), circRNAs=("circrna_id", "nunique"))
    per["curated_miRNA_circRNAs"] = pairs.groupby("fold")["circrna_id"].apply(lambda s: len(set(s) & cur_ids))
    per["gt1022nt_share"] = pairs.groupby("fold")["circrna_id"].apply(
        lambda s: round((mi.loc[s.unique(), "seq_len"] > 1022).mean(), 3))
    # leakage checks: a group or a circRNA must never appear in two folds
    g_ok = int((pairs.groupby("group")["fold"].nunique() > 1).sum())
    c_ok = int((pairs.groupby("circrna_id")["fold"].nunique() > 1).sum())
    print(f"\nseed {seed}: groups in >1 fold: {g_ok} | circRNAs in >1 fold: {c_ok}")
    print(per.to_string())

pair columns: ['circrna_id', 'disease', 'seq_len']
pairs on model-input circRNAs: 2137 (expected 2,137) | circRNAs: 1582
groups: 1251 | singleton placeholder groups: 26 | circRNAs per group max: 9
largest groups by pairs:
         n_pairs  n_circ
group                  
HIPK3        34       3
CDR1         22       1
RPPH1        16       9
ZNF609       15       1
PTK2         15       8

seed 0: groups in >1 fold: 0 | circRNAs in >1 fold: 0
      pairs  circRNAs  curated_miRNA_circRNAs  gt1022nt_share
fold                                                         
0       428       320                     131           0.234
1       427       315                     140           0.210
2       428       315                     133           0.235
3       427       317                     149           0.211
4       427       315                     142           0.216

seed 1: groups in >1 fold: 0 | circRNAs in >1 fold: 0
      pairs  circRNAs  curated_miRNA_circRNAs  gt1022nt_share
fol

**Interpretation.** *(A) Input.* `cda_pairs_primary.csv` has the columns `circrna_id, disease, seq_len`. Restricted to the model-input circRNAs it gives 2,137 pairs over 1,582 circRNAs, as expected. *(B) Groups.* There are 1,251 groups: 1,225 host genes plus 26 singleton groups for circRNAs whose host-gene field is a placeholder (1,582 - 1,556 = 26, matching Cell 28). The largest group holds 9 circRNAs (RPPH1: 9 circRNAs, 16 pairs; PTK2: 8 circRNAs, 15 pairs). The largest group by pairs is HIPK3 (34 pairs, 3 circRNAs), which is 1.6% of all pairs, so no single group can dominate a fold. *(C) Balance.* Every fold holds 427 or 428 pairs in all three seeds (the ideal is 427.4), and the circRNAs per fold range from 315-320 (seed 0), 300-328 (seed 1) and 311-326 (seed 2); each seed's fold sizes add up to 1,582. The share of circRNAs longer than 1,022 nt is 0.199-0.256 per fold (overall 350/1,582 = 0.221). The number of circRNAs with a curated miRNA edge is 129-149 per fold and sums to 695 in every seed, i.e. about 41-47% of the circRNAs in a fold. *(D) Leakage checks.* 0 groups and 0 circRNAs appear in more than one fold, in all three seeds. This check confirms the code, because the fold is derived from the group by construction. It does not show an absence of biological leakage. *(E) Consequence.* Every test circRNA has no training positives, so all test positives are cold-start. This is stricter than the random edge split, where about 64% of test positives were cold-start (Cell 17).

**Not controlled.** (i) The 26 placeholder-gene circRNAs are singletons, so siblings among them (same locus, no gene symbol) can fall in different folds. (ii) Circular RNAs from the same paper, and the same disease in train and test, are not grouped. Diseases are shared across folds by design, and 91 of the 209 diseases have fewer than 3 circRNAs (Cell 16; computed on the 2,206-pair set). (iii) Folds are not stratified by miRNA or RBP label availability. The 41-47% spread in curated-miRNA share is chance variation (inferred).

**Decision.** The host-gene-grouped split is accepted as the working primary split (greedy balancing by pairs, 3 seeds for the spread). Nothing is frozen yet, because no split file has been written. Next come evaluation negatives and the shortcut check (Cell 45).

### 45. Evaluation Negatives and Disease-Popularity Shortcut

**Objective.** Generate evaluation negatives for the CDA task (random and degree-matched) and test a disease-popularity baseline as a shortcut check. The cell is a dry run; nothing is written.

In [ ]:
# Cell 45 - Evaluation negatives for the CDA task and a disease-popularity shortcut check
# Self-contained: rebuilds groups and folds from the saved CSV files. Dry run: nothing is written.
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ---- 0) the two saved label files exist and have the expected shapes ----------------------
rb = pd.read_csv(PROCESSED_DIR / "rbp_labels_encori.csv")
mr = pd.read_csv(PROCESSED_DIR / "mirna_edges_encori.csv")
print("rbp_labels_encori:", rb.shape, "(expected (2495, 8)) | mirna_edges_encori:", mr.shape,
      "(expected (1259, 9))")

# ---- 1) rebuild pairs, groups and folds exactly as in Cell 44 -------------------------------
circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
mi = circ[circ["use_primary"]].set_index("circrna_id")
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
gene_s = fa_idx.set_index("circbase_id")["gene"]
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)

pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
print("pairs:", len(pairs), "(expected 2,137) | circRNAs:", pairs["circrna_id"].nunique(), "(expected 1,582)")

def group_of(cid):
    g = gene_s.get(cid, "")
    return f"solo:{cid}" if PLACE.match(str(g)) else str(g)
pairs["group"] = pairs["circrna_id"].map(group_of)
grp = pairs.groupby("group").agg(n_pairs=("circrna_id", "size"), n_circ=("circrna_id", "nunique"))

def assign(seed, k=5):
    rng = np.random.default_rng(seed)
    order = grp.sample(frac=1, random_state=seed).sort_values("n_pairs", ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for g, n in order["n_pairs"].items():
        f = int(rng.choice(np.flatnonzero(load == load.min())))
        fold[g] = f; load[f] += n
    return fold

SEED = 0
pairs["fold"] = pairs["group"].map(assign(SEED))
sizes = pairs.groupby("fold").size().tolist()
print("seed-0 fold sizes:", sizes, "(Cell 44: [428, 427, 428, 427, 427])")

# ---- 2) negative sampling ---------------------------------------------------------------------
dis = np.array(sorted(pairs["disease"].unique()))
print("diseases:", len(dis), "(expected 207)")
pos_all = set(zip(pairs["circrna_id"], pairs["disease"]))
pos_by_c = pairs.groupby("circrna_id")["disease"].apply(set).to_dict()
dcount = pairs["disease"].value_counts().reindex(dis).astype(float)    # positives per disease, all folds

def sample_negatives(test_pos, scheme, seed, ratio=1):
    """Same circRNAs as the positives, with a disease that is not a known positive of that circRNA."""
    rng = np.random.default_rng(seed)
    out = []
    for c, g in test_pos.groupby("circrna_id", sort=True):
        cand = np.array([d for d in dis if d not in pos_by_c[c]])
        w = dcount.reindex(cand).values
        p = None if scheme == "random" else w / w.sum()
        pick = rng.choice(cand, size=len(g) * ratio, replace=False, p=p)
        out += [(c, d) for d in pick]
    return pd.DataFrame(out, columns=["circrna_id", "disease"])

# ---- 3) checks per fold and scheme ---------------------------------------------------------------
rows = []
for f in range(5):
    test = pairs[pairs["fold"] == f]
    train_cnt = pairs.loc[pairs["fold"] != f, "disease"].value_counts()   # popularity from training folds only
    cold_share = float((~test["disease"].isin(train_cnt.index)).mean())
    for scheme in ("random", "degree_matched"):
        neg = sample_negatives(test, scheme, seed=1000 * SEED + f)
        again = sample_negatives(test, scheme, seed=1000 * SEED + f)
        y = np.r_[np.ones(len(test)), np.zeros(len(neg))]
        s = np.r_[test["disease"].map(train_cnt).fillna(0).values,
                  neg["disease"].map(train_cnt).fillna(0).values]
        rows.append(dict(
            fold=f, scheme=scheme, n_pos=len(test), n_neg=len(neg),
            pop_auroc=round(roc_auc_score(y, s), 3), cold_disease_share=round(cold_share, 3),
            pos_mean_dis_count=round(test["disease"].map(dcount).mean(), 1),
            neg_mean_dis_count=round(neg["disease"].map(dcount).mean(), 1),
            collisions=sum((c, d) in pos_all for c, d in zip(neg["circrna_id"], neg["disease"])),
            wrong_fold=int((~neg["circrna_id"].isin(set(test["circrna_id"]))).sum()),
            dup=int(neg.duplicated().sum()), reproducible=neg.equals(again)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("\nPopularity AUROC by scheme (mean / min / max over folds):")
print(res.groupby("scheme")["pop_auroc"].agg(["mean", "min", "max"]).round(3).to_string())
print("\nDry run: nothing written.")

rbp_labels_encori: (2495, 8) (expected (2495, 8)) | mirna_edges_encori: (1259, 9) (expected (1259, 9))
pairs: 2137 (expected 2,137) | circRNAs: 1582 (expected 1,582)
seed-0 fold sizes: [428, 427, 428, 427, 427] (Cell 44: [428, 427, 428, 427, 427])
diseases: 207 (expected 207)
 fold         scheme  n_pos  n_neg  pop_auroc  cold_disease_share  pos_mean_dis_count  neg_mean_dis_count  collisions  wrong_fold  dup  reproducible
    0         random    428    428      0.811               0.035                49.4                11.1           0           0    0          True
    0 degree_matched    428    428      0.506               0.035                49.4                43.2           0           0    0          True
    1         random    427    427      0.821               0.035                49.4                 9.4           0           0    0          True
    1 degree_matched    427    427      0.517               0.035                49.4                45.7           0          

**Interpretation.** *(A) Rebuild.* The self-contained cell reproduces the earlier state: `rbp_labels_encori.csv` (2,495 × 8) and `mirna_edges_encori.csv` (1,259 × 9) have the expected shapes, the seed-0 fold sizes equal Cell 44 ([428, 427, 428, 427, 427]), and the data hold 2,137 pairs, 1,582 circRNAs and 207 diseases. *(B) Integrity.* In all 10 fold-by-scheme rows there are 0 collisions with known positives, 0 negatives outside the test circRNAs, 0 duplicates, and the sampling is reproducible from its seed. Each fold has exactly as many negatives as positives (427-428). *(C) Popularity shortcut.* A model-free score that uses only how often a disease is positive in the training folds reaches an AUROC of 0.821 (0.807-0.846 over folds) against random negatives, but 0.513 (0.496-0.530) against degree-matched negatives. The reason is visible in the mean disease counts: test positives have a mean of 47.7-54.8 positives per disease, random negatives only 9.4-11.1, and degree-matched negatives 43.2-48.7. Degree-matched negatives are still slightly less popular than the positives (ratio 0.87-0.96 per fold), which explains why the AUROC is slightly above 0.5 in four of five folds (0.496 in fold 2). That explanation is inferred. *(D) Cold diseases.* 3.5% of test positives in folds 0-2, 2.8% in fold 3 and 6.1% in fold 4 belong to a disease with no training positive (about 12-26 pairs per fold), so the evaluation is almost entirely warm-disease. *(E) Pooled scores.* Because every test circRNA contributes as many negatives as positives, a score that depends only on the circRNA (degree, length, embedding norm) has the same distribution for positives and negatives in a pooled AUROC (a consequence of the design, not a measured result). The pooled AUROC can therefore only reward circRNA-disease specificity, while per-circRNA ranking metrics can be computed separately.

**Not checked.** Other shortcuts (disease-side features other than popularity, sequence length interacting with disease) were not tested. The 1:1 negative ratio is a provisional choice, because the negative-sampling protocol of circDis is not verified. A sampled negative may be a true but unknown association (false negative).

**Decision.** (1) Report both negative schemes. Degree-matched negatives are the headline for claims that a model learned circRNA-disease relations. Random negatives are reported only for comparability with the literature, together with the popularity baseline (0.82). (2) The popularity baseline is a required baseline row in every result table. (3) Add per-circRNA ranking metrics (MRR, Hits@10), as already planned. (4) Training negatives are sampled separately, from the training folds only. The degree-matched evaluation negatives use positive counts from all folds including the test fold; this is deliberate (it matches the overall disease distribution) but must be stated. (5) Nothing is frozen yet: negative sets are reproducible from the seed and are not written to disk.

### 46. Sibling Leakage under Three Split Schemes

**Objective.** Evaluate a host-gene sibling baseline under three split schemes (host-gene grouped, circRNA grouped, random edge) in order to measure sibling leakage. The cell is a dry run; nothing is written.

In [ ]:
# Cell 46 - Sibling leakage: does a host-gene sibling baseline work under three split schemes? (dry run)
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ---- rebuild pairs, groups and the host-gene folds (same code as Cell 45) ----------------------
circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
mi = circ[circ["use_primary"]].set_index("circrna_id")
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
gene_s = fa_idx.set_index("circbase_id")["gene"]
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)

pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
pairs["group"] = pairs["circrna_id"].map(
    lambda c: f"solo:{c}" if PLACE.match(str(gene_s.get(c, ""))) else str(gene_s.get(c, "")))

def greedy(units, seed, k=5):
    rng = np.random.default_rng(seed)
    order = units.sample(frac=1, random_state=seed).sort_values(ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for u, n in order.items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[u] = f; load[f] += n
    return fold

SEED = 0
pairs["fold_gene"] = pairs["group"].map(greedy(pairs.groupby("group").size(), SEED))
pairs["fold_circ"] = pairs["circrna_id"].map(greedy(pairs.groupby("circrna_id").size(), SEED))
pairs["fold_edge"] = np.random.default_rng(SEED).permutation(len(pairs)) % 5
print("host-gene fold sizes:", pairs.groupby("fold_gene").size().tolist(), "(Cell 44: [428, 427, 428, 427, 427])")
print("circRNAs in >1 fold -> gene-grouped:", int((pairs.groupby("circrna_id")["fold_gene"].nunique() > 1).sum()),
      "| circRNA-grouped:", int((pairs.groupby("circrna_id")["fold_circ"].nunique() > 1).sum()),
      "| random edge:", int((pairs.groupby("circrna_id")["fold_edge"].nunique() > 1).sum()))

# ---- negatives (degree-matched, as in Cell 45) ---------------------------------------------------
dis = np.array(sorted(pairs["disease"].unique()))
pos_by_c = pairs.groupby("circrna_id")["disease"].apply(set).to_dict()
dcount = pairs["disease"].value_counts().reindex(dis).astype(float)

def sample_negatives(test_pos, seed):
    rng = np.random.default_rng(seed); out = []
    for c, g in test_pos.groupby("circrna_id", sort=True):
        cand = np.array([d for d in dis if d not in pos_by_c[c]])
        w = dcount.reindex(cand).values
        out += [(c, d) for d in rng.choice(cand, size=len(g), replace=False, p=w / w.sum())]
    return pd.DataFrame(out, columns=["circrna_id", "disease"])

# ---- sibling baseline: other circRNAs of the same real host gene, positive for that disease, in training folds ----
cg = pairs.drop_duplicates("circrna_id").set_index("circrna_id")["group"]
gene_circ = (pairs[~pairs["group"].str.startswith("solo:")].groupby("group")["circrna_id"].unique().to_dict())

def sib_score(c, d, f, pf):
    g = cg[c]
    if g.startswith("solo:"):
        return 0
    return sum(1 for s in gene_circ[g] if s != c and (s, d) in pf and pf[(s, d)] != f)

rows = []
for scheme, col in [("host-gene grouped", "fold_gene"), ("circRNA grouped", "fold_circ"), ("random edge", "fold_edge")]:
    pf = dict(zip(zip(pairs["circrna_id"], pairs["disease"]), pairs[col]))
    for f in range(5):
        test = pairs[pairs[col] == f]
        neg = sample_negatives(test, seed=1000 * SEED + f)
        sp = np.array([sib_score(c, d, f, pf) for c, d in zip(test["circrna_id"], test["disease"])])
        sn = np.array([sib_score(c, d, f, pf) for c, d in zip(neg["circrna_id"], neg["disease"])])
        y = np.r_[np.ones(len(sp)), np.zeros(len(sn))]
        rows.append(dict(scheme=scheme, fold=f, n_pos=len(sp), auroc=round(roc_auc_score(y, np.r_[sp, sn]), 3),
                         pos_with_sibling=round(float((sp > 0).mean()), 3),
                         neg_with_sibling=round(float((sn > 0).mean()), 3)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("\nBy scheme (mean over folds):")
print(res.groupby("scheme")[["auroc", "pos_with_sibling", "neg_with_sibling"]].mean().round(3).to_string())
print("\nDry run: nothing written.")

host-gene fold sizes: [428, 427, 428, 427, 427] (Cell 44: [428, 427, 428, 427, 427])
circRNAs in >1 fold -> gene-grouped: 0 | circRNA-grouped: 0 | random edge: 286
           scheme  fold  n_pos  auroc  pos_with_sibling  neg_with_sibling
host-gene grouped     0    428  0.500             0.000             0.000
host-gene grouped     1    427  0.500             0.000             0.000
host-gene grouped     2    428  0.500             0.000             0.000
host-gene grouped     3    427  0.500             0.000             0.000
host-gene grouped     4    427  0.500             0.000             0.000
  circRNA grouped     0    427  0.515             0.042             0.012
  circRNA grouped     1    427  0.531             0.077             0.016
  circRNA grouped     2    428  0.532             0.075             0.012
  circRNA grouped     3    427  0.527             0.070             0.016
  circRNA grouped     4    428  0.532             0.072             0.009
      random edge     

**Interpretation.** *(A) Setup.* The rebuilt host-gene folds equal Cell 44 ([428, 427, 428, 427, 427]). 0 circRNAs fall in more than one fold under both grouped schemes. The random-edge split puts 286 circRNAs into more than one fold, which is expected for a scheme that splits edges, not circRNAs, and is not an error. *(B) Host-gene grouped.* The sibling baseline gives an AUROC of exactly 0.500 in all five folds, with 0.000 of positives and negatives having a sibling in training. This holds by construction, since a same-gene sibling can never be in training, so it confirms the code and is not an empirical finding. *(C) circRNA grouped and random edge.* The sibling baseline gives a mean AUROC of 0.527 and 0.526 (folds 0.515-0.532 and 0.520-0.532). In the circRNA-grouped split 6.7% of positives have a same-gene sibling that is positive for the same disease in training, against 1.3% of degree-matched negatives (random edge: 6.4% against 1.3%). A positive is therefore about five times as likely as a negative to have such a sibling. *(D) Reading the AUROC.* The AUROC is small mainly because the score is nonzero for few pairs (about 7% of positives); with most scores tied at zero the AUROC is close to 0.5 + (0.067 - 0.013) / 2 = 0.527 (hand calculation). A low AUROC therefore understates how much a flexible model could gain from siblings, for instance through shared sequence or graph neighbours. The size of that gain is not measured here.

**Not covered.** The baseline only detects siblings from the same host gene with the same disease. It does not detect similar circRNAs from different genes, shared miRNA or RBP neighbours, or other positives of the same circRNA (the 286 circRNAs split in the random-edge scheme). It scores the 26 placeholder-gene circRNAs as 0, so the 0.000 for the host-gene scheme does not cover them.

**Decision.** (1) The host-gene grouped split stays the primary split. It removes the measured sibling signal completely, and the circRNA-grouped split only partly. (2) Report the random-edge and circRNA-grouped results as secondary rows, to show how much a split matters. (3) The sibling baseline (0.527) is reported next to the popularity baseline in the leakage section. (4) The remaining unmeasured routes (similar sequences across genes, shared graph neighbours) are listed as limitations.

### 47. Frozen Folds and Evaluation Negatives

**Objective.** Freeze the host-gene folds (three seeds) and the seed-0 evaluation negatives, and verify the saved files by reloading them.

In [ ]:
# Cell 47 - Freeze the host-gene folds (3 seeds) and the seed-0 evaluation negatives (dry run unless SAVE = True)
import re
import numpy as np
import pandas as pd

SAVE = True                                           # set True only after approval

circ = pd.read_csv(PROCESSED_DIR / "circrna_table.csv")
mi = circ[circ["use_primary"]].set_index("circrna_id")
fa_idx = pd.read_csv(PROCESSED_DIR / "circbase_hg19_index.csv", keep_default_na=False)
gene_s = fa_idx.set_index("circbase_id")["gene"]
PLACE = re.compile(r"^(TCONS|XLOC|LOC\d|AK\d|ENS[GT]|NONHSA|None$|$)", re.I)

pairs = pd.read_csv(PROCESSED_DIR / "cda_pairs_primary.csv")
pairs = pairs[pairs["circrna_id"].isin(mi.index)].copy()
pairs["group"] = pairs["circrna_id"].map(
    lambda c: f"solo:{c}" if PLACE.match(str(gene_s.get(c, ""))) else str(gene_s.get(c, "")))

def greedy(units, seed, k=5):
    rng = np.random.default_rng(seed)
    order = units.sample(frac=1, random_state=seed).sort_values(ascending=False, kind="stable")
    load = np.zeros(k); fold = {}
    for u, n in order.items():
        f = int(rng.choice(np.flatnonzero(load == load.min()))); fold[u] = f; load[f] += n
    return fold

# ---- 1) folds table: one row per circRNA --------------------------------------------------------
gsize = pairs.groupby("group").size()
folds = (pairs.groupby("circrna_id").agg(group=("group", "first"), n_pairs=("disease", "size")).reset_index())
folds["placeholder_gene"] = folds["group"].str.startswith("solo:")
for seed in (0, 1, 2):
    folds[f"fold_seed{seed}"] = folds["group"].map(greedy(gsize, seed))

print("folds table:", folds.shape, "| circRNAs:", folds["circrna_id"].nunique(), "(expected 1,582)")
for seed in (0, 1, 2):
    col = f"fold_seed{seed}"
    pf = pairs["circrna_id"].map(folds.set_index("circrna_id")[col])
    leak = int((pairs.assign(f=pf).groupby("group")["f"].nunique() > 1).sum())
    print(f"seed {seed}: pairs per fold {pf.value_counts().sort_index().tolist()} | groups in >1 fold: {leak}")
print("seed-0 pair sizes should be [428, 427, 428, 427, 427] (Cell 44)")

# ---- 2) evaluation negatives, seed 0, both schemes ---------------------------------------------
dis = np.array(sorted(pairs["disease"].unique()))
pos_all = set(zip(pairs["circrna_id"], pairs["disease"]))
pos_by_c = pairs.groupby("circrna_id")["disease"].apply(set).to_dict()
dcount = pairs["disease"].value_counts().reindex(dis).astype(float)

def sample_negatives(test_pos, scheme, seed):
    rng = np.random.default_rng(seed); out = []
    for c, g in test_pos.groupby("circrna_id", sort=True):
        cand = np.array([d for d in dis if d not in pos_by_c[c]])
        w = dcount.reindex(cand).values
        p = None if scheme == "random" else w / w.sum()
        out += [(c, d) for d in rng.choice(cand, size=len(g), replace=False, p=p)]
    return pd.DataFrame(out, columns=["circrna_id", "disease"])

pairs["fold"] = pairs["circrna_id"].map(folds.set_index("circrna_id")["fold_seed0"])
parts = []
for scheme in ("random", "degree_matched"):
    for f in range(5):
        neg = sample_negatives(pairs[pairs["fold"] == f], scheme, seed=f)      # same seeds as Cell 45 (SEED=0)
        neg["fold"] = f; neg["scheme"] = scheme
        parts.append(neg)
negs = pd.concat(parts, ignore_index=True)
negs["collision"] = [(c, d) in pos_all for c, d in zip(negs["circrna_id"], negs["disease"])]
negs["in_own_fold"] = negs["circrna_id"].map(folds.set_index("circrna_id")["fold_seed0"]) == negs["fold"]

print("\nnegatives:", negs.shape, "| per scheme:", negs["scheme"].value_counts().to_dict())
print("collisions with positives:", int(negs["collision"].sum()),
      "| negatives outside their circRNA's fold:", int((~negs["in_own_fold"]).sum()),
      "| duplicates (scheme, circRNA, disease):", int(negs.duplicated(["scheme", "circrna_id", "disease"]).sum()))
npos = pairs.groupby("fold").size()
nneg = negs.groupby(["scheme", "fold"]).size().unstack(0)
print("negatives per fold equal positives per fold:", bool((nneg.sub(npos, axis=0) == 0).all().all()))
out_negs = negs[["circrna_id", "disease", "fold", "scheme"]]

# ---- 3) save (only after approval) and verify by reloading ----------------------------------------
if SAVE:
    folds.to_csv(PROCESSED_DIR / "cda_folds_primary.csv", index=False)
    out_negs.to_csv(PROCESSED_DIR / "cda_eval_negatives_primary.csv", index=False)
    f2 = pd.read_csv(PROCESSED_DIR / "cda_folds_primary.csv", keep_default_na=False)
    n2 = pd.read_csv(PROCESSED_DIR / "cda_eval_negatives_primary.csv")
    print("\nsaved. reload equal -> folds:", f2.shape == folds.shape and f2["circrna_id"].tolist() == folds["circrna_id"].tolist()
          and (f2["fold_seed0"].values == folds["fold_seed0"].values).all(),
          "| negatives:", n2.equals(out_negs.reset_index(drop=True)))
else:
    print("\nDry run: nothing written (SAVE = False).")

folds table: (1582, 7) | circRNAs: 1582 (expected 1,582)
seed 0: pairs per fold [428, 427, 428, 427, 427] | groups in >1 fold: 0
seed 1: pairs per fold [428, 428, 427, 427, 427] | groups in >1 fold: 0
seed 2: pairs per fold [427, 427, 428, 427, 428] | groups in >1 fold: 0
seed-0 pair sizes should be [428, 427, 428, 427, 427] (Cell 44)

negatives: (4274, 6) | per scheme: {'random': 2137, 'degree_matched': 2137}
collisions with positives: 0 | negatives outside their circRNA's fold: 0 | duplicates (scheme, circRNA, disease): 0
negatives per fold equal positives per fold: True

saved. reload equal -> folds: True | negatives: True


**Interpretation.** *(A) Folds.* The folds table has one row per model-input circRNA (1,582 rows, 7 columns: circRNA, group, pair count, placeholder-gene flag, and the fold for seeds 0, 1 and 2). Pairs per fold are [428, 427, 428, 427, 427] for seed 0 (identical to Cell 44), [428, 428, 427, 427, 427] for seed 1 and [427, 427, 428, 427, 428] for seed 2. 0 host-gene groups fall in more than one fold in all three seeds. *(B) Negatives.* There are 4,274 evaluation negatives (2,137 per scheme, equal to the 2,137 positives per scheme), with 0 collisions with known positives, 0 negatives outside their circRNA's fold, 0 duplicates, and as many negatives as positives in every fold. The total of 4,274 equals 2 × 2,137. *(C) Saving.* Both files were written and reloaded, and the reload equals what was built: `cda_folds_primary.csv` (1,582 × 7) and `cda_eval_negatives_primary.csv` (4,274 × 4: circRNA, disease, fold, scheme).

**Not checked.** The negatives were built with the same seeds as Cell 45 (seed 0, per-fold seed f), but this cell did not compare them with Cell 45's output, so identity with the Cell 45 sets is expected but not verified. The seed-1 and seed-2 folds are frozen; negatives exist for seed 0 only. Training negatives are not frozen and are sampled from the training folds at training time.

**Decision.** The exploration part is complete. The frozen artifacts for the next stage are `cda_pairs_primary.csv`, `circrna_table.csv`, `mirna_edges_curated.csv`, `rbp_labels_encori.csv`, `mirna_edges_encori.csv`, `cda_folds_primary.csv` and `cda_eval_negatives_primary.csv`, all in `data\processed`.